# Myconn

In [2]:
import pandas as pd
import numpy as np
import time
from datetime import datetime
from datetime import date
import calendar
from dateutil.relativedelta import relativedelta
from contextlib import contextmanager
from sqlalchemy import create_engine
from sqlalchemy import Integer, String, CHAR, Date, Text, text, Float
import os
import gc
import math
from sqlalchemy import Integer, String, CHAR, Date, Text, text, Float
import os
from datetime import datetime
from modules.queries import (df__penju, df__tokoawal, df__transaction, df__cus)
from modules.proses import (save_to_sql, add_index_and_foreign_keys, clean_and_transform_data, 
                            insert_disc2_grouping, insert_margin, rename_col_py_to_sql, insert_pk,
                            get_tgl, get_stock, df__aging)
from modules.queries import (df__tglso, df__tglsa, df__disc2, df__margin, df__artsizecode, df__stock_so,
                             df__stock_sa)

DB_TYPE = 'mysql+pymysql'
DB_USER="dataanalyst"
DB_PASSWORD="PyData*010224"

DB_HOST="36.93.148.91"
DB_HOST2="172.16.11.9"
DB_HOST3="192.168.1.237"
DB_PORT=3307

DB_LOCAL = "gis_local"
DB_SYNC = "gis_db1"
DB_RT = "gis_db"

@contextmanager
def get_engine(stat):
    """Context manager untuk membuka dan menutup SQLAlchemy engine."""
    if stat == 'read':
        engine = create_engine(f'{DB_TYPE}://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_SYNC}')
    elif stat == 'write':
        engine = create_engine(f'{DB_TYPE}://{DB_USER}:{DB_PASSWORD}@{DB_HOST3}:{DB_PORT}/{DB_RT}')
    else:
        engine = create_engine(f'{DB_TYPE}://{DB_USER}:{DB_PASSWORD}@{DB_HOST2}:{DB_PORT}/{DB_LOCAL}')

    try:
        # Mengembalikan engine untuk digunakan dalam blok with
        yield engine
    finally:
        # Menutup koneksi pool engine untuk membersihkan sumber daya
        engine.dispose()
def get_db(stat):
    if stat == 'read':
        db = DB_SYNC
    elif stat == 'write':
        db = DB_RT
    else:
        db = DB_LOCAL
    return db
def df__customers_pb_(colom,distinct=False,where=''):
    if distinct==False:distinct=''
    else :distinct='distinct'

    if where=='': where=''
    else: where=f'{where}'

    query = f'''
        SELECT {distinct} {colom}
        FROM gis_local.customers_pb
        where 1=1 {where}
        '''
    # df = pd.read_sql(query, engine_imp)
    with get_engine('imp') as engine:  # Menggunakan context manager dengan with
        df = pd.read_sql(query, engine)
    return df
# df__customers_pb_(colom,distinct=False,where='')
# df_artikel_pb_(colom,distinct=False,where='')
def df_artikel_pb_(colom,distinct=False,where=''):
    if distinct==False:distinct=''
    else :distinct='distinct'

    if where=='': where=''
    else: where=f'{where}'

    query = f'''
        SELECT {distinct} {colom}
        FROM gis_local.artikel_pb
        where 1=1 {where}
        '''
    # df = pd.read_sql(query, engine_imp)
    with get_engine('imp') as engine:  # Menggunakan context manager dengan with
        df = pd.read_sql(query, engine)
    return df
def list_to_sql_in(df, col_name, astype='str'):
    """
    Mengubah kolom DataFrame menjadi string SQL untuk WHERE IN.
    Contoh hasil: "'A', 'B', 'C'"
    """
    if col_name not in df.columns:
        raise ValueError(f"Kolom '{col_name}' tidak ditemukan di DataFrame.")

    # Ambil kolom dan drop NaN
    values = df[col_name].dropna().astype(astype).tolist()

    # Jika kosong, return string kosong (hindari error di SQL)
    if not values:
        return "''"

    # Format jadi "'A', 'B', 'C'"
    sql_str = ", ".join([f"'{v}'" for v in values])
    return sql_str




✅ .env loaded from: c:\Users\IT-ANALIS\Documents\airflow-DPS\.env
36.93.148.91


## Queri API

In [3]:
def df__customers_pb_n(placeholders):
        query = f'''
            SELECT fn_cusid,fn_whconsiid FROM gis_local.customers_pb
            where fc_code in ({placeholders})
            '''
        # df = pd.read_sql(query, engine_imp)
        with get_engine('imp') as engine:  # Menggunakan context manager dengan with
            df = pd.read_sql(query, engine)
        return df

In [4]:
def add_where(input_param,cus='',art=''):
    store_list = input_param.get('store_list')

    if store_list:  # hanya lanjut kalau ada isi (bukan None, '', atau [])
        placeholders = ", ".join([f"'{s}'" for s in store_list])
        df_cus = df__customers_pb_n(placeholders)
        whconsi_list = df_cus['fn_whconsiid'].tolist()

        if whconsi_list:
            whconsi_placeholders = ", ".join([str(s) for s in whconsi_list])
            whconsi = f"and {cus}fn_whconsiid IN ({whconsi_placeholders})"
        else:
            whconsi = ""
    else:
        whconsi = ""

    # # printol(whconsi)
    if input_param['divisi'] is not None and input_param['divisi']!='':
        divisi = input_param['divisi']
        divisi = f'and {art}fv_divname="{divisi}"'
    else: divisi = ''

    if input_param['ornal'] is not None and input_param['ornal']!='':
        ornal = input_param['ornal']
        ornal = f'and {art}fv_ornal="{ornal}"'
    else: ornal = ''

    if input_param['cat'] is not None and input_param['cat']!='':
        cat = input_param['cat']
        cat = f'and {art}fv_catname="{cat}"'
    else: cat = ''

    if input_param['artikel'] is not None and input_param['artikel']!='':
        artikel = input_param['artikel']
        artikel = f'and {art}fv_barcode="{artikel}"'
    else: artikel = ''

    if input_param['spv_list']: 
        spv_list = input_param['spv_list']
        if spv_list:
            spv_placeholders = ", ".join([f"'{s}'" for s in spv_list])
            spv=f'and {cus}fv_spv IN ({spv_placeholders})'
        else: spv= ''
    else: spv = ''
    
    return {
        'whconsi':whconsi,
        'divisi':divisi,
        'ornal':ornal,
        'cat':cat,
        'artikel':artikel,
        'spv':spv
    }

In [5]:
def df__stock_now_filter(input_param, 
                         col='stock_backdate.fn_whconsiid, fv_artsizecode, fv_barcode, fv_sizename, fv_configname,',
                         gb='stock_backdate.fn_whconsiid, fv_artsizecode',
                         year=None,
                         cus='',
                         art='',
                         times='this_y',
                         having=''):
    where=add_where(input_param,cus=cus, art=art)
    whconsi_list=where['whconsi']
    divisi=where['divisi']
    cat=where['cat']
    artikel=where['artikel']
    ornal=where['ornal']
    if year is None: year = datetime.now().year 
    if times == 'this_y': 
        input_value = input_param['start']
    else: 
        input_value= input_param['start_last']
    sales_thru=f'sales_thru_pb_{str(year)[-2:]}'
    trxstx = f"trxstx_pb{str(year)[-2:]}"
    stock = f"fn_stock_{input_value[5:7]}"
    main_query = f"""
        SELECT {col}
            SUM(stock) AS stocks
        FROM (
            SELECT fn_whconsiid, {stock} AS stock, fv_artsizecode 
            FROM gis_local.{sales_thru}
            where 1=1 {whconsi_list} 
            UNION ALL
            SELECT fn_whconsiid, fn_stock AS stock, fv_artsizecode 
            FROM gis_local.{trxstx} 
            WHERE fd_tgltrx BETWEEN '{str(year)[-2:]}-{input_value[5:7]}-01' AND '{str(year)[-2:]}-{input_value[5:10]}'
            {whconsi_list} 
            UNION ALL
            SELECT fn_whconsiid, fn_stock AS stock, fv_artsizecode 
            FROM gis_local.trxrt_pb
            WHERE fd_tgltrx < '{str(year)[-2:]}-{input_value[5:10]}' {whconsi_list} 
            UNION ALL
            SELECT fn_whconsiid, fn_stock AS stock, fv_artsizecode 
            FROM gis_local.Stock_varian_{str(year)[-2:]}
            WHERE fd_tgl BETWEEN '{str(year)[-2:]}-{input_value[5:7]}-01' AND '{str(year)[-2:]}-{input_value[5:10]}'
            {whconsi_list} 
        ) stock_backdate
        LEFT JOIN artikel_pb a ON a.fv_artsizecode = stock_backdate.fv_artsizecode
        LEFT JOIN customers_pb b ON stock_backdate.fn_whconsiid = b.fn_whconsiid
        WHERE 1=1 {cat} {artikel} {ornal} {divisi}
        GROUP BY {gb}
        {having}
        """
    # Retry up to 5 times if error
    max_retries = 5
    for attempt in range(1, max_retries + 1):
        try:
            with get_engine('imp') as engine:  # Menggunakan context manager dengan with
                df = pd.read_sql(main_query, engine)
            return df # success, langsung return
        except Exception as e:
            print(f"[Attempt {attempt}] Query gagal: {e}")
            if attempt == max_retries:
                raise  # kalau sudah max retry, lempar error
            time.sleep(2)  # tunggu 2 detik sebelum nyoba lagi

# @timed
def df__stock_early_filter(input_param, 
                         col='stock_backdate.fn_whconsiid, fv_artsizecode, fv_barcode, fv_sizename, fv_configname,',
                         gb='stock_backdate.fn_whconsiid, fv_artsizecode',
                         year=None,
                         cus='',
                         art='',
                         times='this_y'):
    where=add_where(input_param,cus=cus, art=art)
    if year is None: year = datetime.now().year 
    if times == 'this_y': input_value = input_param['end']
    else: input_value= input_param['end_last']
    sales_thru=f'sales_thru_pb_{str(year)[-2:]}'
    trxstx = f"trxstx_pb{str(year)[-2:]}"
    stock = f"fn_stock_{input_value[5:7]}"
    whconsi_list=where['whconsi']
    divisi=where['divisi']
    cat=where['cat']
    artikel=where['artikel']
    ornal=where['ornal']
    main_query = f"""
        SELECT {col}
            SUM(stock) AS stocks_early
        FROM (
            SELECT fn_whconsiid, {stock} AS stock, fv_artsizecode 
            FROM gis_local.{sales_thru} 
            WHERE 1=1 {whconsi_list}
            UNION ALL
            SELECT fn_whconsiid, fn_stock AS stock, fv_artsizecode 
            FROM gis_local.{trxstx} 
            WHERE fd_tgltrx BETWEEN '{str(year)[-2:]}-{input_value[5:7]}-01' AND 
            '{str(year)[-2:]}-{input_value[5:10]}' {whconsi_list}
            UNION ALL
            SELECT fn_whconsiid, fn_stock AS stock, fv_artsizecode 
            FROM gis_local.trxrt_pb
            WHERE fd_tgltrx < '{str(year)[-2:]}-{input_value[5:10]}' {whconsi_list}
            UNION ALL
            SELECT fn_whconsiid, fn_stock AS stock, fv_artsizecode 
            FROM gis_local.Stock_varian_{str(year)[-2:]}
            WHERE fd_tgl BETWEEN '{str(year)[-2:]}-{input_value[5:7]}-01' AND 
            '{str(year)[-2:]}-{input_value[5:10]}' {whconsi_list}
        ) stock_backdate
        LEFT JOIN artikel_pb a ON a.fv_artsizecode = stock_backdate.fv_artsizecode
        LEFT JOIN customers_pb b ON stock_backdate.fn_whconsiid = b.fn_whconsiid
        WHERE 1=1 {cat} {artikel} {ornal} {divisi}
        GROUP BY {gb}
        """
    # Retry up to 5 times if error
    max_retries = 5
    for attempt in range(1, max_retries + 1):
        try:
            with get_engine('imp') as engine:  # Menggunakan context manager dengan with
                df = pd.read_sql(main_query, engine)
            return df # success, langsung return
        except Exception as e:
            print(f"[Attempt {attempt}] Query gagal: {e}")
            if attempt == max_retries:
                raise  # kalau sudah max retry, lempar error
            time.sleep(2)  # tunggu 2 detik sebelum nyoba lagi
    # results = DatabaseConnector.execute_raw_sql(main_query, using=dbLoad)
    # df = pd.DataFrame(results)
    # return df

# @timed
def df__count_stock_now_filter(input_param, 
                         col='stock_backdate.fn_whconsiid, fv_artsizecode, fv_barcode, fv_sizename, fv_configname,',
                         gb='stock_backdate.fn_whconsiid, fv_artsizecode',
                         year=None,
                         cus='',
                         art='',
                         col_count='fv_barcode,',
                         times='this_y',
                         first_last='last'):
    where=add_where(input_param,cus=cus, art=art)
    if year is None: year = datetime.now().year 
    if times == 'this_y':
        if first_last == 'last':
            input_value = input_param['input_date']
        else:
            input_value = input_param['end']
    else:
        input_value = input_param['start_last']
    sales_thru=f'sales_thru_pb_{str(year)[-2:]}'
    trxstx = f"trxstx_pb{str(year)[-2:]}"
    stock = f"fn_stock_{input_value[5:7]}"
    whconsi_list=where['whconsi']
    divisi=where['divisi']
    cat=where['cat']
    artikel=where['artikel']
    ornal=where['ornal']
    main_query = f"""
        SELECT {col} COUNT(*) AS art_count
        FROM(   
            SELECT {col} {col_count}
                sum(stock) AS stocks
            FROM (
                SELECT fn_whconsiid, {stock} AS stock, fv_artsizecode 
                FROM gis_local.{sales_thru} WHERE 1=1 {whconsi_list} 
                UNION ALL
                SELECT fn_whconsiid, fn_stock AS stock, fv_artsizecode 
                FROM gis_local.{trxstx} 
                WHERE fd_tgltrx BETWEEN '{str(year)[-2:]}-{input_value[5:7]}-01' AND 
                '{str(year)[-2:]}-{input_value[5:10]}' {whconsi_list}
                UNION ALL
                SELECT fn_whconsiid, fn_stock AS stock, fv_artsizecode 
                FROM gis_local.trxrt_pb
                WHERE fd_tgltrx < '{str(year)[-2:]}-{input_value[5:10]}' {whconsi_list}
                UNION ALL
                SELECT fn_whconsiid, fn_stock AS stock, fv_artsizecode 
                FROM gis_local.Stock_varian_{str(year)[-2:]}
                WHERE fd_tgl BETWEEN '{str(year)[-2:]}-{input_value[5:7]}-01' AND 
                '{str(year)[-2:]}-{input_value[5:10]}' {whconsi_list}
            ) stock_backdate
            LEFT JOIN artikel_pb a ON a.fv_artsizecode = stock_backdate.fv_artsizecode
            LEFT JOIN customers_pb b ON stock_backdate.fn_whconsiid = b.fn_whconsiid
            WHERE 1=1 {cat} {artikel} {ornal} {divisi}
            GROUP BY {col_count} {gb}
            HAVING stocks>0) AS filtered_barcode
        GROUP BY filtered_barcode.{gb}
            """
    # # printol(main_query)
    # # printol(year)
    # Retry up to 5 times if error
    max_retries = 5
    for attempt in range(1, max_retries + 1):
        try:
            with get_engine('imp') as engine:  # Menggunakan context manager dengan with
                df = pd.read_sql(main_query, engine)
            return df # success, langsung return
        except Exception as e:
            print(f"[Attempt {attempt}] Query gagal: {e}")
            if attempt == max_retries:
                raise  # kalau sudah max retry, lempar error
            time.sleep(2)  # tunggu 2 detik sebelum nyoba lagi
    # results = DatabaseConnector.execute_raw_sql(main_query, using=dbLoad)
    # df = pd.DataFrame(results)
    # return df

# @timed
def df__sales_filter(input_param, 
                         col='stock_backdate.fn_whconsiid, fv_artsizecode, fv_barcode, fv_sizename,',
                         gb='fv_barcode, fn_whconsiid',
                         year=None,
                         cus='stock_backdate.',
                         art='',
                         times='this_y'):
    where=add_where(input_param,cus=cus, art=art)
    if year is None: year = datetime.now().year 
    if times == 'this_y':
        start_str = str(year) + (input_param['start'])[4:]
        ends_str = str(year) + (input_param['end'])[4:]
        input_value = input_param['input_date']
    else: 
        start_str = str(year) + (input_param['start_last'])[4:]
        ends_str = str(year) + (input_param['end_last'])[4:]
        input_value= input_param['end_last']
    if year==None: sales = f"pb_penjualan_20{input_value[2:4]}"
    else: sales=f'pb_penjualan_{str(int(year))}'
    trxstx = f"trxstx_pb{input_value[2:4]}"
    stock = f"fn_stock_{input_value[5:7]}"
    whconsi_list=where['whconsi']
    divisi=where['divisi']
    cat=where['cat']
    artikel=where['artikel']
    ornal=where['ornal']
    query = f"""select
        {col} sum(pcs) pcs, sum(sales) sales from (
        SELECT {col} sum(fn_jualpersize) as pcs, sum(fn_totalpenjualan) sales FROM gis_local.{sales} stock_backdate 
        LEFT JOIN artikel_pb a ON a.fv_artsizecode = stock_backdate.fv_artsizecode
        LEFT JOIN customers_pb b ON stock_backdate.fn_whconsiid = b.fn_whconsiid
        where fd_tgltrx between '{ends_str}' and '{start_str}' {whconsi_list} {cat} {artikel} {ornal} {divisi}
        group by {gb}
        union all
        SELECT {col} sum(fn_jualpersize) as pcs, sum(fn_totalpenjualan) sales FROM gis_local.pb_penjualan_today stock_backdate 
        LEFT JOIN artikel_pb a ON a.fv_artsizecode = stock_backdate.fv_artsizecode
        LEFT JOIN customers_pb b ON stock_backdate.fn_whconsiid = b.fn_whconsiid
        where fd_tgltrx between '{ends_str}' and '{start_str}' {whconsi_list} {cat} {artikel} {ornal} {divisi}
        group by {gb})stock_backdate
        group by {gb}
        """
    with get_engine('imp') as engine:  # Menggunakan context manager dengan with
        df = pd.read_sql(query, engine)
    return df # success, langsung return

In [6]:
def df__art_stock_wh(type_show='all'):
    if type_show == 'all':
        whid=''
    else: whid='and a.fn_whtypeid in (6,2)'
    query = f"""
        SELECT distinct
            fv_barcode, sum(fn_qty) fn_qty
        FROM gis_db.stockarticle_tb a 
        LEFT JOIN gis_db.articlesize_tb b ON a.fv_artsizecode = b.fv_artsizecode
        LEFT JOIN gis_db.article_tb c  ON b.fn_articleid = c.fn_articleid
        WHERE 1=1 {whid} AND fn_qty > 0 and fv_barcode is not null
        GROUP BY fv_barcode
    """
    with get_engine('write') as engine:  # Menggunakan context manager dengan with
        df = pd.read_sql(query, engine)
    return df # success, langsung return


## Prosess API

In [7]:
def add_df_main(input_param,y:int,core:str='fv_catname',times:str='this_y'):
    df_stock_now=df__stock_now_filter(input_param, 
                  col=f'{core},',
                  gb=f'{core}',
                  year=y,
                  times=times)
    df_sales=df__sales_filter(input_param, 
                         col=f'{core},',
                         gb=f'{core}',
                         year=y,
                         times=times)
    df_stock_early=df__stock_early_filter(input_param, 
                         col=f'{core},',
                         gb=f'{core}',
                         year=y,
                         times=times)
    df_count_art=df__count_stock_now_filter(input_param, 
                         col=f'{core},',
                         gb=f'{core}',
                         year=y,
                         times=times)
    # kalau semua dataframe kosong → langsung return kosong
    if  df_sales.empty and df_stock_early.empty :
        return pd.DataFrame(
            columns=[
                core, 'Stock Awal','Total Stock','Sales (pcs)','Sales (Rp)',
                'Kontribusi Sales','Sales Thru','Stock Akhir','Kontribusi Stock',
                'Jumlah Artikel','Kontribusi pcs'
            ])
    
    df=pd.merge(df_stock_now,df_sales, how='outer',on=f'{core}').fillna(0)
    print(df.info())
    df=pd.merge(df,df_stock_early, how='outer',on=f'{core}').fillna(0)
    df=pd.merge(df,df_count_art, how='outer',on=f'{core}').fillna(0)
    
    df[['stocks','pcs','sales','stocks_early']] = df[['stocks','pcs','sales','stocks_early']].apply(
        pd.to_numeric, errors="coerce"
    ).fillna(0).astype('int64')
    df['Kontribusi Sales'] = (df['sales'] / df['sales'].sum() * 100).round(2)
    df['Kontribusi Stock'] = (df['stocks'] / df['stocks'].sum() * 100).round(2)
    df['Kontribusi pcs'] = (df['pcs'] / df['pcs'].sum() * 100).round(2)

    df['Total Stock']=df['stocks']+df['pcs']
    df['Sales Thru']=((df['pcs']/df['Total Stock'].fillna(0)) * 100
        ).fillna(0).round(2)
    df=df[[f'{core}','stocks_early','Total Stock','pcs','sales','Kontribusi Sales','Sales Thru','stocks','Kontribusi Stock','art_count','Kontribusi pcs']]
    df=df.rename(columns={'stocks': 'Stock Akhir',
                          'pcs': 'Sales (pcs)',
                          'sales': 'Sales (Rp)',
                          'stocks_early': 'Stock Awal',
                          'art_count':'Jumlah Artikel'})
    return df

In [8]:
from django.db import connection
def execute_raw_sql(query, params=None):
    with connection.cursor() as cursor:
        cursor.execute(query, params or [])
        columns = [col[0] for col in cursor.description]  # Get column names from cursor description
        result = cursor.fetchall()

        # Convert result to a list of dictionaries where column names are keys
        formatted_result = [dict(zip(columns, row)) for row in result]
    return formatted_result

# Test

In [77]:
from modules.queries import (df__penju, df__tokoawal, df__transaction, df__cus, get_monthly_sales_data, 
                             df__sales_thru_24, df__sales_thru_25, df__sales_thru_25_count_artikel, df__artsizecode, df__gdmd, fetch_data_margin,
                             df__sales_, df__customers_pb, df__artikel_pb, df__artikel_pb_read, 
                             df__customers_pb_read, df__cus_user, df__sales_invoice,  df__sales_bz_disc2nd,
                             df__sales_bz, df__sales_consi_disc2nd, df__sales_online_disc2nd, df__otsdm, df__otsdo,
                             df__penju_today, df__artikel_status, df__tglso_, df__tglsa_ ) 
from modules.proses import (save_to_sql, add_index_and_foreign_keys, clean_and_transform_data, 
                            insert_disc2_grouping, insert_margin, rename_col_py_to_sql, insert_pk,
                            get_tgl, get_stock, get_stock_data, merge_and_calculate, merge_detail, 
                            generate_rename_dict,trx_rt, process_data_margin,rename_col_sql_to_xlsx_gis,
                            get_artikel_md, get_new_or_changed_rows,upsert_data, process_data_cus,
                            sls_disctok,proc_pivot_table_n, proc_add_aging)

## md size 1 all store

In [78]:
df_cus= df__cus()
# SELECT fn_sizeid, fv_artsizecode, fn_artsizeid from gis_db1.articlesize_tb
df_artsq= df__artsizecode(col='fv_artsizecode, fv_barcode, fv_sizename,fv_configname',stat='imp')
# print(df_artsq.columns)

# df_sqq24= df__Sales_thru_24('*')
df_sq25= df__sales_thru_25('*')

# df_sq24=df_sqq24.copy()
# df_sq25=df_sqq25.copy()

# df_sq24 = pd.merge(df_sq24, df_artsq, on='fv_artsizecode', how='left')
# df_sq24 = pd.merge(df_sq24,df_cus,on='fn_whconsiid',how='left')

df_sq25 = pd.merge(df_sq25, df_artsq, on='fv_artsizecode', how='left', copy=False)
df_sq25.info(memory_usage='deep')
df_sq25 = pd.merge(df_sq25,df_cus,on='fn_whconsiid',how='left', copy=False)
df_sq25.info(memory_usage='deep')
# df_sq24['fn_artsizeid']=df_sq24['fn_artsizeid'].astype(int)
df_sq25['fn_artsizeid'] = pd.to_numeric(df_sq25['fn_artsizeid'], downcast='integer')
df_sq25.info(memory_usage='deep')

# df_sq24['pk_id']=df_sq24['fn_whconsiid'].astype(str)+"/"+df_sq24['fn_artsizeid'].astype(str)
df_sq25['pk_id']=df_sq25['fn_whconsiid'].astype(str)+"/"+df_sq25['fn_artsizeid'].astype(str)
df_sq25=df_sq25.drop(columns=['fn_cusid'])
#############
# print(df_sq25.columns)  
# df_sq24 = generate_rename_dict(2024, 2024, df_sq24)
df_sq25 = generate_rename_dict(2025, 2025, df_sq25)
# kolom_baru24 = (['pk_id','toko','code toko','artikel','size'] + 
#                 [col for col in df_sq24.columns if col not in 
#                 ['pk_id','toko','code toko','artikel', 'size','barcode','fv_artsizecode','fn_artsizeid','fn_whconsiid']])

kolom_baru25 = (['pk_id','toko','code toko','artikel','size'] + 
                [col for col in df_sq25.columns if col not in 
                ['pk_id','toko','code toko','artikel', 'size','barcode','fv_artsizecode','fn_artsizeid','fn_sizeid','fn_whconsiid']])


# df_sq24 = df_sq24[kolom_baru24]
df_sq25 = df_sq25[kolom_baru25]
del df_artsq, df_cus
# gc.collect()
# df_sq24 = df_sq24.loc[~(df_sq24['artikel'].isnull())]
df_sq25 = df_sq25.loc[~(df_sq25['artikel'].isnull())]
df = proc_pivot_table_n(df_sq25,
                values='stock akhir',
                col_index=['code toko','artikel'],
                columns=['size'],
                col_ket=['fv_configname'],col_select=['code toko','artikel','status'])
df_sq25=pd.merge(df_sq25,df[['code toko','artikel','status']],how='left',on=['code toko','artikel'])
del df
# gc.collect()
df_sq25=df_sq25.rename(columns={'fv_configname':'bot/top'})
# df_result24=df_sq24.select_dtypes(include='number').sum()
df_sq25=proc_add_aging(df_sq25)

# df_sq24.to_excel('/opt/share/For MDstock artikel all store 2024.xlsx', index=False)
total_rows = len(df_sq25)
chunks = 3
chunk_size = math.ceil(total_rows / chunks)  # otomatis bagi 3

with pd.ExcelWriter('/opt/share/For MD/stock artikel all store 2025.xlsx', engine='xlsxwriter') as writer:
    for i in range(chunks):
        start_row = i * chunk_size
        end_row = min(start_row + chunk_size, total_rows)
        sheet_df = df_sq25.iloc[start_row:end_row]
        sheet_df.to_excel(writer, index=False, sheet_name=f'data_{i+1}')
# df_sq24.to_excel('/opt/share/For MDstock artikel all store 2024.xlsx', index=False)
# df_sq25.to_excel('/opt/share/For MD/stock artikel all store 2025.xlsx', index=False)

# df_result24.to_excel('df_result24.xlsx')
# df_result25.to_excel('/opt/share/PB DB/df_result25.xlsx')

KeyboardInterrupt: 

In [ ]:
df_sq25=df_sq25.rename(columns={'artikel':'fv_barcode'})

In [ ]:
df_sq25=proc_add_aging(df_sq25)

# df_sq24.to_excel('/opt/share/For MDstock artikel all store 2024.xlsx', index=False)
total_rows = len(df_sq25)
chunks = 3
chunk_size = math.ceil(total_rows / chunks)  # otomatis bagi 3

# with pd.ExcelWriter('/opt/share/For MD/stock artikel all store 2025.xlsx', engine='xlsxwriter') as writer:
#     for i in range(chunks):
#         start_row = i * chunk_size
#         end_row = min(start_row + chunk_size, total_rows)
#         sheet_df = df_sq25.iloc[start_row:end_row]
#         sheet_df.to_excel(writer, index=False, sheet_name=f'data_{i+1}')

In [ ]:
query=f'''
    sELECT  fv_toko, fv_spv,fv_barcode, a.fv_artsizecode,fv_sizename,  fn_stock, fv_name_doc,fv_stat, fd_tgltrx FROM gis_local.trxstx_pb25 a
    left join artikel_pb b on a.fv_artsizecode=b.fv_artsizecode  
    left join customers_pb c on a.fn_whconsiid=c.fn_whconsiid
    where fv_spv = 'Muhamad Ali Alfaridzy'
    and fd_tgltrx between '2025-06-01' and '2025-08-22' ;
    '''
with get_engine('imp') as engine_imp:
    df = pd.read_sql(query, engine_imp)
    df.to_excel('Exekusi Ali.xlsx')

In [ ]:
df_sq25.info()

In [ ]:
col_index=['code toko','artikel']
col_ket=['fv_configname']
columns=['size']
df=df_sq25
values='stock akhir'

In [ ]:
all_sizes_bottom = ['25', '26', '27', '28', '29', '30', '31', '32', '33', '34', '35', '36', '37', '38',
                        '39', '40', '42', '44', 'ALLSIZE']
all_sizes_top = ['FS', 'XS', 'SS', 'S', 'M', 'L', 'XL', 'XXL', 'XXXL']
all_sizes = all_sizes_bottom + all_sizes_top
df_ket = df[col_index + col_ket].drop_duplicates()
# Buat pivot table
pivot_df = pd.pivot_table(
    df,
    values=values,
    index=col_index,
    columns=columns,
    aggfunc='sum',
    fill_value=0).reset_index()

# Merge kembali kolom tambahan
pivot_table = pd.merge(pivot_df, df_ket, on=col_index, how='left')
for size in all_sizes:
    if size not in pivot_table.columns:
        pivot_table[size] = 0
    pivot_table[size] = pd.to_numeric(pivot_table[size], errors='coerce').fillna(0).astype(int)

# Atur ulang urutan: non-size columns dulu, lalu size columns
size_columns = [size for size in all_sizes if size in pivot_table.columns]
other_columns = [col for col in pivot_table.columns if col not in size_columns]
pivot_table = pivot_table[other_columns + size_columns]

# Hitung status (BROKEN / SEHAT)
top_columns = ["L", "M", "S", "XL", "XS", "XXL", "XXXL"]
bottom_columns = ['25', '26', '27', '28', '29', '30', '31', '32', '33', '34',
                    '35', '36', '37', '38', '39', '40', '42', '44']

pivot_table['status'] = np.where(
    (pivot_table["fv_configname"] == "TOP") & (pivot_table[top_columns].gt(0).sum(axis=1) < 3), "BROKEN",
    np.where(
        (pivot_table["fv_configname"] == "BOTTOM") & (pivot_table[bottom_columns].gt(0).sum(axis=1) < 5), "BROKEN",
        "SEHAT"
    )
)

# Hitung total jumlah qty size (sumQty)
pivot_table['sumQty'] = pivot_table[size_columns].sum(axis=1)
pivot_table=pivot_table.drop(columns=['fv_configname'])

In [ ]:
df=pd.merge(df,pivot_table[['code toko','artikel','status']],how='left',on=['code toko','artikel'])

## md size 2 all artikel

In [ ]:
df_artsq= df__artsizecode(col='fv_artsizecode, fv_barcode, fv_sizename,fv_configname',stat='imp')
df_sqq24= df__sales_thru_24('*')
df_sqq25= df__sales_thru_25('*')
df_sg= df__gdmd()

df_sq24=df_sqq24.copy()
df_sq25=df_sqq25.copy()
df_sq24=df_sq24.drop(columns=[df_sq24.columns[1]])
df_sq25=df_sq25.drop(columns=[df_sq25.columns[1]])
df_sq24['fn_artsizeid']=df_sq24['fn_artsizeid'].astype(str)
df_sq25['fn_artsizeid']=df_sq25['fn_artsizeid'].astype(str)
df_sq24 = df_sq24.groupby(['fv_artsizecode','fn_artsizeid'])[df_sq24.columns[2:]].sum().reset_index()
df_sq25 = df_sq25.groupby(['fv_artsizecode','fn_artsizeid'])[df_sq25.columns[2:]].sum().reset_index()

# Buat dictionary rename berdasarkan data
df_sq24 = generate_rename_dict(2024, 2024, df_sq24)
df_sq25 = generate_rename_dict(2025, 2025, df_sq25)
df=pd.merge(df_sq24,df_sq25,how='outer',on=['fn_artsizeid','fv_artsizecode'])
# df['qty retur']=df['qty retur_x'].fillna(0)+df['qty retur_y'].fillna(0)
print(df.columns)
df=df.rename(columns={
    'qty retur_x': 'retur 2024',
    'qty retur_y': 'retur 2025',})
df['stock akhir']=df['stock akhir_y']
df['total sales qty']=df['total sales qty_x'].fillna(0)+df['total sales qty_y'].fillna(0)
df['total stock']=df['total sales qty']+df['stock akhir'].fillna(0)
df=df.drop(columns=['total stock_x', 'stock akhir_x', 'stock akhir_y',
                    'total stock_y','total sales qty_x','total sales qty_y'])
df_sg['fv_artsizecode']=df_sg['fv_artsizecode'].str.upper()
df = df.groupby(['fv_artsizecode'], as_index=False).sum()
df = pd.merge(df, df_sg, on='fv_artsizecode', how='outer')
df = pd.merge(df, df_artsq, on='fv_artsizecode', how='left')
df =generate_rename_dict(2025, 2025, df)
# df
df = df.loc[~(df['artikel'].isnull())]
df2=df.fillna(0)
df2=df2[df2['artikel']!=0]
kolom_baru24 = ['artikel', 'size'] + [col for col in df2.columns if col not in ['artikel', 'size','fv_artsizecode','fn_artsizeid']]
df2 = df2[kolom_baru24]

In [ ]:
query='''
    SELECT fv_artsizecode, sum(stock) stock, fv_whtypename FROM gis_local.stock_gudang_pb where fv_whtypename in ('RETUR','PUSAT','KONSINETTO','QC') group by fv_artsizecode
    '''
with get_engine('imp') as engine_imp:
    df = pd.read_sql(query, engine_imp)

In [ ]:
(df.loc[~df['fv_artsizecode'].isnull(),'stock']).sum()

In [ ]:
df_sg['stock'].sum()

In [ ]:
df['stock gudang'].sum()

In [ ]:
df.loc[(df['artikel'].isnull()) & df['stock gudang']>0,'stock gudang']

In [ ]:
df2.info()

In [ ]:
df

In [ ]:
df = proc_pivot_table_n(df2,
                    values='stock akhir',
                    col_index=['artikel'],
                    columns=['size'],
                    col_ket=['fv_configname'])
df2=pd.merge(df2,df[['artikel','status']],how='left',on=['artikel'])

In [ ]:
df_sq25

In [ ]:
df2=df2.rename(columns={'fv_configname':'bot/top'})

In [ ]:
df2=pd.merge(df2,df[['artikel','status']],how='left',on=['artikel'])
df2=df2.rename(columns={'fv_configname':'bot/top'})

In [ ]:
# stock_files = generate_stock_files(base_path, start_year, end_year)

In [ ]:
# stock_files

In [ ]:
# stock_data_25

In [ ]:
# sales_data_25

# test sales to sql

In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime
from datetime import date
import calendar
from contextlib import contextmanager
from sqlalchemy import create_engine
from sqlalchemy import Integer, String, CHAR, Date, Text, text, Float
import os
from datetime import datetime
from modules.queries import (df__penju, df__tokoawal, df__transaction, df__cus, get_monthly_sales_data, 
                             df__sales_thru_24, df__sales_thru_25, df__artsizecode, df__gdmd, fetch_data_margin,
                             df__sales_, df__customers_pb, df__artikel_pb) 
from modules.proses import (save_to_sql, add_index_and_foreign_keys, clean_and_transform_data, 
                            insert_disc2_grouping, insert_margin, rename_col_py_to_sql, insert_pk,
                            get_tgl, get_stock, get_stock_data, merge_and_calculate, merge_detail, 
                            generate_rename_dict,trx_rt, process_data_margin,rename_col_sql_to_xlsx_gis)
def margin_tb(year):
    df_trx, df_artimp, df_artred, df_atrxu, df_mrgincat, df_mrgin = fetch_data_margin(year)
    df_trx_final = process_data_margin(df_trx, df_artimp, df_artred, df_atrxu, df_mrgincat, df_mrgin)
    save_to_sql(df_trx_final, f'margin{str(year)[-2:]}_pb', 'imp')

In [ ]:
year=2025
start_time2= datetime.now().strftime('%Y-%m-%d %H:%M:%S')    
print("\n[INFO] Script sales_to_sql mulai dijalankan.")
print(f"[INFO] Timestamp: {start_time2}\n")

df1= df__penju(year)
df= df1.copy()
df= clean_and_transform_data(df)
df=insert_disc2_grouping(df)
margin_tb(year)
df=insert_margin(df,year)
sum_col=['fn_jualpersize','total_penjualan2','total_disc2']
agg_col=['fn_whconsiid', 'fv_artsizecode', 'tgltrx']
fisrt_col=[col for col in df.columns if col not in sum_col + agg_col]
col_col=fisrt_col+[col for col in df.columns if col not in fisrt_col]
df=df[col_col]
df['event']=df['event'].fillna(df['status_artikel'])
df['fn_margin']=df['fn_margin'].fillna(0)
df=rename_col_py_to_sql(df).sort_values(by='fn_idurut')
df=insert_pk(df)
df['fn_nett_amount'] = (df['fn_totalpenjualan']-df['fn_total_disc2']) * (1-df['fn_margin']/100)
df['fn_sim_amount'] = np.where(df['fc_status'] == "X", df['fn_totalpenjualan'], df['fn_nett_amount'])
df['fn_dpp'] = df['fn_nett_amount']/1.11
df['fn_dpp2'] = df['fn_sim_amount']/1.11
table_name = f"pb_penjualan_{year}"
save_to_sql(df, table_name, 'imp')

add_index_and_foreign_keys(
    'imp',
    table_name,
    index_cols=["fn_whconsiid", "fv_artsizecode"],
    foreign_keys=[
        {
            "column": "fn_whconsiid",
            "ref_table": "customers_pb",
            "ref_column": "fn_whconsiid",
            "constraint_name": f"fk_whconsiid_pb_{year}"
        },
        {
            "column": "fv_artsizecode",
            "ref_table": "artikel_tb",
            "ref_column": "fv_artsizecode",
            "constraint_name": f"fk_sales_artsize_pb_{year}"
        }
    ]
)

In [ ]:
df.info()

In [ ]:
df['fn_nett_amount'] = df['fn_totalpenjualan'] * (1-df['fn_margin']/100)

In [ ]:
df

In [ ]:
save_to_sql(df, table_name, 'imp')

add_index_and_foreign_keys(
    'imp',
    table_name,
    index_cols=["fn_whconsiid", "fv_artsizecode"],
    foreign_keys=[
        {
            "column": "fn_whconsiid",
            "ref_table": "customers_pb",
            "ref_column": "fn_whconsiid",
            "constraint_name": f"fk_whconsiid_pb_{year}"
        },
        {
            "column": "fv_artsizecode",
            "ref_table": "artikel_pb",
            "ref_column": "fv_artsizecode",
            "constraint_name": f"fk_artsize_pb_{year}"
        }
    ]
)

In [ ]:
# _df_cek = define_sql_types(df)

In [ ]:
# _df_cek

In [ ]:
df.info()

In [ ]:
from modules.queries import (df__tglso, df__tglsa, df__disc2, df__margin, df__artsizecode, df__stock_so,
                             df__stock_sa, df__retur, df__size, generate_months_for_year, df__rtt)
from modules.db_connection import get_engine
from modules.proses import (save_to_sql, add_index_and_foreign_keys, clean_and_transform_data)
from sqlalchemy import Integer, String, CHAR, Date, Text, text, Float
import pandas as pd
from datetime import datetime
import os

In [ ]:
def trx_rt():
    df_trxstx= df__rtt()
    df_trxstx['tgltrx'] = pd.to_datetime(df_trxstx['tgltrx'], format='%m-%d-%Y')
    df_trxstx['fn_whconsiid']=df_trxstx['fn_whconsiid'].astype(int)
    df_trxstx['stock']=df_trxstx['stock'].astype(int)
    df_trxstx=df_trxstx.rename(columns={
        'stock':'fn_stock', 
        'tgltrx':'fd_tgltrx', 
        'stat':'fv_stat'
    })
    save_to_sql(df_trxstx, 'trxrt_pb', 'imp')

In [ ]:
trx_rt()

In [ ]:
from modules.db_connection import get_engine
from datetime import datetime, date
import pandas as pd

In [ ]:
today_str = date.today().strftime('%Y-%m-%d')
with get_engine('write') as engine:
        query=f'''
        SELECT fn_whconsiid, fv_artsizecode, IFNULL(SUM(totalplus), 0) AS stock, tgltrx, stat
    FROM (
    select
        SUBSTRING_INDEX(a.fv_noreceiving, '/', -1) AS fn_whconsiid,
        b.fv_artsizecode, 
        b.fn_qtyterima AS totalplus,
        DATE_FORMAT(a.fd_tglreceive, '%%m-%%d-%%Y') tgltrx, 'recieve' as stat
        FROM     t_receivingmst a
        INNER JOIN t_receivingdtl b ON a.fv_noreceiving = b.fv_noreceiving
        WHERE a.fc_status <> 'F'
        AND (a.fd_tglsistem BETWEEN '{today_str} 00:00:00' and '{today_str} 23:59:00'))
        AS subquery
    GROUP BY fn_whconsiid, fv_artsizecode, tgltrx;'''
        df= pd.read_sql(query, engine)

In [ ]:
from main import margin_tb

In [ ]:
margin_tb(2025)

# AS1 cosiment sales mini

In [ ]:
# from modules.proses import 
from modules.queries import df__sales_, df__customers_pb, df__artikel_pb
from modules.proses import rename_col_sql_to_xlsx_gis


In [ ]:
df=df__sales_(2023,'fn_whconsiid,fv_artsizecode,fv_event,fd_tgltrx,fn_jualpersize,fn_totalpenjualan,fn_nett_amount')
df_artpb=df__artikel_pb('fv_artsizecode,fv_barcode,fv_catname,fv_fitname,fv_brandname,fv_divname,fv_sizename,fm_price')

In [ ]:
df_cuspb=df__customers_pb('fv_toko,fc_code,fn_whconsiid')

In [ ]:
df=pd.merge(df, df_artpb, on='fv_artsizecode', how='left')
df=pd.merge(df, df_cuspb, on='fn_whconsiid', how='left')


In [ ]:
df=df[['fd_tgltrx','fc_code','fv_toko','fv_barcode','fv_catname','fv_fitname','fv_brandname',
    'fv_divname','fv_sizename','fv_event','fm_price','fn_totalpenjualan','fn_nett_amount','fn_jualpersize']]

In [ ]:
df=rename_col_sql_to_xlsx_gis(df)
# name_col = {
#         'fd_tgltrx': 'Sales Date',
#         'fc_code': 'Code Store',
#         'fv_toko': 'Store',
#         'fv_barcode':'Code Articel',
#         'fv_catname': 'Kategori',
#         'fv_fitname': 'Name Articel',
#         'fv_brandname': 'Brand',
#         'fv_divname': 'Division',
#         'fv_sizename': 'Size',
#         'fv_event': 'Status Article',
#         'fm_price': 'HET',
#         'fn_totalpenjualan': 'Total Nett Kassa',
#         'fn_nett_amount': 'Nett Amount',
#         'fn_jualpersize':'Qty'
#     }
#     # Lakukan rename dan sort
# df = df.rename(columns=name_col)


In [ ]:
output_folder = r'\\192.168.1.254\Share\_From Valdi\For Alokator\Sales cosiment\2023'

In [ ]:
df['Sales Date'] = pd.to_datetime(df['Sales Date'])

In [ ]:
# Tambahkan kolom bulan (format YYYY-MM)
df['bulan_num'] = df['Sales Date'].dt.strftime('%m')
df['bulan'] = df['Sales Date'].dt.strftime('%b')  # Nama bulan 3 huruf
df['tahun'] = df['Sales Date'].dt.strftime('%y')  # Tahun dua digit

# Simpan ke Excel per bulan
for (bulan_num, bulan, tahun), group in df.groupby(['bulan_num', 'bulan', 'tahun']):
    nama_file = f"{output_folder}\Sale Consi {bulan_num} {bulan} {tahun}.xlsx"
    group.drop(columns=['bulan', 'tahun', 'bulan_num']).to_excel(nama_file, index=False)
    print(f"✅ File disimpan: {nama_file}")

# AS2 Consiment Report

## Start

In [ ]:
from modules.db_connection import get_engine
from datetime import datetime, date
import pandas as pd
import numpy as np

In [ ]:
from modules.queries import (fetch_data_margin,df__artikel_pb, df__customers_pb,df__disc2)
from modules.proses import (save_to_sql, clean_and_transform_data,insert_disc2_grouping, insert_margin, process_data_margin, rename_col_py_to_sql, insert_pk,)  

In [ ]:
# Input
bulan_str = 'FEB'
bulan_str = 'MAR'
year = 2025

In [ ]:
import datetime
import calendar

# 1. Konversi nama bulan ke angka
bulan_angka = datetime.datetime.strptime(bulan_str, '%b').month

# 2. Dapatkan awal bulan
awal_bulan = datetime.date(year, bulan_angka, 1)

# 3. Dapatkan akhir bulan
akhir_hari = calendar.monthrange(year, bulan_angka)[1]  # jumlah hari di bulan tsb
akhir_bulan = datetime.date(year, bulan_angka, akhir_hari)
query = f'''SELECT 
    b.fn_idurut,
    a.fv_nopenjualan, 
    DATE_FORMAT(a.fd_tgljual, '%%m-%%d-%%Y') tgltrx,
    DATE_FORMAT(a.fd_tglinput, '%%m-%%d-%%Y %%H:%%i:%%s') AS tglinput,
    a.fn_cusid fn_whconsiid,
    b.fv_artsizecode,
    b.fn_hargajualawal price,
    b.fn_disc disc,
    b.fn_hargasatuan ,
    b.fn_jualpersize ,
    b.fn_hargasatuan*b.fn_jualpersize as total_penjualan2,
    e.fv_nameevent 'EVENT',
    a.fn_userid,
    b.fn_discid disc_id, 
    b.fv_namemark status_artikel, b.fv_descevent "desc",
    d.fv_namedisc 'Disc Name',
    d.fv_namemark 'event',
    b.fn_tabel,
    a.fv_nobonlantai,
    f.fv_articlename,
    b.fc_bonus,
    a.fn_grpcusid, 
    a.fv_namacust
FROM 
    t_penjualanmst a 
INNER JOIN 
    t_penjualandtl_goods b ON a.fv_nopenjualan = b.fv_nopenjualan
LEFT JOIN 
    (SELECT a.fn_discid, b.fv_namemark, a.fv_namedisc FROM discount_tb a
        INNER JOIN markdown_tb b ON a.fn_markid = b.fn_markid) d ON b.fn_discid = d.fn_discid
LEFT JOIN event_tb e ON b.fn_idevent=e.fn_idevent
LEFT JOIN article_tb f on b.fv_articlecode=f.fv_articlecode
WHERE 
    a.fv_nopenjualan LIKE 'PJ%%' and b.fc_status <> 'F' 
    and a.fc_status <> 'F' 
    and a.fd_tgljual BETWEEN '{awal_bulan} 00:00:01' and '{akhir_bulan} 23:59:59'
'''
with get_engine('read') as engine:  # Menggunakan context manager dengan with
    df = pd.read_sql(query, engine)


In [ ]:
df.info()

In [ ]:
df_consi= pd.read_csv(f'CONSI SALES {bulan_str} 2025.csv')

In [ ]:
df_consi.info()

In [ ]:
df_x=df.copy()
df11=df_x.copy()

In [ ]:
df11= clean_and_transform_data(df11)

In [ ]:
df11=insert_disc2_grouping(df11,'sum')

In [ ]:
df11.loc[df11['fv_nopenjualan']=='PJ/DPS/2503/00001/1143',['tglinput','artsize_voucher','artsize_affiliate','artsize_premi','artsize_servisfee']]

In [ ]:
# df11

In [ ]:
# margin_tb(year)
df11=insert_margin(df11,year)

In [ ]:
df_artpb=df__artikel_pb('*')
df_cuspb=df__customers_pb('*')
df_markdown= query = f'''
SELECT * FROM markdown_tb
'''
with get_engine('read') as engine:  # Menggunakan context manager dengan with
    df_markdown = pd.read_sql(query, engine)
    
query = f'''
SELECT * FROM user_tb
'''
with get_engine('read') as engine:  # Menggunakan context manager dengan with
    df_user = pd.read_sql(query, engine)

## 1.0 Margin SHUU

In [ ]:
with get_engine('read') as engine:
        query='''select a.fv_nopenjualan, fn_potonganharga, fn_totalqty qty from
        (SELECT fv_nopenjualan, fn_potonganharga FROM t_penjualanmst
        where fn_potonganharga <> 0) a 
        left join (SELECT fv_nopenjualan, sum(fn_jualpersize) fn_totalqty FROM t_penjualandtl_goods
        group by fv_nopenjualan) b on a.fv_nopenjualan=b.fv_nopenjualan
        '''
        df_disc2nd= pd.read_sql(query, engine)

with get_engine('read') as engine:
        query='''select a.fv_nopenjualan, servisfee, premi, affiliate, voucher, fn_totalqty qty from
        (SELECT fv_nopenjualan, fm_asuransi servisfee, fm_ongkir premi, fm_voucher affiliate, fm_lainlain voucher FROM t_penjualanpotongan
        ) a 
        left join (SELECT fv_nopenjualan, sum(fn_jualpersize) fn_totalqty FROM t_penjualandtl_goods
        group by fv_nopenjualan) b on a.fv_nopenjualan=b.fv_nopenjualan
        '''
        df_disc3nd= pd.read_sql(query, engine)
df_disc2nd=pd.concat([df_disc2nd, df_disc3nd])

# Jumlahkan semua kolom kecuali 'qty', dan ambil 'qty' pertama
agg_dict = {col: 'sum' for col in df_disc2nd.columns if col not in ['fv_nopenjualan', 'qty']}
agg_dict['qty'] = 'first'

# Grouping
df_disc2nd = df_disc2nd.groupby('fv_nopenjualan', as_index=False).agg(agg_dict)

df_disc2nd['artsize_disc2nd'] = np.where(df_disc2nd['fn_potonganharga'] == 0,0,(df_disc2nd['fn_potonganharga'] / df_disc2nd['qty']).round(2))
df_disc2nd['artsize_servisfee'] = np.where(df_disc2nd['servisfee'] == 0,0,(df_disc2nd['servisfee'] / df_disc2nd['qty']).round(2))
df_disc2nd['artsize_premi'] = np.where(df_disc2nd['premi'] == 0,0,(df_disc2nd['premi'] / df_disc2nd['qty']).round(2))
df_disc2nd['artsize_affiliate'] = np.where(df_disc2nd['affiliate'] == 0,0,(df_disc2nd['affiliate'] / df_disc2nd['qty']).round(2))
df_disc2nd['artsize_voucher'] = np.where(df_disc2nd['voucher'] == 0,0,(df_disc2nd['voucher'] / df_disc2nd['qty']))
df_disc2nd.replace([np.inf, -np.inf], 0, inplace=True) 
# # Mendapatkan nama kolom pertama dan terakhir
# col0 = df_disc2nd.columns[0]
# col_last = df_disc2nd.columns[-1]

In [ ]:
df_disc2nd = df_disc2nd.fillna(0)
# df_disc2nd = df_disc2nd.groupby(['fv_nopenjualan'], as_index=False).sum()

In [ ]:
pip install pandas==1.5.3 sqlalchemy==1.4.46


In [ ]:
df_disc2nd.loc[df_disc2nd['fv_nopenjualan']=='PJ/DPS/2501/00113/1128']

In [ ]:
save_to_sql(df_disc2nd, f'disc2artsize_pb', 'imp')

In [ ]:
df
df_disc2= df__disc2()
df = pd.merge(df, df_disc2, on='fv_nopenjualan', how='left')
df = df.loc[~df.duplicated()]
df[['artsize_disc2nd','artsize_servisfee','artsize_premi','artsize_affiliate','artsize_voucher']] = df[
    ['artsize_disc2nd','artsize_servisfee','artsize_premi','artsize_affiliate','artsize_voucher']].fillna(0)
df['total_disc2']=df['artsize_disc2nd']*df['fn_jualpersize']
sum_col=['fn_jualpersize','total_penjualan2','total_disc2',
            'artsize_servisfee','artsize_premi','artsize_affiliate','artsize_voucher']
agg_col=['fv_nopenjualan','fn_whconsiid', 'fv_artsizecode', 'tgltrx']
fisrt_col=[col for col in df.columns if col not in sum_col + agg_col]
print(f" {sum_col}")
print(f" {fisrt_col}")

In [ ]:
df

In [ ]:
df.loc[df['fv_nopenjualan']=='PJ/DPS/2503/00001/1143']

In [ ]:
df_margin= query = f'''
SELECT * FROM margin25_pb
'''
with get_engine('imp') as engine:  # Menggunakan context manager dengan with
    df_margin = pd.read_sql(query, engine)

In [ ]:
from modules.queries import (fetch_data_margin)
df_trx, df_artimp, df_artred, df_atrxu, df_mrgincat, df_mrgin = fetch_data_margin(year)

In [ ]:
with get_engine('read') as engine:
    query_trx = f'''
        SELECT fn_idurut, fn_whconsiid, fd_tgljual fd_tgltrx, fv_artsizecode 
        FROM gis_db1.t_penjualandtl_goods where fd_tgljual BETWEEN '{year}-01-01' AND '{year}-12-31'
        '''
    df_trx = pd.read_sql(query_trx, engine)

In [ ]:
df_trxartcat = pd.merge(df_trx, df_artimp, on='fv_artsizecode', how='left')
df_trxartcat = pd.merge(df_trxartcat, df_artred, on='fv_barcode', how='left')
df_trxartcat = pd.merge(df_trxartcat, df_atrxu, on='fn_idurut', how='left')
df_trxartcat["fn_markid"] = df_trxartcat["fn_markid"].combine_first(df_trxartcat["markid2"])
df_trxartcat['fn_markid'] = df_trxartcat['fn_markid'].astype(int)
df_trxartcat["fd_tgltrx"] = pd.to_datetime(df_trxartcat["fd_tgltrx"])

# Merge with margin data
merged_df = pd.merge(df_trxartcat, df_mrgincat, on=["fn_whconsiid", "fn_catid", "fn_markid"], how="left")
filtered_df = merged_df[(merged_df["fd_tgltrx"] >= merged_df["fd_begindate"]) & (merged_df["fd_tgltrx"] <= merged_df["fd_enddate"])]
df_mgcatfin = filtered_df[['fn_idurut', 'fn_margin']]

merged_df2 = pd.merge(df_trxartcat, df_mrgin, on=["fn_whconsiid", "fn_markid"], how="left")
filtered_df2 = merged_df2[merged_df2["fd_tgltrx"] >= merged_df2["fd_effectivedate"]]
df_mgfin = filtered_df2.loc[filtered_df2.groupby(["fn_idurut", "fn_whconsiid", "fn_catid", "fn_markid"])["fd_effectivedate"].idxmax()]

# Merge final margin data
df_trx_final = df_trx[['fn_idurut']]
df_trx_final = pd.merge(df_trx_final, df_mgcatfin, on="fn_idurut", how="left")
df_trx_final = pd.merge(df_trx_final, df_mgfin[["fn_idurut", 'fn_margin']], on="fn_idurut", how="left", suffixes=("", "_from_mgfin"))
df_trx_final["fn_margin"] = df_trx_final["fn_margin"].combine_first(df_trx_final["fn_margin_from_mgfin"])
df_trx_final.drop(columns=["fn_margin_from_mgfin"], inplace=True, errors="ignore")
df_trx_final = df_trx_final.fillna(0)
df_trx_final = df_trx_final[~df_trx_final.duplicated()]
df_trx_final['fc_status'] = np.where(df_trx_final['fn_margin'] == 0, 'X', 'O')
df_trx_final2= pd.merge(df_trx_final, df_trxartcat[['fn_whconsiid','fn_markid','fn_idurut']], on='fn_idurut', how="left")
df_trx_final3= df_mrgin.sort_values('fd_effectivedate', ascending=False)
df_trx_final3= df_trx_final3.drop_duplicates(subset=['fn_whconsiid','fn_markid'], keep='first')
df_trx_final3 = df_trx_final3.groupby('fn_whconsiid')['fn_margin'].agg(lambda x: x.mode().iloc[0]).reset_index()
df_trx_final4= pd.merge(df_trx_final2, df_trx_final3, on=["fn_whconsiid"], how="left", suffixes=("", "_sim"))
df_trx_final4['fn_margin'] = df_trx_final4.apply(lambda row: row['fn_margin_sim'] if row['fn_margin'] == 0 else row['fn_margin'], axis=1)

In [ ]:
df_trx_final4.loc[df_trx_final4['fn_whconsiid']==186]

In [ ]:
save_to_sql(df_trx_final4, f'margin{str(year)[-2:]}_pb', 'imp')

In [ ]:
# df_consi.info()

In [ ]:
# df['fv_nopenjualan'].info()

## 1.1 Praproses

In [ ]:
df11['tglinput']=pd.to_datetime(df11['tglinput'])

In [ ]:
# Hitung selisih hari
df11['Day Input'] = (df11['tglinput'] - df11['tgltrx']).dt.days

# Beri skor berdasarkan selisih hari
def hitung_score(hari):
    if hari <= 3:
        return 'BAIK'
    elif 4 <= hari <= 10:
        return 'SEDANG'
    else:
        return 'BURUK'

df11['Score'] = df11['Day Input'].apply(hitung_score)


In [ ]:
df11['fv_nobonlantai'] = "'" + df11['fv_nobonlantai']
df11['fv_nobonlantai'] = df11['fv_nobonlantai'].str.replace(',', '', regex=False)
df11['fv_nobonlantai'] = df11['fv_nobonlantai'].str.replace("â€™", "’", regex=False)
df11['fv_nobonlantai'] = df11['fv_nobonlantai'].str.replace("â", "", regex=False)
df11['desc']=df11['desc'].str.replace(',', '', regex=False)
df11.loc[df11['fv_articlename']=='HIRT (CREW NECK) REGULAR','fv_articlename']= 'SHORT SLEEVE T-SHIRT (CREW NECK) REGULAR'
df11['fn_margin']= df11['fn_margin'].fillna(0)
df11['desc']= df11['desc'].fillna('None')


In [ ]:
df11.info()

In [ ]:
df11

In [ ]:
df11.loc[df11['fv_nopenjualan']=='PJ/DPS/2503/00532/325',['fv_nobonlantai','artsize_affiliate','artsize_premi','artsize_servisfee']]

In [ ]:
df.loc[df['fv_nopenjualan']=='PJ/DPS/2502/00094/1128',['fv_artsizecode','fc_bonus','fn_jualpersize','total_penjualan2']]

## 1.2 Main proses

In [ ]:
df12= df11.copy()

In [ ]:
df12.loc[df12['fv_nopenjualan']=='PJ/DPS/2503/00001/1143',['tglinput']]

In [ ]:
df12.info()

In [ ]:
df12=pd.merge(df12, df_cuspb[['fn_whconsiid','fc_code','fv_toko']], on='fn_whconsiid', how='left')
df12 = pd.merge(df12, df_artpb[['fv_artsizecode', 'fv_barcode', 'fv_catname', 'fv_fitname', 'fv_brandname', 'fv_divname', 'fv_sizename','fm_price']], on='fv_artsizecode', how='left')
df12 = pd.merge(df12, df_user[['fn_userid','fv_username']], on='fn_userid', how='left')

In [ ]:
df12['event']=df12['event'].fillna(df12['status_artikel'])
df12['event']=df12['event'].replace('DISC 50+20%','DISC 50%+20%')

In [ ]:
df12 = pd.merge(df12, df_markdown[['fv_namemark','fm_amount']], left_on='event',right_on='fv_namemark', how='left')

In [ ]:
df12['fm_price2'] = np.where(df12['event'].str.contains('SP|NP'), df12['price'], df12['fm_price'])

In [ ]:
conditions = [
    df12['fc_bonus'].str.contains('T'),                    # Kondisi 1
    df12['event'].str.contains('SP|NP'),                     # Kondisi 2
    df12['event'].str.contains('DISC 50%\\+20%') ,                # Kondisi 3
    df12['event'].str.contains('DISC 50%'),                    # Kondisi 4
    df12['event'].str.contains('POTONGAN'),                    # Kondisi 5
    df12['event'].str.contains('DISC 70%'),                    # Kondisi 6
    df12['event'].str.contains('NORMAL')                        # Kondisi 7
]
choices_price = [
    df12['fm_amount'],           # Jika kondisi 1 terpenuhi
    df12['fm_amount'],  # Jika kondisi 2 terpenuhi
    df12['fm_price'],        # Jika kondisi 3 terpenuhi
    df12['fm_price'],        # Jika kondisi 4 terpenuhi
    df12['price'],           # Jika kondisi 5 terpenuhi
    df12['fm_price'],        # Jika kondisi 6 terpenuhi
    df12['fm_price'],        # Jika kondisi 7 terpenuhi
]
choices_disc = [
    df12['disc'],           # Jika kondisi 1 terpenuhi
    df12['disc']*0,         # Jika kondisi 2 terpenuhi
    df12['fm_price']*0.6,   # Jika kondisi 3 terpenuhi
    df12['fm_price']*0.5,   # Jika kondisi 4 terpenuhi
    df12['disc'],           # Jika kondisi 5 terpenuhi
    df12['fm_price']*0.7,   # Jika kondisi 6 terpenuhi
    df12['disc']*0,         # Jika kondisi 7 terpenuhi
]
# Else: pakai nilai default, misalnya fm_price
df12['price'] = np.select(conditions, choices_price, default=df12['fm_price'])
df12['disc'] = np.select(conditions, choices_disc, default=df12['disc'])
df12['disc'] =round(df12['disc']).fillna(0).astype(int)

In [ ]:
df12.loc[df12['fc_bonus']=='T',['event','fm_amount','price','fc_bonus']]

In [ ]:
df12.info()

In [ ]:
df12['Total Brutto'] = np.where(df12['event'].str.contains('SP|NP'), 
                                df12['fm_amount']*df12['fn_jualpersize'], 
                                df12['price']*df12['fn_jualpersize'])
df12['Total Disc'] = df12['disc']*df12['fn_jualpersize']
df12['Total Net Kassa']=df12['Total Brutto']- df12['Total Disc'] - df12['total_disc2']

In [ ]:
df12['Margin Amount (pcs)'] = (df12['Total Net Kassa']/ df12['fn_jualpersize'] * df12['fn_margin'] / 100).round(0) 
df12['Margin Amount'] = (df12['Total Net Kassa'] * df12['fn_margin'] / 100).round(0)
df12['Net Amount'] = df12['Total Net Kassa'] - df12['Margin Amount']
df12['DPP'] = (df12['Net Amount'] / 1.11).round(2)
df12['Ppn'] = (df12['Net Amount'] / 1.11 * 0.11) .round(2)
df12.loc[df12['fn_grpcusid']==404,['DPP','Ppn']]=0

In [ ]:
df12['artsize_servisfee'] = df12['artsize_servisfee'] * df12['fn_jualpersize']
df12['artsize_premi'] = df12['artsize_premi'] * df12['fn_jualpersize']
df12['artsize_affiliate'] = df12['artsize_affiliate'] * df12['fn_jualpersize']

In [ ]:
df12.loc[df12['fn_grpcusid']==404,'DPP+PPN']=df12['Total Net Kassa']-df12['artsize_voucher']
df12['DPP+PPN'] = df12['DPP+PPN'].fillna(0)
df12['DPP Online'] = np.where(df12['DPP+PPN'] == 0,0,((df12['DPP+PPN'] / 1.11)).round(2).fillna(0))
df12['PPN Online'] = np.where(df12['DPP+PPN'] == 0,0,((df12['DPP+PPN'] / 1.11 * 0.11)).round(2).fillna(0))
df12.loc[df12['fn_grpcusid']==404,'Net Amount'] = (df12['DPP+PPN'] - df12['Margin Amount'] - df12['artsize_servisfee']
                                                - df12['artsize_premi'] - df12['artsize_affiliate']) 

In [ ]:
df12.loc[df12['fv_nopenjualan']=='PJ/DPS/2502/00391/1128',['event','fm_amount','DPP Online','PPN Online','DPP+PPN','Total Disc','total_disc2','Total Net Kassa']]

## 1.3 Cleaning and Set up

In [ ]:
df13=df12.copy()

In [ ]:
df13['fv_toko']=df13['fv_toko'].str.replace(',', '', regex=False)
df13.loc[df13['fn_grpcusid']==415,'fv_toko']='VALINO OUTLET'

In [ ]:
# Teks salah encoding
text = "937â€™934â€™941â€™943942â€™932â€™938â€™936"

# Perbaiki encoding: asumsikan teks dibaca sebagai latin-1, ubah ke utf-8
# fixed_text = text.encode('latin1').decode('utf-8')
clean_text = text.replace("â€™", "’")
print(clean_text)

In [ ]:
df13=df13.rename(columns={
    'fv_nopenjualan': 'No. Document',
    'tgltrx': 'Sales Date',
    # 'Day Input'
    # 'Score'
    'fc_code': 'Code Store',
    'fv_toko': 'Store',
    'fv_barcode': 'Code Article',
    'fv_artsizecode': 'Article',
    # 'fv_catname': 'Name Article',
    # 'fv_fitname': 'Name Article',
    'fv_articlename': 'Name Article',
    'fv_brandname': 'Brand',
    'fv_divname': 'Division',
    'fv_sizename': 'Size',
    'event': 'Status Article',
    'fm_price': 'HET',
    'price': 'Price',
    'disc': 'Disc',
    'fn_hargasatuan': 'Net Kassa',  
    'fn_jualpersize': 'Qty',
    'total_disc2':'Total Disc 2',
    # 'total_penjualan2': 'Total Net Kassa',  
    'fn_margin': 'Margin',
    'EVENT': 'Event',
    # 'Disc Name': 'Disc Name'
    'tglinput': 'Input Date',
    'fv_username': 'User Input',
    'desc': 'Description',
    'fn_tabel': 'Table',
    'artsize_servisfee': 'Service Fee',
    'artsize_premi': 'Premi',
    'artsize_affiliate': 'Affiliate',
    'artsize_voucher': 'Voucher',
    'fv_nobonlantai': 'No. Bon Lantai',

    
})

In [ ]:
col_df=['No. Document', 'Sales Date','Day Input','Score', 'Code Store', 'Store',
        'Code Article', 'Article', 'Name Article', 'Brand', 'Division', 'Size','Status Article',
        'HET', 'Price', 'Disc', 'Net Kassa', 'Qty','Total Brutto','Total Disc','Total Disc 2',
        'Total Net Kassa', 'Margin', 'DPP+PPN', 'DPP Online', 'PPN Online',
        'Margin Amount (pcs)','Margin Amount',
        'Service Fee','Premi','Affiliate','Voucher',
        'Net Amount','DPP','Ppn',
        'Event','Disc Name', 'Input Date', 'User Input', 'Description', 'Table', 'No. Bon Lantai',]
df13=df13[col_df]

In [ ]:
df13.info()

In [ ]:
df_consi2=df_consi.copy()

In [ ]:
df_consi.info()

In [ ]:
df_consi2=df_consi.copy()
# Misal: indeks kolom yang ingin dihapus
index_to_drop = [0,27,43,39]

# Ambil nama kolom berdasarkan index
cols_to_drop = df_consi2.columns[index_to_drop]

# Drop kolom tersebut
df_consi2 = df_consi2.drop(columns=cols_to_drop)


## 1.4 CHECKING

In [ ]:
df1=df13.sort_values(['No. Document','Article'])
df2=df_consi2
df2['Article']=df2['Article'].str.upper()
df2=df2.sort_values(['No. Document','Article'])
df2=df2.rename(columns={'Margin Amount / pcs':'Margin Amount (pcs)'})

In [ ]:
df1['Input Date'] = df1['Input Date'].dt.strftime('%#m/%#d/%Y %H:%M')
df1['Input Date']=pd.to_datetime(df1['Input Date'], format='mixed', errors='coerce')

In [ ]:
df2['Input Date']=pd.to_datetime(df2['Input Date'], format='mixed', errors='coerce')
# df2.loc[df2['No. Document']=='PJ/DPS/2503/00001/1143','Input Date']

In [ ]:
# Pastikan index dan kolomnya sama
df1 = df1.reset_index(drop=True).fillna('None')

df2 = df2.reset_index(drop=True).fillna('None')
# df1['Sales Date'] = pd.to_datetime(df1['Sales Date']).dt.strftime('%-m/%-d/%Y')
# df1['Input Date'] = pd.to_datetime(df1['Sales Date']).dt.strftime('%-m/%-d/%Y')

if not df1.columns.equals(df2.columns):
    print("⚠️ Kolom tidak sama")
    print("Kolom df1:", df1.columns)
    print("Kolom df2:", df2.columns)

# Buat DataFrame mask selisih nilai
mask = df1 != df2

# Tampilkan perbedaan (baris dan kolom)
diff = []
kolom_fix=mask.columns[(~mask).all()].tolist()
for row in range(len(df1)):
    for col in df1.columns:
        if col in kolom_fix:
            continue  # lewati kolom yang sudah fix
        if mask.loc[row, col]:
            diff.append({
                "index": row,
                "kolom": col,
                "df1": df1.loc[row, col],
                "df2": df2.loc[row, col]
            })

# Convert ke DataFrame
df_diff = pd.DataFrame(diff)

In [ ]:
def hitung_selisih(row):
    try:
        # Coba konversi ke float
        num1 = float(row['df1'])
        num2 = float(row['df2'])
        selisih = abs(num1 - num2)
        status = "OK" if selisih < 0.1 else "Not OK"
    except:
        selisih = 'N/A'
        status = "OK"
    return pd.Series([selisih, status], index=["selisih", "status"])

# Tambahkan kolom 'selisih' dan 'status'
# df_diff[['selisih', 'status']] = df_diff.apply(hitung_selisih, axis=1)

In [ ]:
# df_diff.loc[df_diff['status'] == 'Not OK', ['index', 'kolom', 'df1', 'df2', 'selisih']].sort_values(by='index')

In [ ]:
df1.info()

In [ ]:
# pd.merge(df_diff,df2[['nomor','No. Document','Article']],how='left',left_on='index',right_on='nomor').to_excel('File Diff.xlsx')

In [ ]:
df_diff

In [ ]:
df_diff.loc[df_diff['kolom']=='Net Kassa']

In [ ]:
df_diff['kolom'].value_counts()

kolom consi feb
Name Article           532
DPP Online             245 
Net Amount             169
Margin Amount (pcs)    165
Margin Amount          164
Margin                  78
DPP                     78
Ppn                     78
Total Net Kassa          8
DPP+PPN                  7
Price                    4
Size                     3

In [ ]:
df1['nomor'] = range(len(df1))
df2['nomor'] = range(len(df2))
# Fungsi untuk mengubah ke float jika bisa, kalau tidak biarkan string
def parse_number(val):
    try:
        # Cek apakah string hanya angka atau notasi ilmiah
        return float(val)
    except:
        return val  # Biarkan tetap string

# Terapkan ke kolom
df2['Article'] = df2['Article'].apply(parse_number)

In [ ]:
# Hapus spasi, newline, dan tab tersembunyi
df1.columns = df1.columns.str.replace('\n', '', regex=False).str.replace('\t', '', regex=False).str.strip()
df2.columns = df2.columns.str.replace('\n', '', regex=False).str.replace('\t', '', regex=False).str.strip()

col_merge=['nomor','No. Document','Article','Status Article'
        #    '
        #    'Qty','HET','Price',
        ]

con_on=['Total Net Kassa','Total Brutto','Total Disc','Total Disc 2']
col_cek = [fr'Premi']
col_cek_x= [col + '_x' for col in col_cek]
col_cek_y= [col + '_y' for col in col_cek]
col_all =col_merge+col_cek+con_on
df_check=pd.merge(df1[col_all],df2[col_all],how='inner',on=col_merge)


In [ ]:
# df_check=pd.merge(df1[col_all],df2[col_all],how='inner',on=col_merge)

# Ambil daftar index unik dari df_diff
# index_list = df_diff.loc[df_diff['status'] == 'Not OK','index'].unique()

# Filter df1 berdasarkan index yang ada di index_list
# df1_filtered = df1.loc[df1.index.isin(index_list)]
# df2_filtered = df2.loc[df2.index.isin(index_list)]
# df_checkall=pd.merge(df1_filtered, df2_filtered, how='outer', on=col_merge)

In [ ]:
mask = (df_check[col_cek_x].values != df_check[col_cek_y].values).any(axis=1)
df_beda=df_check[mask]

In [ ]:
# df1['Input Date'].to_excel('sec.xlsx')

In [ ]:
df_beda

In [ ]:
index_list_beda= df_beda['nomor'].unique()

In [ ]:
def hitung_selisih(row):
    try:
        # Coba konversi ke float
        num1 = float(row[col_cek_x])
        num2 = float(row[col_cek_y])
        selisih = abs(num1 - num2)
        status = "OK" if selisih < 0.1 else "Not OK"
    except:
        selisih = 'N/A'
        status = "OK"
    return pd.Series([selisih, status], index=["selisih", "status"])

In [ ]:
# Tambahkan kolom 'selisih' dan 'status'
# df_beda['']
# df_beda[['selisih', 'status']] = df_beda.apply(hitung_selisih, axis=1)

In [ ]:
df_beda=pd.merge(df_beda,df[['fv_nopenjualan','fn_grpcusid','fn_whconsiid']],how='left',left_on='No. Document',right_on='fv_nopenjualan').drop_duplicates(subset=['nomor'])

In [ ]:
df_beda['fn_grpcusid'].value_counts()

In [ ]:
dloc=((df_beda['fn_grpcusid'] != 2),'fn_grpcusid'  )
df_beda.loc[dloc].value_counts()

In [ ]:
dloc=((df_beda['fn_grpcusid'] == 2),'fn_whconsiid'  )
df_beda.loc[dloc].value_counts()

In [ ]:
df_beda['fn_grpcusid'].value_counts()

In [ ]:
df_beda

In [ ]:
# df_beda.loc[((df_beda['status']!= 'OK') ,['fn_grpcusid'])].value_counts()

In [ ]:
# df_checkall.to_excel('consi.xlsx')

In [ ]:
# df1.loc[df1['nomor'].isin(index_list_beda)].to_excel('dif1.xlsx', index=False)

In [ ]:
# df_beda.to_excel('diff.xlsx', index=False)

In [ ]:
# df1.to_excel('consi feb.xlsx', index=False)

# AS3 max time input

In [ ]:
from modules.db_connection import get_engine, get_db
from datetime import datetime, date
import pandas as pd

In [ ]:
def max_time_input(col_tgl_input,stat,table):
    db=get_db(stat)
    query = f'''
    select max({col_tgl_input}) max_input from {db}.{table}
    '''
    with get_engine(stat) as engine:  # Menggunakan context manager dengan with
        df = pd.read_sql(query, engine)
    return df

In [ ]:
max_time_input('fd_tglinput','write','t_mutasimst')

# AS4 stock ots DO/DM

In [ ]:
from modules.queries import (df__otsdo, df__otsdm, df__otsdo_rt, max_time_input, df__customers_pb, min_time_input,df__otsdo_del)
from modules.proses import (save_to_sql,add_index,add_foreign_keys,add_primary_key,upsert_data)
from modules.db_connection import get_engine, get_db
import pandas as pd

In [ ]:
with get_engine('write') as engine:
    OutstandingDO_query = '''SELECT 
                fn_whconsiid ,fv_artsizecode, sum(fn_qty) OstDO
                FROM gis_db.dodtl_tb a 
                inner join gis_db.domst_tb b on a.fc_nodoc=b.fc_nodoc
                inner join gis_db.t_ordermst c on b.fc_noreff=fv_noorder
                left join gis_db.t_receivingmst d on b.fc_nodoc=d.fc_nosj
                where  fv_noreceiving is null and fd_date <= '{input_value}'and b.fc_status="F" 
                group by fn_whconsiid, fv_artsizecode, fd_date '''
    df= pd.read_sql(OutstandingDO_query, engine)

In [ ]:
store_list=['MD563']
placeholders =  ", ".join([f"'{s}'" for s in store_list])
# df1['kode_list'] = df1['kode'].str.split(',')

# 2. Explode untuk meratakan
# df1_exploded = df1.explode('kode_list')

In [ ]:
def df__customers_pb_n(placeholders):
    query = f'''
        SELECT fn_cusid,fn_whconsiid FROM gis_local.customers_pb
        where fc_code in ({placeholders})
        '''
    with get_engine('imp') as engine:  # Menggunakan context manager dengan with
        df = pd.read_sql(query, engine)
    return df
df_cus=df__customers_pb_n(placeholders)

In [ ]:
df_cus

In [ ]:
cus_list= df_cus['fn_cusid'].tolist()
cus_list =  ", ".join([f"'{s}'" for s in cus_list])
whconsi_list=df_cus['fn_whconsiid'].tolist()
whconsi_list =  ", ".join([f"'{s}'" for s in whconsi_list])

In [ ]:
cus_list

In [ ]:
def df__customers_pb_n(placeholders):
    query = f'''
        SELECT fn_cusid,fn_whconsiid FROM gis_local.customers_pb
        where fc_code in ({placeholders})
        '''
    with get_engine('imp') as engine:  # Menggunakan context manager dengan with
        df = pd.read_sql(query, engine)
    return df
df_cus=df__customers_pb_n(placeholders)
cus_list= df_cus['fn_cusid'].tolist()
cus_list =  ", ".join([f"'{s}'" for s in cus_list])
def change_cusid_to_whconsiid(df):
    df_cus=df__customers_pb('fn_cusid,fn_whconsiid')
    df=pd.merge(df,df_cus,how='left',on='fn_cusid')
    df['fn_cusid']=df['fn_whconsiid']
    df=df.drop(columns='fn_whconsiid')
    df=df.rename(columns={'fn_cusid':'fn_whconsiid'})
    return df

def get_ostdo_(cus_list):
    query = f'''
        SELECT fv_artsizecode,fv_barcode from artikel_pb
        '''
    with get_engine('imp') as engine:  # Menggunakan context manager dengan with
        df_art = pd.read_sql(query, engine)
    with get_engine('write') as engine:
        query= f'''SELECT 
        c.fn_cusid ,fv_artsizecode, sum(fn_qty) fn_qty
        FROM gis_db.dodtl_tb a 
        inner join gis_db.domst_tb b on a.fc_nodoc=b.fc_nodoc
        inner join gis_db.t_ordermst c on b.fc_noreff=fv_noorder
        left join gis_db.t_receivingmst d on b.fc_nodoc=d.fc_nosj
        where  fv_noreceiving is null and c.fn_cusid in ({cus_list}) and b.fc_status="F" 
        group by fn_cusid, fv_artsizecode, fd_date 
        '''
        df= pd.read_sql(query, engine)
        print(query)
    df=change_cusid_to_whconsiid(df)
    df=pd.merge(df, df_art, on='fv_artsizecode', how='left')
    df=df.groupby(['fn_whconsiid','fv_barcode',], as_index=False)['fn_qty'].sum()
    return df
# print(query)
df=get_ostdo_(cus_list)

In [ ]:
def get_ostdm_(whconsi_list):
    query = f'''
        SELECT fv_artsizecode,fv_barcode from artikel_pb
        '''
    with get_engine('imp') as engine:  # Menggunakan context manager dengan with
        df_art = pd.read_sql(query, engine)
    with get_engine('write') as engine:
        query= f'''
        SELECT 
            fn_whconsiid2 fn_whconsiid,fv_artsizecode, sum(fn_qty) fn_qty
            FROM gis_db.dodtl_tb a 
            inner join gis_db.domst_tb b on a.fc_nodoc=b.fc_nodoc
            inner join gis_db.t_mutasimst c on b.fc_noreff=c.fv_nomutasi
            left join gis_db.t_receivingmst d on b.fc_nodoc=d.fc_nosj
            where  fv_noreceiving is null and fn_whconsiid2 in ({whconsi_list})
            and b.fc_status="F" 
            group by fn_whconsiid2, fv_artsizecode, fd_date 
        '''
        df= pd.read_sql(query, engine)
    # df=change_cusid_to_whconsiid(df)
    df=pd.merge(df, df_art, on='fv_artsizecode', how='left')
    df=df.groupby(['fn_whconsiid','fv_barcode',], as_index=False)['fn_qty'].sum()
    return df
df=get_ostdm_(cus_list)

In [ ]:
query = f'''
        SELECT fv_artsizecode, fv_barcode from artikel_pb
        '''
with get_engine('imp') as engine:  # Menggunakan context manager dengan with
    df_art = pd.read_sql(query, engine)

In [ ]:
# OutstandingDM_df=pd.merge(OutstandingDM_df, df_art[['fv_artsizecode','fv_barcode']], on='fv_artsizecode', how='left')
# OutstandingDM_df= OutstandingDM_df.groupby(['fn_whconsiid','fv_barcode',], as_index=False)['OstDM'].sum()

In [ ]:
df=pd.merge(df, df_art[['fv_artsizecode','fv_barcode']], on='fv_artsizecode', how='left')
# df=df.groupby(['fn_whconsiid','fv_barcode',], as_index=False)['fn_qty'].sum()

In [ ]:
df=df.groupby(['fn_whconsiid','fv_barcode',], as_index=False)['fn_qty'].sum()

In [ ]:
df

In [ ]:
def change_cusid_to_whconsiid(df):
    df_cus=df__customers_pb('fn_cusid,fn_whconsiid')
    df=pd.merge(df,df_cus,how='left',on='fn_cusid')
    df['fn_cusid']=df['fn_whconsiid']
    df=df.drop(columns='fn_whconsiid')
    df=df.rename(columns={'fn_cusid':'fn_whconsiid'})
    return df

In [ ]:
max_input_df = max_time_input('fd_dateinput','imp','dt_OSDO')
max_input_str =max_input_df.iloc[0, 0]
max_date = max_input_str.strftime("%Y-%m-%d %H:%M:%S")
min_input_df = min_time_input('fd_dateinput','imp','dt_OSDO')
min_input_str =min_input_df.iloc[0, 0]
min_date = min_input_str.strftime("%Y-%m-%d %H:%M:%S")

In [ ]:
max_date

In [ ]:
df_del

In [ ]:
df_del, df_main =df__otsdo_del(max_date,min_date)

In [ ]:
df_del['pk_id']=df_del['fc_nodoc'].astype(str)+"/"+df_del['fv_artsizecode'].astype(str)

In [ ]:
df_main.loc[~df_main['pk_id'].isin(df_del['pk_id'])]

In [ ]:
df_del

In [ ]:
df=change_cusid_to_whconsiid(df)
df[['fn_qty','fn_whconsiid']]=df[['fn_qty','fn_whconsiid']].fillna(0).astype(int)
df['pk_id']=df['fc_nodoc'].astype(str)+"/"+df['fv_artsizecode'].astype(str)

In [ ]:
df

In [ ]:
# df= df__otsdo()
# df[['fn_qty','fn_whconsiid']]=df[['fn_qty','fn_whconsiid']].fillna(0).astype(int)
# df['pk_id']=df['fc_nodoc'].astype(str)+"/"+df['fv_artsizecode'].astype(str)
# engine='imp'
# table_name='dt_OSDO'
# col_pk='pk_id'
# save_to_sql(df,table_name,engine)
# add_index(engine,table_name,['fn_whconsiid','fv_artsizecode'])
# add_primary_key(engine,table_name,col_pk)

In [ ]:
max_input_df = max_time_input('fd_dateinput','imp','dt_OSDO')
max_input_str =max_input_df.iloc[0, 0]
max_date = max_input_str.strftime("%Y-%m-%d %H:%M:%S")
# max_date= '2025-01-01'
df=df__otsdo_rt(max_date)

In [ ]:
max_date

In [ ]:
df.info()

In [ ]:
df=change_cusid_to_whconsiid(df)
df[['fn_qty','fn_whconsiid']]=df[['fn_qty','fn_whconsiid']].fillna(0).astype(int)
df['pk_id']=df['fc_nodoc'].astype(str)+"/"+df['fv_artsizecode'].astype(str)
engine='imp'
table_name='dt_OSDO'
col_pk='pk_id'

In [ ]:
df.info()

In [ ]:
df

In [ ]:
upsert_data(engine,table_name,df,col_pk)

In [ ]:
df

In [ ]:
# Ini cukup:
max_input_df = max_time_input('fd_dateinput','imp','dt_OSDO')
max_date = max_input_df.iloc[0, 0]  # Sudah bertipe datetime.date

print(max_date)
print(type(max_date))


In [ ]:
from datetime import date

if isinstance(max_date, datetime):
    max_date = max_date.date()
print(type(max_input_df.iloc[0, 0]))  # Bisa datetime.date atau datetime.datetime


In [ ]:
max_date

# AS5 freestock

In [ ]:
store_list=['MD255']
placeholders =  ", ".join([f"'{s}'" for s in store_list])
def df__customers_pb_n(placeholders):
    query = f'''
        SELECT fn_cusid,fn_whconsiid FROM gis_local.customers_pb
        where fc_code in ({placeholders})
        '''
    # results = execute_raw_sql(query)
    # df = pd.DataFrame(results)
    with get_engine('imp') as engine:  # Menggunakan context manager dengan with
        df = pd.read_sql(query, engine) 
    return df
df_cus=df__customers_pb_n(placeholders)
whconsi_list=df_cus['fn_whconsiid'].tolist()
whconsi_list =  ", ".join([f"'{s}'" for s in whconsi_list])
def get_freestok_(whconsi_list):
    query = f'''
        SELECT fn_whconsiid, fn_qtykirim,fn_qtyverified , fv_artsizecode, t.fc_status from t_returdtl_pre t
        LEFT JOIN t_returmst_pre m on t.fv_noretur = m.fv_noretur
        WHERE t.fc_status in ('T','X') and fn_whconsiid in ({whconsi_list})
        '''
    # results = DatabaseConnector.execute_raw_sql(query_1, using=StockDashboardAPIView.dbsave)
    # df = pd.DataFrame(results)
    with get_engine('write') as engine:  # Menggunakan context manager dengan with
        df = pd.read_sql(query, engine) 
    df['stok_f']=np.where(df['fc_status'] == "X", df['fn_qtyverified'], df['fn_qtykirim'])  
    df=df[['fn_whconsiid','stok_f','fv_artsizecode']]
    return df
df=get_freestok_(whconsi_list)
df

In [ ]:
df=get_freestok_(whconsi_list)

In [ ]:
df

# AS5 Update Artikel

In [12]:
from modules.queries import (df__penju, df__tokoawal, df__transaction, df__cus, get_monthly_sales_data, 
                             df__sales_thru_24, df__sales_thru_25, df__artsizecode, df__gdmd, fetch_data_margin,
                             df__sales_, df__customers_pb, df__artikel_pb, df__artikel_pb_read, 
                             df__customers_pb_read, df__cus_user, df__sales_invoice,  df__sales_bz_disc2nd,
                             df__sales_bz, df__sales_consi_disc2nd, df__sales_online_disc2nd, df__otsdm, df__otsdo,df__artikel_status ) 
from modules.proses import (save_to_sql, add_index_and_foreign_keys, clean_and_transform_data, 
                            insert_disc2_grouping, insert_margin, rename_col_py_to_sql, insert_pk,
                            get_tgl, get_stock, get_stock_data, merge_and_calculate, merge_detail, 
                            generate_rename_dict,trx_rt, process_data_margin,rename_col_sql_to_xlsx_gis,
                            get_artikel_md, get_new_or_changed_rows,upsert_data, process_data_cus,
                            sls_disctok,add_ornal)

In [ ]:
def get_artikel_md():
    # Load artikel data
    file_name = fr'\\192.168.1.254\Share\_From Valdi\For MD\FOLLOW UP 2023 (4) (5).xlsx'
    df_nfa = pd.read_excel(file_name, sheet_name=8, header=5).dropna(how='all')
    df_nfa= df_nfa [['Article Code','Artikel/ Detail 1']]
    df_nfa=df_nfa.loc[~df_nfa['Article Code'].isnull()]
    df_nfa= df_nfa.loc[~df_nfa['Artikel/ Detail 1'].isnull()]
    df_nfa=df_nfa.rename(columns={'Article Code':'fv_barcode','Artikel/ Detail 1':'fv_neo_link_foto'})
    df_nfa=df_nfa.loc[~df_nfa['fv_barcode'].duplicated()]   
    return df_nfa

In [ ]:
df= df__artikel_pb_read()

In [ ]:
df= df__artikel_pb_read()
df_nfa= get_artikel_md()
df=pd.merge(df,df_nfa,on='fv_barcode',how='left')
df=df.loc[~df['fv_artsizecode'].isnull()]
df2=df__artikel_pb("*")
colist=df.columns.to_list()
df[['fv_rating','fv_neo_link_foto']]=df[['fv_rating','fv_neo_link_foto']].fillna('none')
df2[['fv_rating','fv_neo_link_foto','fv_ornal']]=df2[['fv_rating','fv_neo_link_foto','fv_ornal']].fillna('none')
df['fv_ornal'] = df.apply(add_ornal, axis=1)
df3=get_new_or_changed_rows(df, df2,subset=colist)
df3=df3.fillna('none')
df_art_status=df__artikel_status()
df3=pd.merge(df3,df_art_status,on='fn_articleid',how='left')
df3=df3[['fv_barcode',
'fv_artsizecode',
'fv_catname',
'fv_brandname',
'fv_divname',
'fv_sizename',
'fm_price',
'fm_amount',
'fv_namemark',
'fv_colorname',
'fv_fitname',
'fv_tomname',
'fv_configname',
'motif',
'fv_picture',
'fv_rating',
'fm_cogs',
'fn_articleid',
'fv_neo_link_foto',
'fv_fitting',
'fv_status',
'fv_ornal']]
table_name = 'artikel_pb'

In [ ]:
df3['fm_amount'] = pd.to_numeric(df3['fm_amount'], errors='coerce') 
df3['fm_amount'] = df3['fm_amount'].fillna(0)
df3['fm_price'] = pd.to_numeric(df3['fm_price'], errors='coerce')  # jadi NaN kalau gak valid
df3['fm_price'] = df3['fm_price'].fillna(0)


In [ ]:
df2=df__artikel_pb("*")

In [ ]:
df2.info()

In [ ]:
df2[df2.duplicated(subset=["fv_artsizecode"], keep=False)]


In [ ]:
# Cari baris yang punya spasi di awal, akhir, atau tengah
mask_spasi = df2['fv_artsizecode'].str.contains(r"\s", regex=True)

# Tampilkan artikel bermasalah
df_spasi = df2.loc[mask_spasi, ['fv_barcode','fv_artsizecode']]
print(df_spasi)

## Self servis

In [ ]:
df2= df__artikel_pb_read()

In [ ]:
df2.loc[df2['fn_articleid']==9134]

In [ ]:
df=df_artikel_pb_('fn_articleid',distinct=False,where='')

In [ ]:
df.loc[df['fn_articleid']==9134]

In [ ]:
df_filtered = df.loc[~df['fn_articleid'].isin(df2['fn_articleid'])]

In [ ]:
df_filtered

In [ ]:
df_filtered['fn_articleid'].unique().tolist()

In [ ]:
queri=f'''
    SELECT * FROM gis_db.article_tb where fv_barcode in ({})'''
with get_engine('read') as engine:  # Menggunakan context manager dengan with
    df = pd.read_sql(queri, engine)

In [ ]:
df3=df.loc[df['fv_barcode']=='ADBC011P051']

In [ ]:
df3=df2.loc[df2['fv_barcode']=='ADBC011P051']

In [ ]:
df3=df3.replace({'ADBC011P051L':'ADBC011P051U',
                 'ADBC011P051M':'ADBC011P051T',
                 'ADBC011P051S	':'ADBC011P051S',
                 'ADBC011P051SS':'ADBC011P051R',
                 'ADBC011P051XS':'ADBC011P0512',})

In [ ]:
df3

In [ ]:
df2.info()

In [ ]:
df= df__artikel_pb("*")
df['fv_ornal'] = df.apply(add_ornal, axis=1)

In [ ]:
df.loc[df['fv_ornal']=='none']

In [ ]:
df.info()

In [ ]:
df3 = (
    df.loc[
        df['fv_fitting'].notna() & df['fv_status'].notna(),
        ['fv_barcode', 'fv_fitting', 'fv_status']
    ]
    .drop_duplicates()
)


In [ ]:
# merge df1 dengan df3 di kolom 'fv_barcode'
df_merge = df.merge(df3, on='fv_barcode', how='left', suffixes=('', '_df3'))

# isi nilai null di df1 dengan dari df3
df_merge['fv_fitting'] = df_merge['fv_fitting'].combine_first(df_merge['fv_fitting_df3'])
df_merge['fv_status'] = df_merge['fv_status'].combine_first(df_merge['fv_status_df3'])

# drop kolom tambahan dari df3
df_merge = df_merge.drop(columns=['fv_fitting_df3', 'fv_status_df3'])


In [ ]:
df=df_merge

In [ ]:
df3['fv_ornal'] = df.apply(add_ornal, axis=1)

In [ ]:
df.info()

In [ ]:
df3=df[['fv_artsizecode','fv_ornal']]

In [ ]:
df3=df[['fv_artsizecode','fv_fitting','fv_status']].fillna('none')

## EXCUTE upsrt

In [ ]:
table_name = 'artikel_pb'
for col in df2.columns:
    if col != 'fv_artsizecode':
        df_sub = df2[['fv_artsizecode', col]].copy()
        upsert_data('imp', table_name, df_sub, primary_key='fv_artsizecode')
        print(col)

In [ ]:
upsert_data('imp', table_name, df3, primary_key="fv_artsizecode")

In [ ]:
table_name = 'artikel_pb'
def df__artikel_status():
    query='''select fn_articleid, fv_status from gis_db.articlestatus_tb'''
    with get_engine('write') as engine:  # Menggunakan context manager dengan with
        df = pd.read_sql(query, engine)
    return df

In [ ]:
df_art_status=df__artikel_status()

In [ ]:
df3=pd.merge(df3,df_art_status,on='fn_articleid',how='left')
df3=df3[['fv_barcode',
 'fv_artsizecode',
 'fv_catname',
 'fv_brandname',
 'fv_divname',
 'fv_sizename',
 'fm_price',
 'fm_amount',
 'fv_namemark',
 'fv_colorname',
 'fv_fitname',
 'fv_tomname',
 'fv_configname',
 'motif',
 'fv_picture',
 'fv_rating',
 'fm_cogs',
 'fn_articleid',
 'fv_neo_link_foto',
 'fv_fitting',
 'fv_status']]

In [ ]:
df_art_status

In [ ]:
df3.info()

# AS6 TARik stok mang

In [ ]:
from services.Queries_API import (df__customers_pb_n,df__customers_pb,df_artikel_pb_,df__main,df__aging,df__sales,
                                  df__retur,df__mutasi,df__obdo,change_cusid_to_whconsiid,df_ostdo_,df_ostdm_)
from services.Process_API import (proc_add_aging, proc_pivot_table, proc_add_ornal, proc_sales_stock,
                                   proc_add_retur, proc_add_mutasi, proc_add_obdo, proc_add_ostdo,
                                   proc_add_ostdm, proc_add_stok_future, proc_add_cus_art, proc_selected_col_detail,
                                   proc_add_summary, proc_cleaning_data_mobile_view,proc_selected_col_mobile_view)

## Input

In [ ]:
input_value = datetime.today().strftime("%Y-%m-%d")
input_date = datetime.strptime(input_value, "%Y-%m-%d")
# Konversi kembali ke string
start=datetime.today()
ends=start - relativedelta(months=3)
# start_str = start.strftime("%Y-%m-%d")
# ends_str = ends.strftime("%Y-%m-%d")
alokator=''
store_list = ['MD255']
input_param={'alokator': alokator,
'store_list': store_list,
'start': start.strftime("%Y-%m-%d"),
'end': ends.strftime("%Y-%m-%d"),
'type': 'default',
# 'format': output_format,
'input_date': input_date.strftime("%Y-%m-%d")}

In [ ]:
def df__main(input_param):
    input_value = input_param['input_date']
    store_list = input_param['store_list']
    if store_list:
        placeholders = ", ".join([f"'{s}'" for s in store_list])
    df_cus = df__customers_pb_n(placeholders)
    cus_list= df_cus['fn_cusid'].tolist()
    cus_list =  ", ".join([f"'{s}'" for s in cus_list])
    whconsi_list=df_cus['fn_whconsiid'].tolist()
    whconsi_list =  ", ".join([f"'{s}'" for s in whconsi_list])
    sales_thru = f"sales_thru_pb_{input_value[2:4]}"
    trxstx = f"trxstx_pb{input_value[2:4]}"
    stock = f"fn_stock_{input_value[5:7]}"

    main_query = f"""
    SELECT 
    stock_backdate.fn_whconsiid,
    SUM(stock) AS stocks,
    stock_backdate.fv_artsizecode, 
    fv_barcode, fv_sizename, fv_configname

    FROM (
    SELECT fn_whconsiid, {stock} AS stock, fv_artsizecode 
    FROM gis_local.{sales_thru}
    UNION ALL
    SELECT fn_whconsiid, fn_stock AS stock, fv_artsizecode 
    FROM gis_local.{trxstx} 
    WHERE fd_tgltrx BETWEEN '{input_value[0:4]}-{input_value[5:7]}-01' AND '{input_value}'
    UNION ALL
    SELECT fn_whconsiid, fn_stock AS stock, fv_artsizecode 
    FROM gis_local.trxrt_pb
    UNION ALL
    SELECT fn_whconsiid, fn_stock AS stock, fv_artsizecode 
    FROM gis_local.Stock_varian_25
    WHERE fd_tgl BETWEEN '{input_value[0:4]}-{input_value[5:7]}-01' AND '{input_value}'
    ) stock_backdate
    left join artikel_pb b on stock_backdate.fv_artsizecode = b.fv_artsizecode
    WHERE 1=1 and fn_whconsiid in ({whconsi_list})
    GROUP BY fn_whconsiid, fv_artsizecode
    """
    # df = pd.read_sql(main_query, engine_imp)
    results = DatabaseConnector.execute_raw_sql(main_query, using=dbLoad)
    df = pd.DataFrame(results)
    return df

In [ ]:
def get_detail(input_param):
    col='fv_barcode'
    type_table = 'detail'
    type = input_param['type']
    #Default API or Detail API
    if type == 'default':
        
        df_main = df__main(input_param)
        # print(f'test df_main')
        df_aging = df__aging(input_param)
        # print('test df_aging')
        df_sales = df__sales(input_param)
        # print('test df_sales')
        df_retur = df__retur(input_param)
        # print('test df_retur')
        df_mutasi = df__mutasi(input_param)
        # print('test df_mutasi')
        df_obdo = df__obdo(input_param)
        # print('test df_obdo')
        df_ostdo = df_ostdo_(input_param,col)
        # print('test df_ostdo')
        df_ostdm=df_ostdm_(input_param,col)
        # print('test df_ostdm')
        df = proc_add_aging(df_main, df_aging)
        # print('test proc_add_aging')
        pivot_table = proc_pivot_table(df)
        # print('test proc_pivot_table')
        pivot_table = proc_add_ornal(pivot_table)
        # print('test proc_add_ornal')
        pivot_table = proc_sales_stock(pivot_table, df_sales)
        # print('test proc_sales_stock')
        pivot_table = proc_add_retur(pivot_table, df_retur)
        # print('test df_osproc_add_returtdm')
        pivot_table = proc_add_mutasi(pivot_table, df_mutasi)
        # print('test proc_add_mutasi')
        pivot_table = proc_add_obdo(pivot_table, df_obdo)
        # print('test proc_add_obdo')
        pivot_table = proc_add_ostdo(pivot_table, df_ostdo,col)
        # print('test proc_add_ostdo')
        pivot_table = proc_add_ostdm(pivot_table, df_ostdm,col)
        # print('test proc_add_ostdm')
        pivot_table = proc_add_stok_future(pivot_table)
        # print('test proc_add_stok_future')
        pivot_table = proc_add_cus_art(pivot_table,input_param,type_table,col)
        # print('test proc_add_cus_art')
        pivot_table = proc_selected_col_detail(pivot_table)
        # pivot_table.info()
        return pivot_table
        

In [ ]:
df=get_detail(input_param)

# AS7 broken 2.0

In [ ]:
def get_top_sizes(row, size_columns):
    # Ambil qty per size
    size_qty = row[size_columns]
    total = size_qty.sum()
    if total == 0:
        return ""
    
    # Urutkan size dari qty terbesar
    sorted_sizes = size_qty.sort_values(ascending=False)
    
    # Ambil size sampai kumulatif >= 50%
    cum_sum = 0
    selected = []
    for size, qty in sorted_sizes.items():
        cum_sum += qty
        selected.append(size)
        if cum_sum >= total * 0.5:
            break
    
    return ", ".join(selected)


In [ ]:

from datetime import datetime, timedelta

date_now = datetime.today().strftime("%Y-%m-%d")
date_last_3m = (datetime.today() - timedelta(days=90)).strftime("%Y-%m-%d")

# periode tahun lalu
date_now_last_year = (datetime.today().replace(year=datetime.today().year-1)).strftime("%Y-%m-%d")
date_3m_last_year = ((datetime.today() + timedelta(days=90)).replace(year=datetime.today().year-1)).strftime("%Y-%m-%d")
query = f'''
        SELECT distinct fv_barcode,fv_catname
        FROM gis_local.artikel_pb
        '''
with get_engine('imp') as engine:  # Menggunakan context manager dengan with
    df_art = pd.read_sql(query, engine)
query = f'''
    SELECT 
    a.fv_artsizecode, 
    fn_cusid, 
    a.fn_whconsiid,
    fv_barcode,
    fv_sizename, 
    c.fv_configname,
    sum(fn_jualpersize) qty, 
    sum(fn_totalpenjualan) net_casa, 
    fd_tgltrx
    FROM gis_local.pb_penjualan_2025 a
    left join gis_local.customers_pb b on a.fn_whconsiid=b.fn_whconsiid
    left join gis_local.artikel_pb c on a.fv_artsizecode=c.fv_artsizecode
    where fd_tgltrx between '{date_last_3m}' and '{date_now}'
    group by fv_artsizecode, a.fn_whconsiid, fd_tgltrx;
        '''
with get_engine('imp') as engine:  # Menggunakan context manager dengan with
    df_cons25 = pd.read_sql(query, engine)
query = f'''
    SELECT 
    a.fv_artsizecode, 
    fn_cusid, 
    a.fn_whconsiid,
    fv_barcode,
    fv_sizename, 
    c.fv_configname,
    sum(fn_jualpersize) qty, 
    sum(fn_totalpenjualan) net_casa, 
    fd_tgltrx
    FROM gis_local.pb_penjualan_2024 a
    left join gis_local.customers_pb b on a.fn_whconsiid=b.fn_whconsiid
    left join gis_local.artikel_pb c on a.fv_artsizecode=c.fv_artsizecode
    where fd_tgltrx between '{date_now_last_year}' and '{date_3m_last_year}'
    group by fv_artsizecode, a.fn_whconsiid, fd_tgltrx;
        '''
with get_engine('imp') as engine:  # Menggunakan context manager dengan with
    df_cons24 = pd.read_sql(query, engine)

df=df_cons24
values:str='qty'
col_index:list=['fv_barcode', 'fn_whconsiid']
columns:list=['fv_sizename']
col_ket:list=['fv_configname']
    
all_sizes_bottom = ['25', '26', '27', '28', '29', '30', '31', '32', '33', '34', '35', '36', '37', '38',
                    '39', '40', '42', '44', 'ALLSIZE']
all_sizes_top = ['FS', 'XS', 'SS', 'S', 'M', 'L', 'XL', 'XXL', 'XXXL']
all_sizes = all_sizes_bottom + all_sizes_top
df_ket = df[col_index + col_ket].drop_duplicates()
# Buat pivot table
pivot_df = pd.pivot_table(
    df,
    values=values,
    index=col_index,
    columns=columns,
    aggfunc='sum',
    fill_value=0).reset_index()

# Merge kembali kolom tambahan
pivot_table = pd.merge(pivot_df, df_ket, on=col_index, how='left')
for size in all_sizes:
    if size not in pivot_table.columns:
        pivot_table[size] = 0
    pivot_table[size] = pd.to_numeric(pivot_table[size], errors='coerce').fillna(0).astype(int)

# Atur ulang urutan: non-size columns dulu, lalu size columns
size_columns = [size for size in all_sizes if size in pivot_table.columns]
other_columns = [col for col in pivot_table.columns if col not in size_columns]
pivot_table = pivot_table[other_columns + size_columns]

# Hitung status (BROKEN / SEHAT)
top_columns = ["L", "M", "S", "XL", "XS", "XXL", "XXXL"]
bottom_columns = ['27', '28', '29', '30', '31', '32', '33', '34',
                    '35', '36', '37', '38', '39', '40']

pivot_table['status'] = np.where(
    (pivot_table["fv_configname"] == "TOP") & (pivot_table[top_columns].gt(0).sum(axis=1) < 3), "BROKEN",
    np.where(
        (pivot_table["fv_configname"] == "BOTTOM") & (pivot_table[bottom_columns].gt(0).sum(axis=1) < 5), "BROKEN",
        "SEHAT"
    )
)

# Hitung total jumlah qty size (sumQty)
pivot_table['sumQty'] = pivot_table[size_columns].sum(axis=1)
pivot_table=pivot_table.drop(columns=['fv_configname'])

df=pivot_table
# size kecil = kolom 25–31 + kolom S + M
df["size_kecil"] = df['25'] + df['26'] + df["27"] + df["28"] + df['29'] + df['30'] + df["S"] + df["M"]

# size sedang = kolom 30–34 + kolom M + L
df["size_sedang"] = df["34"] + df["33"] + df['32'] + df['30'] + df["31"] + df["L"] + df["M"]

# size besar = kolom 34 ke atas + kolom L, XL, XXL
df["size_besar"] = df["35"] + df["36"] + df['37'] + df['38'] + df["39"] + df['40'] + df['42'] + df["44"] + df["L"] + df["XL"] + df["XXL"]
df_detail=df
df=df[['fv_barcode','fn_whconsiid','size_kecil','size_sedang','size_besar']]
df=pd.merge(df,df_art, how='left',on='fv_barcode')
import numpy as np

def get_dominant_size(row):
    k, s, b = row["size_kecil"], row["size_sedang"], row["size_besar"]

    # semua sama
    if k == s == b:
        return "ksb"
    # dua sama besar
    if k == s and k > b:
        return "ks"
    if k == b and k > s:
        return "kb"
    if s == b and s > k:
        return "sb"
    # salah satu paling besar
    if k > s and k > b:
        return "k"
    if s > k and s > b:
        return "s"
    if b > k and b > s:
        return "b"
    return None  # fallback

df["dominant_size"] = df.apply(get_dominant_size, axis=1)

In [ ]:

from datetime import datetime, timedelta

date_now = datetime.today().strftime("%Y-%m-%d")
date_last_3m = (datetime.today() - timedelta(days=90)).strftime("%Y-%m-%d")

# periode tahun lalu
date_now_last_year = (datetime.today().replace(year=datetime.today().year-1)).strftime("%Y-%m-%d")
date_3m_last_year = ((datetime.today() + timedelta(days=90)).replace(year=datetime.today().year-1)).strftime("%Y-%m-%d")
query = f'''
        SELECT distinct fv_barcode,fv_catname
        FROM gis_local.artikel_pb
        '''
with get_engine('imp') as engine:  # Menggunakan context manager dengan with
    df_art = pd.read_sql(query, engine)
query = f'''
    SELECT 
    a.fv_artsizecode, 
    fn_cusid, 
    a.fn_whconsiid,
    fv_barcode,
    fv_sizename, 
    c.fv_configname,
    sum(fn_jualpersize) qty, 
    sum(fn_totalpenjualan) net_casa, 
    fd_tgltrx
    FROM gis_local.pb_penjualan_2025 a
    left join gis_local.customers_pb b on a.fn_whconsiid=b.fn_whconsiid
    left join gis_local.artikel_pb c on a.fv_artsizecode=c.fv_artsizecode
    where fd_tgltrx between '{date_last_3m}' and '{date_now}'
    group by fv_artsizecode, a.fn_whconsiid, fd_tgltrx;
        '''
with get_engine('imp') as engine:  # Menggunakan context manager dengan with
    df_cons25 = pd.read_sql(query, engine)
query = f'''
    SELECT 
    a.fv_artsizecode, 
    fn_cusid, 
    a.fn_whconsiid,
    fv_catname,
    fv_sizename, 
    c.fv_configname,
    sum(fn_jualpersize) qty, 
    sum(fn_totalpenjualan) net_casa, 
    fd_tgltrx
    FROM gis_local.pb_penjualan_2024 a
    left join gis_local.customers_pb b on a.fn_whconsiid=b.fn_whconsiid
    left join gis_local.artikel_pb c on a.fv_artsizecode=c.fv_artsizecode
    where fd_tgltrx between '{date_now_last_year}' and '{date_3m_last_year}'
    group by fv_artsizecode, a.fn_whconsiid, fd_tgltrx;
        '''
with get_engine('imp') as engine:  # Menggunakan context manager dengan with
    df_cons24 = pd.read_sql(query, engine)

df=df_cons24
values:str='qty'
col_index:list=['fv_catname', 'fn_whconsiid']
columns:list=['fv_sizename']
col_ket:list=['fv_configname']
    
all_sizes_bottom = ['25', '26', '27', '28', '29', '30', '31', '32', '33', '34', '35', '36', '37', '38',
                    '39', '40', '42', '44', 'ALLSIZE']
all_sizes_top = ['FS', 'XS', 'SS', 'S', 'M', 'L', 'XL', 'XXL', 'XXXL']
all_sizes = all_sizes_bottom + all_sizes_top
df_ket = df[col_index + col_ket].drop_duplicates()
# Buat pivot table
pivot_df = pd.pivot_table(
    df,
    values=values,
    index=col_index,
    columns=columns,
    aggfunc='sum',
    fill_value=0).reset_index()

# Merge kembali kolom tambahan
pivot_table = pd.merge(pivot_df, df_ket, on=col_index, how='left')
for size in all_sizes:
    if size not in pivot_table.columns:
        pivot_table[size] = 0
    pivot_table[size] = pd.to_numeric(pivot_table[size], errors='coerce').fillna(0).astype(int)

# Atur ulang urutan: non-size columns dulu, lalu size columns
size_columns = [size for size in all_sizes if size in pivot_table.columns]
other_columns = [col for col in pivot_table.columns if col not in size_columns]
pivot_table = pivot_table[other_columns + size_columns]

# Hitung status (BROKEN / SEHAT)
top_columns = ["L", "M", "S", "XL", "XS", "XXL", "XXXL"]
bottom_columns = ['27', '28', '29', '30', '31', '32', '33', '34',
                    '35', '36', '37', '38', '39', '40']

pivot_table['status'] = np.where(
    (pivot_table["fv_configname"] == "TOP") & (pivot_table[top_columns].gt(0).sum(axis=1) < 3), "BROKEN",
    np.where(
        (pivot_table["fv_configname"] == "BOTTOM") & (pivot_table[bottom_columns].gt(0).sum(axis=1) < 5), "BROKEN",
        "SEHAT"
    )
)

# Hitung total jumlah qty size (sumQty)
pivot_table['sumQty'] = pivot_table[size_columns].sum(axis=1)
pivot_table=pivot_table.drop(columns=['fv_configname'])

df=pivot_table
# size kecil = kolom 25–31 + kolom S + M
df["size_kecil"] = df['25'] + df['26'] + df["27"] + df["28"] + df['29'] + df['30'] + df["S"] + df["M"]

# size sedang = kolom 30–34 + kolom M + L
df["size_sedang"] = df["34"] + df["33"] + df['32'] + df['30'] + df["31"] + df["L"] + df["M"]

# size besar = kolom 34 ke atas + kolom L, XL, XXL
df["size_besar"] = df["35"] + df["36"] + df['37'] + df['38'] + df["39"] + df['40'] + df['42'] + df["44"] + df["L"] + df["XL"] + df["XXL"]
df_detail=df
df=df[['fv_barcode','fn_whconsiid','size_kecil','size_sedang','size_besar']]
df=pd.merge(df,df_art, how='left',on='fv_barcode')
import numpy as np

def get_dominant_size(row):
    k, s, b = row["size_kecil"], row["size_sedang"], row["size_besar"]

    # semua sama
    if k == s == b:
        return "ksb"
    # dua sama besar
    if k == s and k > b:
        return "ks"
    if k == b and k > s:
        return "kb"
    if s == b and s > k:
        return "sb"
    # salah satu paling besar
    if k > s and k > b:
        return "k"
    if s > k and s > b:
        return "s"
    if b > k and b > s:
        return "b"
    return None  # fallback

df["dominant_size"] = df.apply(get_dominant_size, axis=1)

In [ ]:
# Terapkan ke pivot_table
df_detail["dominant_sizes"] = df_detail.apply(lambda row: get_top_sizes(row, size_columns), axis=1)

In [ ]:
df_detail[df_detail.columns[20:]].sort_values('sumQty',ascending=False)

In [ ]:
# Terapkan ke pivot_table
pivot_table["dominant_sizes"] = pivot_table.apply(lambda row: get_top_sizes(row, size_columns), axis=1)

In [ ]:
pivot_table.loc[pivot_table['dominant_sizes']=='XL, L']

In [ ]:
df_detail['dominant_sizes'].value_counts().head(20)

In [ ]:
df=df[['fv_barcode','fn_whconsiid','size_kecil','size_sedang','size_besar']]
df=pd.merge(df,df_art, how='left',on='fv_barcode')

In [ ]:
df_detail

In [ ]:
df

In [ ]:
pivot_table

In [ ]:
df_detail

In [ ]:
pivot_table

In [ ]:
df_detail

In [ ]:
# Terapkan ke pivot_table
pivot_table["dominant_sizes"] = pivot_table.apply(lambda row: get_top_sizes(row, size_columns), axis=1)

In [ ]:
df_cat_dom

In [ ]:
df_cat_dom = (
    df.groupby(['fn_whconsiid', 'fv_catname'], as_index=False)
      .agg({
          'size_kecil': 'sum',
          'size_sedang': 'sum',
          'size_besar': 'sum'
      })
)

In [ ]:
df_cat_dom["dominant_size"] = df_cat_dom.apply(get_dominant_size, axis=1)

In [ ]:
df_cat_dom['dominant_size'].value_counts()

In [ ]:
df_cat_dom.loc[(df_cat_dom['dominant_size']=='ksb'),'fv_catname'].value_counts()

In [ ]:
df

In [ ]:
df.loc[df['fn_whconsiid']==19]

# AS8 Update whtype

In [ ]:
from modules.queries import (df__whtype_name) 
from modules.proses import (save_to_sql)

from sqlalchemy import Integer, String, CHAR, DateTime, Text, text, Float, Numeric

In [ ]:
def define_sql_types(df):
    """Menentukan tipe data untuk SQL sesuai dengan tipe kolom."""
    sql_types = {}
    for col in df.columns:
        col_dtype = df[col].dtype

        # Jika kolom tipe float
        if col_dtype == 'float64':
            sql_types[col] = Numeric(15, 2)
        # Jika kolom diawali 'fn' dan bukan float
        elif col.startswith('fn'):
            sql_types[col] = Integer
        # Jika kolom diawali 'fv'
        elif col.startswith('fv'):
            max_length = df[col].astype(str).map(len).max()
            if max_length > 255:
                sql_types[col] = Text
            else:
                sql_types[col] = String(255)
        # Jika kolom diawali 'fc'
        elif col.startswith('fc'):
            sql_types[col] = CHAR(20)
        # Jika kolom diawali 'fd'
        elif col.startswith('fd'):
            sql_types[col] = DateTime
        # Kalau tidak masuk kriteria di atas
        else:
            sql_types[col] = String(50)
    
    return sql_types


def save_to_sql(df, table_name, engine):
    """Menyimpan data ke dalam database menggunakan SQLAlchemy."""
    with get_engine(engine) as engine:
        sql_types = define_sql_types(df)
        df.to_sql(table_name, con=engine, if_exists='replace', index=False, dtype=sql_types)

In [ ]:
df=df__whtype_name()
nama_tabel='whtype'
save_to_sql(df, nama_tabel, 'imp')

# AS9 Aging gudang

In [9]:
query_gr = '''
    SELECT a.fc_nodoc, fd_arrivaldate, fv_nopo, fn_articleid, fn_qty FROM gis_db1.grdtl_tb a
    inner join gis_db1.grmst_tb b on a.fc_nodoc = b.fc_nodoc
    '''
with get_engine('read') as engine:  # Menggunakan context manager dengan with
    df_gr = pd.read_sql(query_gr, engine)
# Cari tanggal dengan qty tertinggi per store & product
result = df_gr.sort_values('fn_qty', ascending=False).groupby(['fv_nopo', 'fn_articleid']).first().reset_index()

# Urutkan dari tanggal terbaru
df_sorted = result.sort_values('fd_arrivaldate', ascending=False)

# Ambil baris pertama (arrival terbaru) untuk tiap articleid
df_latest = df_sorted.drop_duplicates(subset='fn_articleid', keep='first')

query_gr = '''
    SELECT distinct fn_articleid, fv_barcode FROM gis_local.artikel_pb 
    '''
with get_engine('imp') as engine:  # Menggunakan context manager dengan with
    df_art = pd.read_sql(query_gr, engine)

# unique_ids = pd.DataFrame(df_art['fn_articleid'].unique(), columns=[['fn_articleid','fv_barcode']])
df_latest = pd.merge(df_latest, df_art, on='fn_articleid', how='right')

df_latest=df_latest.fillna({'fn_qty':0, 'fd_arrivaldate':'2023-01-01', 'fv_nopo':'', 'fc_nodoc':''})


today = pd.Timestamp(datetime.now().date())
df_latest["fn_hari"] = (today - df_latest["fd_arrivaldate"]).dt.days
df_latest["fn_bulan"] = (today.year - df_latest["fd_arrivaldate"].dt.year) * 12 + (today.month - df_latest["fd_arrivaldate"].dt.month)

query = '''
SELECT distinct fn_articleid, a.fd_dateinput FROM gis_db1.dodtl_tb a
left join gis_db1.domst_tb b on a.fc_nodoc=b.fc_nodoc 
where a.fc_nodoc like "DO%%"'''
with get_engine('read') as engine:  # Menggunakan context manager dengan with
    df_fdo = pd.read_sql(query, engine)
df_unique = (
    df_fdo
    .sort_values('fd_dateinput')
    .drop_duplicates(subset='fn_articleid', keep='first')
)
# df_latest.to_sql(f'aging_gudang_pb', engine_imp, if_exists='replace', index=False)
df_latest = pd.merge(df_latest, df_unique, on='fn_articleid', how='left')
df_latest["fn_hari_fdo"] = (today - df_latest["fd_dateinput"]).dt.days
df_latest["fn_bulan_fdo"] = (today.year - df_latest["fd_dateinput"].dt.year
                         ) * 12 + (today.month - df_latest["fd_dateinput"].dt.month)
df_latest = df_latest.assign(
    fd_dateinput = df_latest['fd_dateinput'].combine_first(df_latest['fd_arrivaldate']),
    fn_hari_fdo  = df_latest['fn_hari_fdo'].combine_first(df_latest['fn_hari']),
    fn_bulan_fdo = df_latest['fn_bulan_fdo'].combine_first(df_latest['fn_bulan'])
)
nama_tabel='aging_gudang_pb'
save_to_sql(df_latest, nama_tabel, 'imp')

In [ ]:
df_sq25= df__sales_thru_25('*')

# AS10 insentif

In [ ]:
def add_ornal(row):
    acara = row.get('event', '')  # diasumsikan ini sama dengan Artikel[acara]
    kategori = row.get('fv_catname', '')
    harga = row.get('fn_hargasatuan', 0)

    # Acara diskon langsung jadi "Obral"
    if acara in ["DISC 50%", "DISC 70%", "DISC 50+20%"]:
        return "Obral"
    if acara in ["NORMAL"]:
        return "Normal"
    rules = {
        "LONG SLEEVE SHIRT": 229000,
        "SHORT SLEEVE SHIRT": 229000,
        "LONG SLEEVE KOKO SHIRT": 229000,
        "SHORT SLEEVE KOKO SHIRT": 229000,
        "SHORT SLEEVE POLO SHIRT": 189000,
        "SHORT SLEEVE HENLEY SHIRT": 189000,
        "SHORT SLEEVE T-SHIRT (CREW NECK)": 139000,
        "LONG SLEEVE T-SHIRT (CREW NECK)": 192000,
        "TRUCKER JACKET PIECE DYED": 300000,
        "TRUCKER JACKET DENIM":300000,
        "HERINGTON JACKET": 300000,
        "BOMBER JACKET": 300000,
        "UTILITIES JACKET": 300000,
        "HOODIE JACKET (KNIT)": 250000,
        "SWEAT SHIRT": 200000,
        "JOGER PANTS DENIM": 300000,
        "JOGER PANTS PIECE DYED": 300000,
        "JOGER SHORT DENIM": 279000,
        "5 POCKET PANT DENIM": 300000,
        "5 POCKET PANT PIECE DYED": 300000,
        "CHINO PANT PIECE DYED": 300000,
        "5 POCKET SHORT DENIM": 279000,
        "5 POCKET SHORT PIECE DYED": 279000,
        "CARGO PANT PIECE DYED": 300000,
        "CARGO SHORT PIECE DYED": 300000,
        "CHINO SHORT PIECE DYED": 279000,
        "CARGO PANT DENIM": 300000,
        "CARGO SHORT DENIM": 300000,
        "LONG SLEEVE UTILITIES SHIRT": 229000
    }

    batas_harga = rules.get(kategori)
    if batas_harga is not None:
        return "Normal" if harga >= batas_harga else "Obral"

    return None  # jika tidak ada kategori atau aturan

In [ ]:
awal_bulan = '2025-01-01'
akhir_bulan = '2025-06-30'
whlist='18,27,73,116,117,134,137,151,1279,1331'
query = f'''SELECT 
    fv_namasales, 
    fn_cusid fn_whconsiid, 
    fv_artsizecode, 
    b.fv_namemark status_artikel, 
    d.fv_namemark 'event',
    b.fv_descevent "desc",
    b.fn_jualpersize ,
    b.fn_hargasatuan ,
    b.fn_hargasatuan*b.fn_jualpersize as total_penjualan2,
    DATE_FORMAT(a.fd_tgljual, '%%m-%%d-%%Y') tgltrx
FROM 
    t_penjualanmst a 
INNER JOIN 
    t_penjualandtl_goods b ON a.fv_nopenjualan = b.fv_nopenjualan
LEFT JOIN 
        (SELECT a.fn_discid, b.fv_namemark FROM discount_tb a
            INNER JOIN markdown_tb b ON a.fn_markid = b.fn_markid) d ON b.fn_discid = d.fn_discid
LEFT JOIN t_salesmanmst z on a.fn_idsales=z.fn_idsales

WHERE 
    a.fv_nopenjualan LIKE 'PJ%%' and b.fc_status <> 'F' 
    and a.fc_status <> 'F' 
    and a.fd_tgljual BETWEEN '{awal_bulan} 00:00:01' and '{akhir_bulan} 23:59:59'
    and fn_cusid in ({whlist})
'''
with get_engine('read') as engine:  # Menggunakan context manager dengan with
    df = pd.read_sql(query, engine)

In [ ]:
df['event']=df['event'].fillna(df['status_artikel'])

In [ ]:
df_cus=df__customers_pb_("fn_whconsiid,fv_kota,concat(fc_code,' - ', fv_toko) fv_toko",distinct=True,where=f"and fn_whconsiid in ({whlist})")

In [ ]:
artsize_list=list_to_sql_in(df,'fv_artsizecode',str)
df_artsize=df_artikel_pb_('fv_artsizecode,fv_catname',distinct=True,where=f"and fv_artsizecode in ({artsize_list})")

In [ ]:
df=pd.merge(df,df_cus,on='fn_whconsiid',how='left')
df=pd.merge(df,df_artsize,on='fv_artsizecode',how='left')

In [ ]:
df

In [ ]:
# df['fm_amount']=df['fn_hargasatuan']
df['fv_ornal'] = df.apply(add_ornal, axis=1)

In [ ]:
df['fv_ornal'].value_counts()

In [ ]:
df['tgltrx'] = pd.to_datetime(df['tgltrx'], errors='coerce')
df['tgltrx'].dt.strftime('%Y-%m-%d')
# Buat kolom hari dalam minggu (0=Senin, 6=Minggu)
df['dayofweek'] = df['tgltrx'].dt.dayofweek

# Tandai weekend (Sabtu dan Minggu)
df['is_weekend'] = df['dayofweek'].isin([5,6]).astype(int)

In [ ]:
ramadan_start = '2025-03-01'
ramadan_end = '2025-03-31'
df['season'] = 'normal'  # default
df.loc[df['tgltrx'].between(ramadan_start, ramadan_end), 'season'] = 'ramadan'

In [ ]:
def kategori_hari(row):
    if row['is_weekend'] == 1 and row['season'] == 'natal':
        return 'weekend_natal'
    elif row['is_weekend'] == 0 and row['season'] == 'natal':
        return 'weekday_natal'
    elif row['is_weekend'] == 1 and row['season'] == 'ramadan':
        return 'weekend_ramadan'
    elif row['is_weekend'] == 0 and row['season'] == 'ramadan':
        return 'weekday_ramadan'
    elif row['is_weekend'] == 1 and row['season'] == 'chineseyear':
        return 'weekend_chineseyear'
    elif row['is_weekend'] == 0 and row['season'] == 'chineseyear':
        return 'weekday_chineseyear'
    else:
        return 'weekday_normal' if row['is_weekend'] == 0 else 'weekend_normal'
def kategori_day(row):
    if row['is_weekend'] == 1: return 'weekend'
    else:return 'weekday'

In [ ]:
df['kategori_hari'] = df.apply(kategori_hari, axis=1)
df['day_type'] = df.apply(kategori_day, axis=1)

In [ ]:
df['bulan'] = df['tgltrx'].dt.month
df['nama_bulan'] = df['tgltrx'].dt.strftime('%B')
df['tahun'] = df['tgltrx'].dt.year

In [ ]:
df_monthly = (
    df.groupby(['fv_toko', 'fv_namasales', 'fv_catname', 'fv_ornal', 'day_type','season', 'nama_bulan','bulan'])
      .agg({'fn_jualpersize': 'sum', 'total_penjualan2': 'sum'})
      .reset_index()
).sort_values(['fv_namasales','bulan'])

In [ ]:
UMK_LOKAL = 5069708.36
UMK_ACUAN = 5000000
adjustment = UMK_LOKAL / UMK_ACUAN 

In [ ]:
df_monthly['insentif'] = np.where(
    df_monthly['fv_ornal'] == 'Obral',
    df_monthly['total_penjualan2'] * 0.01,   # jika 'Obral'
    df_monthly['total_penjualan2'] * 0.03    # jika bukan
)

In [ ]:
conditions = [
    df_monthly['day_type'] == 'weekday',
    df_monthly['day_type'] == 'weekend'
]

choices = [
    1.00,  # kalau obral
    1.10   # kalau weekend
]

df_monthly['Multiplier_day_type'] = np.select(conditions, choices, 0)
conditions = [
    df_monthly['season'] == 'normal',
    df_monthly['season'] == 'ramadan'
]

choices = [
    1.20,  # kalau obral
    1.00   # kalau weekend
]

df_monthly['Multiplier_season'] = np.select(conditions, choices, 0)

In [ ]:
df_monthly.info()

In [ ]:
df_monthly['insentif total']=df_monthly['insentif']*df_monthly['Multiplier_day_type']*df_monthly['Multiplier_season']

In [ ]:
df.loc[(df['nama_bulan']=='March')&(df['fv_namasales']=='AMEL')]['fn_jualpersize'].sum()

In [ ]:
df_monthly.loc[(df_monthly['nama_bulan']=='March')&(df_monthly['fv_namasales']=='AMEL')]['insentif total'].sum()

In [ ]:
df_monthly.to_excel('Detail_insentif.xlsx',index=False)
df_sum=(df_monthly.groupby(['fv_toko', 'fv_namasales', 'nama_bulan','bulan'])
      .agg({'insentif total': 'sum','fn_jualpersize': 'sum', 'total_penjualan2': 'sum'})
      .reset_index()
).sort_values(['fv_toko','fv_namasales','bulan'])

In [ ]:
df_sum['presentasi insentif per net casa']=(df_sum['insentif total']/df_sum['total_penjualan2'])* 100

In [ ]:
with pd.ExcelWriter('Insentif SPG.xlsx', engine='openpyxl') as writer:
    df_sum.to_excel(writer, sheet_name='Summary', index=False)
    df_monthly.to_excel(writer, sheet_name='Detail', index=False)

# AS 11 Report excel bedah counter

In [ ]:
def df__penjualan(input_param):
    store_list =input_param['store_list']
    if store_list  is not None:
        if store_list:
            placeholders = ", ".join([f"'{s}'" for s in store_list])
            df_cus = df__customers_pb_n(placeholders)
            whconsi_list = df_cus['fn_whconsiid'].tolist()
            
            if whconsi_list:
                whconsi_placeholders = ", ".join([str(s) for s in whconsi_list])
                whconsi = f"AND fn_whconsiid IN ({whconsi_placeholders})"
            else: whconsi = ""  # atau kondisi default kalau kosong
    else: whconsi = ""  # jika input_param['store_list'] None

    if input_param['divisi'] is not None and input_param['divisi']!='':
        divisi = input_param['divisi']
        divisi = f'and fv_divname="{divisi}"'
    else: divisi = ''

    if input_param['ornal'] is not None and input_param['ornal']!='':
        ornal = input_param['ornal']
        ornal = f'and fv_ornal="{ornal}"'
    else: ornal = ''
    
    whconsi_list=list_to_sql_in(
        df__customers_pb_(
            'fn_whconsiid',distinct=True,where='and fc_detail_tok="MDS"'), 'fn_whconsiid', astype='str')
    whconsi_placeholders = ", ".join([str(s) for s in whconsi_list])
    whconsi = f"AND fn_whconsiid IN ({whconsi_list})"
    main_query = f"""
    SELECT
    bulan,
    penjualan_2024,
    ifnull(penjualan_2025,0) penjualan_2025,
    ifnull((penjualan_2025 - penjualan_2024),0) AS offer_under,ifnull(
    CASE
        WHEN penjualan_2024 = 0 THEN NULL
        ELSE ROUND(((penjualan_2025 - penjualan_2024) / penjualan_2024) * 100, 2)
    END,0) AS growth_percent
    FROM (
    SELECT
        'January' AS bulan,
        (SELECT SUM(fn_totalpenjualan) FROM gis_local.pb_penjualan_2024 a 
            left join artikel_pb b on a.fv_artsizecode=b.fv_artsizecode 
            WHERE fc_monthname = 'January' {whconsi} {divisi} {ornal}) AS penjualan_2024,
        (SELECT SUM(fn_totalpenjualan) FROM gis_local.pb_penjualan_2025 a 
            left join artikel_pb b on a.fv_artsizecode=b.fv_artsizecode 
            WHERE fc_monthname = 'January' {whconsi} {divisi} {ornal}) AS penjualan_2025
    union all
        SELECT
        'February' AS bulan,
        (SELECT SUM(fn_totalpenjualan) FROM gis_local.pb_penjualan_2024 a 
            left join artikel_pb b on a.fv_artsizecode=b.fv_artsizecode 
            WHERE fc_monthname = 'February' {whconsi} {divisi} {ornal}) AS penjualan_2024,
        (SELECT SUM(fn_totalpenjualan) FROM gis_local.pb_penjualan_2025 a 
            left join artikel_pb b on a.fv_artsizecode=b.fv_artsizecode 
            WHERE fc_monthname = 'February' {whconsi} {divisi} {ornal}) AS penjualan_2025
    union all
        SELECT
        'March'AS bulan,
        (SELECT SUM(fn_totalpenjualan) FROM gis_local.pb_penjualan_2024 a 
            left join artikel_pb b on a.fv_artsizecode=b.fv_artsizecode 
            WHERE fc_monthname = 'March' {whconsi} {divisi} {ornal}) AS penjualan_2024,
        (SELECT SUM(fn_totalpenjualan) FROM gis_local.pb_penjualan_2025 a 
            left join artikel_pb b on a.fv_artsizecode=b.fv_artsizecode 
            WHERE fc_monthname = 'March' {whconsi} {divisi} {ornal}) AS penjualan_2025
    union all
        SELECT
        'April' AS bulan,
        (SELECT SUM(fn_totalpenjualan) FROM gis_local.pb_penjualan_2024 a 
            left join artikel_pb b on a.fv_artsizecode=b.fv_artsizecode 
            WHERE fc_monthname = 'April' {whconsi} {divisi} {ornal}) AS penjualan_2024,
        (SELECT SUM(fn_totalpenjualan) FROM gis_local.pb_penjualan_2025 a 
            left join artikel_pb b on a.fv_artsizecode=b.fv_artsizecode 
            WHERE fc_monthname = 'April' {whconsi} {divisi} {ornal}) AS penjualan_2025
    union all
        SELECT
        'May' AS bulan,
        (SELECT SUM(fn_totalpenjualan) FROM gis_local.pb_penjualan_2024 a 
            left join artikel_pb b on a.fv_artsizecode=b.fv_artsizecode 
            WHERE fc_monthname = 'May' {whconsi} {divisi} {ornal}) AS penjualan_2024,
        (SELECT SUM(fn_totalpenjualan) FROM gis_local.pb_penjualan_2025 a 
            left join artikel_pb b on a.fv_artsizecode=b.fv_artsizecode 
            WHERE fc_monthname = 'May' {whconsi} {divisi} {ornal}) AS penjualan_2025
    union all
        SELECT
        'June' AS bulan,
        (SELECT SUM(fn_totalpenjualan) FROM gis_local.pb_penjualan_2024 a 
            left join artikel_pb b on a.fv_artsizecode=b.fv_artsizecode 
            WHERE fc_monthname = 'June' {whconsi} {divisi} {ornal}) AS penjualan_2024,
        (SELECT SUM(fn_totalpenjualan) FROM gis_local.pb_penjualan_2025 a 
            left join artikel_pb b on a.fv_artsizecode=b.fv_artsizecode 
            WHERE fc_monthname = 'June' {whconsi} {divisi} {ornal}) AS penjualan_2025
    union all
        SELECT
        'July' AS bulan,
        (SELECT SUM(fn_totalpenjualan) FROM gis_local.pb_penjualan_2024 a 
            left join artikel_pb b on a.fv_artsizecode=b.fv_artsizecode 
            WHERE fc_monthname = 'July' {whconsi} {divisi} {ornal}) AS penjualan_2024,
        (SELECT SUM(fn_totalpenjualan) FROM gis_local.pb_penjualan_2025 a 
            left join artikel_pb b on a.fv_artsizecode=b.fv_artsizecode 
            WHERE fc_monthname = 'July' {whconsi} {divisi} {ornal}) AS penjualan_2025
    union all
        SELECT
        'August' AS bulan,
        (SELECT SUM(fn_totalpenjualan) FROM gis_local.pb_penjualan_2024 a 
            left join artikel_pb b on a.fv_artsizecode=b.fv_artsizecode 
            WHERE fc_monthname = 'August' {whconsi} {divisi} {ornal}) AS penjualan_2024,
        (SELECT SUM(fn_totalpenjualan) FROM gis_local.pb_penjualan_2025 a 
            left join artikel_pb b on a.fv_artsizecode=b.fv_artsizecode 
            WHERE fc_monthname = 'August' {whconsi} {divisi} {ornal}) AS penjualan_2025
    union all
        SELECT
        'September' AS bulan,
        (SELECT SUM(fn_totalpenjualan) FROM gis_local.pb_penjualan_2024 a 
            left join artikel_pb b on a.fv_artsizecode=b.fv_artsizecode 
            WHERE fc_monthname = 'September' {whconsi} {divisi} {ornal}) AS penjualan_2024,
        (SELECT SUM(fn_totalpenjualan) FROM gis_local.pb_penjualan_2025 a 
            left join artikel_pb b on a.fv_artsizecode=b.fv_artsizecode 
            WHERE fc_monthname = 'September' {whconsi} {divisi} {ornal}) AS penjualan_2025
    union all
        SELECT
        'October' AS bulan,
        (SELECT SUM(fn_totalpenjualan) FROM gis_local.pb_penjualan_2024 a 
            left join artikel_pb b on a.fv_artsizecode=b.fv_artsizecode 
            WHERE fc_monthname = 'October' {whconsi} {divisi} {ornal}) AS penjualan_2024,
        (SELECT SUM(fn_totalpenjualan) FROM gis_local.pb_penjualan_2025 a 
            left join artikel_pb b on a.fv_artsizecode=b.fv_artsizecode 
            WHERE fc_monthname = 'October' {whconsi} {divisi} {ornal}) AS penjualan_2025
    union all
        SELECT
        'November' AS bulan,
        (SELECT SUM(fn_totalpenjualan) FROM gis_local.pb_penjualan_2024 a 
            left join artikel_pb b on a.fv_artsizecode=b.fv_artsizecode 
            WHERE fc_monthname = 'November' {whconsi} {divisi} {ornal}) AS penjualan_2024,
        (SELECT SUM(fn_totalpenjualan) FROM gis_local.pb_penjualan_2025 a 
            left join artikel_pb b on a.fv_artsizecode=b.fv_artsizecode 
            WHERE fc_monthname = 'November' {whconsi} {divisi} {ornal}) AS penjualan_2025
    union all
        SELECT
        'December' AS bulan,
        (SELECT SUM(fn_totalpenjualan) FROM gis_local.pb_penjualan_2024 a 
            left join artikel_pb b on a.fv_artsizecode=b.fv_artsizecode 
            WHERE fc_monthname = 'December' {whconsi} {divisi} {ornal}) AS penjualan_2024,
        (SELECT SUM(fn_totalpenjualan) FROM gis_local.pb_penjualan_2025 a 
            left join artikel_pb b on a.fv_artsizecode=b.fv_artsizecode 
            WHERE fc_monthname = 'December' {whconsi} {divisi} {ornal}) AS penjualan_2025
    union all
        SELECT
        'Total' AS bulan,
        (SELECT SUM(fn_totalpenjualan) FROM gis_local.pb_penjualan_2024 a 
            left join artikel_pb b on a.fv_artsizecode=b.fv_artsizecode 
            WHERE 1=1 {whconsi} {divisi} {ornal}) AS penjualan_2024,
        (SELECT SUM(fn_totalpenjualan) FROM gis_local.pb_penjualan_2025 a 
            left join artikel_pb b on a.fv_artsizecode=b.fv_artsizecode 
            WHERE 1=1 {whconsi} {divisi} {ornal}) AS penjualan_2025
    ) AS penju;
    """
    with get_engine('imp') as engine:  # Menggunakan context manager dengan with
        df = pd.read_sql(main_query, engine)
    return df


## input param

In [ ]:
input_param={'input_value':'2025-10-27',
    'test':None,
        'alokator': None,
        'store_list': None,
        'start': '2025-10-27', #start.strftime("%Y-%m-%d"),
        'end': '2025-01-01',#ends.strftime("%Y-%m-%d"),
        'type': None,
        'format': None,
        'artikel': None,
        'input_date': '2025-10-27', #input_date.strftime("%Y-%m-%d"),
        'subid': None,
        's_all': None,
        'divisi': None,
        'ornal': None,
        'bulan': None,
        'cat': None,
        'mark': None,
        'art': None,
        'core': 'fv_ornal',
        'start_last': None,
        'end_last': None,
        'time': None,
        'stock_ideal': None,
        'spv_list':None,
        'list_selected':None,
        'spv':None,
        'export':None,}

In [ ]:
# input_param={'store_list':None, 'divisi':None, 'ornal':None,
            #  'core':'ornal', 'cat':None, 'artikel':None, 'spv_list':None}

## Penjualan tahunan

In [ ]:
df=df__penjualan(input_param)

## Obral normal

In [ ]:
def build_offer_under(
        df_this_year: pd.DataFrame,
        df_last_year: pd.DataFrame,
        kolom_dibandingkan: list,
        tahun_label: str = 'Offer Under',
        main_on:str = 'ornal'
    ) -> pd.DataFrame:
        # Merge berdasarkan kolom "ornal"
        df_offer_under = pd.merge(
            df_this_year,
            df_last_year,
            on=main_on,
            suffixes=('_this', '_last'),
            how='outer')
        # Selisih antar tahun untuk kolom yang diminta
        for kolom in kolom_dibandingkan:
            df_offer_under[kolom] = df_offer_under[f'{kolom}_this'] - df_offer_under[f'{kolom}_last']

        # Tambahkan kolom total_stock (jika sales dan stock_akhir dihitung)
        # if 'sales' in kolom_dibandingkan and 'stock_akhir' in kolom_dibandingkan:
        #     df_offer_under['total_stock'] = df_offer_under['sales'] + df_offer_under['stock_akhir']

        # # Hitung kontribusi hanya untuk kolom sales dan stock_akhir (jika ada)
        # if 'sales' in kolom_dibandingkan:
        #     df_offer_under['sales'] = pd.to_numeric(df_offer_under['sales'], errors='coerce')
        #     total_sales = df_offer_under['sales'].sum()
        #     df_offer_under['sales_kontribusi'] = round(df_offer_under['sales'] / total_sales * 100, 2)

        # if 'stock_akhir' in kolom_dibandingkan:
        #     df_offer_under['stock_akhir'] = pd.to_numeric(df_offer_under['stock_akhir'], errors='coerce')
        #     total_stock_akhir = df_offer_under['stock_akhir'].sum()
        #     df_offer_under['stock_kontribusi'] = round(df_offer_under['stock_akhir'] / total_stock_akhir * 100, 2)

        # Tambahkan kolom tahun
        df_offer_under['tahun'] = tahun_label

        # Ambil kolom akhir yang dibutuhkan
        kolom_akhir = [main_on] + kolom_dibandingkan
        # if 'total_stock' in df_offer_under.columns:
        #     kolom_akhir.append('total_stock')
        kolom_akhir.append('tahun')
        return df_offer_under[kolom_akhir]

In [ ]:
# Ambil core (misalnya 'fv_ornal' atau 'fv_catname') dari input_param
core=input_param['core']
# Ambil data per tahun sesuai core
df_main_this_year = add_df_main(input_param, y='2025', core=core)
df_main_last_year = add_df_main(input_param, y='2024', core=core)
# Gabungkan dan hitung Offer Under
df_main_ou = build_offer_under(
    df_this_year=df_main_this_year,
    df_last_year=df_main_last_year,
    kolom_dibandingkan=['Stock Awal', 'Sales (Rp)', 'Sales (pcs)', 'Stock Akhir','Jumlah Artikel'],
    tahun_label='Offer Under 2024-2025',
    main_on=core
).fillna(0)
# Jika semua kosong, kembalikan 204
df_main_this_year=df_main_this_year.loc[df_main_this_year['Total Stock']>0]
df_main_last_year=df_main_last_year.loc[df_main_last_year['Total Stock']>0]
df_main_ou=df_main_ou.loc[((df_main_ou['Stock Akhir'] + df_main_ou['Sales (pcs)'])!=0)]
df_main_this_year=df_main_this_year.loc[df_main_this_year['fv_ornal']!=0]
df_main_ou=df_main_ou.loc[df_main_ou['fv_ornal']!=0]
df_ornal_24=df_main_last_year.copy()
df_ornal_25=df_main_this_year.copy()
df_ornal_ou=df_main_ou.copy()
# df_main_this_year[['Kontribusi Sales','Kontribusi Stock']] = df_main_this_year[['Kontribusi Sales','Kontribusi Stock']].astype(str) + '%'
# df_main_last_year[['Kontribusi Sales','Kontribusi Stock']] = df_main_last_year[['Kontribusi Sales','Kontribusi Stock']]
# Kembalikan hasil JSON
# output_dict = {
#     '2025': df_main_this_year.to_dict(orient='records'),
#     '2024': df_main_last_year.to_dict(orient='records'),
#     'ou': df_main_ou.to_dict(orient='records'),
# }

In [ ]:
df_main_this_year

In [ ]:
df_main_last_year

In [ ]:
df_main_ou

## Kategory

In [ ]:
core='fv_catname'
# Ambil data per tahun sesuai core
df_main_this_year = add_df_main(input_param, y='2025', core=core)
df_main_last_year = add_df_main(input_param, y='2024', core=core)
# Gabungkan dan hitung Offer Under
df_main_ou = build_offer_under(
    df_this_year=df_main_this_year,
    df_last_year=df_main_last_year,
    kolom_dibandingkan=['Stock Awal', 'Sales (Rp)', 'Sales (pcs)', 'Stock Akhir','Jumlah Artikel'],
    tahun_label='Offer Under 2024-2025',
    main_on=core
).fillna(0)
# Jika semua kosong, kembalikan 204
df_main_this_year=df_main_this_year.loc[df_main_this_year['Total Stock']>0].sort_values('Sales (Rp)',ascending=False).head(5)
df_main_last_year=df_main_last_year.loc[df_main_last_year['Total Stock']>0].sort_values('Sales (Rp)',ascending=False).head(5)
df_main_ou=df_main_ou.loc[((df_main_ou['Stock Akhir'] + df_main_ou['Sales (pcs)'])!=0)].sort_values('Sales (Rp)',ascending=False).head(5)

In [ ]:

df_main_this_year=df_main_this_year.loc[df_main_this_year['Total Stock']>0].sort_values('Sales (Rp)',ascending=False).head(5)
df_main_last_year=df_main_last_year.loc[df_main_last_year['Total Stock']>0].sort_values('Sales (Rp)',ascending=False).head(5)
df_main_ou=df_main_ou.loc[((df_main_ou['Stock Akhir'] + df_main_ou['Sales (pcs)'])!=0)].sort_values('Sales (Rp)',ascending=False).head(5)

In [ ]:
df_main_this_year
df_main_last_year
df_main_ou

In [ ]:
df_main_last_year

In [ ]:
df_main_ou

## Top 5 artikel

In [ ]:
df_sales=df__sales_filter(input_param, 
                  col=f'fv_catname,fv_barcode,',
                  gb=f'fv_catname,fv_barcode',
                  year=2025)
df_top5_25 = (
    df_sales.sort_values(['fv_catname', 'sales'], ascending=[True, False])
      .groupby('fv_catname')
      .head(5)
)
df_sales=df__sales_filter(input_param, 
                  col=f'fv_catname,fv_barcode,',
                  gb=f'fv_catname,fv_barcode',
                  year=2024)
df_top5_24 = (
    df_sales.sort_values(['fv_catname', 'sales'], ascending=[True, False])
      .groupby('fv_catname')
      .head(5)
)


In [ ]:
df_top5_24

## Export excel

In [ ]:
with pd.ExcelWriter("laporan_gabungan.xlsx", engine='openpyxl') as writer:
    # ===============================
    # 2️⃣  Tulis beberapa DataFrame di sheet berbeda
    # ===============================
    df.to_excel(writer, index=False, sheet_name="Penjualan 2024-2025")
    ## Sheet Penjualan Obral-Normal
    start_row = 0  # posisi tulis awal
    df_ornal_24.to_excel(writer, index=False, sheet_name="Penjualan Obral-Normal", startrow=start_row)
    start_row += len(df_ornal_24) + 3  # beri jarak 3 baris kosong

    df_ornal_25.to_excel(writer, index=False, sheet_name="Penjualan Obral-Normal", startrow=start_row)
    start_row += len(df_ornal_25) + 3

    df_ornal_ou.to_excel(writer, index=False, sheet_name="Penjualan Obral-Normal", startrow=start_row)
    start_row += len(df_ornal_ou) + 3
    ## Sheet Penjualan Kategori Top 5
    start_row = 0  # posisi tulis awal
    df_main_last_year.to_excel(writer, index=False, sheet_name="Penjualan Kategori Top 5", startrow=start_row)
    start_row += len(df_main_last_year) + 3  # beri jarak 3 baris kosong

    df_main_this_year.to_excel(writer, index=False, sheet_name="Penjualan Kategori Top 5", startrow=start_row)
    start_row += len(df_main_this_year) + 3

    df_main_ou.to_excel(writer, index=False, sheet_name="Penjualan Kategori Top 5", startrow=start_row)
    start_row += len(df_main_ou) + 3
    
    ## Sheet Penjualan Artikel Top 5
    start_row = 0  # posisi tulis awal
    df_top5_24.to_excel(writer, index=False, sheet_name="Penjualan Artikel Top 5", startrow=start_row)
    start_row += len(df_top5_24) + 3  # beri jarak 3 baris kosong

    df_top5_25.to_excel(writer, index=False, sheet_name="Penjualan Artikel Top 5", startrow=start_row)
    start_row += len(df_top5_25) + 3



# AS 12 Stock store

In [ ]:
input_param={'input_value':'2025-10-29',
    'test':None,
        'alokator': None,
        'store_list': None,
        'start': '2025-10-29', #start.strftime("%Y-%m-%d"),
        'end': '2025-01-01',#ends.strftime("%Y-%m-%d"),
        'type': None,
        'format': None,
        'artikel': None,
        'input_date': '2025-10-29', #input_date.strftime("%Y-%m-%d"),
        'subid': None,
        's_all': None,
        'divisi': None,
        'ornal': None,
        'bulan': None,
        'cat': None,
        'mark': None,
        'art': None,
        'core': 'fv_ornal',
        'start_last': None,
        'end_last': None,
        'time': None,
        'stock_ideal': None,
        'spv_list':None,
        'list_selected':None,
        'spv':None,
        'export':None,}

In [ ]:
df=df__stock_now_filter(input_param, 
                         col='fv_barcode,',
                         gb='fv_barcode',
                         year=None,
                         cus='',
                         art='',
                         times='this_y',
                         having='')


In [ ]:
df_wh=df__art_stock_wh(type_show='all')

## list art

In [ ]:
list_art=['11213429010',
'11213429030',
'11213429083',
'11213429090',
'11223429005',
'11223429006',
'11223429007',
'11223429009',
'11223429011',
'11223429028',
'11223429031',
'11223429038',
'11223429041',
'11223429045',
'11223429081',
'11223429082',
'11223429087',
'11223429091',
'11223429092',
'11223429093',
'11238429004',
'11238429042',
'11513259001',
'11544129010',
'11734269057',
'12213429001',
'12213429009',
'12213429010',
'12213429024',
'12213429026',
'12213429027',
'12213429028',
'12213429036',
'12213429037',
'12213429042',
'12213429050',
'12213429053',
'12223259022',
'12223429007',
'12223429021',
'12223429023',
'12238429004',
'12443169001',
'12443269023',
'12443269024',
'12443439040',
'12443439041',
'12523519050',
'12523719052',
'12523759053',
'12538429010',
'12538429066',
'12613719043',
'12613739045',
'12613779044',
'12614569002',
'12614579067',
'12614579068',
'12614579075',
'12614589056',
'12614589058',
'12614629069',
'12614719060',
'12614729026',
'12614749017',
'12614749061',
'12614749062',
'12614749072',
'12614749074',
'12614769016',
'12614769018',
'12614769019',
'12614769025',
'12614769073',
'12614789031',
'12614829027',
'12615569003',
'12615589057',
'12615589059',
'12615749032',
'12615769029',
'12623549036',
'12623569035',
'12623569037',
'12623769014',
'12623789015',
'12654589011',
'12654769012',
'12663769042',
'12735129021',
'16313269019',
'16313269020',
'16313269021',
'16323129018',
'16323269019',
'16323269020',
'16323269021',
'16323269022',
'16443179053',
'21238429008',
'21313269013',
'21313269014',
'21313269015',
'21323269017',
'21743189016',
'21743189017',
'21743189018',
'21743189019',
'22213429002',
'22613719022',
'22613739024',
'22613759023',
'22614769028',
'22615769030',
'23443269016',
'31313129075',
'31313179074',
'31313269076',
'31323269077',
'31743129026',
'81223429072',
'81223429073',
'81223429087',
'81223429095',
'81714269058',
'82213429041',
'82213429046',
'82223429042',
'82443269025',
'82523519051',
'82523719054',
'82523759055',
'82614789033',
'82615719034',
'86623769038',
'86623769039',
'86623769040',
'86623769041',
'86713129020',
'86713219022',
'86713259021',
'86723189025',
'86723219023',
'86723219024',
'86743129037',
'86743129041',
'86743129043',
'86743189032',
'86743189034',
'86743219031',
'86743219042',
'86743219044',
'86743239033',
'86743239035',
'86743269036',
'86743429039',
'86743429040',
'1261458B9058P',
'AAAB438M0224',
'AAAC021M091',
'AAAC031C111',
'AAAC040F111',
'AAAC041A111',
'AAAC042M111',
'AAAC068M022',
'AAAC078M032',
'AAAC096H0822',
'AAAC102A0722',
'AAAC109A0922',
'AAAC115F1022',
'AAAC129K1222',
'AAAC149A1222',
'AAAC157M0123',
'AAAC174M0323',
'AAAC196L0123',
'AAAC230M0223',
'AAAC241M0323',
'AAAC261M0523',
'AAAC263L0323',
'AAAC276M0323',
'AAAC287M0323',
'AAAC288M0323',
'AAAC397M0124',
'AAAC419M0224',
'AAAC425M0224',
'AAAC429M0224',
'AAAC431M0324',
'AAAC450M0324',
'AAAC472M0224',
'AAAC600M0225',
'AAAC602M0125',
'AAAC607M0325',
'AAAG004A061',
'AAAG005K061',
'AAAG007F061',
'AAAG012C061',
'AABC003K061',
'AABC006G061',
'AABC008F061',
'AABC009N061',
'AABC010N061',
'AABC011N061',
'AABC013A061',
'AABC014N061',
'AABC027L101',
'AABC034M111',
'AABC035D111',
'AABC036A111',
'AABC037K111',
'AABC038A111',
'AABC039M111',
'AABC061M022',
'AABC077L032',
'AABC085M0722',
'AABC086K0722',
'AABC087A0722',
'AABC095M0822',
'AABC103L0722',
'AABC108K0922',
'AABC115A1022',
'AABC116K1022',
'AABC120G1122',
'AABC128A1222',
'AABC143L1222',
'AABC170M0323',
'AABC176M0423',
'AABC194L0323',
'AABC222M0423',
'AABC226D0123',
'AABC228M0223',
'AABC229M0223',
'AABC231M0223',
'AABC242M0323',
'AABC243M0323',
'AABC265F0323',
'AABC275M0323',
'AABC349L1023',
'AABC400M0124',
'AABC422M0124',
'AABC424M0224',
'AABC426M0224',
'AABC445M0224',
'AABC449M0324',
'AABC461M0324',
'AABC462M0324',
'AABC464K1223',
'AABC471M0224',
'AABC497M1124',
'AABC503M1024',
'AABC504M1124',
'AABC540M1224',
'AABC598M0225',
'AABC601M0125',
'AACC076M032',
'AACC124K1222',
'AACC159A0123',
'AACC200K0123',
'AACC586J0225',
'AACC612K0225',
'AADC063H042',
'AADC073K042',
'AADC075M032',
'AADC081H042',
'AADC125K1222',
'AADC140L0123',
'AADC158M0123',
'AADC216J0223',
'AADC244L0323',
'AADC245E0323',
'AADC246G0323',
'AADC371L1223',
'AADC465K1223',
'AADC611K0225',
'AAFC046L111',
'AAFC047L111',
'AAFC233D0223',
'AAFC236H0223',
'AAFC237A0223',
'AAFC238A0223',
'AAFC292L1223',
'AAFC295L1223',
'AAHC017K071',
'AAHC018A071',
'AAHC019K071',
'AAHC044I111',
'AAHC045P111',
'AAHC053K022',
'AAHC054K022',
'AAHC081A032',
'AAHC082A032',
'AAHC094F032',
'AAHC108J032',
'AAHC114C032',
'AAHC115D022',
'AAHC116K032',
'AAHC117H032',
'AAHC176D0822',
'AAHC180K0822',
'AAHC195I0822',
'AAHC203C0922',
'AAHC222D1122',
'AAHC228D1122',
'AAHC256E0223',
'AAHC338L0124',
'AAJC076F101',
'AAJC148N0423',
'AAJC168E1223',
'AAKC234A1124',
'AALC189A1123',
'AALC226J1124',
'AALD191H1123',
'AAMC218A0822',
'AASC024A101',
'AASC030D111',
'AASC034P121',
'AASC105J032',
'AASC118H032',
'AASC182P0822',
'AASC219D0922',
'AASC246D0223',
'AAUC074A101',
'AAUC075H101',
'AAVC194A1023',
'AAWD150E0423',
'AAWD151H0423',
'AAWD152A0423',
'AAWD153J0423',
'AAWE184A1023',
'AAXC061A081',
'AAXC062H081',
'AAXC063F081',
'AAXC067H081',
'AAXC068H081',
'AAXC069F081',
'AAXC094A022',
'AAXC095G022',
'AAXC096G022',
'AAXC097N022',
'AAXC098N022',
'AAXC118A1022',
'AAXC129A1122',
'AAZC099N022',
'AAZC137N0123',
'AAZC138N0123',
'ABAB437M0324',
'ABAB468M0224',
'ABAB485C0324',
'ABAB487C0324',
'ABAC023M091',
'ABAC025C101',
'ABAC029C111',
'ABAC033C111',
'ABAC050D111',
'ABAC051A111',
'ABAC056M111',
'ABAC090A0722',
'ABAC092H0722',
'ABAC098C0822',
'ABAC105M0722',
'ABAC111H0922',
'ABAC115A1022',
'ABAC121K1122',
'ABAC123H1122',
'ABAC137C1222',
'ABAC147M1222',
'ABAC148M1222',
'ABAC150M0223',
'ABAC163M0123',
'ABAC175M0323',
'ABAC183L0123',
'ABAC184E0223',
'ABAC185L0223',
'ABAC186C0223',
'ABAC187M0223',
'ABAC188M0223',
'ABAC203C0123',
'ABAC211M0223',
'ABAC212M0623',
'ABAC215L1123',
'ABAC218M0623',
'ABAC219M0623',
'ABAC220H0323',
'ABAC235M0223',
'ABAC236M0223',
'ABAC238M0223',
'ABAC251M0323',
'ABAC252M0323',
'ABAC253M0323',
'ABAC262L0323',
'ABAC269M0223',
'ABAC277M0323',
'ABAC278L0323',
'ABAC279C0323',
'ABAC281M0323',
'ABAC289M0323',
'ABAC290M0323',
'ABAC295M0623',
'ABAC319M1123',
'ABAC321M1123',
'ABAC323M1123',
'ABAC326M1123',
'ABAC327M1123',
'ABAC328M1123',
'ABAC339M0923',
'ABAC343C1023',
'ABAC347C0823',
'ABAC351M1123',
'ABAC360D1123',
'ABAC365M1123',
'ABAC366L0624',
'ABAC372M1223',
'ABAC374M1123',
'ABAC385A1123',
'ABAC387C1223',
'ABAC389M1223',
'ABAC399M0124',
'ABAC403M0124',
'ABAC412M0124',
'ABAC436M0324',
'ABAC446M0224',
'ABAC453M0324',
'ABAC454M0324',
'ABAC458M0224',
'ABAC459M0224',
'ABAC476M0324',
'ABAC498M1124',
'ABAC499M1124',
'ABAC539M1124',
'ABAC546M1224',
'ABAC604M0225',
'ABAC606M0125',
'ABAC614A1124',
'ABAC616A0125',
'ABAD338M0923',
'ABAD417M0224',
'ABAG001C061',
'ABAG002C061',
'ABBC017N061',
'ABBC018N061',
'ABBC019N061',
'ABBC020N061',
'ABBC024C101',
'ABBC049C111',
'ABBC060M032',
'ABBC070M022',
'ABBC071M022',
'ABBC091A0722',
'ABBC104L0722',
'ABBC112C0922',
'ABBC115P1022',
'ABBC254M0323',
'ABBC320M1123',
'ABBC322M1123',
'ABBC324M1123',
'ABBC325M1123',
'ABBC348L1023',
'ABBC386M1223',
'ABBD344L0923',
'ABCC069C022',
'ABCC160L0123',
'ABCC162C0123',
'ABCC182G0123',
'ABCC206K0123',
'ABCC214C0223',
'ABCC227C0123',
'ABCC234L0223',
'ABCC250C0323',
'ABCC255C0323',
'ABCC258C0123',
'ABCC270L0223',
'ABCC271C0223',
'ABCC363C1123',
'ABCC378L1223',
'ABCC413M0124',
'ABCC481L1223',
'ABDC059M032',
'ABDC164A0123',
'ABDC213M0223',
'ABDC237M0223',
'ABDC248C0323',
'ABDC249L0323',
'ABDC250J0323',
'ABDC260C0223',
'ABDC377M1223',
'ABDC421L0124',
'ABGC209A0922',
'ABGC210A0822',
'ABHC021K071',
'ABHC022A071',
'ABHC039H111',
'ABHC040A111',
'ABHC051J111',
'ABHC052K071',
'ABHC056K032',
'ABHC059D032',
'ABHC060D032',
'ABHC061J032',
'ABHC062A032',
'ABHC069A032',
'ABHC070C032',
'ABHC071C032',
'ABHC072B032',
'ABHC073P032',
'ABHC084F032',
'ABHC085J032',
'ABHC086H032',
'ABHC087H032',
'ABHC111J032',
'ABHC112O032',
'ABHC113P032',
'ABHC168A0822',
'ABHC170D0822',
'ABHC173J1222',
'ABHC195C0822',
'ABHC197K0822',
'ABHC204J0922',
'ABHC223J1122',
'ABHC224F1122',
'ABHC227K1022',
'ABHC248D0123',
'ABHC249J0123',
'ABHC250H0123',
'ABHC254D0123',
'ABHC258D0223',
'ABHC260P0223',
'ABHC261F0223',
'ABHC262D0223',
'ABHC266D0723',
'ABHC281K0823',
'ABHC286J0923',
'ABHC287F0923',
'ABHC288A0923',
'ABHC296K1123',
'ABHC297H1123',
'ABHC301J1123',
'ABHC305K1023',
'ABHC307L0124',
'ABHC310K1223',
'ABHC316K1223',
'ABHC317J1223',
'ABHC318A0124',
'ABHC325L0224',
'ABHC328G0224',
'ABHC340J0324',
'ABHC341H0324',
'ABHC361H0424',
'ABHC372N0524',
'ABHC385N0824',
'ABHC390A0624',
'ABHC391K1124',
'ABHG363L0824',
'ABJC021Z121',
'ABJC091V0223',
'ABJC125V1023',
'ABJC175V0524',
'ABJC176V0524',
'ABJC177X0524',
'ABJC181X0524',
'ABKR232M0125',
'ABMC025P111',
'ABMC026H111',
'ABMC027C111',
'ABMC036C101',
'ABMC097C032',
'ABMC098A032',
'ABMC186A0822',
'ABMC187F0822',
'ABMC190D0922',
'ABMC342P111',
'ABMC343C111',
'ABMC344C101',
'ABMC345C032',
'ABMC346A032',
'ABMC347A0822',
'ABMC348F0822',
'ABMC349D0922',
'ABRC068X1122',
'ABRC072V1222',
'ABRC073T1222',
'ABRC075V1222',
'ABRC076X1222',
'ABRC093V0323',
'ABRC094X0323',
'ABRC114X0623',
'ABRC150V0124',
'ABRC151V0124',
'ABRC152X0124',
'ABRC153X0224',
'ABRC184Y1024',
'ABRC185W1024',
'ABRC186Y1124',
'ABRC187T1124',
'ABRC188V1124',
'ABRD040V1022',
'ABRD055X1122',
'ABRD063V1222',
'ABRD088X0123',
'ABRD092V0323',
'ABRD111X0423',
'ABRD115V0623',
'ABRD119V0423',
'ABRD120X0723',
'ABRD121V0723',
'ABRD124X0623',
'ABRD148X1223',
'ABRD149V0124',
'ABRD163Q1223',
'ABRD164X1223',
'ABRD165V0224',
'ABRD192V1224',
'ABRD194W1224',
'ABRD199V0924',
'ABRE049X1022',
'ABRE063V0123',
'ABRE166V0224',
'ABRE193V1224',
'ABRF023V012',
'ABRF074V1222',
'ABSC032C121',
'ABSC037J111',
'ABSC101J032',
'ABSC102P032',
'ABSC188H0922',
'ABSC189A0922',
'ABSC192L0922',
'ABSC220L0323',
'ABSC221A0323',
'ABSC244L0223',
'ABSC245L0223',
'ABSC276A0723',
'ABSC285L1023',
'ABSC313L1123',
'ABTC047X1022',
'ABTC057X1122',
'ABTC162V1223',
'ABTC196V1224',
'ABTC197V1224',
'ABTC198T1224',
'ADBC001K051',
'ADBC002B051',
'ADBC003B051',
'ADBC004D051',
'ADBC005D051',
'ADBC006K051',
'ADBC007B051',
'ADBC008D051',
'ADBC009J051',
'ADBC010H051',
'ADBC011P051',
'ADBC012A051',
'ADBC013A051',
'ADBC014P051',
'ADHC119P042',
'ADHC120P042',
'ADHC121O042',
'ADHC122J042',
'ADHC123J042',
'ADHC124D042',
'ADMC131A042',
'ADMC132P042',
'ADMC133C042',
'ADRE001X041',
'ADRE002U041',
'ADRE003W041',
'ADRE004Y041',
'ADRE005S041',
'ADRE006Z041',
'ADRE007V041',
'ADRE008T041',
'ADRE009X041',
'ADRE010X041',
'ADRE016X121',
'ADRE017X121',
'ADRE018V121',
'ADRE019V121',
'ADRE020X121',
'ADRE031V0622',
'ADRE032V0622',
'ADRE033V0622',
'ADRE034X0622',
'ADRE035V0622',
'ADRE036T0622',
'ADRE037T0622',
'ADRE038W0622',
'ADRE140X1123',
'ADRE141V1123',
'ADRE142T1123',
'ADRE143T1123',
'ADRE144V1123',
'ADRE145X1123',
'ADSC128J042',
'ADSC129J042',
'ADSC130P042',
'ADUE090C022',
'ADUE091F022',
'ADUE092A022',
'ADUE093E022',
'AF0691P',
'AMRD128V1123',
'ARBC058M012',
'ARBC058M111',
'ARBC107H0422',
'ARRC014V0622',
'ARRC027U022',
'ARRD109V1122',
'ARRD110V1122',
'ARTC080V1222',
'ARTC081V1222',
'ARTC082V1222',
'ARTC083V1222',
'ARTD109V1122',
'ARTD110V1122',
'ARUA114K0422',
'ARXD008F022',
'ARXD008G022',
'B1037Z',
'BAAC028A101',
'BAAC046A111',
'BAAC047G111',
'BAAC089A0722',
'BAAC098A0822',
'BAAC154K1222',
'BAAC155K1222',
'BAAC172M0323',
'BAAC201M0123',
'BAAC210L0223',
'BAAC233M0223',
'BAAC311M1123',
'BAAC312M1123',
'BAAC432M0224',
'BAAC456M0224',
'BAAG007F061',
'BABC005K061',
'BABC008F061',
'BABC015N061',
'BABC016N061',
'BABC022M091',
'BABC027L101',
'BABC032L111',
'BABC043H111',
'BABC044C111',
'BABC045M111',
'BABC086D0822',
'BABC088L0722',
'BABC097H0822',
'BABC099D0822',
'BABC110K0922',
'BABC117K1022',
'BABC132H1122',
'BABC171M0323',
'BABC192M0523',
'BABC202M0123',
'BABC223M0423',
'BABC232M0223',
'BABC247M0323',
'BABC292M0323',
'BABC296L0623',
'BABC309M1123',
'BABC310M1123',
'BABC315M1123',
'BABC316M1123',
'BABC352M1123',
'BABC457M0224',
'BABC463M0324',
'BABC474M0224',
'BABC496M1124',
'BABC527M1124',
'BABC603M0125',
'BACC080M032',
'BACC200K0123',
'BADC079G032',
'BADC082H042',
'BADC083A042',
'BADC158M0323',
'BADC161L0123',
'BADC179H0123',
'BAFC233D0223',
'BAFC241H0223',
'BAFC242A0223',
'BAFC243A0223',
'BAHC020A071',
'BAHC048I111',
'BAHC049H111',
'BAHC050P111',
'BAHC055A022',
'BAHC058K032',
'BAHC092A032',
'BAHC093P032',
'BAHC095C032',
'BAHC096A032',
'BAHC109D032',
'BAHC110P032',
'BAHC134A042',
'BAHC135J042',
'BAHC137H042',
'BAHC138C042',
'BAHC139D042',
'BAHC140H042',
'BAHC141K042',
'BAHC150F052',
'BAHC154A052',
'BAHC155A052',
'BAHC157H052',
'BAHC158D052',
'BAHC159H052',
'BAHC160C052',
'BAHC161K052',
'BAHC162C052',
'BAHC166H042',
'BAHC171D0822',
'BAHC177H0822',
'BAHC178A0822',
'BAHC179A0922',
'BAHC208D0822',
'BAHC257E0223',
'BAJC001A071',
'BAJC076F101',
'BAJC148N0423',
'BAJC186E0923',
'BALC190A1123',
'BALD192H1123',
'BAMC142A032',
'BASC031C111',
'BASC035F121',
'BASC106J032',
'BASC107H032',
'BASC181J0822',
'BASC183H0822',
'BASC191A0922',
'BASC194J0922',
'BASC205J0822',
'BASC247D0223',
'BAUC077A101',
'BAUC078H101',
'BAVC179H0923',
'BAVC180H0923',
'BAVC183G0923',
'BAWD124A1122',
'BAWD156A0523',
'BAXC002A051',
'BAXC064A081',
'BAXC065H081',
'BAXC066F081',
'BAXC070H081',
'BAXC071F081',
'BAXC101N022',
'BAXC102N022',
'BAXC108H0522',
'BAXC142N0123',
'BAXC143N0123',
'BAXC174A0823',
'BAXC175E0823',
'BAXC176H0823',
'BAYC214A0524',
'BAZC100N022',
'BAZC138N0123',
'BAZC139N0123',
'BBAC054C111',
'BBAC057M111',
'BBAC094C0722',
'BBAC100A0822',
'BBAC119A1022',
'BBAC132H1222',
'BBAC133L1122',
'BBAC151M0123',
'BBAC152M0123',
'BBAC153M0223',
'BBAC173M0323',
'BBAC177M0323',
'BBAC191M0223',
'BBAC204C0123',
'BBAC207M0123',
'BBAC209M0123',
'BBAC225M0423',
'BBAC239M0223',
'BBAC240M0223',
'BBAC264L0323',
'BBAC282M0323',
'BBAC283M0323',
'BBAC284M0323',
'BBAC285L0323',
'BBAC286M0323',
'BBAC291M0323',
'BBAC306M1123',
'BBAC340M0923',
'BBAC345C1023',
'BBAC346C0823',
'BBAC350M1023',
'BBAC385A1123',
'BBAC423M0224',
'BBAC447M0224',
'BBAC448M0224',
'BBAC451M0224',
'BBAC452M0224',
'BBAC455M0324',
'BBAC460M0224',
'BBAC477M0324',
'BBAC495M1024',
'BBAC550M1224',
'BBAD341M0923',
'BBBC024C101',
'BBBC026L101',
'BBBC052M111',
'BBBC053F111',
'BBBC072M022',
'BBBC074M032',
'BBBC101L0822',
'BBBC113A0922',
'BBBC114C0922',
'BBBC118H1022',
'BBBC144M1222',
'BBBC145M1222',
'BBBC146M1222',
'BBBC166L0123',
'BBBC208M0123',
'BBBC217M0223',
'BBBC224M0423',
'BBBC301M1123',
'BBBC302M1123',
'BBBC303M1123',
'BBBC304M1123',
'BBBC305M1123',
'BBBC545M1224',
'BBBD342M0923',
'BBCC062M032',
'BBCC165L0123',
'BBCC198C0123',
'BBCC227C0123',
'BBCC256C0323',
'BBDC167H0123',
'BBHC016A071',
'BBHC041C111',
'BBHC042J111',
'BBHC043A111',
'BBHC057K032',
'BBHC063J032',
'BBHC064A022',
'BBHC065J032',
'BBHC066D032',
'BBHC067K032',
'BBHC068C032',
'BBHC074P032',
'BBHC075C032',
'BBHC076A032',
'BBHC077D032',
'BBHC078H032',
'BBHC079P032',
'BBHC080A032',
'BBHC088F032',
'BBHC089J032',
'BBHC090C032',
'BBHC091C032',
'BBHC125H042',
'BBHC126J042',
'BBHC143C042',
'BBHC144J042',
'BBHC145A042',
'BBHC146J042',
'BBHC147C042',
'BBHC148P042',
'BBHC149P042',
'BBHC153A042',
'BBHC167042',
'BBHC178C0922',
'BBHC198A0922',
'BBHC199H042',
'BBHC200D0822',
'BBHC201D0822',
'BBHC202D0822',
'BBHC207K0822',
'BBHC225K1122',
'BBHC229K1222',
'BBHC230J1222',
'BBHC235L0123',
'BBHC251D0123',
'BBHC252J0123',
'BBHC253H0123',
'BBHC255D0123',
'BBHC259D0223',
'BBHC263F0223',
'BBHC264D0223',
'BBHC265P0223',
'BBHC283K0823',
'BBHC289J0923',
'BBHC290F0923',
'BBHC302J1123',
'BBHC362H0424',
'BBHC386A0824',
'BBJC022Z121',
'BBJC086V1222',
'BBJC087V1222',
'BBMC023H101',
'BBMC028D111',
'BBMC033P121',
'BBMC099A032',
'BBMC100C032',
'BBMC163P052',
'BBMC190F0922',
'BBMC193C0922',
'BBMC213A0922',
'BBMC350F0922',
'BBMC351C0922',
'BBMC352A0922',
'BBMC353D0922',
'BBMC354F0922',
'BBMC355K0922',
'BBRC069V1122',
'BBRC070V1122',
'BBRC077X1222',
'BBRC078V1222',
'BBRC079V1222',
'BBRC122X0723',
'BBRC154X0224',
'BBRC189Y1124',
'BBRC190T1124',
'BBRC191V1124',
'BBRD026V042',
'BBRD028V052',
'BBRD029V052',
'BBRD030V052',
'BBRD042V1022',
'BBRD059X1122',
'BBRD066V1222',
'BBRD067X0123',
'BBRD071V1122',
'BBRD103V0423',
'BBRD104X0423',
'BBRD105V0823',
'BBRD106X0823',
'BBRD112X0423',
'BBRD113V0423',
'BBRD116X0623',
'BBRD117V0623',
'BBRE043X1022',
'BBRE058V1122',
'BBRE066V0123',
'BBRE067X1222',
'BBRE090X0223',
'BBRF024V012',
'BBRF044V1022',
'BBSC029A111',
'BBSC103P032',
'BBSC104J032',
'BBSC164P052',
'BBSC183J0822',
'BBSC184A0822',
'BBSC244L0223',
'BBSC245L0223',
'BBSC277A0723',
'BBTC048V1022',
'BBTC060V1122',
'BBTC061X1122',
'BBTC062V1122',
'BCBC405M1023',
'BCRD178V0424',
'BCUC136H1222',
'BMRD138X1123',
'BRAC106L0422',
'BRAC577E0824',
'BRAC578E0824',
'BRAC579J0824',
'BRAC580E0824',
'BRBC058M012',
'BRBC509C0424',
'BRBC510J0424',
'BRBC511G0424',
'BRBC513A0424',
'BRBC514M0424',
'BRBC516E0424',
'BRBC520D0424',
'BRBC522P0424',
'BRBC523H0424',
'BRBC524G0424',
'BRBC525P0424',
'BRBC526N0424',
'BRBC528C0524',
'BRBC529C0524',
'BRBC530C0524',
'BRBC531C0524',
'BRBC532C0524',
'BRBC533C0524',
'BRBC534C0524',
'BRBC535C0524',
'BRBC536C0524',
'BRBC537C0524',
'BRBC538C0524',
'BRBC541N0524',
'BRBC542N0524',
'BRBC543N0524',
'BRBC551D0724',
'BRBC552H0724',
'BRBC553G0724',
'BRBC554O0724',
'BRBC555A0724',
'BRBC556C0724',
'BRBC557C0724',
'BRBC558A0724',
'BRBC559J0724',
'BRBC560D0724',
'BRBC561A0724',
'BRBC562A0724',
'BRBC563D0724',
'BRBC564C0724',
'BRBC565D0724',
'BRBC566O0724',
'BRBC567A0724',
'BRBC568F0724',
'BRBC569P0724',
'BRBC570D0724',
'BRBC571E0724',
'BRBC572J0824',
'BRBC573C0824',
'BRBC574K0824',
'BRBC575L0824',
'BRBC576C0824',
'BRBC581G0824',
'BRBC582J0824',
'BRBC583G0824',
'BRHC206A0422',
'BRRC014V0622',
'BRRD027U012',
'BRRE109V1122',
'BRRE110V1122',
'BRTC014U1024',
'BRTC027U1024',
'BRTC102V0323',
'BRTC114V0323',
'BRTD013U051',
'BRTD015U051',
'BRUA114K0422',
'BRUC004A061',
'BRUC005A051',
'BRUC006A051',
'BRUC007H051',
'BRUC012G051',
'BRUC035F051',
'BRUC039F051',
'BRUC046E051',
'BRUC079F081',
'BRUC081H081',
'BRUC083G081',
'BRUC084G081',
'BRUC088G081',
'BRUC089F081',
'BRVC004A051',
'BRVC017G051',
'BRVC019H051',
'BRVC020H051',
'BRVC023E051',
'BRVC025F051',
'BRVC032C051',
'BRVC033C051',
'BRVC034F051',
'BRVC043G051',
'BRVC047A061',
'BRVC082G081',
'BRVC087E081',
'BRVD010H012',
'BRVD018G012',
'BRWC073H061',
'BRXC002A061',
'BRXC022G051',
'BRXC024F051',
'BRXC040M051',
'BRXC042F051',
'BRXC045K051',
'BRXC048A061',
'BRXC072H061',
'BRXC080F081',
'BRXC085G081',
'BRXC086G081',
'BZ01701P',
'BZ01701S',
'BZ01750F',
'C0207',
'C0465',
'C0475',
'C1004',
'C1033Z',
'CBRC131V1123',
'CBRC136X1123',
'CBRC137V1123',
'CBRC146V1123',
'CBRC167X0224',
'CBRC168X0224',
'CBRD129X1123',
'CBRD130X1123',
'CBRD134X1123',
'CBRD135V1123',
'CBRD158X1123',
'CBRD160X1123',
'CBRD161T1123',
'CBRE169X0224',
'CBRE170X0224',
'CBRE171X0224',
'CC01069M',
'CC01601R',
'CRUC115A0422',
'CZ01701A',
'CZ01701R',
'CZ01745M',
'CZ01751V',
'DF0818A',
'EBRD172X0324',
'EBRD173X0324',
'EBRE174V0524',
'EF0734B',
'EF0850A',
'EF1096F',
'EMRD132V0124',
'GA01530L',
'GA01708K',
'GA01712K',
'GA01745L',
'GA01745R',
'GA01805F',
'GA01883U',
'GA01943K',
'GA01946L',
'GA02032F',
'GA02095F',
'GA02113F',
'GA02117F',
'GA02187K',
'GA02215R',
'GA02231F',
'GA02232F',
'GA02233F',
'GA02234F',
'GA02288L',
'GA02289K',
'GA02352F',
'GA02353F',
'GA02354F',
'GA02355F',
'GA02357R',
'GA02388F',
'GA02389F',
'GA02402F',
'GA02405F',
'GA02419F',
'GA02420F',
'GA02421F',
'GA02422F',
'GA02438F',
'GA02447F',
'GA02449F',
'GA02489F',
'GA02499F',
'GA02500F',
'GA02521R',
'GA02522F',
'GA02532F',
'GA02540F',
'GA02545R',
'GA02547F',
'GA02548F',
'GA02559F',
'GA02560R',
'GA02580F',
'GA02581F',
'GA02599R',
'GA02600F',
'GA02601F',
'GA02603F',
'GA02604R',
'GA02605F',
'GA02606F',
'GA02607F',
'GA02613F',
'GA02631F',
'GA02632F',
'GA02634F',
'GA02635F',
'GA02636F',
'GA02637F',
'GA02644K',
'GA02648F',
'GA02650F',
'GA02652F',
'GA02657K',
'GA02658K',
'GA02659K',
'GA02660F',
'GA02661K',
'GA02662K',
'GA02694R',
'GA02705L',
'GA02706K',
'GA02707K',
'GA02708F',
'GA02710F',
'GA02734F',
'GA02842R',
'GA02969L',
'GA03083F',
'GA03164R',
'GA03240L',
'GA03440F',
'GA0464L',
'GA0801R',
'GA0892G',
'GA0988S',
'GA1042R',
'GA1044F',
'GA1101S',
'GA1103S',
'GA1171S',
'GA1217L',
'GA1245F',
'GA1413F',
'GA1485K',
'GAA540S',
'GAA571L',
'GAB533L',
'GB01521L',
'GB01811F',
'GB01844G',
'GB01927F',
'GB01950L',
'GB01952K',
'GB01953K',
'GB01954K',
'GB01966K',
'GB01970K',
'GB02008U',
'GB02009U',
'GB02018F',
'GB02022T',
'GB02083F',
'GB02101F',
'GB02118F',
'GB02119F',
'GB02237F',
'GB02261R',
'GB02282F',
'GB02283F',
'GB02285F',
'GB02286F',
'GB02298K',
'GB02356F',
'GB02362F',
'GB02450F',
'GB02735K',
'GB02812F',
'GB02813F',
'GB02828F',
'GB03299L',
'GB03300L',
'GB03303L',
'GB0987S',
'GB1135G',
'GB1226K',
'GB1494K',
'GBA365F',
'GBE225K',
'GBRE090X0324',
'GE0842F',
'GF0788B',
'GF1029L',
'GF1030L',
'GS7438F',
'GS7442F',
'GT01401P',
'GT11407F',
'H05266',
'H5268',
'HA01484F',
'HA01495F',
'HA01558F',
'HA01559F',
'HA01618F',
'HA01621F',
'HA01622F',
'HA01623S',
'HA01641F',
'HA01652S',
'HA01653F',
'HA01672F',
'HA01675S',
'HA01678F',
'HA01680F',
'HA01703R',
'HA01762R',
'HA01765S',
'HA01767F',
'HA01790S',
'HA01792F',
'HA01794F',
'HA01804S',
'HA01831F',
'HA01836S',
'HA01861S',
'HA01993F',
'HA01994F',
'HA01995F',
'HA02004F',
'HA02005F',
'HA02006T',
'HA02007T',
'HA02014T',
'HA02040F',
'HA02041F',
'HA02075F',
'HA02076F',
'HA02109F',
'HA02139F',
'HA02154F',
'HA02168F',
'HA02170F',
'HA02227R',
'HA02228F',
'HA02271F',
'HA02273F',
'HA02291F',
'HA02292F',
'HA02293F',
'HA02295R',
'HA02320F',
'HA02331F',
'HA02332F',
'HA02369F',
'HA02371F',
'HA02448F',
'HA02459F',
'HA02473F',
'HA02490R',
'HA02491R',
'HA02498R',
'HA02501R',
'HA02502R',
'HA02519F',
'HA02520R',
'HA02558F',
'HA02576F',
'HA02577R',
'HA02591F',
'HA02633F',
'HA02638R',
'HA02647R',
'HA02649R',
'HA02696R',
'HA02698R',
'HA02711R',
'HA02723F',
'HA02727R',
'HA02750F',
'HA02751R',
'HA02753R',
'HA02777F',
'HA02788R',
'HA02790F',
'HA02805F',
'HA02806F',
'HA02807F',
'HA02809F',
'HA02810F',
'HA02814R',
'HA02815R',
'HA02816R',
'HA02817R',
'HA02818R',
'HA02829R',
'HA02830F',
'HA02831F',
'HA02832F',
'HA02833F',
'HA02834F',
'HA02835F',
'HA02843R',
'HA02844R',
'HA02845R',
'HA02846F',
'HA02847F',
'HA02848F',
'HA02849F',
'HA02850R',
'HA02851F',
'HA02883L',
'HA02886R',
'HA02889L',
'HA02890F',
'HA02891F',
'HA02892F',
'HA02893F',
'HA02928R',
'HA02929F',
'HA02930F',
'HA02931F',
'HA02949F',
'HA02952R',
'HA02953R',
'HA02954F',
'HA02955F',
'HA02956R',
'HA03009F',
'HA03018R',
'HA03021F',
'HA03022F',
'HA03023R',
'HA03071R',
'HA03073F',
'HA03080F',
'HA03082R',
'HA03119R',
'HA03318R',
'HA03336R',
'HA0718R',
'HA1004S',
'HA1032S',
'HA1121S',
'HA1150S',
'HA1244F',
'HA1275R',
'HA1278F',
'HA1301S',
'HA1382F',
'HA1456F',
'HA1489F',
'HA1518F',
'HA1564F',
'HA1567G',
'HA1568F',
'HAA1274R',
'HAA1321F',
'HAA1565F',
'HAA252F',
'HAA4B0001-',
'HAA4B0002-',
'HAA4B0003-',
'HAA4B0006-',
'HAA542F',
'HAB4B0004-',
'HAE4B0005-',
'HB01499F',
'HB01560F',
'HB01561F',
'HB01562F',
'HB01589R',
'HB01620G',
'HB01627S',
'HB01659F',
'HB01684F',
'HB01687G',
'HB01716R',
'HB01717S',
'HB01718F',
'HB01719F',
'HB01747S',
'HB01751R',
'HB01753F',
'HB01782S',
'HB01784S',
'HB01786R',
'HB01795R',
'HB01809S',
'HB01898F',
'HB01899F',
'HB01929F',
'HB02020F',
'HB02042F',
'HB02062F',
'HB02115F',
'HB02141F',
'HB02174R',
'HB02300F',
'HB02492R',
'HB02493R',
'HB02506F',
'HB02507R',
'HB02541R',
'HB02695F',
'HB02789F',
'HB0293F',
'HB02983R',
'HB03026R',
'HB03099R',
'HB03320L',
'HB0386F',
'HB0633F',
'HB0707F',
'HB0763F',
'HB0992R',
'HB1155G',
'HB1174F',
'HB1248F',
'HB1285F',
'HB1309S',
'HB1314S',
'HB1327R',
'HB1349F',
'HB1410F',
'HB1427F',
'HB1465F',
'HB1496F',
'HBA0544S',
'HBA1272R',
'HBA1328F',
'HBA1462F',
'HBA292F',
'HBE194F',
'HBE269S',
'HC0359F',
'HD01813S',
'HE0330F',
'HE0903F',
'HE1318G',
'HE1319G',
'HF01639G',
'HF01717R',
'HF01832F',
'HF01847G',
'HF01881F',
'HF01881G',
'HF01909F',
'HF02043F',
'HF02044R',
'HF02045F',
'HF02046R',
'HF02079F',
'HF0392F',
'HF0792A',
'HF0820A',
'HF0951B',
'HF0953A',
'HF0994B',
'HF0995B',
'HF0997B',
'HF0999B',
'HF1000B',
'HF1001B',
'HF1002B',
'HF1057S',
'HF1058S',
'HF1075S',
'HF1099G',
'HF1164G',
'HF1165G',
'HF1167G',
'HF1168G',
'HF1373G',
'HF1402G',
'HF1404G',
'HF1405G',
'HF1522G',
'HF1523S',
'HF1524G',
'HFA1711F',
'HFB1700F',
'HI0918E',
'HI0919E',
'HI0920E',
'HI0920Q',
'HJ0916F',
'HJ0917F',
'HJ0919F',
'HJ0920R',
'HJ0921R',
'HR01745F',
'HR01750F',
'HR02738F',
'HR02766F',
'HR02767R',
'HR02792F',
'HR02793F',
'HR03175R',
'HR03379L',
'HR03380K',
'HR03493R',
'HR03494R',
'HR03495F',
'HR03496F',
'HR03497F',
'HR03497R',
'HRRC179R0424',
'HRRD180R0424',
'HS7439F',
'HT01410G',
'HT01410S',
'HT01601G',
'HT01606F',
'HT01882F',
'HT0487F',
'HT1026F',
'HT1101Y',
'HT11605F',
'HT11605P',
'HT8001F',
'HT8002F',
'HT8005F',
'HT8006F',
'HT8008F',
'HT8011F',
'HT8012F',
'HV01405S',
'HV01460F',
'HV01485F',
'HY0004E',
'HY1005E',
'HZ0002F',
'HZ0003F',
'HZ0004F',
'HZ01647G',
'HZ1001F',
'HZ1004F',
'HZ1004P',
'HZ11611F',
'HZ11613P',
'HZ7002F',
'IA0009E',
'IA01517F',
'IA01576F',
'IA01577S',
'IA01578F',
'IA01644F',
'IA01645F',
'IA01646G',
'IA01647F',
'IA01648R',
'IA01655F',
'IA01656F',
'IA01657F',
'IA01673S',
'IA01681R',
'IA01683S',
'IA01705L',
'IA01710L',
'IA01724S',
'IA01739F',
'IA01740K',
'IA01763G',
'IA01764S',
'IA01789S',
'IA01793R',
'IA01806S',
'IA01827S',
'IA01828S',
'IA01863R',
'IA0187D',
'IA01885S',
'IA01888T',
'IA01910T',
'IA01911T',
'IA01939K',
'IA01944L',
'IA01964R',
'IA01964U',
'IA02071F',
'IA02111F',
'IA02112F',
'IA02116K',
'IA02128L',
'IA02129R',
'IA02130R',
'IA02133F',
'IA02134F',
'IA02192R',
'IA02205F',
'IA02206F',
'IA02207F',
'IA02208F',
'IA02210R',
'IA02216F',
'IA02220L',
'IA02313K',
'IA02333F',
'IA02358R',
'IA02455F',
'IA02457F',
'IA02457R',
'IA02458F',
'IA02458R',
'IA02474R',
'IA02477R',
'IA02511F',
'IA02569F',
'IA02570F',
'IA02571F',
'IA02575F',
'IA02672L',
'IA02687L',
'IA02691K',
'IA0270P',
'IA0271P',
'IA02736L',
'IA02737L',
'IA02754L',
'IA02887L',
'IA02888L',
'IA02913L',
'IA02914L',
'IA02966K',
'IA02967K',
'IA02968K',
'IA02970L',
'IA02971K',
'IA02972K',
'IA02984K',
'IA02985L',
'IA03037K',
'IA03040R',
'IA03041R',
'IA03064R',
'IA03233R',
'IA03239L',
'IA03254K',
'IA03255L',
'IA03288L',
'IA03289L',
'IA03290L',
'IA03314L',
'IA03316L',
'IA03384K',
'IA03384L',
'IA03459L',
'IA03461F',
'IA0423L',
'IA0693G',
'IA0715S',
'IA0800F',
'IA0807S',
'IA1070F',
'IA1161G',
'IA1180S',
'IA1247F',
'IA1256F',
'IA1257F',
'IA1258G',
'IA1304F',
'IA1324F',
'IA1325F',
'IA1348F',
'IA1359F',
'IA1361F',
'IA1362R',
'IA1363F',
'IA1375G',
'IA1376F',
'IA1379R',
'IA1420F',
'IA1436F',
'IA1476F',
'IA1477F',
'IA1480L',
'IA1481K',
'IA1487K',
'IA1515L',
'IA1516L',
'IA1527K',
'IAA1478K',
'IAA1480L',
'IAA271R',
'IAA924F',
'IAA980L',
'IAB1175F',
'IAB1376F',
'IAE048F',
'IAE367F',
'IB0055F',
'IB0078R',
'IB01615F',
'IB01638F',
'IB01650G',
'IB01651F',
'IB01697S',
'IB01713K',
'IB01723S',
'IB01744F',
'IB01760F',
'IB01770S',
'IB01812G',
'IB01829S',
'IB01840S',
'IB01886U',
'IB01900G',
'IB01965K',
'IB01967K',
'IB02019F',
'IB02023K',
'IB02024K',
'IB02121F',
'IB02137F',
'IB02159R',
'IB02344R',
'IB02429R',
'IB02495F',
'IB02508R',
'IB02562R',
'IB02572F',
'IB02986K',
'IB02987L',
'IB03017K',
'IB03042R',
'IB03043R',
'IB03055R',
'IB03058R',
'IB03059F',
'IB03060R',
'IB03297L',
'IB03298L',
'IB03312L',
'IB03313K',
'IB03321L',
'IB03322L',
'IB03385K',
'IB03386K',
'IB03427F',
'IB03427R',
'IB0937G',
'IB1087F',
'IB1088R',
'IB1159G',
'IB1199G',
'IB1330F',
'IB1332F',
'IB1360K',
'IB1368G',
'IB1369F',
'IB1377G',
'IB1378F',
'IB1490F',
'IB1519L',
'IB1520F',
'IBA1463L',
'IBA1491K',
'IBA1529K',
'IBA1546S',
'IBA1557L',
'IBA1615R',
'IBB0976L',
'IBB1554F',
'ID0269F',
'IE01780F',
'IE03462F',
'IE1027L',
'IF01905F',
'IF02462R',
'IF0521L',
'IF1061F',
'IF1062F',
'IF1098F',
'II0915E',
'II0916E',
'II0917E',
'II0918E',
'II0918Q',
'IJ0916F',
'IR02738F',
'IR02768F',
'IR02769K',
'IR02770K',
'IR02771K',
'IR02773K',
'IR02774F',
'IR02775F',
'IR02775R',
'IS7388F',
'IT01409F',
'IT01409K',
'IT11401P',
'IT11406F',
'IV01405P',
'IV11411F',
'IV21411S',
'IW0455F',
'IY0002E',
'IY0003E',
'IY0004E',
'IZ0001F',
'IZ0002E',
'IZ0002F',
'IZ0003F',
'IZ1004P',
'IZ1006F',
'JA00547K',
'JA00571E',
'JA00577M',
'JA00660N',
'JA00661M',
'JA00663J',
'JA00676A',
'JA00687A',
'JA00688M',
'JA00689A',
'JA00689K',
'JA00696K',
'JA00702A',
'JA00723K',
'JA00723M',
'JA00739A',
'JA00740A',
'JA00741A',
'JA00751A',
'JA00751E',
'JA00751K',
'JA00752A',
'JA00752K',
'JA00791A',
'JA00791E',
'JA00791M',
'JA00792A',
'JA00829A',
'JA00837A',
'JA00846A',
'JA00846K',
'JA00891J',
'JA00907M',
'JA00909M',
'JA0092U',
'JA00965M',
'JA00966M',
'JA00973M',
'JA01020M',
'JA01073K',
'JA01097J',
'JA0109U',
'JA0110U',
'JA01198E',
'JA01276K',
'JA01276M',
'JA01278M',
'JA01279K',
'JA0340M',
'JA0349M',
'JA0393M',
'JA0407B',
'JA0410B',
'JA0464I',
'JA0506I',
'JA0508M',
'JA0538M',
'JA0570I',
'JA0576I',
'JA0597M',
'JA0605I',
'JA0605K',
'JAA0570A',
'JAA0578A',
'JAA0605A',
'JAA375M',
'JAB0571A',
'JB0011U',
'JB0019H',
'JB00309A',
'JB0036U',
'JB00389H',
'JB00452H',
'JB0051J',
'JB00610M',
'JB00642K',
'JB00692A',
'JB00715A',
'JB00727A',
'JB00728A',
'JB00729A',
'JB00768A',
'JB00793M',
'JB0079U',
'JB0088H',
'JB00914H',
'JB0097H',
'JB0097N',
'JB01098J',
'JB01142M',
'JB01143E',
'JB01144K',
'JB01145M',
'JB01146E',
'JB01147M',
'JB0310M',
'JB0364Y',
'JB0414I',
'JB0453I',
'JBA020Y',
'JBA0310M',
'JBA0549A',
'JBA079H',
'JBA452M',
'JBB0415A',
'JBB0452M',
'JBB197M',
'JBB310H',
'JBC0310H',
'JBC170I',
'JBD0326H',
'JC0007H',
'JC0008H',
'JE0053J',
'JE0055J',
'JE0072G',
'JE01100M',
'JE01191D',
'JE01192D',
'JE01256Q',
'JE0143J',
'JE0231J',
'JE0253J',
'JE0374M',
'JE0969K',
'JEA332H',
'JF00357G',
'JF00630D',
'JF00650Y',
'JF00669Y',
'JF00750A',
'JF00756Y',
'JF00757Y',
'JF0079M',
'JF00913M',
'JF00931M',
'JF01002E',
'JF01003E',
'JF01026E',
'JF0355O',
'JF0355Q',
'JF0434H',
'JF0515I',
'JF0563D',
'JH01133D',
'JI08013M',
'JI8007M',
'JK00639M',
'JL00706K',
'JO7527M',
'JO7537M',
'JO7582M',
'JP00713K',
'JP00735K',
'JP00764M',
'JP00795M',
'JP00796M',
'JP00816M',
'JP00817M',
'JP00818M',
'JP00833M',
'JP00834M',
'JP00842Q',
'JP00855M',
'JP00856M',
'JP00887M',
'JP00888M',
'JP00915M',
'JP00916M',
'JP00952M',
'JP00953M',
'JP00954M',
'JP00972M',
'JP8287H',
'JP8470Y',
'JQ00737M',
'JQ00738M',
'JQ00889M',
'JQ00890M',
'JR01125K',
'JR01138E',
'JR01148K',
'JR01149E',
'JR01150K',
'JR01159K',
'JR01186A',
'JR01188A',
'JR01188K',
'JR01209A',
'JR01213J',
'JR01240J',
'JR01240K',
'JR01241K',
'JR01285K',
'JR01286K',
'JR01287Q',
'JR01288E',
'JR01290K',
'JR01290M',
'JR01291M',
'JR01292J',
'JR01701J',
'JR01701Y',
'JR01711A',
'JR01711M',
'JR01745P',
'JT01407Y',
'JT01414M',
'JT01445J',
'JT01606J',
'JT01606V',
'JT01606Y',
'JT01647J',
'JT11415H',
'JT11605M',
'JT11607K',
'JT21409M',
'JT21411Y',
'JT61483M',
'JT8006J',
'JT8434M',
'JT8459M',
'JT8467M',
'JT8468M',
'JT8472M',
'JT8478M',
'JT8480M',
'JU00474H',
'JU00621J',
'JU00627J',
'JU00649M',
'JU00655A',
'JU00656M',
'JU00668Y',
'JU00669G',
'JU00700Y',
'JU00703Y',
'JU00705A',
'JU00711H',
'JU00711M',
'JU00726A',
'JU00742A',
'JU00742K',
'JU00743M',
'JU00744A',
'JU00746A',
'JU00746E',
'JU00753A',
'JU00758M',
'JU00759M',
'JU00760M',
'JU00762M',
'JU00763A',
'JU00774A',
'JU00788M',
'JU00789M',
'JU00790A',
'JU00790M',
'JU00797A',
'JU00801M',
'JU00804A',
'JU00805A',
'JU00813A',
'JU00827Y',
'JU00828A',
'JU00828M',
'JU00830H',
'JU00830K',
'JU00830M',
'JU00854E',
'JU00865A',
'JU00865M',
'JU00867E',
'JU00894M',
'JU00905E',
'JU00905K',
'JU00911H',
'JU00912M',
'JU00922M',
'JU00928E',
'JU00933H',
'JU00935H',
'JU00937H',
'JU00939H',
'JU00942M',
'JU00943M',
'JU00946M',
'JU00947E',
'JU00950M',
'JU00970Q',
'JU00993J',
'JU00994J',
'JU00996E',
'JU00997E',
'JU00997M',
'JU00998M',
'JU00999H',
'JU01004H',
'JU01015J',
'JU01016J',
'JU01018E',
'JU01025E',
'JU01033H',
'JU01041K',
'JU01041M',
'JU01053J',
'JU01088K',
'JU01115K',
'JU01117E',
'JU01118E',
'JU01121E',
'JU01122E',
'JU01194J',
'JU01204Q',
'JU01214E',
'JU01215E',
'JU01216D',
'JU01217D',
'JU01219D',
'JU01227J',
'JU01238D',
'JU01239D',
'JU01241D',
'JU01254K',
'JU01255K',
'JU01269M',
'JU01281D',
'JU0255O',
'JU0363M',
'JU0385Y',
'JU0401I',
'JU0402I',
'JU0411M',
'JU0450B',
'JU0499M',
'JU0519M',
'JU0564D',
'JU0567Y',
'JU0598H',
'JU0602I',
'JU0612I',
'JUA0709A',
'JUA0866E',
'JV01611D',
'JV21411M',
'JW00771M',
'JW00814A',
'JW00815A',
'JW00825A',
'JW00826A',
'JW00975M',
'JW00977M',
'JW00979M',
'JW01006M',
'JW01027Q',
'JW01157E',
'JW01171M',
'JW01190Q',
'JW01272Q',
'JX00609K',
'JX00613A',
'JX00614M',
'JX00631A',
'JX00633K',
'JX00634A',
'JX00646A',
'JX00652D',
'JX00654A',
'JX00664J',
'JX00665J',
'JX00674M',
'JX00708K',
'JX00710E',
'JX00710M',
'JX00712A',
'JX00718D',
'JX00721D',
'JX00722D',
'JX00724A',
'JX00725D',
'JX00770A',
'JX0077U',
'JX00783M',
'JX00784D',
'JX00785D',
'JX00786D',
'JX00787D',
'JX00802Q',
'JX00803Q',
'JX00808A',
'JX00808E',
'JX00808M',
'JX00809A',
'JX00810A',
'JX00811A',
'JX00811E',
'JX00811M',
'JX00812A',
'JX00819A',
'JX00820A',
'JX00821A',
'JX00822A',
'JX00823A',
'JX00838A',
'JX00839A',
'JX00839E',
'JX00839K',
'JX00839M',
'JX00847A',
'JX00847E',
'JX00847K',
'JX00847M',
'JX00848A',
'JX00849K',
'JX00849M',
'JX00850A',
'JX00850E',
'JX00851D',
'JX00892K',
'JX00929E',
'JX00932M',
'JX00934M',
'JX00936M',
'JX00938M',
'JX00940M',
'JX00941M',
'JX00944M',
'JX00949E',
'JX00951M',
'JX00958D',
'JX00958Q',
'JX00962T',
'JX00963T',
'JX00967E',
'JX00968M',
'JX0097U',
'JX00981E',
'JX00982E',
'JX00983E',
'JX00984E',
'JX00985E',
'JX00986Q',
'JX00987Q',
'JX00988M',
'JX00990E',
'JX00991J',
'JX00992J',
'JX00995M',
'JX01000M',
'JX01021J',
'JX01022J',
'JX01023E',
'JX01024K',
'JX01028J',
'JX01029J',
'JX01031Q',
'JX01032E',
'JX0103U',
'JX01040M',
'JX01048E',
'JX01050E',
'JX01051M',
'JX01054J',
'JX01062J',
'JX01063E',
'JX01064E',
'JX01066D',
'JX01067D',
'JX01074M',
'JX01078E',
'JX01078K',
'JX01095K',
'JX01096E',
'JX01099M',
'JX01101E',
'JX01102E',
'JX01104D',
'JX01105E',
'JX01116E',
'JX01126K',
'JX01128M',
'JX01131D',
'JX01139E',
'JX01140E',
'JX01141E',
'JX01165E',
'JX01181Q',
'JX01182Q',
'JX01201D',
'JX01202D',
'JX01207E',
'JX01218D',
'JX01224D',
'JX01225D',
'JX01229D',
'JX01230D',
'JX01231D',
'JX01242D',
'JX01243D',
'JX01245K',
'JX01250J',
'JX01252K',
'JX01253K',
'JX01257Q',
'JX01259D',
'JX01260D',
'JX01263D',
'JX01271M',
'JX0305M',
'JX0424H',
'JX0428I',
'JX0442B',
'JX0443I',
'JX0446I',
'JX0447I',
'JX0466N',
'JX0496I',
'JX0516M',
'JX0517M',
'JX0518M',
'JX0540M',
'JX0541M',
'JX0545M',
'JX0565I',
'JX0579M',
'JX0584I',
'JX0594I',
'JX0595I',
'JX0607I',
'JXA0609A',
'JXA0719D',
'JXA260M',
'JXA474M',
'JXB0431M',
'JY00541M',
'JY00589C',
'JY00637M',
'JY00657M',
'JY00678M',
'JY00680M',
'JY00681M',
'JY00682M',
'JY00683M',
'JY00694D',
'JY00704M',
'JY00717M',
'JY00754M',
'JY00755M',
'JY00799M',
'JY00806M',
'JY00807M',
'JY00826M',
'JY00832M',
'JY00978M',
'JY01019Q',
'JY11625M',
'JY8498M',
'JY8521H',
'JY8523M',
'JY8524M',
'JY8553M',
'JY8556M',
'JY8557M',
'JY8559H',
'JY8572M',
'JY8611M',
'JYA0472M',
'JYA0521M',
'JYA0573M',
'JYA0589M',
'JYA0611M',
'JZ01421K',
'JZ1010M',
'JZ1094U',
'JZ1099U',
'JZ11601Y',
'JZ11617A',
'JZ11617K',
'JZ11625A',
'JZ61511M',
'KA01821G',
'KA01926C',
'KA01986C',
'KA02053G',
'KA02054G',
'KA02152G',
'KA02279C',
'KA02349G',
'KA02397S',
'KA02514G',
'KA02536G',
'KA02554G',
'KA02563G',
'KA02574G',
'KA02579G',
'KA02586G',
'KA02676C',
'KA02684G',
'KA02685G',
'KA02690G',
'KA02697S',
'KA02714C',
'KA02715D',
'KA02722C',
'KA02724S',
'KA02725G',
'KA02872S',
'KA02874G',
'KA02919D',
'KA02936G',
'KA03065S',
'KA03131S',
'KA0401S',
'KA1288S',
'KA1482D',
'KAA583G',
'KB02050G',
'KB02069G',
'KB02281C',
'KB02553G',
'KB02693S',
'KV01611D',
'KZ01743D',
'LA01338G',
'LA01539S',
'LA01540S',
'LA01541G',
'LA01542G',
'LA01543G',
'LA01545S',
'LA01591G',
'LA01592S',
'LA01597G',
'LA01598S',
'LA01600S',
'LA01664S',
'LA01665S',
'LA01666S',
'LA01690S',
'LA01691S',
'LA01692S',
'LA01694S',
'LA01728S',
'LA01858S',
'LA01859S',
'LA01878G',
'LA01879G',
'LA01972C',
'LA01985S',
'LA02240G',
'LA02241S',
'LA02348G',
'LA02372S',
'LA02398S',
'LA02410S',
'LA02411G',
'LA02413S',
'LA02433G',
'LA02454G',
'LA02466S',
'LA02467S',
'LA02468S',
'LA02481S',
'LA02488D',
'LA02513S',
'LA02538S',
'LA02555S',
'LA02564S',
'LA02567S',
'LA02582S',
'LA02587S',
'LA02588S',
'LA02596S',
'LA02675S',
'LA02681G',
'LA02682G',
'LA02683G',
'LA02717S',
'LA02733S',
'LA02739S',
'LA02742S',
'LA02743S',
'LA02757S',
'LA02759S',
'LA02779S',
'LA02780S',
'LA02794G',
'LA02796S',
'LA02798G',
'LA02854S',
'LA02855S',
'LA02856S',
'LA02859S',
'LA02860G',
'LA02861G',
'LA02870S',
'LA02871S',
'LA02873G',
'LA02902S',
'LA02903G',
'LA02909D',
'LA02934S',
'LA02935S',
'LA02937G',
'LA02939S',
'LA02989G',
'LA02990S',
'LA02996S',
'LA02997S',
'LA02998S',
'LA02999G',
'LA03002G',
'LA03029G',
'LA03032G',
'LA03047G',
'LA03088S',
'LA03089S',
'LA03090G',
'LA03342S',
'LA03454S',
'LA03455R',
'LA03456G',
'LA0958S',
'LA0960S',
'LA1064S',
'LA1076S',
'LA1078S',
'LA1084G',
'LA1129S',
'LA1188G',
'LA1232S',
'LA1238S',
'LA1468S',
'LA1469S',
'LA1503G',
'LA1506S',
'LAA1338S',
'LAA1467S',
'LAA1601S',
'LAA577S',
'LAA744G',
'LAB0586S',
'LAE138S',
'LB01547G',
'LB01606S',
'LB01608S',
'LB01610G',
'LB01669S',
'LB01670S',
'LB01671S',
'LB01696S',
'LB01730S',
'LB01732S',
'LB01734G',
'LB01735G',
'LB01736S',
'LB01737S',
'LB01801G',
'LB01824S',
'LB01873S',
'LB01916G',
'LB01978D',
'LB02058G',
'LB02059G',
'LB02060G',
'LB02089G',
'LB02102S',
'LB02105G',
'LB02106G',
'LB02122S',
'LB02308S',
'LB02310G',
'LB02385S',
'LB02400S',
'LB02470G',
'LB02515S',
'LB02539S',
'LB02550G',
'LB02551G',
'LB02552G',
'LB02726S',
'LB02744S',
'LB02745G',
'LB02749G',
'LB02758S',
'LB02875G',
'LB02943S',
'LB02944S',
'LB02945S',
'LB02946G',
'LB02947G',
'LB02948S',
'LB03005S',
'LB03029S',
'LB03138G',
'LB03287D',
'LB03457S',
'LB1143G',
'LB1237G',
'LB1241S',
'LB1472G',
'LB1473S',
'LB1474R',
'LB1511S',
'LBA1372S',
'LBB0579S',
'LBE204G',
'LE02741S',
'LE1339S',
'LF01613G',
'LF01729S',
'LF01881G',
'LF01882F',
'LF1533S',
'LI8002P',
'LI8003P',
'LR01745G',
'LR01750G',
'LR02625G',
'LR02626G',
'LR02627G',
'LR02628G',
'LR02763G',
'LT01606G',
'LT02013F',
'LT11607G',
'LT61402G',
'LT8003F',
'LT8004F',
'LT8004P',
'LV01485S',
'LV21401G',
'LV21401S',
'LV21407G',
'LZ01411F',
'LZ01411P',
'LZ01701F',
'LZ11613S',
'LZ61405G',
'LZ61495F',
'LZ61495P',
'MA01579S',
'MA01580S',
'MA01584S',
'MA01587S',
'MA01632G',
'MA01640S',
'MA01662G',
'MA01663S',
'MA01667S',
'MA01689S',
'MA01693S',
'MA01726L',
'MA01727L',
'MA01757L',
'MA01758G',
'MA01773D',
'MA01815L',
'MA01816S',
'MA01818S',
'MA01819S',
'MA01868S',
'MA01869S',
'MA01870S',
'MA01876S',
'MA01877S',
'MA01890S',
'MA01925S',
'MA01930S',
'MA01931S',
'MA01958S',
'MA01959S',
'MA02030S',
'MA02049S',
'MA02055S',
'MA02066G',
'MA02088G',
'MA02107S',
'MA02163S',
'MA02196G',
'MA02238G',
'MA02304G',
'MA02381S',
'MA02412G',
'MA02412S',
'MA02483S',
'MA02573G',
'MA02573S',
'MA02597S',
'MA02669C',
'MA02669D',
'MA02670D',
'MA02671D',
'MA02756D',
'MA02762C',
'MA02781S',
'MA02783G',
'MA02904C',
'MA02905C',
'MA02906D',
'MA02907C',
'MA02908D',
'MA02921C',
'MA02922D',
'MA02991D',
'MA03000G',
'MA03008G',
'MA03010R',
'MA03011R',
'MA03038D',
'MA03045G',
'MA03046S',
'MA03052G',
'MA03077S',
'MA03078S',
'MA03208S',
'MA03249D',
'MA03261G',
'MA03262G',
'MA03263G',
'MA03264G',
'MA03267D',
'MA03268C',
'MA03269L',
'MA03270D',
'MA03282D',
'MA03284D',
'MA03305C',
'MA03306C',
'MA03307C',
'MA03308C',
'MA03378D',
'MA03381D',
'MA03433F',
'MA03433R',
'MA0694G',
'MA0827G',
'MA1343G',
'MA1406L',
'MA1420F',
'MA1466L',
'MA1497L',
'MA1498L',
'MA1500G',
'MA1535D',
'MA1538G',
'MA1552S',
'MAA1548S',
'MAA1551S',
'MAB0340S',
'MAC1264C',
'MAE264C',
'MB01604S',
'MB01738C',
'MB01802D',
'MB01823S',
'MB01875S',
'MB01923G',
'MB01935S',
'MB01936S',
'MB01990C',
'MB02103S',
'MB02198G',
'MB02256G',
'MB02382S',
'MB02617S',
'MB02787S',
'MB02921D',
'MB02922C',
'MB03013R',
'MB03038C',
'MB03048G',
'MB03251D',
'MB03275D',
'MB03276D',
'MB03278D',
'MB03279C',
'MB03286D',
'MB03382D',
'MB03383C',
'MB0697G',
'MB1265L',
'MB1513G',
'MB1514D',
'MB1546S',
'MB1573G',
'MB1573S',
'MB1574G',
'MB1574S',
'MBA1550S',
'MBA1573S',
'ME03015R',
'ME03016R',
'ME03182S',
'MF01759L',
'MK00001J',
'MK00019F',
'MK01650Y',
'MK21401F',
'MK21405P',
'MK21450K',
'MK21501F',
'MK21505P',
'MK21505R',
'MK21511M',
'MK21511Q',
'MK21511R',
'MK21516Q',
'MK21605S',
'MKH1601B',
'MKK1601F',
'MR02688G',
'MR02776G',
'MR02776S',
'MT21405G',
'MT21405S',
'MT21407G',
'MV21410L',
'MV21411G',
'MV61495G',
'MV61495S',
'MZ01401L',
'PA0256V',
'PB01012M',
'PB01056F',
'PE01997W',
'PE01998W',
'PE01999W',
'PF01055F',
'PL00746M',
'PL01059V',
'PL01060M',
'PL01071F',
'PL01072F',
'PL01072M',
'PR01742C',
'PR01743B',
'PR01744B',
'PT01413V',
'PT8439M',
'PT8617M',
'PU00859M',
'PU00881Y',
'PU00903V',
'PU01058V',
'PU01093D',
'PU01112R',
'PU01699W',
'PU01911C',
'PU01992W',
'PU01993W',
'PU01994W',
'PU02016W',
'PV21407V',
'PV21441J',
'PW01447M',
'PW01448M',
'PW01449M',
'PW01745E',
'PW01746E',
'PW01747E',
'PX00858V',
'PX00874D',
'PX00900D',
'PX00901D',
'PX00902V',
'PX00920B',
'PX00925D',
'PX00959V',
'PX00976W',
'PX01008V',
'PX01009K',
'PX01009W',
'PX01010W',
'PX01011J',
'PX01110V',
'PX01111V',
'PX01113D',
'PX01347M',
'PX01348M',
'PX01349W',
'PX01350T',
'PX01692T',
'PX01712C',
'PX01741C',
'PX01761C',
'PX01762C',
'PX01763C',
'PX01780W',
'PX01801W',
'PX01802W',
'PX01803W',
'PX01829W',
'PX01830W',
'PX01892W',
'PX01893W',
'PX01895C',
'PX01896C',
'PX01897C',
'PX02018W',
'PX02020C',
'PX02021C',
'PX02080W',
'PX02081W',
'PX02084W',
'PX0542A',
'PX0565J',
'PX0579Y',
'PX0580Y',
'PX0581Y',
'PX0593Y',
'PX0686M',
'PX0693M',
'PX0721J',
'PX0786J',
'PXA0816J',
'PY01014M',
'PY01497M',
'PY01498M',
'PY01581E',
'PY01725E',
'PY01726E',
'PY01727E',
'PY8768M',
'PYA0789M',
'PZ01700Z',
'QA00840Q',
'QA00841J',
'QA00845S',
'QA00859S',
'QA00866J',
'QA00868L',
'QA00868Q',
'QA00872Q',
'QA0088Q',
'QA0184X',
'QA0280L',
'QA0455Q',
'QA0496V',
'QA0499S',
'QA0533S',
'QA0548S',
'QA0550G',
'QA0569O',
'QA0570L',
'QA0608S',
'QA0681Q',
'QA0686I',
'QA0687I',
'QA0691G',
'QA0714R',
'QA0737L',
'QA0791S',
'QA0792S',
'QAA0571X',
'QAA0587P',
'QAA0691G',
'QAA0691H',
'QAA0806J',
'QAA263J',
'QAA397G',
'QAA397H',
'QAA485L',
'QAA489R',
'QAA500L',
'QAA569Q',
'QAA570Q',
'QAB178R',
'QB0058J',
'QB01130R',
'QB01702V',
'QB01703R',
'QB01704R',
'QB0193X',
'QB0264J',
'QB0285J',
'QB0568L',
'QB0735J',
'QBA508G',
'QBA508H',
'QBB0508H',
'QBB0509P',
'QE00846R',
'QE00853X',
'QE00854X',
'QE00865R',
'QE00933F',
'QE01021F',
'QE01025S',
'QE01101V',
'QE01104F',
'QE01126R',
'QE01138R',
'QE01139R',
'QE01141R',
'QE01142R',
'QE01156S',
'QE01163S',
'QE01189R',
'QE01197R',
'QE01371R',
'QE01372R',
'QE01373R',
'QE01374R',
'QE01375R',
'QE01376R',
'QE01406R',
'QE01407R',
'QE01408R',
'QE01409R',
'QE01416R',
'QE01417R',
'QE01418R',
'QE01419R',
'QE01426R',
'QE01427R',
'QE01428R',
'QE01433R',
'QE01667R',
'QE01715W',
'QE01716W',
'QE01824R',
'QE01825R',
'QE01826R',
'QE01834R',
'QE01835R',
'QE01836R',
'QE01837R',
'QE01930R',
'QE01930W',
'QE0265W',
'QE0435H',
'QG01665R',
'QG01738R',
'QG01739R',
'QG01831R',
'QG01832R',
'QG01833R',
'QH01666R',
'QH01684R',
'QH01693R',
'QI0921U',
'QI0924Q',
'QI1018X',
'QJ0919U',
'QJ0922U',
'QJ0939J',
'QJ1021X',
'QN0566H',
'QN0705X',
'QN0726S',
'QNA486X',
'QR01681B',
'QR01732B',
'QR01779V',
'QR02037R',
'QR02038V',
'QR02039J',
'QR02134M',
'QR02135C',
'QR02136V',
'QR8642R',
'QRA347R',
'QT01415H',
'QT01416H',
'QT01416V',
'QT01417R',
'QT01421V',
'QT0272W',
'QT11411R',
'QT11411S',
'QT11417X',
'QT21491V',
'QT61483R',
'QT8010W',
'QT8013J',
'QT8239R',
'QT8504S',
'QT8602S',
'QT8617S',
'QT8649R',
'QU00834X',
'QU00836V',
'QU00865S',
'QU00873V',
'QU00883J',
'QU00921P',
'QU00921V',
'QU00922R',
'QU00931V',
'QU00932P',
'QU00932V',
'QU00947P',
'QU00948G',
'QU00948H',
'QU00949F',
'QU00949U',
'QU00950P',
'QU00971F',
'QU00972F',
'QU00972U',
'QU00973R',
'QU00989R',
'QU00989S',
'QU01007Q',
'QU01018J',
'QU01019G',
'QU01019H',
'QU01020G',
'QU01020H',
'QU01029G',
'QU01050K',
'QU01051K',
'QU01069F',
'QU01076K',
'QU01077R',
'QU01088V',
'QU01097V',
'QU01106K',
'QU01123V',
'QU01124V',
'QU01125K',
'QU01128R',
'QU01135K',
'QU01136R',
'QU01152R',
'QU01154K',
'QU01158L',
'QU01159V',
'QU01194K',
'QU01195K',
'QU01196K',
'QU01208R',
'QU01224V',
'QU01234V',
'QU01243R',
'QU01248K',
'QU01256V',
'QU01257V',
'QU01268B',
'QU01269V',
'QU01270V',
'QU01272K',
'QU01274K',
'QU01280J',
'QU01283K',
'QU01294Q',
'QU01297H',
'QU01301Q',
'QU01303R',
'QU01308K',
'QU01309K',
'QU01311V',
'QU01313V',
'QU01319H',
'QU01328R',
'QU01329Q',
'QU01330J',
'QU01341H',
'QU01342H',
'QU01343H',
'QU01352H',
'QU01384R',
'QU01385R',
'QU01386R',
'QU01387R',
'QU01388R',
'QU01389R',
'QU01390R',
'QU01421J',
'QU01422B',
'QU01451K',
'QU01457R',
'QU01459R',
'QU01460R',
'QU01464V',
'QU01474B',
'QU01481B',
'QU01489V',
'QU01496B',
'QU01499K',
'QU01499V',
'QU01504B',
'QU01513B',
'QU01517K',
'QU01518V',
'QU01519V',
'QU01525F',
'QU01526F',
'QU01533K',
'QU01535R',
'QU01536R',
'QU01546K',
'QU01561V',
'QU01562V',
'QU01563V',
'QU01565R',
'QU01567R',
'QU01570B',
'QU01576R',
'QU01599V',
'QU01616K',
'QU01625W',
'QU01632W',
'QU01635V',
'QU01641W',
'QU01643B',
'QU01644B',
'QU01650V',
'QU01652B',
'QU01653B',
'QU01657W',
'QU01658W',
'QU01672V',
'QU01720V',
'QU01721V',
'QU01728V',
'QU01729V',
'QU01730V',
'QU01731W',
'QU01771V',
'QU01772V',
'QU01773W',
'QU01888F',
'QU01890F',
'QU01906F',
'QU01908F',
'QU01954W',
'QU01955W',
'QU01956W',
'QU01979W',
'QU01980W',
'QU01981W',
'QU02007V',
'QU02008V',
'QU02009V',
'QU02029W',
'QU02033W',
'QU02056B',
'QU02057B',
'QU02061R',
'QU02062R',
'QU02127V',
'QU02128V',
'QU02129V',
'QU02130K',
'QU02130V',
'QU0298J',
'QU0566G',
'QU0576S',
'QU0656X',
'QU0794S',
'QU0836X',
'QUA0611X',
'QUA0616P',
'QUA0619L',
'QUA0678G',
'QUA0743V',
'QUA1319K',
'QUA480X',
'QUA554P',
'QUA634L',
'QV01401H',
'QV01421R',
'QV01429J',
'QV11415B',
'QV21413R',
'QV21413V',
'QV21417H',
'QW01209R',
'QW01210R',
'QW01211R',
'QW01332R',
'QW01333R',
'QW01334R',
'QW01364K',
'QW01365K',
'QW01366K',
'QW01378R',
'QW01379R',
'QW01380R',
'QW01381R',
'QW01411R',
'QW01412R',
'QW01413R',
'QW01414R',
'QW01430R',
'QW01431R',
'QW01435K',
'QW01437K',
'QW01485R',
'QW01502R',
'QW01503R',
'QW01506K',
'QW01507K',
'QW01539R',
'QW01556R',
'QW01559R',
'QW01821W',
'QW01838F',
'QW01839R',
'QW01840R',
'QW01841R',
'QW02105W',
'QW02107W',
'QW02108W',
'QX00829X',
'QX00830Q',
'QX00871P',
'QX00871V',
'QX00884B',
'QX00886X',
'QX00911V',
'QX00912C',
'QX00912T',
'QX00913P',
'QX00914P',
'QX00915V',
'QX00916S',
'QX00917R',
'QX00926R',
'QX00927C',
'QX00927T',
'QX00928P',
'QX00928V',
'QX00929P',
'QX00929V',
'QX00933U',
'QX00936R',
'QX00938V',
'QX00941P',
'QX00941V',
'QX00942P',
'QX00942V',
'QX00943G',
'QX00943H',
'QX00944G',
'QX00945G',
'QX00945H',
'QX00951P',
'QX00952P',
'QX00953G',
'QX00954G',
'QX00954H',
'QX00956P',
'QX00956V',
'QX00961P',
'QX00961V',
'QX00962P',
'QX00963C',
'QX00963T',
'QX00966F',
'QX00977H',
'QX00977V',
'QX00978H',
'QX00978V',
'QX00979H',
'QX00979V',
'QX00991G',
'QX01017S',
'QX01031R',
'QX01032H',
'QX01034V',
'QX01038F',
'QX01039F',
'QX01043Z',
'QX01044Z',
'QX01061Y',
'QX01086R',
'QX01094Y',
'QX01095R',
'QX01105V',
'QX01115Y',
'QX01116V',
'QX01117V',
'QX01118W',
'QX01119R',
'QX01121V',
'QX01122J',
'QX01132W',
'QX01133R',
'QX01144R',
'QX01145R',
'QX01146R',
'QX01147R',
'QX01148R',
'QX01149R',
'QX01150R',
'QX01151V',
'QX01156R',
'QX01174R',
'QX01179R',
'QX01180R',
'QX01181R',
'QX01183R',
'QX01184R',
'QX01212K',
'QX01213V',
'QX01214V',
'QX01215R',
'QX01216R',
'QX01219V',
'QX01221R',
'QX01222R',
'QX01223R',
'QX01225K',
'QX01226K',
'QX01229V',
'QX01230R',
'QX01231B',
'QX01231V',
'QX01232V',
'QX01235V',
'QX01236V',
'QX01238R',
'QX01239R',
'QX01240R',
'QX01252V',
'QX01253V',
'QX01254K',
'QX01254V',
'QX01258R',
'QX01259R',
'QX01262V',
'QX01264V',
'QX01266B',
'QX01276R',
'QX01277R',
'QX01278C',
'QX01279J',
'QX01281K',
'QX01282K',
'QX01289R',
'QX01290R',
'QX01291R',
'QX01292R',
'QX01293R',
'QX01296V',
'QX01298R',
'QX01299R',
'QX01304J',
'QX01305J',
'QX01307W',
'QX01310V',
'QX01314R',
'QX01315B',
'QX01315C',
'QX01316W',
'QX01320V',
'QX01326R',
'QX01336H',
'QX01337H',
'QX01338H',
'QX01339V',
'QX01340V',
'QX01344R',
'QX01345R',
'QX01346R',
'QX01351H',
'QX01353W',
'QX01354V',
'QX01355V',
'QX01356V',
'QX01363B',
'QX01363Q',
'QX01383R',
'QX01398V',
'QX01399V',
'QX01400V',
'QX01401V',
'QX01404R',
'QX01405R',
'QX01420J',
'QX01441V',
'QX01442V',
'QX01443V',
'QX01445C',
'QX01446B',
'QX01452V',
'QX01455R',
'QX01456R',
'QX01461C',
'QX01462B',
'QX01463V',
'QX01473B',
'QX01475R',
'QX01479B',
'QX01480B',
'QX01483R',
'QX01490R',
'QX01491R',
'QX01492R',
'QX01495B',
'QX01501V',
'QX01508K',
'QX01511B',
'QX01512B',
'QX01516K',
'QX01521B',
'QX01522B',
'QX01523B',
'QX01527B',
'QX01528K',
'QX01529K',
'QX01530K',
'QX01537R',
'QX01538R',
'QX01540R',
'QX01541R',
'QX01543R',
'QX01544B',
'QX01545K',
'QX01547V',
'QX01548K',
'QX01549K',
'QX01550V',
'QX01551R',
'QX01554V',
'QX01555R',
'QX01557B',
'QX01558J',
'QX01566R',
'QX01568R',
'QX01573V',
'QX01574K',
'QX01577B',
'QX01578B',
'QX01595K',
'QX01596K',
'QX01597K',
'QX01598V',
'QX01603K',
'QX01618F',
'QX01624K',
'QX01636F',
'QX01640B',
'QX01645V',
'QX01647B',
'QX01648B',
'QX01649B',
'QX01655R',
'QX01695R',
'QX01696R',
'QX01717V',
'QX01718F',
'QX01719F',
'QX01722F',
'QX01723V',
'QX01756W',
'QX01757W',
'QX01769F',
'QX01769V',
'QX01770V',
'QX01777R',
'QX01778R',
'QX01788B',
'QX01791B',
'QX01810F',
'QX01811F',
'QX01812F',
'QX01813F',
'QX01814F',
'QX01815F',
'QX01823V',
'QX01846F',
'QX01871F',
'QX01872F',
'QX01873F',
'QX01914V',
'QX01915V',
'QX01926B',
'QX01927B',
'QX01928B',
'QX01932W',
'QX01933W',
'QX01934W',
'QX01937W',
'QX01938W',
'QX01939W',
'QX01940W',
'QX01941W',
'QX01942W',
'QX01943C',
'QX01944C',
'QX01946C',
'QX01947B',
'QX01948B',
'QX01949F',
'QX01950F',
'QX01951F',
'QX01960F',
'QX01961F',
'QX01964F',
'QX01965F',
'QX01977K',
'QX01978K',
'QX01987W',
'QX01988W',
'QX01990W',
'QX01991W',
'QX0199J',
'QX02003W',
'QX02004K',
'QX02004W',
'QX02005K',
'QX02005W',
'QX02010V',
'QX02011V',
'QX02022V',
'QX02023V',
'QX02024B',
'QX02026V',
'QX02027B',
'QX02043K',
'QX02044K',
'QX02048J',
'QX02054B',
'QX02055B',
'QX02058K',
'QX02059R',
'QX02060R',
'QX02063V',
'QX02064V',
'QX02072J',
'QX02073J',
'QX02074B',
'QX02075B',
'QX02076K',
'QX02077K',
'QX02078K',
'QX02086W',
'QX02087W',
'QX02088W',
'QX02094R',
'QX02095V',
'QX02096R',
'QX02097R',
'QX02098W',
'QX02099W',
'QX02123V',
'QX02124V',
'QX02125V',
'QX02126V',
'QX02133V',
'QX0219Q',
'QX0243G',
'QX0381J',
'QX0529G',
'QX0553S',
'QX0573R',
'QX0574S',
'QX0591S',
'QX0596Q',
'QX0610P',
'QX0629Q',
'QX0664R',
'QX0665S',
'QX0668V',
'QX0690S',
'QX0702O',
'QX0702X',
'QX0711R',
'QX0711S',
'QX0715J',
'QX0725R',
'QX0766X',
'QX0769H',
'QX0785J',
'QX0815L',
'QX0815Q',
'QX0821J',
'QX0823S',
'QX0831V',
'QX0832P',
'QX0832V',
'QX0833X',
'QXA0563S',
'QXA0610P',
'QXA0610V',
'QXA0629L',
'QXA0629Q',
'QXA0655S',
'QXA0671R',
'QXA0711S',
'QXA0723V',
'QXA0766X',
'QXA0769H',
'QXA1036F',
'QXA1062V',
'QXA349J',
'QXA468O',
'QXA654G',
'QXB0243G',
'QXB0654G',
'QY0005Q',
'QY0006U',
'QY00636S',
'QY00796R',
'QY00848R',
'QY00850R',
'QY00893R',
'QY00893S',
'QY00894F',
'QY00894U',
'QY00896R',
'QY00981S',
'QY00983S',
'QY00988M',
'QY00988N',
'QY00993R',
'QY00993S',
'QY00994R',
'QY00994S',
'QY00996R',
'QY00996S',
'QY00997R',
'QY00997S',
'QY01000R',
'QY01000S',
'QY01003R',
'QY01005R',
'QY01005S',
'QY01022F',
'QY01022S',
'QY01023S',
'QY01027S',
'QY01028R',
'QY01054R',
'QY01063V',
'QY01065V',
'QY01067R',
'QY01098V',
'QY01099V',
'QY01155R',
'QY01169R',
'QY01172R',
'QY01175R',
'QY01176R',
'QY01369R',
'QY01391R',
'QY01392R',
'QY01393R',
'QY01394R',
'QY01395R',
'QY01396R',
'QY01397R',
'QY01423R',
'QY01424R',
'QY01425R',
'QY01472R',
'QY01493R',
'QY01842R',
'QY01843R',
'QY01844R',
'QY01874R',
'QY01921W',
'QY01957W',
'QY01958W',
'QY01959W',
'QY01982W',
'QY01983W',
'QY01984W',
'QY01985W',
'QY01986W',
'QY02028W',
'QY02030W',
'QY02031W',
'QY02110R',
'QY1013X',
'QY8805S',
'QYA0518R',
'QYA0518S',
'QYA0776S',
'QYA0784R',
'QYA0982R',
'QYA1168R',
'QZ01401W',
'QZ01650V',
'QZ1005U',
'QZ1007U',
'QZ1007X',
'QZ11607R',
'QZ11611V',
'QZ11617B',
'QZ11617R',
'QZ11621Q',
'RA03033S',
'RA03266D',
'RA03346R',
'RB03277D',
'TA0006B',
'TA00163F',
'TA00164F',
'TA00167F',
'TA00168F',
'TA00169F',
'TA00182F',
'TA00184F',
'TA00189F',
'TA0020D',
'TA00304F',
'TA00317F',
'TA00325F',
'TA00327F',
'TA00338F',
'TA00339F',
'TA00348F',
'TA00358F',
'TA00361F',
'TA00362F',
'TA00370F',
'TA00378F',
'TA00381F',
'TA00405F',
'TA0051R',
'TA0054R',
'TA0075D',
'TA0088D',
'TA0091D',
'TA0093F',
'TA0095F',
'TA0099D',
'TA0101D',
'TA01046P',
'TA01047C',
'TA01048C',
'TA01049F',
'TA01067P',
'TA01068T',
'TA01069T',
'TA01071T',
'TA01077T',
'TA01081T',
'TA01082T',
'TA01084T',
'TA01085T',
'TA01086T',
'TA01089T',
'TA01090T',
'TA01091T',
'TA01092P',
'TA01093P',
'TA01101T',
'TA01102T',
'TA01109T',
'TA01114P',
'TA01115P',
'TA01124T',
'TA01125T',
'TA01126P',
'TA01135J',
'TA01136J',
'TA01137J',
'TA01151T',
'TA01159T',
'TA01160T',
'TA01161T',
'TA01171T',
'TA01186T',
'TA01187T',
'TA01188T',
'TA01189T',
'TA01192T',
'TA01196T',
'TA0127D',
'TA0132F',
'TA0133F',
'TA0146F',
'TA0841D',
'TA0842D',
'TA0845D',
'TAA0140F',
'TAA0160F',
'TAA0172F',
'TAA0173F',
'TAA0324D',
'TAA110F',
'TAA118F',
'TAB0142F',
'TAB111F',
'TAZ0414F',
'TAZ0415F',
'TAZ0416F',
'TAZ0418F',
'TAZ0429F',
'TAZ0430F',
'TAZ0439F',
'TAZ0440F',
'TAZ0445F',
'TAZ0446F',
'TAZ0447F',
'TAZ0460F',
'TAZ0463F',
'TAZ0487F',
'TAZ0493F',
'TAZ0496F',
'TAZ0499F',
'TAZ0500F',
'TAZ0501F',
'TAZ0502F',
'TAZ0506F',
'TAZ0507F',
'TAZ0510F',
'TAZ0512F',
'TAZ0513F',
'TAZ0516F',
'TAZ0517F',
'TAZ0519F',
'TAZ0544F',
'TAZ0545F',
'TAZ0548D',
'TAZ0555F',
'TAZ0557F',
'TAZ0559F',
'TAZ0566F',
'TAZ0573F',
'TAZ0576F',
'TAZ0577R',
'TAZ0600F',
'TAZ0601F',
'TAZ0602F',
'TAZ0620D',
'TAZ0621F',
'TAZ0632F',
'TAZ0691S',
'TAZ0692S',
'TAZ0743S',
'TAZ0802D',
'TAZ0829D',
'TAZ0830D',
'TAZ9192F',
'TAZ9332D',
'TAZ9408F',
'TAZ9414F',
'TAZ9415F',
'TAZ9430F',
'TAZ9439F',
'TAZ9445F',
'TAZ9545D',
'TAZ9545F',
'TAZ9587D',
'TAZ9600D',
'TAZ9601D',
'TAZ9603D',
'TAZ9620D',
'TAZ9621D',
'TAZ9642D',
'TAZ9704D',
'TB0001D',
'TB00171F',
'TB00333F',
'TB00337F',
'TB00340F',
'TB00359F',
'TB0063R',
'TB01050T',
'TB01051T',
'TB01052T',
'TB01053T',
'TB0105D',
'TB01070T',
'TB01078T',
'TB01088T',
'TB01103T',
'TB01110T',
'TB01111T',
'TB01112T',
'TB01113T',
'TB01147T',
'TB01148T',
'TB01154T',
'TB01155T',
'TB01156T',
'TB01162T',
'TB01163T',
'TB01193T',
'TB01194T',
'TB01197T',
'TB01198T',
'TB01199T',
'TB01200T',
'TB01201T',
'TB01202T',
'TB0150F',
'TB0833D',
'TB0838D',
'TB0847D',
'TB0848D',
'TBA0121F',
'TBA0193F',
'TBA0302F',
'TBY9328F',
'TBZ0409F',
'TBZ0432F',
'TBZ0433F',
'TBZ0434F',
'TBZ0437F',
'TBZ0455F',
'TBZ0458F',
'TBZ0459F',
'TBZ0508F',
'TBZ0556F',
'TBZ0565F',
'TBZ0578F',
'TBZ0604F',
'TBZ0605F',
'TBZ0607F',
'TBZ0690S',
'TBZ0825D',
'TBZ9377F',
'TBZ9547F',
'TBZ9549F',
'TBZ9604D',
'TBZ9608D',
'TBZ9625D',
'TBZ9626D',
'TBZ9643D',
'TBZ9644D',
'TF0062D',
'TJ0008D',
'TN00380F',
'TNZ0682D',
'TNZ0683D',
'TNZ0684D',
'TNZ0685D',
'TNZ0726D',
'TNZ0727D',
'TNZ0730D',
'TNZ0732D',
'TNZ0734D',
'TNZ0736D',
'TT01450D',
'TT01643D',
'TT09005D',
'TV09007B',
'TV21447D',
'UA00079T',
'UA00090T',
'UA00091T',
'UA00094T',
'UA00095T',
'UA00098T',
'UA00101P',
'UA00102P',
'UA00105P',
'UA00106P',
'UA00108T',
'UA00137D',
'UA00140D',
'UA00142D',
'UA00155D',
'UA0016Q',
'UA00308D',
'UA00365D',
'UA00366D',
'UA00384D',
'UA00398D',
'UA00399D',
'UA00403D',
'UA0044A',
'UA0057P',
'UA0066P',
'UA0071T',
'UA0077P',
'UA0086D',
'UA0096F',
'UA0099F',
'UA01002T',
'UA01003T',
'UA01004T',
'UA01007P',
'UA01009P',
'UA01010P',
'UA01011P',
'UA01014P',
'UA01015T',
'UA01016T',
'UA01017P',
'UA01018P',
'UA01019T',
'UA01020T',
'UA01021T',
'UA01022P',
'UA01023T',
'UA01024P',
'UA01025P',
'UA01026T',
'UA01027T',
'UA01028P',
'UA01029P',
'UA01030P',
'UA01031T',
'UA01032P',
'UA01033P',
'UA01034T',
'UA01035P',
'UA01036P',
'UA01037P',
'UA01039P',
'UA01040P',
'UA01041T',
'UA01042P',
'UA01058P',
'UA01059P',
'UA01060P',
'UA01061P',
'UA01062T',
'UA01063T',
'UA01064P',
'UA01065P',
'UA01072P',
'UA01073T',
'UA01075T',
'UA01079T',
'UA01080P',
'UA01083T',
'UA01087T',
'UA01094P',
'UA01095P',
'UA01099P',
'UA01100P',
'UA01104T',
'UA01105P',
'UA01106P',
'UA01107P',
'UA01108P',
'UA01116P',
'UA01120P',
'UA01121T',
'UA01122T',
'UA01123T',
'UA01127P',
'UA01128P',
'UA01129P',
'UA01130T',
'UA01131T',
'UA01132T',
'UA01133T',
'UA01134P',
'UA01138P',
'UA01139P',
'UA01143T',
'UA01149T',
'UA01150T',
'UA01164P',
'UA01165P',
'UA01166P',
'UA01167P',
'UA01168P',
'UA01169P',
'UA01170P',
'UA01172T',
'UA01173T',
'UA01174T',
'UA01184T',
'UA01185T',
'UA01190T',
'UA01191T',
'UA0130D',
'UA05000D',
'UA0832D',
'UA0840D',
'UA0843D',
'UA0844D',
'UAA0078T',
'UAA0162D',
'UAA087D',
'UAC0112D',
'UAY9331D',
'UAY9385D',
'UAZ0335D',
'UAZ0442D',
'UAZ0448D',
'UAZ0449D',
'UAZ0450D',
'UAZ0451D',
'UAZ0469D',
'UAZ0470D',
'UAZ0475D',
'UAZ0476D',
'UAZ0477D',
'UAZ0478D',
'UAZ0479D',
'UAZ0481D',
'UAZ0482D',
'UAZ0488D',
'UAZ0492D',
'UAZ0494D',
'UAZ0523D',
'UAZ0524D',
'UAZ0525D',
'UAZ0526D',
'UAZ0543D',
'UAZ0551D',
'UAZ0569D',
'UAZ0571D',
'UAZ0579D',
'UAZ0580D',
'UAZ0581D',
'UAZ0588D',
'UAZ0589D',
'UAZ0590D',
'UAZ0593D',
'UAZ0594D',
'UAZ0597D',
'UAZ0636D',
'UAZ0806D',
'UAZ0831D',
'UAZ8388D',
'UAZ8634D',
'UAZ9350D',
'UAZ9388D',
'UAZ9389D',
'UAZ9435D',
'UAZ9569D',
'UAZ9571D',
'UAZ9619D',
'UAZ9634D',
'UAZ9745D',
'UB00092T',
'UB00093T',
'UB00097T',
'UB00107T',
'UB00109T',
'UB00160D',
'UB00305D',
'UB00345D',
'UB00367D',
'UB00369D',
'UB00390D',
'UB00394D',
'UB0061P',
'UB0062P',
'UB0064P',
'UB0105F',
'UB01074P',
'UB01076T',
'UB0110D',
'UB01117P',
'UB01118T',
'UB01119T',
'UB01140P',
'UB01141P',
'UB01152T',
'UB01153T',
'UB01157T',
'UB01158T',
'UB01175P',
'UB01176P',
'UB01177P',
'UB01178P',
'UB01179P',
'UB01180P',
'UB01181T',
'UB01182T',
'UB01183T',
'UB01195T',
'UB01203T',
'UB01204T',
'UB01205T',
'UB0834D',
'UB0835D',
'UB0836D',
'UB0837D',
'UB0839D',
'UB0849D',
'UBA0129D',
'UBA0152D',
'UBY9322D',
'UBZ0406D',
'UBZ0413D',
'UBZ0421D',
'UBZ0422D',
'UBZ0424D',
'UBZ0426D',
'UBZ0465D',
'UBZ0483D',
'UBZ0486D',
'UBZ0491D',
'UBZ0554D',
'UBZ0563D',
'UBZ0584D',
'UBZ0595D',
'UBZ0596D',
'UBZ0599D',
'UBZ0801D',
'UBZ0826D',
'UBZ0827D',
'UBZ8406D',
'UBZ9406D',
'UBZ9413D',
'UBZ9436D',
'UBZ9486D',
'UBZ9527D',
'UBZ9538D',
'UBZ9570D',
'UBZ9572D',
'UBZ9582D',
'UBZ9595D',
'UBZ9618D',
'UBZ9638D',
'UBZ9640D',
'UBZ9733D',
'UBZ9760D',
'UF0048D',
'UI0909E',
'UI0916E',
'UN00115F',
'UN00116F',
'UN00153F',
'UN00313F',
'UN00353F',
'UN00354F',
'UN00393D',
'UN00401D',
'UN0074E',
'UNZ0407D',
'UNZ0598D',
'UNZ0737D',
'UNZ0738D',
'UR01054T',
'UR01055T',
'UR01206T',
'UT01457D',
'UT1100D',
'UT8002D',
'UT8009D',
'UT8010D',
'UV01451Z',
'UV01460D',
'UY1003Z',
'UZ1005Z',
'UZ9900Z',
'UZ9901Z',
'UZ9903Z',
'VA01892R',
'VA01893R',
'VA01894G',
'VA01896R',
'VA01897R',
'VA01938T',
'VA01991F',
'VA01992F',
'VA03063F',
'VA03074F',
'VA03225F',
'VA03226F',
'VA03241F',
'VA03242F',
'VA03295F',
'VA03296F',
'VA03419F',
'VA03420F',
'VA03421F',
'VB01902G',
'VB02099F',
'VB02114R',
'VB03230F',
'VB03245F',
'VB03246F',
'WA01851S',
'WA01852G',
'WA01973D',
'WA02027S',
'WA02048G',
'WA02074T',
'WA02081G',
'WA02087G',
'WA02104S',
'WA02164G',
'WA02165G',
'WA02177G',
'WA02180S',
'WA02496G',
'WA03050G',
'WA03066G',
'WA03067G',
'WA03206G',
'WA03207G',
'WA03214G',
'WA03215F',
'WA03347G',
'WA03348G',
'WA03349G',
'WA03393G',
'WB01918G',
'WB01919G',
'WB02051S',
'WB02056G',
'WB02057G',
'WB02067G',
'WB02068G',
'WB02184G',
'WB03069G',
'WB03070G',
'WB03217F',
'WB03395G',
'WB03396G',
'XT01745X',
'Y0026Z',
'Y1028Z',
'Y1034Z',
'Y1038Z',
'Y1042Z',
'Y1074Z',
'Y1076Z',
'Y1078Z',
'Y1083Z',
'YA00071Z',
'YA00072Z',
'YA01445Z',
'YB00070Z',
'Z0022Z',
'Z6006Z',
'ZA01445Z',
'AAAX999B1223',
'BAAX999B1223',
'EAAX999B1223',
'AABC064E032',
'AABC064G032',
'AABC065E032',
'AABC065F032',
'BABC066F032',
'BABC066J032',
'BABC067G032',
'BABC067J032',
'AABC126A1222',
'AABC126D1222',
'AABC126G1222',
'BABC130A1122',
'BABC130C1122',
'BABC130D1122',
'AABC134F1222',
'AABC134G1222',
'BABC135A1222',
'BABC135G1222',
'AABC141D1222',
'AABC141J1222',
'AABC141K1222',
'AABC141P1222',
'AABC195F0123',
'AABC195J0123',
'AABC266E0323',
'AABC266F0323',
'AABC268D0223',
'AABC268J0223',
'AABC280C0323',
'AABC280F0323',
'AABX999B1223',
'BABX999B1223',
'EABX999B1223',
'AACC199F0223',
'AACC199P0223',
'BACC199H0223',
'BACC199P0223',
'AACX999B1223',
'BACX999B1223',
'EACX999B1223',
'BADC169F0223',
'BADC169G0223',
'BADC169H0223',
'BADC169K0223',
'AADC181A0123',
'AADC181K0123',
'BADC205C0123',
'BADC205H0123',
'BADC205J0123',
'AADC257F0123',
'AADC257H0123',
'AADX999B1223',
'BADX999B1223',
'EADX999B1223',
'AAFC226A1122',
'AAFC226D1122',
'AAFC226J1122',
'AAFC226K1122',
'BAFC240I0223',
'BAFC240J0223',
'BAFC240P0223',
'AAFX999B1223',
'BAFX999B1223',
'EAFX999B1223',
'BAHC083H032',
'BAHC083J052',
'AAHC175C0822',
'AAHC175K0922',
'AAHC216A1022',
'AAHC216D1022',
'AAHC216F1022',
'AAHC216J1022',
'AAHC216K1022',
'AAHC216P1022',
'BAHC217A1022',
'BAHC217D1022',
'BAHC217F1022',
'BAHC217J1022',
'BAHC217K1022',
'BAHC217P1022',
'AAHX999B1223',
'BAHX999B1223',
'EAHX999B1223',
'AAJC119A1022',
'AAJC119G1022',
'AAJC133E1222',
'AAJC133F1222',
'AAJX999B1223',
'BAJX999B1223',
'EAJX999B1223',
'AAMC186D0822',
'AAMC186P0922',
'AASC181A0822',
'AASC181H0822',
'AAUC106G0522',
'AAUC106H0522',
'BAUC107A0522',
'BAUC107G0522',
'BAUC107H0522',
'AAUX999B1223',
'BAUX999B1223',
'EAUX999B1223',
'AAVX999B1223',
'BAVX999B1223',
'EAVX999B1223',
'AAWC116A1022',
'AAWC116F1022',
'AAWC116G1022',
'AAWC116H1022',
'AAWC122E1122',
'AAWC122F1122',
'AAWC122H1122',
'BAWC141F0123',
'BAWC141G0123',
'AAWD109A0922',
'AAWD109F0922',
'AAWD109H0922',
'BAWD110A0922',
'BAWD110F0922',
'BAWD110H0922',
'AAWD113A0822',
'AAWD113A1222',
'AAWD113C0822',
'AAWD113C1222',
'AAWD113F0822',
'AAWD113F1222',
'AAWD113H0822',
'BAWD123E1122',
'BAWD123F1122',
'BAWD128F1222',
'BAWD128G1222',
'BAWD128H1222',
'BAWD130A1222',
'BAWD130G1222',
'BAWD130J1222',
'BAWD131F1222',
'BAWD131H1222',
'BAWD141A0123',
'BAWD141C0123',
'BAWD141H0123',
'BAWD141J0123',
'AAWD146A0323',
'AAWD146F0323',
'AAWD146G0323',
'AAWD146H0323',
'AAWX999B1223',
'BAWX999B1223',
'EAWX999B1223',
'AAXC111F0822',
'AAXC111H0822',
'BAXC112F0822',
'BAXC112H0822',
'AAXC117A1022',
'AAXC117F1022',
'AAXC117G1022',
'AAXC117H1022',
'BAXC125A1122',
'BAXC125F1122',
'BAXC125H1122',
'BAXC132C1222',
'BAXC132F1222',
'BAXC132H1222',
'BAXC132J1222',
'AAXC140A0123',
'AAXC140C0123',
'AAXX999B1223',
'BAXX999B1223',
'EAXX999B1223',
'AAZC120A1022',
'AAZC120H1022',
'BAZC121G1022',
'BAZC121H1022',
'AAZC134F1222',
'AAZC134H1222',
'BAZC135A1222',
'BAZC135E1222',
'AAZC144H0223',
'AAZC144J0223',
'BAZC145H0223',
'BAZC145J0223',
'ABAC100A0822',
'ABAC100H0722',
'ABAC127A1222',
'ABAC127C1222',
'ABAC127F1222',
'BBAC131C1122',
'BBAC131H1122',
'ABAC136C1222',
'ABAC136I1222',
'BBAC138C1222',
'BBAC138I1222',
'ABAC189E0223',
'ABAC189F0223',
'ABAC197C0123',
'ABAC197H0123',
'ABAC267C0323',
'ABAC267J0323',
'ABAX999B1223',
'BBAX999B1223',
'EBAX999B1223',
'ABBX999B1223',
'BBBX999B1223',
'EBBX999B1223',
'ABCC180C0123',
'ABCC180H0123',
'ABCC198A0123',
'ABCC198C0123',
'BBCC259A0123',
'BBCC259C0123',
'BBCC271A0223',
'BBCC271C0223',
'ABCX999B1223',
'BBCX999B1223',
'EBCX999B1223',
'BBDC168C0223',
'BBDC168J0223',
'BBDC168K0223',
'BBDC168P0223',
'ABDX999B1223',
'BBDX999B1223',
'EBDX999B1223',
'ABFX999B1223',
'BBFX999B1223',
'EBFX999B1223',
'BBHC215A0922',
'BBHC215C0922',
'BBHC215H0922',
'BBHC215J0922',
'BBHC215K0922',
'ABHC234L0123',
'ABHC234L0423',
'ABHC234L1023',
'HBHG331A1223',
'HBHG331K1223',
'HBHG332A1223',
'HBHG332K1223',
'HBHG333A1223',
'HBHG333K1223',
'HBHG334A1223',
'HBHG334K1223',
'HBHG335A1223',
'HBHG335K1223',
'ABHX999B1223',
'BBHX999B1223',
'EBHX999B1223',
'BBIC231D1222',
'BBIC231P1222',
'ABIC232C1222',
'ABIC232P1222',
'AALD219A0824',
'AALD219F0824',
'ABJX999B1223',
'BBJX999B1223',
'EBJX999B1223',
'BBMC214D0922',
'BBMC214F0922',
'BBMC214K0922',
'ABRD052V0123',
'ABRD052V1122',
'BBRD096V0323',
'BBRD096V0423',
'ABRD099V0323',
'ABRD099V0423',
'EBRD182X0924',
'HBRD182X0924',
'BBRD183V0924',
'GBRD183V0924',
'ABRE045V0123',
'ABRE045V1022',
'ABRE051X0123',
'ABRE051X1122',
'BBRE212X012',
'BBRE212X121',
'ABRF046V0123',
'ABRF046V1022',
'ABRX999B1223',
'BBRX999B1223',
'EBRX999B1223',
'ABSC185A0922',
'ABSC185B0822',
'ABSC185P0822',
'BBSC211A0922',
'BBSC211B0922',
'BBSC211P0922',
'BBSC212H0922',
'BBSC212J0922',
'ABTC053X0123',
'ABTC053X1122',
'ABTC056V1122',
'ABTC056V1223',
'ABTX999B1223',
'BBTX999B1223',
'EBTX999B1223',
'GBA1216L',
'GDA1216L',
'GFA1216L',
'GT11405K',
'GV11405K',
'GT11405L',
'GV11405L',
'GA01674F',
'GB01674F',
'GA01709K',
'GAA1709K',
'GAB1709K',
'GE01709K',
'GB01721L',
'GBA1721L',
'GB01745S',
'GF01745S',
'GA01830S',
'GD01830S',
'GF01830S',
'GA01845G',
'GF01845G',
'GA01862F',
'GF01862F',
'GA01928T',
'GF01928T',
'GB01948K',
'GBA1948K',
'GF01948K',
'GFA1948K',
'GB01951L',
'GBA1951L',
'GF01951L',
'GB01968K',
'GBA1968K',
'GF01968K',
'GB01969K',
'GD01969K',
'GF01969K',
'GB02001F',
'GF02001F',
'GB02021F',
'GD02021F',
'GF02021F',
'GA02033R',
'GF02033R',
'GA02039F',
'GAA2039F',
'GA02153F',
'GAA2153F',
'GE02153F',
'GA02153R',
'GAA2153R',
'GF02153R',
'GA1216L',
'GAE216L',
'GA02188K',
'GAA2188K',
'GE02188K',
'GB02189K',
'GBA2189K',
'GE02189K',
'GF02189K',
'GB02190K',
'GF02190K',
'GA02195R',
'GF02195R',
'GA02199F',
'GAA2199F',
'GB02199F',
'GE02199F',
'GF02199F',
'GFA2199F',
'GA02200F',
'GF02200F',
'GA02201F',
'GF02201F',
'GA02202F',
'GAA2202F',
'GE02202F',
'GF02202F',
'GB02203F',
'GF02203F',
'GB02204F',
'GF02204F',
'GA02214F',
'GF02214F',
'GB02224F',
'GBA2224F',
'GE02224F',
'GF02224F',
'GB02225F',
'GF02225F',
'GA02229F',
'GAA2229F',
'GF02229F',
'GA02230F',
'GAA2230F',
'GF02230F',
'GA02235R',
'GAA2235R',
'GF02235R',
'GB02236F',
'GBA2236F',
'GE02236F',
'GF02236F',
'GA02258F',
'GF02258F',
'GB02259F',
'GBA2259F',
'GF02259F',
'GFA2259F',
'GB02260F',
'GBA2260F',
'GE02260F',
'GF02260F',
'GB02267F',
'GBA2267F',
'GF02267F',
'GFA2267F',
'GB02268K',
'GBA2268K',
'GE02268K',
'GF02268K',
'GB02269L',
'GBA2269L',
'GE02269L',
'GF02269L',
'GB02275F',
'GF02275F',
'GB02276F',
'GBA2276F',
'GE02276F',
'GF02276F',
'GB02277K',
'GF02277K',
'GB02278K',
'GF02278K',
'GB02290K',
'GF02290K',
'GB02297K',
'GBA2297K',
'GB02299R',
'GF02299R',
'GA02311K',
'GAA2311K',
'GE02311K',
'GA02312K',
'GAA2312K',
'GB02314K',
'GBA2314K',
'GE02314K',
'GF02314K',
'GB02315K',
'GBA2315K',
'GE02315K',
'GB02316K',
'GF02316K',
'GB02317R',
'GBA2317R',
'GE02317R',
'GF02317R',
'GA02319F',
'GAA2319F',
'GE02319F',
'GB02321F',
'GBA2321F',
'GE02321F',
'GF02321F',
'GB02322F',
'GF02322F',
'GB02323F',
'GF02323F',
'GB02325F',
'GF02325F',
'GB02326F',
'GBA2326F',
'GE02326F',
'GF02326F',
'GB02336R',
'GF02336R',
'GB02337R',
'GBA2337R',
'GE02337R',
'GF02337R',
'GB02338R',
'GF02338R',
'GB02339F',
'GBA2339F',
'GE02339F',
'GF02339F',
'GB02340F',
'GBA2340F',
'GE02340F',
'GF02340F',
'GB02341F',
'GF02341F',
'GB02347K',
'GBA2347K',
'GE02347K',
'GB02360F',
'GF02360F',
'GA02361F',
'GF02361F',
'GB02363F',
'GF02363F',
'GB02364F',
'GBA2364F',
'GE02364F',
'GF02364F',
'GB02365F',
'GF02365F',
'GB02366R',
'GF02366R',
'GB02367R',
'GBA2367R',
'GE02367R',
'GF02367R',
'GB02368R',
'GBA2368R',
'GE02368R',
'GF02368R',
'GB02373F',
'GBA2373F',
'GE02373F',
'GF02373F',
'GA02375F',
'GAA2375F',
'GAB2375F',
'GE02375F',
'GF02375F',
'GB02376F',
'GBA2376F',
'GE02376F',
'GF02376F',
'GB02377F',
'GBA2377F',
'GE02377F',
'GF02377F',
'GB02378F',
'GF02378F',
'GB02379R',
'GF02379R',
'GA02387F',
'GAA2387F',
'GE02387F',
'GB02390F',
'GF02390F',
'GB02391F',
'GF02391F',
'GB02392F',
'GF02392F',
'GB02393F',
'GF02393F',
'GB02394F',
'GBA2394F',
'GE02394F',
'GB02395F',
'GF02395F',
'GA02403F',
'GAA2403F',
'GA02404F',
'GAA2404F',
'GE02404F',
'GB02406F',
'GF02406F',
'GB02407F',
'GF02407F',
'GB02408F',
'GF02408F',
'GB02409F',
'GF02409F',
'GB02423F',
'GF02423F',
'GB02424F',
'GF02424F',
'GB02425F',
'GF02425F',
'GB02426F',
'GF02426F',
'GB02427F',
'GF02427F',
'GB02428F',
'GF02428F',
'GB02430F',
'GF02430F',
'GB02431F',
'GF02431F',
'GB02432F',
'GF02432F',
'GB02439F',
'GF02439F',
'GB02440R',
'GF02440R',
'GB02442F',
'GBA2442F',
'GE02442F',
'GF02442F',
'GB02444F',
'GBA2444F',
'GE02444F',
'GF02444F',
'GB02460F',
'GF02460F',
'GB02461R',
'GF02461R',
'GB02503F',
'GE02503F',
'GB02504F',
'GE02504F',
'GB02505F',
'GE02505F',
'GB02523R',
'GF02523R',
'GB02524F',
'GF02524F',
'GB02525R',
'GF02525R',
'GB02526F',
'GF02526F',
'GB02531F',
'GE02531F',
'GB02546F',
'GE02546F',
'GB02578F',
'GE02578F',
'GB02590F',
'GE02590F',
'GA02595F',
'GE02595F',
'GB02598F',
'GE02598F',
'GB02609R',
'GE02609R',
'GB02610F',
'GE02610F',
'GB02616R',
'GE02616R',
'GB02629F',
'GE02629F',
'GB02630K',
'GE02630K',
'GB02639K',
'GE02639K',
'GB02640K',
'GE02640K',
'GB02641K',
'GE02641K',
'GB02642F',
'GE02642F',
'GB02643K',
'GE02643K',
'GB02651F',
'GE02651F',
'GB02653F',
'GE02653F',
'GB02654F',
'GE02654F',
'GB02655F',
'GE02655F',
'GB02656R',
'GE02656R',
'GB02664K',
'GE02664K',
'GB02665K',
'GE02665K',
'GB02666L',
'GE02666L',
'GB02667K',
'GE02667K',
'GB02712F',
'GE02712F',
'GB02713F',
'GE02713F',
'GB02874R',
'GE02874R',
'GB02884K',
'GE02884K',
'GB02961F',
'GE02961F',
'GA03072R',
'GB03072R',
'GE03072R',
'GB03085F',
'GE03085F',
'GB03154F',
'GE03154F',
'GA03199F',
'GE03199F',
'GA03222R',
'GE03222R',
'GB03229F',
'GE03229F',
'GA03258L',
'GE03258L',
'GA03294L',
'GE03294L',
'GB03302L',
'GE03302L',
'GB03309L',
'GE03309L',
'GA03329F',
'GE03329F',
'GB03334R',
'GE03334R',
'GAA428L',
'GBA428L',
'GBA570G',
'GBB570G',
'GA0798F',
'GE0798F',
'HT01401F',
'HT11401F',
'HV11401F',
'HT01401P',
'HV11401P',
'HT01405F',
'HT11405F',
'HT21407F',
'HV11407F',
'HV21407F',
'HBB1409F',
'HBC1409F',
'HT21411F',
'HV11411F',
'HZ01411F',
'HT21411S',
'HV21411S',
'HAA1452F',
'HAB1452F',
'HFA1452F',
'HAA1566F',
'HF01566F',
'HAA1567G',
'HB01567G',
'HF01567G',
'HAA1568F',
'HB01568F',
'HB01590F',
'HE01590F',
'HV11601F',
'HZ01601F',
'HV11601P',
'HZ01601P',
'HT11607F',
'HV11607F',
'HT11611P',
'HV11611P',
'HAA1617R',
'HB01617R',
'HF01617R',
'HB01625S',
'HF01625S',
'HB01626G',
'HF01626G',
'HT01647F',
'HZ01647F',
'HA01649R',
'HAA1649R',
'HF01649R',
'HA01676F',
'HAA1676F',
'HAB1676F',
'HA01677F',
'HAA1677F',
'HA01679F',
'HAA1679F',
'HA01682F',
'HAA1682F',
'HDA1682F',
'HB01685F',
'HF01685F',
'HF01701F',
'HFA1701F',
'HR01701F',
'HA01704F',
'HAA1704F',
'HAB1704F',
'HF01704F',
'HB01714S',
'HF01714S',
'HBA1715S',
'HFA1715S',
'HAA1741U',
'HD01741U',
'HF01741U',
'HB01743F',
'HF01743F',
'HB01749S',
'HF01749S',
'HB01750S',
'HF01750S',
'HB01752R',
'HF01752R',
'HB01754S',
'HF01754S',
'HA01761F',
'HB01761F',
'HB01768F',
'HBA1768F',
'HF01768F',
'HFA1768F',
'HB01769R',
'HF01769R',
'HB01783S',
'HF01783S',
'HB01785R',
'HF01785R',
'HB01796S',
'HF01796S',
'HB01797S',
'HF01797S',
'HB01798S',
'HF01798S',
'HB01799G',
'HF01799G',
'HB01800F',
'HF01800F',
'HA01803R',
'HB01803R',
'HF01803R',
'HB01808F',
'HD01808F',
'HF01808F',
'HB01810S',
'HF01810S',
'HA01825S',
'HB01825S',
'HF01825S',
'HA01826F',
'HF01826F',
'HB01837S',
'HD01837S',
'HB01838S',
'HD01838S',
'HF01838S',
'HB01839S',
'HD01839S',
'HF01839S',
'HB1183G',
'HBE183G',
'HB01841S',
'HF01841S',
'HB01842G',
'HF01842G',
'HB01843G',
'HF01843G',
'HB01864G',
'HF01864G',
'HA1191G',
'HAE191G',
'HF01937G',
'HT01937G',
'HB01940F',
'HE01940F',
'HA01942F',
'HF01942F',
'HA01947F',
'HB01947F',
'HF01947F',
'HB02026G',
'HF02026G',
'HA02031F',
'HD02031F',
'HF02031F',
'HB02065F',
'HF02065F',
'HB02082F',
'HBA2082F',
'HF02082F',
'HFA2082F',
'HFB2082F',
'HA02096F',
'HAA2096F',
'HB02100F',
'HF02100F',
'HB02120F',
'HF02120F',
'HA02131F',
'HF02131F',
'HA02132F',
'HF02132F',
'HB02135F',
'HF02135F',
'HB02136F',
'HF02136F',
'HA02140F',
'HAA2140F',
'HF02140F',
'HA02142F',
'HAA2142F',
'HF02142F',
'HB02156F',
'HF02156F',
'HB02157F',
'HF02157F',
'HB02158F',
'HF02158F',
'HB02160F',
'HF02160F',
'HB02161F',
'HF02161F',
'HB02162F',
'HF02162F',
'HA02169F',
'HF02169F',
'HB02171F',
'HF02171F',
'HB02172F',
'HBA2172F',
'HE02172F',
'HF02172F',
'HA02194F',
'HAA2194F',
'HB02194F',
'HE02194F',
'HF02194F',
'HFA2194F',
'HB02194R',
'HE02194R',
'HB02213F',
'HF02213F',
'HA02217F',
'HF02217F',
'HA02218F',
'HF02218F',
'HB02222F',
'HBA2222F',
'HE02222F',
'HB02223F',
'HBA2223F',
'HE02223F',
'HF02223F',
'HB02226F',
'HBA2226F',
'HE02226F',
'HB02262F',
'HF02262F',
'HA02264R',
'HAA2264R',
'HE02264R',
'HB02265R',
'HBA2265R',
'HE02265R',
'HB02266F',
'HF02266F',
'HA02270F',
'HAA2270F',
'HA02272F',
'HAA2272F',
'HE02272F',
'HB02284F',
'HBA2284F',
'HE02284F',
'HA02294F',
'HAA2294F',
'HE02294F',
'HA02296F',
'HAA2296F',
'HE02296F',
'HB02301F',
'HF02301F',
'HB02302F',
'HF02302F',
'HB02303F',
'HF02303F',
'HB02324F',
'HBA2324F',
'HBB2324F',
'HE02324F',
'HEA2324F',
'HF02324F',
'HA02334F',
'HAA2334F',
'HA02335R',
'HAA2335R',
'HE02335R',
'HB02342F',
'HF02342F',
'HB02343F',
'HF02343F',
'HB02345F',
'HF02345F',
'HB02346F',
'HF02346F',
'HB02359F',
'HBA2359F',
'HE02359F',
'HF02359F',
'HA02370F',
'HAA2370F',
'HE02370F',
'HB02374F',
'HBA2374F',
'HF02374F',
'HFA2374F',
'HB02418F',
'HE02418F',
'HB02436F',
'HF02436F',
'HB02437F',
'HF02437F',
'HB02441F',
'HF02441F',
'HB02443F',
'HF02443F',
'HB02445F',
'HF02445F',
'HA02456F',
'HE02456F',
'HA02456R',
'HAA2456R',
'HE02456R',
'HB02475F',
'HF02475F',
'HB02476R',
'HF02476R',
'HA02478F',
'HE02478F',
'HB02494R',
'HBB2494R',
'HE02494R',
'HB02527F',
'HF02527F',
'HB02528F',
'HF02528F',
'HB02529R',
'HF02529R',
'HA02533R',
'HAA2533R',
'HE02533R',
'HB02534F',
'HE02534F',
'HB02535R',
'HBA2535R',
'HE02535R',
'HEA2535R',
'HB02542F',
'HE02542F',
'HA02543R',
'HE02543R',
'HA02544R',
'HE02544R',
'HB02561R',
'HE02561R',
'HA02592R',
'HE02592R',
'HB02593F',
'HE02593F',
'HB02594R',
'HE02594R',
'HA02612F',
'HE02612F',
'HA02612R',
'HE02612R',
'HB02614F',
'HE02614F',
'HB02614R',
'HE02614R',
'HB02615F',
'HE02615F',
'HB02699R',
'HE02699R',
'HB02700R',
'HBA2700R',
'HE02700R',
'HB02703F',
'HE02703F',
'HB02704R',
'HE02704R',
'HA02728F',
'HE02728F',
'HB02778F',
'HE02778F',
'HA1277F',
'HAE277F',
'HB02789R',
'HE02789R',
'HB02791F',
'HE02791F',
'HB02802F',
'HE02802F',
'HB02803F',
'HE02803F',
'HB02804F',
'HE02804F',
'HB02808F',
'HE02808F',
'HB02819R',
'HE02819R',
'HB02820R',
'HE02820R',
'HB02821F',
'HE02821F',
'HB02822R',
'HE02822R',
'HB02823F',
'HE02823F',
'HB02824F',
'HE02824F',
'HB02825F',
'HE02825F',
'HB02826F',
'HE02826F',
'HB02827F',
'HE02827F',
'HB02836F',
'HE02836F',
'HB02837F',
'HE02837F',
'HB02838F',
'HE02838F',
'HB02839F',
'HE02839F',
'HB02840F',
'HE02840F',
'HB02853F',
'HE02853F',
'HB02894F',
'HE02894F',
'HB02895F',
'HE02895F',
'HB02896F',
'HE02896F',
'HB02899R',
'HE02899R',
'HB02900R',
'HE02900R',
'HB02932R',
'HE02932R',
'HB02933F',
'HE02933F',
'HB02950F',
'HE02950F',
'HB02951F',
'HE02951F',
'HB02957F',
'HE02957F',
'HB02958R',
'HE02958R',
'HB02959F',
'HE02959F',
'HB02960R',
'HBA2960R',
'HE02960R',
'HEA2960R',
'HB02973F',
'HE02973F',
'HB02974F',
'HE02974F',
'HB03019F',
'HE03019F',
'HB03020F',
'HE03020F',
'HB03024F',
'HE03024F',
'HB03027R',
'HE03027R',
'HB03081F',
'HE03081F',
'HB03084R',
'HBA3084R',
'HE03084R',
'HB03087F',
'HE03087F',
'HB03098R',
'HE03098R',
'HB03100F',
'HE03100F',
'HA03115R',
'HE03115R',
'HA03116R',
'HE03116R',
'HA03117R',
'HE03117R',
'HA03118F',
'HE03118F',
'HA03119F',
'HAA3119F',
'HE03119F',
'HEA3119F',
'HA03120F',
'HE03120F',
'HA03121F',
'HE03121F',
'HA03122F',
'HE03122F',
'HB03123R',
'HE03123R',
'HB03124R',
'HE03124R',
'HB03125F',
'HE03125F',
'HB03126F',
'HE03126F',
'HB03127F',
'HE03127F',
'HB03128F',
'HE03128F',
'HA03141R',
'HE03141R',
'HA03142F',
'HE03142F',
'HA03143F',
'HE03143F',
'HB03144R',
'HE03144R',
'HB03145F',
'HE03145F',
'HB03147F',
'HE03147F',
'HB03148F',
'HE03148F',
'HA03149R',
'HE03149R',
'HB03150R',
'HE03150R',
'HA03152F',
'HE03152F',
'HB03153F',
'HE03153F',
'HA03155R',
'HE03155R',
'HA03156R',
'HE03156R',
'HA03157F',
'HE03157F',
'HB03158R',
'HE03158R',
'HB03160F',
'HE03160F',
'HB03161F',
'HE03161F',
'HA03168F',
'HE03168F',
'HB03174F',
'HE03174F',
'HA03188R',
'HE03188R',
'HA03189R',
'HE03189R',
'HA03190R',
'HE03190R',
'HA03191F',
'HE03191F',
'HB03194F',
'HE03194F',
'HA03196F',
'HE03196F',
'HB03197R',
'HE03197R',
'HB03198R',
'HE03198R',
'HA03200F',
'HE03200F',
'HA03223F',
'HE03223F',
'HA03224F',
'HE03224F',
'HB03227R',
'HE03227R',
'HB03228R',
'HE03228R',
'HA03231F',
'HE03231F',
'HA03232F',
'HE03232F',
'HB03235R',
'HE03235R',
'HB03237R',
'HE03237R',
'HA03327F',
'HE03327F',
'HA03328R',
'HE03328R',
'HA03330R',
'HE03330R',
'HA03331R',
'HE03331R',
'HB03332R',
'HE03332R',
'HB03335R',
'HE03335R',
'HB03336F',
'HE03336F',
'HB03338R',
'HE03338R',
'HB03339R',
'HE03339R',
'HB03356R',
'HE03356R',
'HA03366R',
'HE03366R',
'HA03367F',
'HE03367F',
'HA03368F',
'HE03368F',
'HB03370R',
'HE03370R',
'HB03371R',
'HE03371R',
'HB03372R',
'HE03372R',
'HB03373F',
'HE03373F',
'HB03374F',
'HE03374F',
'HB03376R',
'HE03376R',
'HB03401R',
'HE03401R',
'HB03403R',
'HE03403R',
'HB03404F',
'HE03404F',
'HA03410R',
'HE03410R',
'HB03415R',
'HE03415R',
'HA03438R',
'HE03438R',
'HB03443R',
'HE03443R',
'HB03444R',
'HE03444R',
'HB03445F',
'HE03445F',
'HB03446F',
'HE03446F',
'HA03460F',
'HE03460F',
'HA1346F',
'HAA346F',
'HA1422F',
'HE1422F',
'HA1453F',
'HT0453F',
'HB0991S',
'HBA991S',
'12463239049',
'12463259049',
'ABIC394G0125',
'ABIC394H0125',
'ABIC394J0125',
'32463259050',
'82463259050',
'BBIC395F0125',
'BBIC395G0125',
'BBIC395J0125',
'GBIC395F0125',
'GBIC395G0125',
'GBIC395J0125',
'IA1047R',
'IB1047R',
'IBB1337K',
'IF01337K',
'IBA1368G',
'IF01368G',
'IBB1368T',
'IDB1368T',
'IFA1368T',
'IT01401F',
'IT11401F',
'IV01401F',
'IV11401F',
'IT01405F',
'IT11405F',
'IV01405F',
'IV11405F',
'IT11405K',
'IV11405K',
'IT11405L',
'IV11405L',
'IT11407F',
'IT21407F',
'IV21407F',
'IT01409P',
'IV01409P',
'IBA1528L',
'IF01528L',
'IA1153S',
'IBE153S',
'IA1154F',
'IAE154F',
'IB1154F',
'IBE154F',
'IE1154F',
'IA01582G',
'IAA1582G',
'IF01582G',
'IA01583G',
'IAA1583G',
'IT11607F',
'IV11607F',
'IT11611P',
'IV11611P',
'IA01706L',
'IAA1706L',
'IA01707K',
'IB01707K',
'IAE176F',
'IAF176F',
'IBC1770R',
'IFB1770R',
'IBA1770U',
'IBB1770U',
'IF01770U',
'IFA1770U',
'IA1178F',
'IE1178F',
'IA01834R',
'IAA1834R',
'IA01835F',
'IAA1835F',
'IAB1835F',
'IE01835F',
'IA01884U',
'IAA1884U',
'IBA1900F',
'IBB1900F',
'IFA1900F',
'IA01912T',
'IB01912T',
'IF01912T',
'IB01941K',
'ID01941K',
'IF01941K',
'IA01945K',
'IAA1945K',
'IB01949K',
'ID01949K',
'IF01949K',
'IB01955K',
'ID01955K',
'IF01955K',
'IB01956K',
'IF01956K',
'IA01956L',
'IB01956L',
'IBA1956L',
'ID01956L',
'IFA1956L',
'IB02017F',
'ID02017F',
'IF02017F',
'IB02035F',
'ID02035F',
'IB02036F',
'IF02036F',
'IB02037F',
'ID02037F',
'IF02037F',
'IB02038F',
'ID02038F',
'IF02038F',
'IB02070F',
'IBA2070F',
'IF02070F',
'IA02071R',
'IAA2071R',
'IF02071R',
'IAA2073F',
'IAB2073F',
'IFA2073F',
'IA02073T',
'IF02073T',
'IA02092F',
'IAA2092F',
'IA02093F',
'IAA2093F',
'IF02093F',
'IA02094F',
'IAA2094F',
'IF02094F',
'IB02097F',
'IBA2097F',
'IF02097F',
'IFA2097F',
'IA02108F',
'IF02108F',
'IA02110F',
'IAA2110F',
'IB02110F',
'IBA2110F',
'IA02110R',
'IAA2110R',
'IB02110R',
'IB02127F',
'IF02127F',
'IB02138F',
'IF02138F',
'IA02155F',
'IAA2155F',
'IB02155F',
'IF02155F',
'IB02175R',
'IBA2175R',
'IF02175R',
'IA02191R',
'IAA2191R',
'IE02191R',
'IA02193R',
'IF02193R',
'IB02211K',
'IF02211K',
'IB02212F',
'IBA2212F',
'IE02212F',
'IF02212F',
'IA02219L',
'IAA2219L',
'IE02219L',
'IB02221L',
'IF02221L',
'IA02244K',
'IAA2244K',
'IB02245K',
'IBA2245K',
'IFA2245K',
'IA02246K',
'IAA2246K',
'IB02247K',
'IBA2247K',
'IFA2247K',
'IB02263F',
'IBA2263F',
'IF02263F',
'IFA2263F',
'IB02274F',
'IF02274F',
'IB02287F',
'IF02287F',
'IB02344F',
'IF02344F',
'IB02380R',
'IF02380R',
'IB02463F',
'IE02463F',
'IB02463R',
'IBA2463R',
'IE02463R',
'IF02463R',
'IA02557F',
'IE02557F',
'IA02568F',
'IAA2568F',
'IB02568F',
'IA02568R',
'IAA2568R',
'IB02645K',
'IE02645K',
'IA02645L',
'IB02645L',
'IE02645L',
'IB02646K',
'IE02646K',
'IA02646L',
'IB02646L',
'IE02646L',
'IA02692K',
'IB02692K',
'IA02701K',
'IAA2701K',
'IE02701K',
'IA02701L',
'IB02701L',
'IE02701L',
'IA02702K',
'IB02702K',
'IE02702K',
'IA02737K',
'IB02737K',
'IE02737K',
'IB02752R',
'IE02752R',
'IB02755L',
'IE02755L',
'IA02782F',
'IB02782F',
'IE02782F',
'IA02782R',
'IB02782R',
'IE02782R',
'IB02897L',
'IE02897L',
'IB02898L',
'IE02898L',
'IA02964K',
'IE02964K',
'IA02965K',
'IE02965K',
'IB02977K',
'IE02977K',
'IB02978K',
'IE02978K',
'IB02979K',
'IE02979K',
'IB02980K',
'IE02980K',
'IB02981K',
'IE02981K',
'IB02982K',
'IE02982K',
'IA03056F',
'IB03056F',
'IA03061F',
'IAA3061F',
'IE03061F',
'IB03086F',
'IE03086F',
'IAA308F',
'IAB308F',
'IB0308F',
'IBA308F',
'IW8308F',
'IA03140F',
'IE03140F',
'IB03151R',
'IE03151R',
'IA03162R',
'IE03162R',
'IA03163R',
'IE03163R',
'IA03165F',
'IE03165F',
'IB03166R',
'IE03166R',
'IA03169R',
'IB03169R',
'IA03170F',
'IB03170F',
'IA03171F',
'IB03171F',
'IA03172F',
'IB03172F',
'IA03173F',
'IB03173F',
'IA03195F',
'IB03195F',
'IE03195F',
'IA03195R',
'IB03195R',
'IE03195R',
'IA03234F',
'IB03234F',
'IE03234F',
'IA03234R',
'IB03234R',
'IE03234R',
'IA03238L',
'IE03238L',
'IA1323F',
'IW8323F',
'IB03243L',
'IE03243L',
'IB03244K',
'IE03244K',
'IA03253K',
'IB03253K',
'IA03256L',
'IB03256L',
'IE03256L',
'IB03257L',
'IE03257L',
'IB03301L',
'IE03301L',
'IA03315L',
'IE03315L',
'IB03319L',
'IE03319L',
'IB03323L',
'IE03323L',
'IB03324L',
'IE03324L',
'IA03325R',
'IE03325R',
'IA03326F',
'IE03326F',
'IB03337F',
'IE03337F',
'IB03340R',
'IE03340R',
'IA03369R',
'IE03369R',
'IA03399F',
'IE03399F',
'IA03400R',
'IE03400R',
'IA03408R',
'IE03408R',
'IA03409F',
'IE03409F',
'IA03411F',
'IE03411F',
'IB03416R',
'IE03416R',
'IB03417L',
'IE03417L',
'IB03418L',
'IE03418L',
'IA03422R',
'IE03422R',
'IB03423F',
'IE03423F',
'IB03424R',
'IE03424R',
'IB03425R',
'IE03425R',
'IB03426R',
'IE03426R',
'IAA344S',
'IAB344S',
'IB0344S',
'IBA344S',
'IA03465F',
'IB03465F',
'IA03466R',
'IB03466R',
'IA1347R',
'IW8347R',
'JAB0308M',
'JBB0308M',
'JBC0309H',
'JBD0309H',
'JF00309H',
'JB00309M',
'JBA0309M',
'JF00309M',
'JAA0334M',
'JAB0334M',
'JAC0334M',
'JBC0334M',
'JBD0334M',
'JBE0334M',
'JBF0334M',
'JBG0334M',
'JFC0334M',
'JFE0334M',
'JAA0389M',
'JAB0389M',
'JAC0389M',
'JBE0389M',
'JBF0389M',
'JBG0389M',
'JBH0389M',
'JF00389M',
'JFA0389M',
'JU00474M',
'JXB0474M',
'JYA0522M',
'JYB0522M',
'JF00545M',
'JXA0545M',
'JAA0547A',
'JAB0547A',
'JF00547A',
'JF00555M',
'JYA0555M',
'JYA0556M',
'JYB0556M',
'JUA0563D',
'JXA0563D',
'JA00571K',
'JAA0571K',
'JAB0571K',
'JAC0571K',
'JAD0571K',
'JE00571K',
'JAA0576A',
'JAB0576A',
'JA00576K',
'JAA0576K',
'JP00629M',
'JY00629M',
'JX00632A',
'JXA0632A',
'JL00636M',
'JP00636M',
'JPA0636M',
'JPB0636M',
'JB00641M',
'JE00641M',
'JU00645E',
'JX00645E',
'JXA0645E',
'JXB0645E',
'JU00645H',
'JUA0645H',
'JUC0645H',
'JFA0645M',
'JFB0645M',
'JU00645M',
'JUA0645M',
'JUB0645M',
'JUC0645M',
'JUD0645M',
'JX00645M',
'JXA0645M',
'JXB0645M',
'JXC0645M',
'JXD0645M',
'JXE0645M',
'JU00645N',
'JX00645N',
'JXA0645N',
'JF00647M',
'JX00647M',
'JA00660M',
'JAA0660M',
'JAB0660M',
'JB00660M',
'JBA0660M',
'JE00660M',
'JF00660M',
'JA00662J',
'JB00662J',
'JF00667Y',
'JK00667Y',
'JA00675M',
'JB00675M',
'JBA0675M',
'JBC0675M',
'JF00675M',
'JB00677A',
'JBA0677A',
'JD00677A',
'JB00677K',
'JBA0677K',
'JE00677K',
'JA00687K',
'JE00687K',
'JA00688A',
'JAA0688A',
'JA00688E',
'JF00688E',
'JA00688K',
'JAA0688K',
'JE00688K',
'JA00690J',
'JAA0690J',
'JU00695A',
'JUA0695A',
'JX00695A',
'JE00695K',
'JU00695K',
'JF00697Y',
'JU00697Y',
'JX00708A',
'JXA0708A',
'JFA0708E',
'JFB0708E',
'JX00708E',
'JXA0708E',
'JU00711A',
'JUA0711A',
'JF00711E',
'JU00711E',
'JA00723A',
'JAA0723A',
'JF00723A',
'JA00723E',
'JF00723E',
'JB00731A',
'JBA0731A',
'JB00731H',
'JBA0731H',
'JW00732R',
'JWA0732R',
'JYA0732R',
'JYB0732R',
'JP00734M',
'JPA0734M',
'JPB0734M',
'JPC0734M',
'JW00734M',
'JWA0734M',
'JY00734M',
'JYA0734M',
'JU00745A',
'JUA0745A',
'JF00761W',
'JX00761W',
'JU00772D',
'JX00772D',
'JU00773D',
'JX00773D',
'JE00776M',
'JU00776M',
'JUA0776M',
'JUB0776M',
'JX00776M',
'JXA0776M',
'JU00776N',
'JX00776N',
'JW00777M',
'JY00777M',
'JYA0777M',
'JYB0777M',
'JW00778R',
'JWA0778R',
'JY00778R',
'JYA0778R',
'JYB0778R',
'JYC0778R',
'JF00779M',
'JY00779M',
'JF00780M',
'JY00780M',
'JE00781D',
'JEA0781D',
'JF00781D',
'JFA0781D',
'JG00781D',
'JGA0781D',
'JW00781D',
'JWA0781D',
'JY00781D',
'JYA0781D',
'JYB0781D',
'JYC0781D',
'JE00781Q',
'JEA0781Q',
'JF00781Q',
'JG00781Q',
'JGA0781Q',
'JW00781Q',
'JY00781Q',
'JF00782D',
'JY00782D',
'JD00788A',
'JU00788A',
'JF00788E',
'JU00788E',
'JD00789A',
'JF00789A',
'JU00789A',
'JF00789E',
'JU00789E',
'JE00790E',
'JU00790E',
'JUA0790E',
'JU00794D',
'JX00794D',
'JY00798M',
'JYA0798M',
'JD00800E',
'JF00800E',
'JU00800E',
'JFA0801E',
'JU00801E',
'JUA0801E',
'JW00826E',
'JY00826E',
'JF00828E',
'JU00828E',
'JF00830E',
'JUA0830E',
'JE00835D',
'JEA0835D',
'JF00835D',
'JH00835D',
'JPA0835D',
'JW00835D',
'JWA0835D',
'JWB0835D',
'JWC0835D',
'JY00835D',
'JYA0835D',
'JE00836Q',
'JEA0836Q',
'JF00836Q',
'JH00836Q',
'JPA0836Q',
'JW00836Q',
'JWA0836Q',
'JWB0836Q',
'JY00836Q',
'JYA0836Q',
'JW00841Q',
'JY00841Q',
'JYA0841Q',
'JD00843Q',
'JF00843Q',
'JFA0843Q',
'JU00843Q',
'JUA0843Q',
'JD00844Q',
'JFA0844Q',
'JU00844Q',
'JUA0844Q',
'JD00845A',
'JF00845A',
'JU00845A',
'JX00849E',
'JXA0849E',
'JF00850K',
'JX00850K',
'JF00850M',
'JX00850M',
'JD00852M',
'JU00852M',
'JD00853M',
'JF00853M',
'JU00853M',
'JF00854A',
'JU00854A',
'JP00861M',
'JW00861M',
'JY00861M',
'JD00862M',
'JF00862M',
'JU00862M',
'JF00864M',
'JU00864M',
'JF00865E',
'JU00865E',
'JD00866A',
'JF00866A',
'JU00866A',
'JF00866M',
'JU00866M',
'JF00867A',
'JU00867A',
'JB00868A',
'JF00868A',
'JF00869A',
'JX00869A',
'JX00872E',
'JXA0872E',
'JA00883K',
'JF00883K',
'JF00885A',
'JU00885A',
'JF00886Q',
'JX00886Q',
'JXA0886Q',
'JF00893M',
'JX00893M',
'JF00894E',
'JFA0894E',
'JU00894E',
'JUA0894E',
'JE00904E',
'JU00904E',
'JUA0904E',
'JF00905M',
'JU00905M',
'JUA0905M',
'JA00906M',
'JAA0906M',
'JF00906M',
'JU00908M',
'JUA0908M',
'JA00914M',
'JAA0914M',
'JF00914M',
'JX00914M',
'JF00917E',
'JX00917E',
'JA00918E',
'JAA0918E',
'JE00918E',
'JA00918M',
'JAA0918M',
'JF00918M',
'JFA0918M',
'JA00919M',
'JF00919M',
'JF00921M',
'JU00921M',
'JF00927E',
'JU00927E',
'JU00930J',
'JUA0930J',
'JX00930J',
'JXA0930J',
'JF00945M',
'JU00945M',
'JE00948E',
'JU00948E',
'JUA0948E',
'JE00948K',
'JU00948K',
'JE00948M',
'JU00948M',
'JU00955M',
'JX00955M',
'JU00956M',
'JX00956M',
'JU00957M',
'JX00957M',
'JU00959M',
'JX00959M',
'JU00961M',
'JX00961M',
'JU00964M',
'JX00964M',
'JE00969E',
'JU00969E',
'JE00969K',
'JU00969K',
'JE00969M',
'JU00969M',
'JUA0969M',
'JW00971M',
'JY00971M',
'JU00980J',
'JX00980J',
'JW01007M',
'JY01007M',
'JU01017J',
'JX01017J',
'JE01031D',
'JEA1031D',
'JEB1031D',
'JEC1031D',
'JX01031D',
'JXA1031D',
'JXB1031D',
'JXC1031D',
'JXD1031D',
'JXE1031D',
'JF01034M',
'JU01034M',
'JW01036D',
'JY01036D',
'JF01041E',
'JU01041E',
'JU01042J',
'JX01042J',
'JE01046E',
'JEA1046E',
'JU01046E',
'JUA1046E',
'JX01046E',
'JXA1046E',
'JXB1046E',
'JE01046M',
'JF01046M',
'JU01046M',
'JUA1046M',
'JX01046M',
'JXA1046M',
'JF01047E',
'JX01047E',
'JA01056J',
'JAA1056J',
'JB0105U',
'JX0105U',
'JE01065D',
'JX01065D',
'JXA1065D',
'JU01068R',
'JX01068R',
'JE01074E',
'JX01074E',
'JE01074K',
'JX01074K',
'JE01075E',
'JX01075E',
'JE01075K',
'JX01075K',
'JE01076E',
'JX01076E',
'JE01077E',
'JX01077E',
'JE01085E',
'JW01085E',
'JE01085M',
'JF01085M',
'JG01085M',
'JH01085M',
'JW01085M',
'JY01085M',
'JA01086K',
'JE01086K',
'JA01087K',
'JE01087K',
'JE01088E',
'JU01088E',
'JE01094D',
'JEA1094D',
'JU01094D',
'JUA1094D',
'JUB1094D',
'JE01095E',
'JX01095E',
'JE01096K',
'JX01096K',
'JE01097E',
'JU01097E',
'JE01098E',
'JU01098E',
'JE01099K',
'JX01099K',
'JE01100E',
'JU01100E',
'JU01100K',
'JUA1100K',
'JE01103D',
'JU01103D',
'JX01106D',
'JXA1106D',
'JXB1106D',
'JE01107D',
'JX01107D',
'JE01107Q',
'JEA1107Q',
'JEB1107Q',
'JX01107Q',
'JXA1107Q',
'JXB1107Q',
'JXC1107Q',
'JA01108E',
'JAA1108E',
'JAB1108E',
'JE01108E',
'JE01109M',
'JU01109M',
'JE01120E',
'JU01120E',
'JE01123Q',
'JEA1123Q',
'JEB1123Q',
'JU01123Q',
'JUA1123Q',
'JUB1123Q',
'JE01124D',
'JEA1124D',
'JU01124D',
'JUA1124D',
'JX01127E',
'JXA1127E',
'JXB1127E',
'JE01129E',
'JEA1129E',
'JE01130E',
'JU01130E',
'JA01132M',
'JAA1132M',
'JAB1132M',
'JB01132M',
'JE01132M',
'JEA1132M',
'JEB1132M',
'JU01132M',
'JUA1132M',
'JE01134E',
'JU01134E',
'JE01135E',
'JU01135E',
'JE01136E',
'JU01136E',
'JE01152E',
'JU01152E',
'JE01153E',
'JU01153E',
'JE01154M',
'JEA1154M',
'JEB1154M',
'JX01154M',
'JXA1154M',
'JXB1154M',
'JXC1154M',
'JE01155M',
'JEA1155M',
'JEB1155M',
'JU01155M',
'JUA1155M',
'JUB1155M',
'JX01156D',
'JXA1156D',
'JXB1156D',
'JXC1156D',
'JE01157M',
'JEA1157M',
'JEB1157M',
'JG01157M',
'JGA1157M',
'JW01157M',
'JWA1157M',
'JWB1157M',
'JY01157M',
'JE01158E',
'JU01158E',
'JUA1158E',
'JB01161E',
'JE01161E',
'JB01162E',
'JE01162E',
'JE01163D',
'JEA1163D',
'JX01163D',
'JXA1163D',
'JX01164K',
'JXA1164K',
'JX01166E',
'JXA1166E',
'JE01167D',
'JEA1167D',
'JW01167D',
'JWA1167D',
'JE01167Q',
'JEA1167Q',
'JW01167Q',
'JWA1167Q',
'JE01170Q',
'JW01170Q',
'JE01171E',
'JW01171E',
'JE01172E',
'JX01172E',
'JXA1172E',
'JE01173E',
'JEA1173E',
'JX01173E',
'JXA1173E',
'JXB1173E',
'JE01177E',
'JX01177E',
'JXA1177E',
'JE01178E',
'JEA1178E',
'JU01178E',
'JUA1178E',
'JE01179E',
'JEA1179E',
'JU01179E',
'JUA1179E',
'JE01180E',
'JEA1180E',
'JU01180E',
'JUA1180E',
'JE01183D',
'JX01183D',
'JXA1183D',
'JE01184Q',
'JU01184Q',
'JE01185Q',
'JU01185Q',
'JE01187Q',
'JEA1187Q',
'JU01187Q',
'JUA1187Q',
'JE01189Q',
'JEA1189Q',
'JW01189Q',
'JWA1189Q',
'JWB1189Q',
'JY01189Q',
'JE01190D',
'JEA1190D',
'JW01190D',
'JWA1190D',
'JWB1190D',
'JY01190D',
'JE01193J',
'JU01193J',
'JX01193J',
'JXA1193J',
'JE01195Q',
'JU01195Q',
'JUA1195Q',
'JX01195Q',
'JE01196Q',
'JEA1196Q',
'JE01197E',
'JEA1197E',
'JEB1197E',
'JU01197E',
'JUA1197E',
'JX01197E',
'JXA1197E',
'JXB1197E',
'JE01197M',
'JEA1197M',
'JEB1197M',
'JU01197M',
'JUA1197M',
'JUB1197M',
'JX01197M',
'JXA1197M',
'JE01199E',
'JX01199E',
'JA01200E',
'JAA1200E',
'JE01200E',
'JB01203E',
'JBA1203E',
'JE01203E',
'JEA1203E',
'JE01205E',
'JEA1205E',
'JU01205E',
'JUA1205E',
'JE01205M',
'JEA1205M',
'JEB1205M',
'JU01205M',
'JUA1205M',
'JUB1205M',
'JE01206E',
'JX01206E',
'JE01208E',
'JW01208E',
'JY01208E',
'JE01210Q',
'JEA1210Q',
'JW01210Q',
'JWA1210Q',
'JE01212D',
'JEA1212D',
'JX01212D',
'JXA1212D',
'JG01221D',
'JW01221D',
'JY01221D',
'JG01222D',
'JY01222D',
'JG01222Q',
'JY01222Q',
'JG01223Q',
'JW01223Q',
'JY01223Q',
'JE01226J',
'JX01226J',
'JE01232D',
'JY01232D',
'JE01233D',
'JY01233D',
'JE01233Q',
'JY01233Q',
'JE01234Q',
'JG01234Q',
'JE01235Q',
'JW01235Q',
'JE01237D',
'JG01237D',
'JE01244E',
'JX01244E',
'JE01246D',
'JU01246D',
'JE01247D',
'JW01247D',
'JE01248D',
'JY01248D',
'JA01251E',
'JE01251E',
'JE01258Q',
'JX01258Q',
'JE01261Q',
'JU01261Q',
'JE01262Q',
'JU01262Q',
'JW01265Q',
'JY01265Q',
'JW01267Q',
'JY01267Q',
'JE01268M',
'JX01268M',
'JE01270E',
'JU01270E',
'JA01275E',
'JE01275E',
'JT11408K',
'JV11408K',
'JT11415M',
'JV11415M',
'JT01416H',
'JV01416H',
'JT11417A',
'JV11417A',
'JC0142J',
'JE0142J',
'JZ01611M',
'JZ11611M',
'JT11613A',
'JV01613A',
'JV11613A',
'JT01613M',
'JT11613M',
'JV11613M',
'JZ01617M',
'JZ11617M',
'JM01701V',
'JR01701V',
'JM01705M',
'JR01705M',
'JM01706A',
'JR01706A',
'JAA197H',
'JBB197H',
'JXA222M',
'JXB222M',
'JX0251O',
'JXA251O',
'JBA309H',
'JBB309H',
'JA0326H',
'JAA326H',
'JAA328I',
'JAB328I',
'JAD328I',
'JB0328I',
'JF0328I',
'JA0334M',
'JBA334M',
'JBB334M',
'JB0347M',
'JBA347M',
'JB0389M',
'JBC389M',
'JX0424M',
'JXA424M',
'JBA454I',
'JF0454I',
'JA0513F',
'JF0513F',
'JU0531D',
'JX0531D',
'JX0539M',
'JXA539M',
'KA01774D',
'KF01774D',
'KB01778C',
'KD01778C',
'KA01820S',
'KF01820S',
'KA01971C',
'KB01971C',
'KBA1971C',
'KE01971C',
'KB01976D',
'KD01976D',
'KB01977C',
'KF01977C',
'KB01980D',
'KD01980D',
'KB01989C',
'KD01989C',
'KF01989C',
'KB02050S',
'KD02050S',
'KF02050S',
'KB02069S',
'KBA2069S',
'KF02069S',
'KA02084G',
'KAA2084G',
'KF02084G',
'KA02084S',
'KAA2084S',
'KE02084S',
'KA02143G',
'KF02143G',
'KB02151G',
'KF02151G',
'KA02248C',
'KAA2248C',
'KE02248C',
'KA02252G',
'KAA2252G',
'KE02252G',
'KA02280C',
'KAA2280C',
'KE02280C',
'KA02305G',
'KAA2305G',
'KE02305G',
'KA02327G',
'KAA2327G',
'KE02327G',
'KA02328S',
'KAA2328S',
'KE02328S',
'KB02330G',
'KBA2330G',
'KE02330G',
'KA02396G',
'KAA2396G',
'KE02396G',
'KB02399G',
'KF02399G',
'KA02464G',
'KF02464G',
'KB02516G',
'KE02516G',
'KB02517G',
'KE02517G',
'KB02537G',
'KE02537G',
'KB02556G',
'KE02556G',
'KB02565G',
'KE02565G',
'KB02566G',
'KE02566G',
'KA02585G',
'KE02585G',
'KB02589G',
'KE02589G',
'KB02674C',
'KE02674C',
'KB02678D',
'KE02678D',
'KB02686G',
'KE02686G',
'KB02721C',
'KE02721C',
'KB02729C',
'KE02729C',
'KB02731S',
'KE02731S',
'KB02732G',
'KE02732G',
'KA02857S',
'KAA2857S',
'KE02857S',
'KB02869G',
'KE02869G',
'KB02917K',
'KE02917K',
'KA03109G',
'KE03109G',
'KA03186S',
'KE03186S',
'KAA587S',
'KEA587S',
'AACC416A0124',
'AACC416F0124',
'AACC416K0124',
'AACC584A0125',
'AACC584G0125',
'AACC584K0125',
'BACC416A0124',
'BACC416K0124',
'BACC584A0125',
'BACC584H0125',
'BACC584J0125',
'BACC584K0125',
'GACC416A0124',
'GACC416F0124',
'GACC584A0125',
'GACC584H0125',
'GACC584J0125',
'GACC584K0125',
'32313139030',
'82313139030',
'BBCC395A0124',
'BBCC395C0124',
'BBCC395C0225',
'BBCC395E0124',
'BBCC395J0124',
'BBCC395P0124',
'BBCC589C0225',
'GBCC395A0124',
'GBCC395C0124',
'GBCC395C0225',
'GBCC395E0124',
'GBCC395J0124',
'GBCC395P0124',
'GBCC589C0225',
'BBCC407A1223',
'BBCC407K1223',
'GBCC407A1223',
'GBCC407K1223',
'AACC406A0124',
'AACC406G0124',
'AACC406J0124',
'ABCC410A0124',
'ABCC410K0124',
'ABCC597E0125',
'ABCC597G0125',
'ABCC597H0125',
'ABCC597J0125',
'11313129049',
'11313219049',
'11313269049',
'31313189050',
'31313259050',
'31313269050',
'81313189050',
'81313259050',
'81313269050',
'32313189031',
'32313219031',
'32313239031',
'82313189031',
'82313219031',
'82313239031',
'11313189067',
'11313259067',
'11313269067',
'31313219069',
'31313259069',
'31313269069',
'81313219069',
'81313259069',
'81313269069',
'12313189033',
'12313229033',
'12313239033',
'ABCC415E0124',
'ABCC415J0124',
'ABCC415P0124',
'BACC414E0124',
'BACC414G0124',
'GACC414E0124',
'GACC414G0124',
'AACC469A0224',
'AACC469E0224',
'BACC470A0224',
'BACC470E0224',
'BACC470J0224',
'GACC470A0224',
'GACC470E0224',
'GACC470J0224',
'BACC587E0225',
'BACC587J0225',
'12313149029',
'12313259029',
'ABCC408A0225',
'ABCC408C0225',
'ABCC408P0225',
'11313189001',
'11313239001',
'11313259001',
'21313219001',
'21313279001',
'31313219001',
'31313279001',
'BACC612K0225',
'GACC612K0225',
'BADC480C0224',
'BADC480F0224',
'BADC480J0224',
'GADC480C0224',
'GADC480F0224',
'BADC272D0423',
'BADC272H0423',
'BADC272J0423',
'BBDC411C0124',
'BBDC411J0124',
'BBDC411P0124',
'GBDC411C0124',
'GBDC411J0124',
'GBDC411P0124',
'ABDC408A0124',
'ABDC408C0124',
'ABDC408P0124',
'AADC390G0124',
'AADC390J0124',
'AADC390P0124',
'AADC391J0124',
'BADC396A0124',
'BADC396K0124',
'GADC396A0124',
'GADC396K0124',
'BADC392H0124',
'BADC392J0124',
'BADC392P0124',
'BADC393P0124',
'BADC394H0124',
'GADC392H0124',
'GADC392J0124',
'GADC392P0124',
'GADC393P0124',
'GADC394H0124',
'AADC409A0124',
'AADC409K0124',
'BADC585C0125',
'BADC585G0125',
'BADC585J0125',
'11323129037',
'11323269037',
'31323429060',
'31323429061',
'31323429062',
'81323429060',
'81323429061',
'81323429062',
'11323419056',
'11323419057',
'11323419058',
'11323419059',
'11323129068',
'11323239068',
'11323269068',
'31323129070',
'31323239070',
'31323269070',
'81323129070',
'81323239070',
'81323269070',
'12323139032',
'12323149032',
'12323139034',
'12323239034',
'12323259034',
'81323129051',
'81323269051',
'AADC439D0224',
'AADC439H0224',
'AADC439J0224',
'AADC440J0224',
'BADC439J0224',
'BADC441D0224',
'BADC441H0224',
'BADC441J0224',
'GADC439J0224',
'GADC441D0224',
'GADC441H0224',
'GADC441J0224',
'AADC178H0123',
'AADC473F0224',
'AADC473H0224',
'BADC478K0224',
'BADC479J0224',
'GADC478K0224',
'GADC479J0224',
'BADC594L0225',
'BADC595L0225',
'BADC596L0225',
'AADC590M0225',
'AADC591M0225',
'AADC592M0225',
'AADC593M0225',
'ABDC410A0225',
'ABDC410C0225',
'ABDC410H0126',
'ABDC410K0225',
'BBDC410A0225',
'BBDC410C0126',
'BBDC410H0225',
'BBDC410K0225',
'GBDC410A0225',
'GBDC410C0126',
'GBDC410H0225',
'GBDC410K0225',
'BADC611K0225',
'GADC611K0225',
'BBDC441E0225',
'BBDC441H0225',
'BBDC441I0225',
'BBDC588G0225',
'GBDC441E0225',
'GBDC441H0225',
'GBDC441I0225',
'GBDC588G0225',
'ABDC439J0225',
'ABDC441D0225',
'ABDC441G0225',
'ABDC441H0225',
'LT01401F',
'LT21401F',
'LT01401P',
'LT21401P',
'LV01401P',
'LT21411S',
'LV21411S',
'LT01413G',
'LV01413G',
'LT01413S',
'LV01413S',
'LAA1504G',
'LD01504G',
'LF01504G',
'LAA1506S',
'LD01506S',
'LF01506S',
'LAA1507S',
'LF01507S',
'LBA1511S',
'LD01511S',
'LF01511S',
'LBA1512S',
'LF01512S',
'LA01594S',
'LF01594S',
'LA01595S',
'LF01595S',
'LA01599S',
'LF01599S',
'LT01605G',
'LT11605G',
'LV01605G',
'LV11605G',
'LZ01605G',
'LT01605S',
'LT11605S',
'LV01605S',
'LV11605S',
'LZ01605S',
'LB01607S',
'LT11607S',
'LB01611G',
'LF01611G',
'LZ11611G',
'LA01661S',
'LF01661S',
'LAA1688G',
'LF01688G',
'LB01699S',
'LBA1699S',
'LF01699S',
'LM01701G',
'LR01701G',
'LB01853S',
'LF01853S',
'LB01854G',
'LD01854G',
'LF01854G',
'LB01855S',
'LD01855S',
'LF01855S',
'LB01856G',
'LF01856G',
'LB01908G',
'LF01908G',
'LA01917G',
'LF01917G',
'LA1201G',
'LE1201G',
'LA02052G',
'LAA2052G',
'LA02085G',
'LAA2085G',
'LE02085G',
'LA02085S',
'LAA2085S',
'LF02085S',
'LA02086G',
'LAA2086G',
'LE02086G',
'LA02086S',
'LAA2086S',
'LF02086S',
'LBA2089S',
'LF02089S',
'LB02090S',
'LBA2090S',
'LF02090S',
'LB02091G',
'LBA2091G',
'LA02124G',
'LF02124G',
'LB02146S',
'LBA2146S',
'LFA2146S',
'LB02149G',
'LBA2149G',
'LF02149G',
'LFA2149G',
'LB02150S',
'LBA2150S',
'LF02150S',
'LA02167G',
'LAA2167G',
'LF02167G',
'LA02179S',
'LAA2179S',
'LE02179S',
'LB02182S',
'LBA2182S',
'LF02182S',
'LB02183S',
'LF02183S',
'LB02185S',
'LBA2185S',
'LE02185S',
'LF02185S',
'LB02186S',
'LBA2186S',
'LE02186S',
'LF02186S',
'LAA2209S',
'LE02209S',
'LA02239G',
'LAA2239G',
'LF02239G',
'LB02242S',
'LBA2242S',
'LE02242S',
'LB02243S',
'LBA2243S',
'LF02243S',
'LA02253G',
'LAA2253G',
'LF02253G',
'LB02255S',
'LBA2255S',
'LE02255S',
'LF02255S',
'LB02257G',
'LBA2257G',
'LE02257G',
'LA02306S',
'LAA2306S',
'LE02306S',
'LB02309S',
'LBA2309S',
'LE02309S',
'LA02329S',
'LAA2329S',
'LE02329S',
'LA1234S',
'LE1234S',
'LB02350G',
'LF02350G',
'LB02383S',
'LBA2383S',
'LE02383S',
'LB02384S',
'LBA2384S',
'LE02384S',
'LA02386S',
'LAA2386S',
'LB02401S',
'LBA2401S',
'LE02401S',
'LB02414S',
'LBA2414S',
'LE02414S',
'LF02414S',
'LB02415S',
'LF02415S',
'LB02416S',
'LBA2416S',
'LE02416S',
'LF02416S',
'LB02418S',
'LBA2418S',
'LE02418S',
'LF02418S',
'LB02434G',
'LBA2434G',
'LE02434G',
'LF02434G',
'LB02434S',
'LE02434S',
'LA02465G',
'LE02465G',
'LA02465S',
'LAA2465S',
'LE02465S',
'LA02469S',
'LAA2469S',
'LE02469S',
'LB02471S',
'LF02471S',
'LB02472S',
'LF02472S',
'LA02480G',
'LE02480G',
'LA02480S',
'LAA2480S',
'LE02480S',
'LA02497S',
'LE02497S',
'LB02549G',
'LE02549G',
'LB02583G',
'LE02583G',
'LB02584G',
'LE02584G',
'LB02619G',
'LE02619G',
'LB02620S',
'LE02620S',
'LB02621S',
'LE02621S',
'LB02622S',
'LE02622S',
'LB02623G',
'LE02623G',
'LB02624G',
'LE02624G',
'LB02677G',
'LE02677G',
'LA02680G',
'LE02680G',
'LA02680S',
'LE02680S',
'LB02718S',
'LE02718S',
'LB02730G',
'LE02730G',
'LB02740S',
'LE02740S',
'LA02747S',
'LAA2747S',
'LB02747S',
'LE02747S',
'LB02748S',
'LBA2748S',
'LE02748S',
'LB02784S',
'LE02784S',
'LB02785G',
'LE02785G',
'LA02795S',
'LAA2795S',
'LB02797S',
'LE02797S',
'LB02799S',
'LE02799S',
'LB02800S',
'LE02800S',
'LB02801S',
'LE02801S',
'LA02858S',
'LAA2858S',
'LB02858S',
'LB02865G',
'LE02865G',
'LB02866S',
'LE02866S',
'LB02867S',
'LE02867S',
'LB02868S',
'LE02868S',
'LB02876G',
'LE02876G',
'LB02877G',
'LE02877G',
'LB02878S',
'LE02878S',
'LB02879S',
'LE02879S',
'LB02881S',
'LE02881S',
'LA02901S',
'LB02901S',
'LB02911S',
'LE02911S',
'LB02912S',
'LE02912S',
'LB02924S',
'LE02924S',
'LB02925G',
'LE02925G',
'LB02940G',
'LE02940G',
'LB02941G',
'LBA2941G',
'LE02941G',
'LB02942S',
'LE02942S',
'LB02992S',
'LBA2992S',
'LE02992S',
'LB02993S',
'LE02993S',
'LA02995S',
'LB02995S',
'LE02995S',
'LB03003S',
'LE03003S',
'LB03004S',
'LE03004S',
'LB03006S',
'LE03006S',
'LB03007G',
'LE03007G',
'LA03014R',
'LB03014R',
'LE03014R',
'LB03034G',
'LBA3034G',
'LE03034G',
'LB03068S',
'LE03068S',
'LB03091S',
'LE03091S',
'LB03092S',
'LE03092S',
'LB03093G',
'LE03093G',
'LB03093S',
'LE03093S',
'LB03094G',
'LE03094G',
'LB03095G',
'LE03095G',
'LA03101S',
'LE03101S',
'LA03102S',
'LE03102S',
'LA03103S',
'LE03103S',
'LA03104S',
'LE03104S',
'LB03105S',
'LE03105S',
'LB03106S',
'LE03106S',
'LB03107G',
'LE03107G',
'LB03108G',
'LE03108G',
'LA03110G',
'LE03110G',
'LA03111G',
'LE03111G',
'LB03112G',
'LE03112G',
'LB03113S',
'LE03113S',
'LB03114S',
'LE03114S',
'LA03129S',
'LE03129S',
'LA03130S',
'LE03130S',
'LA03134S',
'LE03134S',
'LA03135G',
'LE03135G',
'LB03136G',
'LE03136G',
'LB03137G',
'LE03137G',
'LB03139S',
'LE03139S',
'LA03176S',
'LE03176S',
'LA03177S',
'LE03177S',
'LA03178G',
'LE03178G',
'LB03179S',
'LE03179S',
'LA03181S',
'LE03181S',
'LB03185S',
'LE03185S',
'LA03202S',
'LE03202S',
'LB03203S',
'LE03203S',
'LB03204S',
'LE03204S',
'LB03205S',
'LE03205S',
'LB03210G',
'LE03210G',
'LA03216R',
'LE03216R',
'LB03218S',
'LE03218S',
'LB03220R',
'LE03220R',
'LB03272D',
'LE03272D',
'LA03283D',
'LE03283D',
'LA03341R',
'LE03341R',
'LA03343S',
'LE03343S',
'LA03345G',
'LE03345G',
'LB03350S',
'LE03350S',
'LB03351S',
'LE03351S',
'LB03352S',
'LE03352S',
'LB03355S',
'LE03355S',
'LA03358S',
'LE03358S',
'LA03359S',
'LE03359S',
'LB03361R',
'LE03361R',
'LB03363G',
'LE03363G',
'LA03387S',
'LE03387S',
'LA03388S',
'LE03388S',
'LA03389S',
'LE03389S',
'LA03391R',
'LE03391R',
'LB03434G',
'LE03434G',
'LB03435F',
'LE03435F',
'LA03447S',
'LE03447S',
'LA03449S',
'LE03449S',
'LB03451R',
'LE03451R',
'32213259062',
'82213259062',
'BBAC492A1125',
'BBAC492C1024',
'BBAC492D1125',
'BBAC492H1024',
'BBAC492J1024',
'BBAC492K1125',
'GBAC492A1125',
'GBAC492C1024',
'GBAC492D1125',
'GBAC492H1024',
'GBAC492J1024',
'GBAC492K1125',
'BBAC293D1123',
'BBAC293J1123',
'BBAC293P1123',
'ABAC294A1123',
'ABAC294D1123',
'ABAC294J1123',
'ABAC294P1123',
'BBAC360D1123',
'BBAC360P1123',
'GBAC360D1123',
'GBAC360P1123',
'ABAC491H1124',
'ABAC491J1124',
'ABAC494A1125',
'ABAC494D1125',
'ABAC494J1124',
'ABAC494K1125',
'31213129027',
'31213169027',
'31213269027',
'81213129027',
'81213169027',
'81213269027',
'AAAC507A1224',
'AAAC507D1125',
'AAAC507K1224',
'32213429025',
'82213429025',
'31213429048',
'81213429048',
'32213429039',
'32213429040',
'82213429039',
'82213429040',
'MAB1264K',
'MF01264K',
'MK21405L',
'MZ01405L',
'MT21411S',
'MV21411S',
'MBA1546S',
'MF01546S',
'MBA1556C',
'MF01556C',
'MBA1573G',
'MF01573G',
'MA1158G',
'ME1158G',
'MT01611K',
'MV01611K',
'MZ01611K',
'MT01611L',
'MV01611L',
'MZ01611L',
'MBA1616S',
'MF01616S',
'MB01725L',
'MF01725L',
'MB01733L',
'MF01733L',
'MB01738L',
'MF01738L',
'MA01756L',
'MF01756L',
'MA01772D',
'MAA1772D',
'MF01772D',
'MB01775C',
'MD01775C',
'MF01775C',
'MB01776D',
'MD01776D',
'MF01776D',
'MA01857S',
'MAA1857S',
'MF01857S',
'MB01866S',
'MBA1866S',
'MBB1866S',
'ME01866S',
'MF01866S',
'MA01871G',
'MAA1871G',
'MF01871G',
'MA01874S',
'MB01874S',
'MF01874S',
'MB01880S',
'MF01880S',
'MA01906S',
'MF01906S',
'MA01907G',
'MF01907G',
'MB01920S',
'MF01920S',
'MB01921S',
'MF01921S',
'MB01922S',
'MF01922S',
'MBA1963S',
'MF01963S',
'MB01974C',
'MD01974C',
'MB01975D',
'MD01975D',
'MB01979D',
'MD01979D',
'MF01979D',
'MA01981D',
'MAA1981D',
'MA01982C',
'MAA1982C',
'MB01983D',
'MD01983D',
'MF01983D',
'MB01984D',
'MBA1984D',
'MD01984D',
'MF01984D',
'MA1208S',
'ME1208S',
'MA02125S',
'MAA2125S',
'MF02125S',
'MB1212G',
'ME1212G',
'MA02144G',
'MAA2144G',
'MF02144G',
'MB02145G',
'MF02145G',
'MA02166G',
'MF02166G',
'MB02197D',
'MF02197D',
'MA02249C',
'MAA2249C',
'MA02249D',
'MAA2249D',
'MA02250D',
'MB02250D',
'MA02251D',
'MAA2251D',
'MB02254C',
'MBA2254C',
'ME02254C',
'MF02254C',
'MA02254D',
'MB02254D',
'ME02254D',
'MB02351G',
'MF02351G',
'MB02417S',
'MF02417S',
'MA02451S',
'MAA2451S',
'ME02451S',
'MA02452G',
'MAA2452G',
'ME02452G',
'MB02453S',
'MF02453S',
'MB02672D',
'ME02672D',
'MB02673D',
'ME02673D',
'MA02679C',
'MB02679C',
'ME02679C',
'MB02679D',
'ME02679D',
'MA02782S',
'MAA2782S',
'MA02786S',
'MAA2786S',
'ME02786S',
'MB02863G',
'ME02863G',
'MB02864G',
'ME02864G',
'MB02913D',
'ME02913D',
'MB02914D',
'ME02914D',
'MB02915D',
'ME02915D',
'MA02920D',
'MB02920D',
'MBA2920D',
'MB02926K',
'ME02926K',
'MA02938S',
'MAA2938S',
'MB02938S',
'MB02994S',
'MBA2994S',
'ME02994S',
'MA03001G',
'MAA3001G',
'MA03012R',
'MB03012R',
'MA03028G',
'MB03028G',
'MA03044S',
'MB03044S',
'ME03044S',
'MA03049G',
'MAA3049G',
'ME03049G',
'MA03132G',
'ME03132G',
'MA03133S',
'ME03133S',
'MA03180S',
'MB03180S',
'ME03180S',
'MA03183S',
'ME03183S',
'MA03209G',
'ME03209G',
'MA03247R',
'MB03247R',
'ME03247R',
'MA03248C',
'MB03248C',
'MB03252S',
'ME03252S',
'MA03259D',
'ME03259D',
'MA03265D',
'ME03265D',
'MA03271D',
'MB03271D',
'MB03273D',
'ME03273D',
'MB03274D',
'ME03274D',
'MA03280D',
'ME03280D',
'MA03281D',
'ME03281D',
'MB03285L',
'ME03285L',
'MA03304D',
'ME03304D',
'MB03310D',
'ME03310D',
'MA03344R',
'ME03344R',
'MB03353R',
'ME03353R',
'MB03354R',
'ME03354R',
'MA03357R',
'ME03357R',
'MA03392S',
'ME03392S',
'MA0340G',
'MAA340G',
'MA0342G',
'MA3342G',
'MAB342G',
'MA3342S',
'ME3342S',
'MA03432S',
'ME03432S',
'MB03436R',
'ME03436R',
'MB03437S',
'ME03437S',
'MB1344G',
'ME1344G',
'MA03450S',
'ME03450S',
'MA03465F',
'MB03465F',
'MA03466R',
'MB03466R',
'MAE379S',
'ME1379S',
'MA1380G',
'ME1380G',
'MF1380G',
'MA0750S',
'ME0750S',
'MA0826S',
'MAA826S',
'MB0917G',
'MBA917G',
'MA0955L',
'MB0955L',
'PX00875D',
'PXA0875D',
'PA01030M',
'PB01030M',
'PF01070V',
'PX01070V',
'PF01091D',
'PU01091D',
'PF01114D',
'PX01114D',
'PD01143D',
'PF01143D',
'PX01143D',
'PE01449E',
'PY01449E',
'PF01505D',
'PU01505D',
'PX01505D',
'PF01514D',
'PU01514D',
'PX01514D',
'PE01571E',
'PW01571E',
'PE01579E',
'PY01579E',
'PE01580E',
'PY01580E',
'PE01582E',
'PW01582E',
'PE01583E',
'PW01583E',
'PE01584E',
'PY01584E',
'PE01614E',
'PEA1614E',
'PEB1614E',
'PEC1614E',
'PF01614E',
'PH01614E',
'PW01614E',
'PWA1614E',
'PWB1614E',
'PE01614M',
'PW01614M',
'PE01615E',
'PEA1615E',
'PEB1615E',
'PEC1615E',
'PF01615E',
'PG01615E',
'PGA1615E',
'PGB1615E',
'PY01615E',
'PE01615M',
'PY01615M',
'PE01622E',
'PEA1622E',
'PEB1622E',
'PF01622E',
'PH01622E',
'PW01622E',
'PWA1622E',
'PWB1622E',
'PE01622M',
'PW01622M',
'PE01623E',
'PEA1623E',
'PEB1623E',
'PEC1623E',
'PF01623E',
'PG01623E',
'PGA1623E',
'PGB1623E',
'PY01623E',
'PE01623M',
'PY01623M',
'PE01630E',
'PEA1630E',
'PEB1630E',
'PF01630E',
'PH01630E',
'PW01630E',
'PWA1630E',
'PWB1630E',
'PE01630M',
'PW01630M',
'PE01631E',
'PEB1631E',
'PF01631E',
'PG01631E',
'PGA1631E',
'PY01631E',
'PE01631M',
'PY01631M',
'PF01668E',
'PG01668E',
'PF01669E',
'PG01669E',
'PF01670E',
'PH01670E',
'PE01671D',
'PX01671D',
'PE01675D',
'PU01675D',
'PX01675D',
'PE01676D',
'PU01676D',
'PX01676D',
'PE01677D',
'PU01677D',
'PX01677D',
'PE01697C',
'PX01697C',
'PXA1697C',
'PE01697W',
'PX01697W',
'PXA1697W',
'PE01698C',
'PX01698C',
'PXA1698C',
'PE01698W',
'PX01698W',
'PXA1698W',
'PE01699C',
'PU01699C',
'PE01700C',
'PU01700C',
'PE01713C',
'PU01713C',
'PE01714C',
'PU01714C',
'PE01724E',
'PY01724E',
'PE01764C',
'PU01764C',
'PE01765C',
'PU01765C',
'PE01795W',
'PU01795W',
'PE01796W',
'PU01796W',
'PE01797W',
'PU01797W',
'PE01798E',
'PEA1798E',
'PY01798E',
'PYA1798E',
'PE01799E',
'PEA1799E',
'PY01799E',
'PYA1799E',
'PE01800E',
'PEA1800E',
'PY01800E',
'PYA1800E',
'PE01826W',
'PU01826W',
'PE01827W',
'PU01827W',
'PE01828W',
'PU01828W',
'PE01860C',
'PU01860C',
'PE01861C',
'PU01861C',
'PE01862C',
'PU01862C',
'PE01863C',
'PU01863C',
'PE01864C',
'PX01864C',
'PE01865C',
'PX01865C',
'PE01866C',
'PX01866C',
'PE01867C',
'PX01867C',
'PE01875C',
'PX01875C',
'PE01876C',
'PX01876C',
'PE01877C',
'PX01877C',
'PE01878C',
'PX01878C',
'PX01881C',
'PXA1881C',
'PE01882W',
'PU01882W',
'PE01891C',
'PEA1891C',
'PU01891C',
'PUA1891C',
'PE01898W',
'PU01898W',
'PE01971C',
'PX01971C',
'PE01972C',
'PX01972C',
'PE01973W',
'PX01973W',
'PE01974W',
'PX01974W',
'PE01975W',
'PX01975W',
'PE01976W',
'PX01976W',
'PE01995W',
'PU01995W',
'PE01996W',
'PU01996W',
'PE02017W',
'PX02017W',
'PE02034W',
'PU02034W',
'PE02035W',
'PU02035W',
'PE02036W',
'PU02036W',
'PE02049C',
'PU02049C',
'PE02050C',
'PU02050C',
'PE02082W',
'PU02082W',
'PE02083W',
'PU02083W',
'PE02085W',
'PU02085W',
'31423129324',
'31423169323',
'31423189324',
'31423219051',
'31423219324',
'31423259051',
'31423299324',
'81423129324',
'81423169323',
'81423189324',
'81423219051',
'81423219324',
'81423259051',
'81423299324',
'BAFC323D0125',
'BAFC324A0125',
'BAFC324C0125',
'BAFC324F0125',
'BAFC324G0125',
'BAFC324G0224',
'BAFC324H0224',
'BAFC324J0125',
'BAFC324J0224',
'BAFC324K0125',
'BAFC324P0125',
'BAFC393H0125',
'GAFC323D0125',
'GAFC324A0125',
'GAFC324C0125',
'GAFC324F0125',
'GAFC324G0125',
'GAFC324G0224',
'GAFC324H0224',
'GAFC324J0125',
'GAFC324J0224',
'GAFC324K0125',
'GAFC324P0125',
'GAFC393H0125',
'11423129323',
'11423169323',
'11423219040',
'11423219324',
'11423239040',
'11423239323',
'11423259323',
'11423269323',
'11423279323',
'11423299323',
'AAFC323A0125',
'AAFC323C0125',
'AAFC323D0125',
'AAFC323G0224',
'AAFC323H0125',
'AAFC323H0224',
'AAFC323J0125',
'AAFC323J0224',
'AAFC323K0125',
'AAFC323P0125',
'AAFC324G0224',
'AAFC393H0125',
'21423219008',
'21423239008',
'21423259008',
'31423219008',
'31423239008',
'31423259008',
'11423129007',
'11423189007',
'11423259007',
'11423219006',
'11423229006',
'11423239006',
'11423259006',
'21423189010',
'21423239010',
'21423259010',
'31423189010',
'31423229035',
'31423239010',
'31423259010',
'81423189010',
'81423229035',
'81423239010',
'12428129018',
'12428179018',
'BAFC336F0324',
'BAFC336H0324',
'BAFC336J0324',
'GAFC336F0324',
'GAFC336H0324',
'GAFC336J0324',
'AAFC336F0324',
'AAFC336G0324',
'AAFC336J0324',
'31423169013',
'31423269013',
'81423169013',
'81423269013',
'BAFC226F1122',
'BAFC226H1122',
'BAFC226J1023',
'BAFC226K1023',
'BAFC226P1023',
'BAFC226P1122',
'AAFC291H1123',
'AAFC291J1123',
'AAFC291K1123',
'11423419014',
'11423419015',
'11423129033',
'11423189033',
'11423259033',
'AAFC293D1223',
'AAFC293I1223',
'AAFC293P1223',
'BAFC294A1223',
'BAFC294D1223',
'BAFC294P1223',
'BAFC322P1223',
'11423419016',
'11423419017',
'11423419043',
'11423419044',
'QE00555R',
'QUB0555R',
'QN00664S',
'QX00664S',
'QX00838V',
'QXA0838V',
'QX00934K',
'QXA0934K',
'QXB0934K',
'QE00936S',
'QX00936S',
'QU00947V',
'QX00947V',
'QE00980R',
'QUA0980R',
'QE01017R',
'QX01017R',
'QY01022R',
'QYA1022R',
'QYB1022R',
'QYC1022R',
'QYD1022R',
'QA01025R',
'QB01025R',
'QU01033V',
'QX01033V',
'QE01048R',
'QY01048R',
'QYA1048R',
'QYB1048R',
'QE01049R',
'QEA1049R',
'QY01049R',
'QYA1049R',
'QYB1049R',
'QYC1049R',
'QE01052F',
'QU01052F',
'QUA1052F',
'QE01053R',
'QU01053R',
'QE01066V',
'QU01066V',
'QUA1066V',
'QE01068R',
'QU01068R',
'QX01074V',
'QXA1074V',
'QUA1075V',
'QX01075V',
'QX01102K',
'QXA1102K',
'QE01103F',
'QX01103F',
'QE01107F',
'QUA1107F',
'QUB1107F',
'QE01120R',
'QX01120R',
'QE01137R',
'QU01137R',
'QE01153V',
'QU01153V',
'QUA1153V',
'QUB1153V',
'QE01160R',
'QU01160R',
'QE01161R',
'QU01161R',
'QE01162R',
'QU01162R',
'QE01163R',
'QU01163R',
'QE01164R',
'QEA1164R',
'QY01164R',
'QYA1164R',
'QYB1164R',
'QYC1164R',
'QY01165R',
'QYA1165R',
'QY01166R',
'QYA1166R',
'QY01167R',
'QYA1167R',
'QE01170R',
'QEA1170R',
'QW01170R',
'QY01170R',
'QYA1170R',
'QYB1170R',
'QYC1170R',
'QYD1170R',
'QE01171R',
'QEA1171R',
'QY01171R',
'QYA1171R',
'QYB1171R',
'QE01173R',
'QEA1173R',
'QY01173R',
'QYA1173R',
'QYC1173R',
'QX01177R',
'QXA1177R',
'QX01178R',
'QXA1178R',
'QX01182R',
'QXA1182R',
'QE01185R',
'QX01185R',
'QX01186K',
'QXA1186K',
'QX01187K',
'QXA1187K',
'QX01188K',
'QXA1188K',
'QE01190R',
'QEA1190R',
'QU01190R',
'QUA1190R',
'QUB1190R',
'QUC1190R',
'QE01191R',
'QEA1191R',
'QU01191R',
'QUA1191R',
'QUB1191R',
'QUC1191R',
'QE01192R',
'QU01192R',
'QE01193R',
'QU01193R',
'QUA1193R',
'QUB1193R',
'QB01198R',
'QE01198R',
'QB01199R',
'QE01199R',
'QE01200R',
'QU01200R',
'QE01201R',
'QU01201R',
'QE01202R',
'QU01202R',
'QE01203R',
'QU01203R',
'QE01204R',
'QU01204R',
'QE01205R',
'QU01205R',
'QE01206R',
'QU01206R',
'QE01207R',
'QU01207R',
'QB01217R',
'QBA1217R',
'QBB1217R',
'QBC1217R',
'QEA1217R',
'QEB1217R',
'QB01218R',
'QBA1218R',
'QBB1218R',
'QE01218R',
'QEB1218R',
'QA01220R',
'QAA1220R',
'QX01227W',
'QXA1227W',
'QE01228W',
'QX01228W',
'QXA1228W',
'QXB1228W',
'QU01233R',
'QUA1233R',
'QX01237R',
'QXA1237R',
'QE01244R',
'QU01244R',
'QU01245K',
'QUA1245K',
'QU01246K',
'QUA1246K',
'QU01247K',
'QUA1247K',
'QE01249R',
'QU01249R',
'QE01250R',
'QU01250R',
'QE01251R',
'QU01251R',
'QX01255V',
'QXA1255V',
'QE01260R',
'QU01260R',
'QE01261R',
'QU01261R',
'QX01263V',
'QXA1263V',
'QE01271V',
'QX01271V',
'QXA1271V',
'QXB1271V',
'QE01272V',
'QEA1272V',
'QU01272V',
'QUA1272V',
'QE01275R',
'QU01275R',
'QX01284V',
'QXA1284V',
'QX01285V',
'QXA1285V',
'QU01286V',
'QUA1286V',
'QU01287V',
'QUA1287V',
'QU01288V',
'QX01288V',
'QXA1288V',
'QU01294B',
'QUA1294B',
'QE01295R',
'QU01295R',
'QE01300R',
'QX01300R',
'QXA1300R',
'QE01301B',
'QU01301B',
'QUA1301B',
'QE01302R',
'QU01302R',
'QX01306K',
'QXA1306K',
'QE01312V',
'QU01312V',
'QUA1312V',
'QX01317K',
'QXA1317K',
'QX01318K',
'QXA1318K',
'QE01321V',
'QU01321V',
'QE01322V',
'QEA1322V',
'QU01322V',
'QUA1322V',
'QE01323V',
'QEA1323V',
'QU01323V',
'QUA1323V',
'QE01324R',
'QX01324R',
'QE01325R',
'QX01325R',
'QE01327R',
'QEA1327R',
'QU01327R',
'QUA1327R',
'QE01329B',
'QEA1329B',
'QU01329B',
'QUA1329B',
'QUB1329B',
'QX01329B',
'QE01331J',
'QU01331J',
'QUA1331J',
'QF01335R',
'QH01335R',
'QW01335R',
'QWA1335R',
'QEA1357V',
'QU01357V',
'QUA1357V',
'QE01358V',
'QEA1358V',
'QU01358V',
'QUA1358V',
'QX01359R',
'QXA1359R',
'QE01360R',
'QX01360R',
'QXA1360R',
'QB01361R',
'QE01361R',
'QB01362R',
'QE01362R',
'QE01367R',
'QEA1367R',
'QY01367R',
'QYA1367R',
'QE01368R',
'QEA1368R',
'QW01368R',
'QY01368R',
'QYA1368R',
'QYB1368R',
'QE01370R',
'QEA1370R',
'QW01370R',
'QY01370R',
'QYA1370R',
'QYB1370R',
'QW01377R',
'QWA1377R',
'QW01382R',
'QWA1382R',
'QT01401R',
'QV01401R',
'QT01401V',
'QV01401V',
'QE01402V',
'QEA1402V',
'QU01402V',
'QUA1402V',
'QE01403V',
'QEA1403V',
'QU01403V',
'QUA1403V',
'QW01410R',
'QWA1410R',
'QT11411J',
'QV11411J',
'QT01411V',
'QT11411V',
'QZ21411V',
'QE01415R',
'QEA1415R',
'QEB1415R',
'QT21421X',
'QV21421X',
'QT11429Q',
'QV11429Q',
'QW01429R',
'QWA1429R',
'QE01432R',
'QEA1432R',
'QE01434R',
'QY01434R',
'QW01436K',
'QWA1436K',
'QWB1436K',
'QW01438K',
'QWA1438K',
'QWB1438K',
'QE01439V',
'QX01439V',
'QE01440V',
'QX01440V',
'QT11441J',
'QV01441J',
'QV11441J',
'QU01444J',
'QX01444J',
'QE01450W',
'QX01450W',
'QXA1450W',
'QE01453V',
'QU01453V',
'QE01454V',
'QU01454V',
'QU01458R',
'QX01458R',
'QE01465V',
'QU01465V',
'QE01466R',
'QU01466R',
'QUA1466R',
'QE01467R',
'QU01467R',
'QUA1467R',
'QE01468R',
'QW01468R',
'QE01469R',
'QY01469R',
'QE01470R',
'QW01470R',
'QE01471R',
'QW01471R',
'QE01476R',
'QW01476R',
'QE01477V',
'QU01477V',
'QE01478V',
'QU01478V',
'QX01482R',
'QXA1482R',
'QE01484R',
'QW01484R',
'QY01484R',
'QE01486R',
'QY01486R',
'QE01487R',
'QEA1487R',
'QW01487R',
'QY01487R',
'QYA1487R',
'QE01488V',
'QU01488V',
'QE01494R',
'QY01494R',
'QE01500V',
'QU01500V',
'QE01509R',
'QW01509R',
'QE01510R',
'QW01510R',
'QE01515V',
'QU01515V',
'QE01524R',
'QX01524R',
'QE01528V',
'QX01528V',
'QE01528W',
'QEA1528W',
'QU01528W',
'QX01528W',
'QXA1528W',
'QXB1528W',
'QXC1528W',
'QE01529V',
'QX01529V',
'QE01529W',
'QEA1529W',
'QEB1529W',
'QU01529W',
'QUA1529W',
'QUB1529W',
'QX01529W',
'QXA1529W',
'QXB1529W',
'QXC1529W',
'QE01530V',
'QX01530V',
'QE01530W',
'QEA1530W',
'QEB1530W',
'QU01530W',
'QUA1530W',
'QUB1530W',
'QX01530W',
'QXA1530W',
'QXB1530W',
'QXC1530W',
'QE01531R',
'QY01531R',
'QE01532V',
'QU01532V',
'QE01534V',
'QU01534V',
'QE01542R',
'QW01542R',
'QY01542R',
'QE01552R',
'QY01552R',
'QE01553R',
'QY01553R',
'QE01564R',
'QX01564R',
'QXA1564R',
'QE01569R',
'QU01569R',
'QUA1569R',
'QE01572V',
'QX01572V',
'QE01575V',
'QU01575V',
'QX01600F',
'QXA1600F',
'QE01600R',
'QEA1600R',
'QX01600R',
'QXA1600R',
'QXB1600R',
'QU01604V',
'QX01604V',
'QU01605V',
'QX01605V',
'QE01606V',
'QU01606V',
'QE01608R',
'QU01608R',
'QE01609R',
'QEA1609R',
'QEB1609R',
'QF01609R',
'QG01609R',
'QGA1609R',
'QGB1609R',
'QY01609R',
'QYA1609R',
'QE01610R',
'QEA1610R',
'QEB1610R',
'QF01610R',
'QG01610R',
'QGA1610R',
'QGB1610R',
'QY01610R',
'QYA1610R',
'QV11611R',
'QZ11611R',
'QT11613R',
'QV11613R',
'QT11613V',
'QV11613V',
'QZ11613V',
'QT11617H',
'QV11617H',
'QZ11617H',
'QE01617V',
'QU01617V',
'QE01618R',
'QX01618R',
'QXA1618R',
'QE01619R',
'QU01619R',
'QE01620R',
'QF01620R',
'QH01620R',
'QW01620R',
'QE01621R',
'QF01621R',
'QH01621R',
'QW01621R',
'QT11621X',
'QV11621X',
'QE01626V',
'QU01626V',
'QE01627R',
'QU01627R',
'QE01628R',
'QY01628R',
'QE01629R',
'QEA1629R',
'QF01629R',
'QG01629R',
'QGA1629R',
'QGB1629R',
'QY01629R',
'QU01633V',
'QX01633V',
'QU01634V',
'QX01634V',
'QE01636R',
'QX01636R',
'QXA1636R',
'QE01637R',
'QU01637R',
'QE01638R',
'QF01638R',
'QH01638R',
'QW01638R',
'QE01639R',
'QF01639R',
'QH01639R',
'QW01639R',
'QA01642V',
'QB01642V',
'QE01646R',
'QU01646R',
'QE01647K',
'QU01647K',
'QU01648V',
'QX01648V',
'QU01649V',
'QX01649V',
'QE01651R',
'QX01651R',
'QE01656W',
'QEA1656W',
'QEB1656W',
'QU01656W',
'QUA1656W',
'QUB1656W',
'QX01656W',
'QXA1656W',
'QX01659K',
'QXA1659K',
'QX01660K',
'QXA1660K',
'QX01661K',
'QXA1661K',
'QE01662K',
'QU01662K',
'QE01663K',
'QU01663K',
'QE01664R',
'QU01664R',
'QE01673W',
'QEA1673W',
'QX01673W',
'QXA1673W',
'QB01674R',
'QE01674R',
'QE01678R',
'QF01678R',
'QG01678R',
'QGA1678R',
'QGB1678R',
'QE01679R',
'QU01679R',
'QE01680W',
'QEA1680W',
'QU01680W',
'QUA1680W',
'QUB1680W',
'QX01680W',
'QE01683R',
'QU01683R',
'QE01685R',
'QEA1685R',
'QH01685R',
'QHA1685R',
'QHB1685R',
'QHC1685R',
'QE01686R',
'QEA1686R',
'QH01686R',
'QHA1686R',
'QHB1686R',
'QHC1686R',
'QE01687R',
'QH01687R',
'QHA1687R',
'QHB1687R',
'QE01688R',
'QEA1688R',
'QG01688R',
'QGA1688R',
'QE01689R',
'QEA1689R',
'QEB1689R',
'QG01689R',
'QGA1689R',
'QGB1689R',
'QE01690R',
'QEA1690R',
'QG01690R',
'QGA1690R',
'QE01691R',
'QEA1691R',
'QG01691R',
'QGA1691R',
'QGB1691R',
'QE01694R',
'QEA1694R',
'QH01694R',
'QHA1694R',
'QHB1694R',
'QHC1694R',
'QB01701R',
'QM01701R',
'QR01701R',
'QE01705W',
'QU01705W',
'QE01708R',
'QEA1708R',
'QU01708R',
'QUA1708R',
'QE01709R',
'QEA1709R',
'QU01709R',
'QUA1709R',
'QR01733R',
'QRA1733R',
'QE01734R',
'QG01734R',
'QE01735R',
'QG01735R',
'QE01736R',
'QG01736R',
'QE01737R',
'QG01737R',
'QE01740R',
'QG01740R',
'QE01748F',
'QEA1748F',
'QU01748F',
'QUA1748F',
'QE01749F',
'QEA1749F',
'QU01749F',
'QUA1749F',
'QE01750R',
'QW01750R',
'QE01751R',
'QW01751R',
'QE01752R',
'QW01752R',
'QA01753H',
'QAA1753H',
'QE01753H',
'QA01754H',
'QAA1754H',
'QE01754H',
'QA01755H',
'QAA1755H',
'QE01755H',
'QE01758W',
'QU01758W',
'QE01759W',
'QU01759W',
'QC0175X',
'QE0175X',
'QE01760W',
'QU01760W',
'QE01766W',
'QEA1766W',
'QU01766W',
'QUA1766W',
'QE01767W',
'QU01767W',
'QE01768F',
'QEA1768F',
'QU01768F',
'QUA1768F',
'QE01774R',
'QX01774R',
'QXA1774R',
'QE01775R',
'QEA1775R',
'QW01775R',
'QWA1775R',
'QWB1775R',
'QY01775R',
'QE01776R',
'QW01776R',
'QY01776R',
'QE01781F',
'QU01781F',
'QE01782F',
'QU01782F',
'QE01783F',
'QU01783F',
'QE01785R',
'QG01785R',
'QGA1785R',
'QE01786R',
'QG01786R',
'QE01787R',
'QG01787R',
'QE01790F',
'QX01790F',
'QXA1790F',
'QXB1790F',
'QE01792R',
'QW01792R',
'QE01793R',
'QW01793R',
'QE01794R',
'QW01794R',
'QE01804R',
'QG01804R',
'QE01805R',
'QG01805R',
'QE01806R',
'QG01806R',
'QE01807R',
'QG01807R',
'QE01808R',
'QG01808R',
'QE01809R',
'QG01809R',
'QE01817F',
'QU01817F',
'QE01818F',
'QEA1818F',
'QU01818F',
'QE01819F',
'QU01819F',
'QE01820F',
'QU01820F',
'QE01822W',
'QW01822W',
'QE01847W',
'QU01847W',
'QX01847W',
'QE01848W',
'QU01848W',
'QX01848W',
'QE01849W',
'QU01849W',
'QE01850W',
'QU01850W',
'QE01851F',
'QU01851F',
'QE01852F',
'QU01852F',
'QE01853F',
'QU01853F',
'QE01854F',
'QU01854F',
'QE01855R',
'QY01855R',
'QE01856W',
'QU01856W',
'QX01856W',
'QE01857W',
'QU01857W',
'QX01857W',
'QE01858W',
'QEA1858W',
'QU01858W',
'QX01858W',
'QXA1858W',
'QE01859W',
'QEA1859W',
'QU01859W',
'QX01859W',
'QXA1859W',
'QU01868B',
'QX01868B',
'QU01870B',
'QX01870B',
'QE01883W',
'QU01883W',
'QE01884F',
'QX01884F',
'QE01885F',
'QX01885F',
'QE01886F',
'QX01886F',
'QE01887F',
'QX01887F',
'QE01889F',
'QEA1889F',
'QY01889F',
'QYA1889F',
'QE01902W',
'QU01902W',
'QE01903C',
'QX01903C',
'QE01904L',
'QX01904L',
'QE01905C',
'QX01905C',
'QE01909R',
'QX01909R',
'QH01919W',
'QY01919W',
'QU01921W',
'QX01921W',
'QE01922W',
'QU01922W',
'QX01922W',
'QU01924W',
'QX01924W',
'QE01925R',
'QX01925R',
'QE01929W',
'QU01929W',
'QE01931W',
'QU01931W',
'QE01935W',
'QU01935W',
'QE01936W',
'QU01936W',
'QE01962F',
'QX01962F',
'QE01963F',
'QX01963F',
'QE01966F',
'QU01966F',
'QE01967F',
'QU01967F',
'QE01968F',
'QU01968F',
'QE01970F',
'QU01970F',
'QE02013W',
'QU02013W',
'QE02014W',
'QU02014W',
'QE02015W',
'QU02015W',
'QE02019R',
'QG02019R',
'QU02032W',
'QX02032W',
'QE02040F',
'QX02040F',
'QE02041F',
'QX02041F',
'QE02046W',
'QX02046W',
'QE02047W',
'QX02047W',
'QE02065W',
'QY02065W',
'QE02066W',
'QY02066W',
'QE02067W',
'QY02067W',
'QE02079W',
'QU02079W',
'QE02091R',
'QX02091R',
'QE02092R',
'QX02092R',
'QE02093R',
'QX02093R',
'QE02102W',
'QU02102W',
'QU02103V',
'QX02103V',
'QU02104V',
'QX02104V',
'QU02131V',
'QX02131V',
'QU02132V',
'QX02132V',
'QN0468X',
'QX0468X',
'QE0491J',
'QX0491J',
'QA0551G',
'QAA551G',
'QB0551G',
'QUB554V',
'QXA554V',
'QX0565J',
'QXA565J',
'QA0569L',
'QAA569L',
'QI0909U',
'QJ0909U',
'RBA2869G',
'REA2869G',
'RB03036G',
'RBA3036G',
'RE03036G',
'RAA3065S',
'RE03065S',
'RB03458G',
'RE03458G',
'ARAC106H0422',
'ARAC106K0422',
'BRBC544B0524',
'BRBC544L0524',
'BRBC544M0524',
'BRBC609B0824',
'BRBC609L0824',
'BRBC609M0824',
'BRBC609N0824',
'DRDC084A032',
'DRDC084L032',
'ARHC206D0422',
'ARHC206H0422',
'ARHC206K0422',
'ARHC206P0422',
'BRRD014U012',
'BRRD014U051',
'BRUC008F0723',
'BRUC008H051',
'BRUC008H061',
'BRUC013G051',
'BRUC013G081',
'BRVC003A051',
'BRVC003A061',
'BRVC008A0523',
'BRVC008E022',
'BRVC008E0523',
'BRVC008F022',
'BRVC008F0423',
'BRVC008F0523',
'BRVC008G022',
'BRVC008G0523',
'BRVC008H022',
'BRVC008H0523',
'BRVC008J022',
'BRVC008K022',
'BRVC009H051',
'BRVC009H061',
'BRVC010H051',
'BRVC010H061',
'BRVC014G051',
'BRVC014G081',
'BRVC018G051',
'BRVC018G081',
'BRVC026G051',
'BRVC026G081',
'BRVC029F051',
'BRVC029F081',
'BRVC030F051',
'BRVC030F081',
'BRVC037F051',
'BRVC037F081',
'ARVD007A0823',
'ARVD007C0823',
'ARVD007F0823',
'BRXC007A022',
'BRXC007B022',
'BRXC007C022',
'BRXC007D022',
'BRXC007E022',
'BRXC007F022',
'BRXC007G022',
'BRXC007H022',
'BRXC007J022',
'BRXC008A022',
'BRXC008A0423',
'BRXC008A0523',
'BRXC008D022',
'BRXC008E022',
'BRXC008E0523',
'BRXC008F022',
'BRXC008F0423',
'BRXC008F0523',
'BRXC008G022',
'BRXC008G0523',
'BRXC008H022',
'BRXC008H0523',
'BRXC011H051',
'BRXC011H061',
'BRXC015G051',
'BRXC015G081',
'BRXC016G051',
'BRXC016G081',
'BRXC021H051',
'BRXC021H081',
'BRXC027G051',
'BRXC027G081',
'BRXC028G051',
'BRXC028G081',
'BRXC031F051',
'BRXC031F081',
'BRXC036E051',
'BRXC036E081',
'BRXC038F051',
'BRXC038F081',
'BRXC041F051',
'BRXC041F081',
'BRXC044C051',
'BRXC044C081',
'BRXD007F0823',
'BRXD007G0823',
'32223129008',
'32223139008',
'32223239008',
'32223269008',
'GABC490A0524',
'GABC490K0524',
'11223189002',
'11223239002',
'11223259002',
'AABC488E0126',
'AABC488F0624',
'AABC488J0624',
'21223179003',
'21223189003',
'21223259003',
'31223179003',
'31223189003',
'31223189084',
'31223259003',
'31223259084',
'81223179003',
'81223189003',
'81223189084',
'81223259003',
'81223259084',
'BABC489A0624',
'BABC489G0624',
'GABC489A0624',
'GABC489G0624',
'BBBC337A0723',
'BBBC337C0723',
'BABC353A0823',
'BABC353F0823',
'BABC353J0823',
'AABC353A0823',
'AABC354F0823',
'AABC354H0823',
'12223169014',
'12223269014',
'11223189356',
'AABC356F1123',
'AABC356H1123',
'AABC356J1123',
'AABC357A1024',
'BABC357A1123',
'BABC357E1123',
'BABC357F1123',
'GABC357A1123',
'GABC357E1123',
'GABC357F1123',
'AABC358D1123',
'AABC358P1123',
'BABC359D1123',
'BABC359P1123',
'GABC359D1123',
'GABC359P1123',
'BABC493D1124',
'BABC493F1124',
'BABC493G1124',
'BABC493J1124',
'GABC493D1124',
'GABC493F1124',
'GABC493G1124',
'GABC493J1124',
'AABC502F1124',
'AABC502J1124',
'AABC502P1124',
'AABC506A1124',
'AABC506D1124',
'AABC506F0925',
'AABC506G0925',
'AABC506J0925',
'AABC506K1124',
'31223189029',
'31223239029',
'31223259029',
'81223189029',
'81223239029',
'81223259029',
'BABC506A1124',
'BABC506D1125',
'BABC506K1124',
'GABC506A1124',
'GABC506D1125',
'GABC506K1124',
'BABC370L1223',
'GABC370L1223',
'11223189039',
'11223219039',
'11223259039',
'31223129040',
'31223189040',
'31223219040',
'81223129040',
'81223189040',
'81223219040',
'31223429043',
'81223429043',
'11223139032',
'11223169032',
'11223239032',
'AABC605A0125',
'AABC605C0125',
'AABC605D0825',
'AABC605F0125',
'AABC605G0125',
'AABC605H0125',
'AABC605J0125',
'AABC605K0125',
'AABC605P0125',
'31223139032',
'31223169032',
'81223139032',
'81223169032',
'BABC605A0125',
'BABC605C1125',
'BABC605D0925',
'BABC605F0125',
'BABC605G1125',
'BABC605K0125',
'BABC608G0125',
'BABC610G0125',
'BABC610H0125',
'GABC605A0125',
'GABC605C1125',
'GABC605D0925',
'GABC605F0125',
'GABC605G1125',
'GABC605K0125',
'GABC608G0125',
'GABC610H0125',
'11223429052',
'11223429053',
'11223429054',
'11223429055',
'31223419063',
'31223419064',
'31223419065',
'31223419066',
'81223419063',
'81223419064',
'81223419065',
'81223419066',
'31223429046',
'81223429046',
'31223429047',
'81223429047',
'TA00156F',
'TAA0156F',
'TAC0192F',
'TAY0192F',
'TAZ0192F',
'TBA0328F',
'TBY0328F',
'TB00329F',
'TBA0329F',
'TN00329F',
'TAA0332D',
'TAB0332D',
'TB00334F',
'TBA0334F',
'TBA0341F',
'TN00341F',
'TA00355F',
'TAA0355F',
'TA00360F',
'TAA0360F',
'TBY0360F',
'TB00363F',
'TBA0363F',
'TA00371F',
'TAZ0371F',
'TAY0410F',
'TAZ0410F',
'TBZ0411F',
'TNZ0411F',
'TBZ0412F',
'TNZ0412F',
'TBZ0417F',
'TNZ0417F',
'TAZ0419F',
'TNZ0419F',
'TBZ0420F',
'TNZ0420F',
'TBZ0431F',
'TNZ0431F',
'TBZ0443F',
'TNZ0443F',
'TBZ0444F',
'TNZ0444F',
'TBZ0453F',
'TNZ0453F',
'TBZ0454F',
'TNZ0454F',
'TBZ0457F',
'TNZ0457F',
'TBZ0461F',
'TNZ0461F',
'TBZ0474F',
'TNZ0474F',
'TBZ0497F',
'TNZ0497F',
'TBZ0504F',
'TNZ0504F',
'TBZ0505F',
'TNZ0505F',
'TBZ0509F',
'TNZ0509F',
'TBZ0514F',
'TNZ0514F',
'TBZ0515F',
'TNZ0515F',
'TAZ0518F',
'TNZ0518F',
'TBZ0521F',
'TNZ0521F',
'TBZ0522F',
'TNZ0522F',
'TBZ0531F',
'TNZ0531F',
'TBZ0532F',
'TNZ0532F',
'TBZ0546F',
'TNZ0546F',
'TBZ0547F',
'TNZ0547F',
'TBZ0549F',
'TNZ0549F',
'TBZ0550D',
'TNZ0550D',
'TBZ0560F',
'TNZ0560F',
'TBZ0561F',
'TNZ0561F',
'TAZ0567F',
'TNZ0567F',
'TBZ0568F',
'TNZ0568F',
'TAZ0574F',
'TNZ0574F',
'TBZ0575F',
'TNZ0575F',
'TBZ0585F',
'TNZ0585F',
'TAZ0587F',
'TNZ0587F',
'TAZ0603F',
'TNZ0603F',
'TBZ0608F',
'TNZ0608F',
'TAZ0609F',
'TNZ0609F',
'TBZ0610F',
'TNZ0610F',
'TBZ0611F',
'TNZ0611F',
'TBZ0612F',
'TNZ0612F',
'TBZ0614F',
'TNZ0614F',
'TBZ0615F',
'TNZ0615F',
'TAZ0622F',
'TNZ0622F',
'TBZ0623F',
'TNZ0623F',
'TBZ0625F',
'TNZ0625F',
'TBZ0626F',
'TNZ0626F',
'TBZ0631D',
'TNZ0631D',
'TAZ0641D',
'TNZ0641D',
'TAZ0642D',
'TNZ0642D',
'TBZ0643D',
'TNZ0643D',
'TBZ0644D',
'TNZ0644D',
'TBZ0645D',
'TNZ0645D',
'TBZ0649D',
'TNZ0649D',
'TAZ0653D',
'TNZ0653D',
'TBZ0654D',
'TNZ0654D',
'TBZ0655D',
'TNZ0655D',
'TAZ0656D',
'TNZ0656D',
'TAZ0657D',
'TNZ0657D',
'TAZ0658D',
'TNZ0658D',
'TBZ0659D',
'TNZ0659D',
'TBZ0660D',
'TNZ0660D',
'TBZ0661D',
'TNZ0661D',
'TAZ0672D',
'TNZ0672D',
'TAZ0673D',
'TNZ0673D',
'TAZ0674D',
'TNZ0674D',
'TBZ0675D',
'TNZ0675D',
'TAZ0676D',
'TNZ0676D',
'TBZ0677D',
'TNZ0677D',
'TBZ0678D',
'TNZ0678D',
'TAZ0693D',
'TNZ0693D',
'TAZ0694D',
'TNZ0694D',
'TAZ0695D',
'TNZ0695D',
'TBZ0696D',
'TNZ0696D',
'TBZ0697D',
'TNZ0697D',
'TAZ0699P',
'TNZ0699P',
'TBZ0700D',
'TNZ0700D',
'TBZ0701D',
'TNZ0701D',
'TBZ0702D',
'TNZ0702D',
'TAZ0704D',
'TNZ0704D',
'TAZ0705D',
'TNZ0705D',
'TAZ0706D',
'TNZ0706D',
'TBZ0715D',
'TNZ0715D',
'TAZ0718D',
'TNZ0718D',
'TBZ0724D',
'TNZ0724D',
'TBZ0739D',
'TNZ0739D',
'TAZ0740D',
'TNZ0740D',
'TAZ0742D',
'TNZ0742D',
'TAZ0746D',
'TNZ0746D',
'TBZ0750D',
'TNZ0750D',
'TBZ0751D',
'TNZ0751D',
'TAZ0752D',
'TNZ0752D',
'TAZ0753D',
'TNZ0753D',
'TAZ0755D',
'TNZ0755D',
'TBZ0759D',
'TNZ0759D',
'TAZ0773D',
'TNZ0773D',
'TAZ0774D',
'TNZ0774D',
'TAZ0775D',
'TNZ0775D',
'TBZ0776D',
'TNZ0776D',
'TBZ0777D',
'TNZ0777D',
'TBZ0778D',
'TNZ0778D',
'TAZ0779D',
'TNZ0779D',
'TBZ0780D',
'TNZ0780D',
'TBZ0781D',
'TNZ0781D',
'TAZ0795D',
'TNZ0795D',
'TAZ0796D',
'TNZ0796D',
'TAZ0797D',
'TNZ0797D',
'TBZ0798D',
'TNZ0798D',
'TBZ0799D',
'TNZ0799D',
'TBZ0800D',
'TNZ0800D',
'TAZ0808D',
'TNZ0808D',
'TAZ0809D',
'TNZ0809D',
'TAZ0810D',
'TNZ0810D',
'TBZ0811D',
'TNZ0811D',
'TBZ0812D',
'TNZ0812D',
'TBZ0813D',
'TNZ0813D',
'TAZ0820D',
'TNZ0820D',
'TAZ0821D',
'TNZ0821D',
'TBZ0823D',
'TNZ0823D',
'TBZ0824D',
'TNZ0824D',
'TA01004T',
'TB01004T',
'TE01004T',
'TA01043T',
'TAA1043T',
'TA0114D',
'TAA114D',
'TA0117F',
'TAA117F',
'TBZ9409F',
'TNZ9409F',
'TBZ9412F',
'TNZ9412F',
'TBZ9417F',
'TNZ9417F',
'TBZ9431F',
'TNZ9431F',
'TBZ9432F',
'TNZ9432F',
'TBZ9433F',
'TNZ9433F',
'ABHG337H0224',
'ABHG337I0224',
'ABHG337J0224',
'12443149007',
'12443149060',
'12443189007',
'12443239007',
'12443239060',
'11448129042',
'11448169003',
'11448169042',
'11448189003',
'11448189042',
'11448239042',
'11448259003',
'11448259042',
'11448269042',
'AAHG356A0424',
'AAHG356G0424',
'AAHG356H0424',
'11443429011',
'11443429012',
'21443429011',
'21443429012',
'31443429011',
'31443429012',
'81443429011',
'81443429012',
'22443269023',
'82443269023',
'BBHC373N0524',
'GBHC373N0825',
'23443129045',
'23443169046',
'BBHG364L0824',
'BBHG365L0824',
'BBHC384A0824',
'BBHC384K0824',
'GBHC384A0824',
'GBHC384K0824',
'12443419037',
'12443419038',
'11443439019',
'11443439020',
'11443439021',
'11443439022',
'12443129389',
'12443169044',
'12443179389',
'12443219389',
'12443239047',
'12443239389',
'12443259044',
'12443259389',
'12443269389',
'ABHC389A1024',
'ABHC389C1125',
'ABHC389D0725',
'ABHC389E1024',
'ABHC389G1024',
'ABHC389H1024',
'ABHC389J1024',
'ABHC389K1024',
'81443439023',
'81443439024',
'81443219025',
'81443239025',
'81443439026',
'81443439027',
'32443129389',
'32443169044',
'32443179389',
'32443219389',
'32443239044',
'32443239048',
'32443239389',
'32443259044',
'32443259048',
'32443259389',
'32443269389',
'82443129389',
'82443169044',
'82443179389',
'82443219048',
'82443219389',
'82443239048',
'82443239389',
'82443259044',
'82443259048',
'82443259389',
'82443269389',
'BBHC389A1124',
'BBHC389D0825',
'BBHC389E1124',
'BBHC389G1124',
'BBHC389H1124',
'BBHC389J1124',
'BBHC389K1124',
'GBHC389A1124',
'GBHC389D0825',
'GBHC389E1124',
'GBHC389G1124',
'GBHC389H1124',
'GBHC389J1124',
'GBHC389K1124',
'12443129039',
'12443269039',
'12443189056',
'12443269056',
'12443189057',
'12443269057',
'12443129058',
'12443189058',
'31443129041',
'31443179041',
'31443189041',
'31443229041',
'31443259041',
'81443129041',
'81443179041',
'81443189041',
'81443229041',
'81443259041',
'11443419036',
'11443419037',
'31443419038',
'31443419039',
'81443419038',
'81443419039',
'12443419051',
'12443419054',
'ABHC311L0124',
'32443419052',
'32443419055',
'82443419052',
'82443419055',
'BBHC312L0124',
'GBHC312L1123',
'22443439005',
'22443439006',
'32443439005',
'32443439006',
'82443439005',
'31443189032',
'31443219032',
'31443259032',
'81443189032',
'81443219032',
'81443259032',
'32443439042',
'82443439042',
'32443439043',
'82443439043',
'UBA0081T',
'UBC0081T',
'UA00100P',
'UAA0100P',
'UA00103P',
'UAA0103P',
'UA00104P',
'UAA0104P',
'UBA0313D',
'UBX0313D',
'UBY0313D',
'UAA0314D',
'UAB0314D',
'UB00314D',
'UA00316D',
'UAA0316D',
'UAB0316D',
'UBA0322D',
'UBY0322D',
'UA00331D',
'UAA0331D',
'UAB0331D',
'UAY0331D',
'UA00343D',
'UAA0343D',
'UB00344D',
'UBA0344D',
'UBY0344D',
'UA00349D',
'UAA0349D',
'UA00350D',
'UAA0350D',
'UAX0350D',
'UAY0350D',
'UAZ0350D',
'UB00356D',
'UBY0356D',
'UN00356D',
'UA00357D',
'UAA0357D',
'UBY0357D',
'UA00385D',
'UAA0385D',
'UAY0385D',
'UA00387D',
'UAZ0387D',
'UA00388D',
'UAZ0388D',
'UA00389D',
'UAY0389D',
'UAZ0389D',
'UB00391D',
'UBZ0391D',
'UB00400D',
'UN00400D',
'UBZ0404D',
'UNZ0404D',
'UBZ0423D',
'UNZ0423D',
'UAZ0425D',
'UNZ0425D',
'UBZ0428D',
'UNZ0428D',
'UAZ0435D',
'UNZ0435D',
'UBZ0436D',
'UNZ0436D',
'UAZ0441D',
'UNZ0441D',
'UBZ0452D',
'UNZ0452D',
'UAZ0464D',
'UNZ0464D',
'UBZ0466D',
'UNZ0466D',
'UBZ0468D',
'UNZ0468D',
'UBZ0471D',
'UNZ0471D',
'UBZ0472D',
'UNZ0472D',
'UBZ0473D',
'UNZ0473D',
'UBZ0484D',
'UNZ0484D',
'UBZ0485D',
'UNZ0485D',
'UBZ0489D',
'UNZ0489D',
'UBZ0490D',
'UNZ0490D',
'UAZ0495D',
'UNZ0495D',
'UBZ0511D',
'UNZ0511D',
'UBZ0528D',
'UNZ0528D',
'UBZ0529D',
'UNZ0529D',
'UBZ0538D',
'UNZ0538D',
'UAZ0552D',
'UNZ0552D',
'UBZ0553D',
'UNZ0553D',
'UAZ0562D',
'UNZ0562D',
'UAZ0564D',
'UNZ0564D',
'UBZ0570D',
'UNZ0570D',
'UBZ0572D',
'UNZ0572D',
'UBZ0582D',
'UNZ0582D',
'UBZ0583D',
'UNZ0583D',
'UAZ0586D',
'UNZ0586D',
'UAZ0591D',
'UNZ0591D',
'UAZ0592D',
'UNZ0592D',
'UAZ0616D',
'UNZ0616D',
'UBZ0617D',
'UNZ0617D',
'UBZ0618D',
'UNZ0618D',
'UAZ0619D',
'UNZ0619D',
'UAZ0633D',
'UNZ0633D',
'UAZ0634D',
'UNZ0634D',
'UBZ0635D',
'UNZ0635D',
'UAZ0637D',
'UNZ0637D',
'UBZ0638D',
'UNZ0638D',
'UBZ0639D',
'UNZ0639D',
'UBZ0640D',
'UNZ0640D',
'UAZ0646D',
'UNZ0646D',
'UBZ0647D',
'UNZ0647D',
'UBZ0648D',
'UNZ0648D',
'UAZ0650D',
'UNZ0650D',
'UAZ0651D',
'UNZ0651D',
'UAZ0652D',
'UNZ0652D',
'UAZ0662D',
'UNZ0662D',
'UAZ0663D',
'UNZ0663D',
'UAZ0664D',
'UNZ0664D',
'UBZ0665D',
'UNZ0665D',
'UBZ0666D',
'UNZ0666D',
'UBZ0667D',
'UNZ0667D',
'UAZ0668D',
'UNZ0668D',
'UBZ0669D',
'UNZ0669D',
'UBZ0670D',
'UNZ0670D',
'UBZ0671D',
'UNZ0671D',
'UAZ0680D',
'UNZ0680D',
'UAZ0681D',
'UNZ0681D',
'UAZ0688D',
'UNZ0688D',
'UAZ0707D',
'UNZ0707D',
'UAZ0708D',
'UNZ0708D',
'UBZ0709D',
'UNZ0709D',
'UBZ0710D',
'UNZ0710D',
'UBZ0711D',
'UNZ0711D',
'UAZ0716D',
'UNZ0716D',
'UAZ0717D',
'UNZ0717D',
'UBZ0719D',
'UNZ0719D',
'UBZ0720D',
'UNZ0720D',
'UAZ0721D',
'UNZ0721D',
'UAZ0722D',
'UNZ0722D',
'UAZ0725D',
'UNZ0725D',
'UBZ0728D',
'UNZ0728D',
'UBZ0729D',
'UNZ0729D',
'UBZ0731D',
'UNZ0731D',
'UBZ0733D',
'UNZ0733D',
'UAZ0735D',
'UNZ0735D',
'UAZ0744D',
'UNZ0744D',
'UAZ0745D',
'UNZ0745D',
'UAZ0747D',
'UNZ0747D',
'UBZ0748D',
'UNZ0748D',
'UBZ0749D',
'UNZ0749D',
'UBZ0756D',
'UNZ0756D',
'UAZ0757D',
'UNZ0757D',
'UAZ0758D',
'UNZ0758D',
'UBZ0760D',
'UNZ0760D',
'UAZ0761D',
'UNZ0761D',
'UAZ0762D',
'UNZ0762D',
'UAZ0763D',
'UNZ0763D',
'UBZ0764D',
'UNZ0764D',
'UBZ0765D',
'UNZ0765D',
'UBZ0766D',
'UNZ0766D',
'UAZ0767D',
'UNZ0767D',
'UAZ0768D',
'UNZ0768D',
'UAZ0769D',
'UNZ0769D',
'UBZ0770D',
'UNZ0770D',
'UBZ0771D',
'UNZ0771D',
'UBZ0772D',
'UNZ0772D',
'UBZ0783D',
'UNZ0783D',
'UAZ0784D',
'UNZ0784D',
'UAZ0785D',
'UNZ0785D',
'UBZ0786D',
'UNZ0786D',
'UBZ0787D',
'UNZ0787D',
'UAZ0789D',
'UNZ0789D',
'UAZ0790D',
'UNZ0790D',
'UAZ0791D',
'UNZ0791D',
'UBZ0792D',
'UNZ0792D',
'UBZ0793D',
'UNZ0793D',
'UBZ0794D',
'UNZ0794D',
'UAZ0803D',
'UNZ0803D',
'UAZ0804D',
'UNZ0804D',
'UAZ0805D',
'UNZ0805D',
'UAZ0807D',
'UNZ0807D',
'UAZ0814D',
'UNZ0814D',
'UAZ0815D',
'UNZ0815D',
'UAZ0816D',
'UNZ0816D',
'UBZ0819D',
'UNZ0819D',
'UA01001P',
'UB01001P',
'UA01005T',
'UB01005T',
'UB01006T',
'UBA1006T',
'UA01008P',
'UAA1008P',
'UA01012P',
'UAA1012P',
'UA01013P',
'UAA1013P',
'UA01096P',
'UAA1096P',
'UBY8322D',
'UNY8322D',
'UBX9313D',
'UBY9313D',
'UNY9313D',
'UBY9344D',
'UNY9344D',
'UBY9357D',
'UNY9357D',
'VB01901G',
'VF01901G',
'VB01903R',
'VF01903R',
'VB01904R',
'VF01904R',
'VB01924R',
'VD01924R',
'VF01924R',
'VB01996F',
'VD01996F',
'VF01996F',
'VB01997F',
'VD01997F',
'VF01997F',
'VB01998F',
'VF01998F',
'VB01999F',
'VF01999F',
'VB02000F',
'VF02000F',
'VB02002F',
'VF02002F',
'VB02003F',
'VF02003F',
'VB02010R',
'VD02010R',
'VF02010R',
'VB02011R',
'VD02011R',
'VA02034T',
'VB02034T',
'VF02034T',
'VB02063F',
'VD02063F',
'VA03062F',
'VB03062F',
'VE03062F',
'VB03076F',
'VBA3076F',
'VB03146F',
'VE03146F',
'VB03413F',
'VE03413F',
'VB03414F',
'VE03414F',
'VB03428F',
'VE03428F',
'VB03429F',
'VE03429F',
'VB03430F',
'VE03430F',
'VB03431F',
'VE03431F',
'WB01960G',
'WD01960G',
'WF01960G',
'WB01961G',
'WD01961G',
'WB01962G',
'WD01962G',
'WF01962G',
'WB01987C',
'WD01987C',
'WF01987C',
'WB01988D',
'WD01988D',
'WF01988D',
'WA02028G',
'WD02028G',
'WF02028G',
'WB02029G',
'WD02029G',
'WF02029G',
'WA02047G',
'WD02047G',
'WF02047G',
'WA02074G',
'WF02074G',
'WA02123G',
'WF02123G',
'WA02147G',
'WF02147G',
'WBA2148G',
'WF02148G',
'WA02178G',
'WAA2178G',
'WE02178G',
'WB02435G',
'WF02435G',
'WA02482G',
'WF02482G',
'WA03051G',
'WB03051G',
'WE03051G',
'WA03448G',
'WE03448G',
'WB03452G',
'WE03452G',
'WB03453G',
'WE03453G',
'11423419047',
'11423419048',
'12223139052',
'12223189052',
'12223229052',
'12223139055',
'12223149055',
'12223189055',
'12223229055',
'12223239055',
'12223259055',
'12248149058',
'12248189058',
'12248259058',
'12443439019',
'12443439020',
'31423419045',
'31423419046',
'81423419045',
'81423419046',
'31423419049',
'31423419050',
'81423419049',
'81423419050',
'32223139056',
'32223149056',
'32223189056',
'32223229056',
'32223239056',
'32223259056',
'82223139056',
'82223149056',
'82223189056',
'82223229056',
'82223239056',
'82223259056',
'32323149035',
'32323229035',
'32323239035',
'82323149035',
'82323229035',
'82323239035',
'32323149057',
'32323189057',
'32323259057',
'82323149057',
'82323189057',
'82323259057',
'22443439021',
'22443439022',
'32443439021',
'32443439022',
'82443439021',
'82443439022',
'32443439027',
'32443439028',
'82443439027',
'82443439028',
'82223229047',
'82223239047',
'82223259047',
'11223269355',
'AABC355K1123',
'11423169013',
'11423269013',
'11443189028',
'11443239028',
'11443259028',
'11443219018',
'11443239018',
'12443129029',
'12443269030',
'22443149008',
'22443239008',
'32443149008',
'32443149061',
'32443239008',
'32443239061',
'82443149061',
'82443239061',
'31213429085',
'81213429085',
'31213429088',
'81213429088',
'21213429010',
'31213429010',
'31213429089',
'81213429010',
'31223269355',
'81223269355',
'BABC355K1123',
'GABC355K1123',
'31223429033',
'31223429034',
'81223429033',
'81223429034',
'31223429035',
'81223429035',
'31223429036',
'81223429036',
'31223429080',
'81223429080',
'31223429086',
'81223429086',
'31223429098',
'81223429098',
'31223429099',
'31223429100',
'81223429099',
'81223429100',
'31423219034',
'31423239034',
'31423259034',
'81423219034',
'81423239034',
'81423259034',
'31443189029',
'31443189030',
'31443259029',
'81443189029',
'81443189030',
'81443259029',
'32213429017',
'82213429017',
'32213429018',
'32213429019',
'82213429018',
'82213429019',
'32213429020',
'82213429020',
'32213429038',
'82213429038',
'32223429043',
'82223429043',
'32223429051',
'82223429051',
'32223429054',
'82223429054',
'AADC478G0224',
'AADC478J0224',
'AADC478K0224',
'AAFC239I0223',
'AAFC239J0223',
'AAFC239P0223',
'AAFC239P1023',
'11213429025',
'11213429026',
'11223429078',
'11223429079',
'11223429096',
'11223429097',
'11553129038',
'11553259038',
'12213429011',
'12213429012',
'12213429048',
'12213429049',
'12443439009',
'12443439010',
'12443439012',
'12443439013',
'ABHC387N1024',
'12443439017',
'12443439018',
'21213129023',
'31213129023',
'21213429009',
'31213429009',
'21213429012',
'31213429012',
'21223129023',
'31223129023',
'21443189004',
'31443189004',
'21443259005',
'31443259005',
'21743229012',
'21743229013',
'21743229014',
'21743229015',
'22213429003',
'32213429003',
'22223429006',
'32223429006',
'22443139011',
'32443139011',
'22443169001',
'32443169001',
'22443419004',
'32443419004',
'BBHC306L0124',
'BBHC308L1223',
'BBHC309L1223',
'GBHC306L1124',
'GBHC309L0824',
'31223429094',
'81223429094',
'32213429013',
'82213429013',
'32443269026',
'82443269026',
'82213429044',
'82213429045',
'AAAC335M1123',
'AAAC336M1123',
'AAAC442A0823',
'AAAC442K0823',
'AAAC444C0823',
'AABC329M1123',
'AABC330M1123',
'AABC331M1123',
'AABC332M1123',
'AABC333M1123',
'AABC334M1123',
'AABC442A0823',
'AABC442C0823',
'AABC442K0823',
'AABC443C0823',
'AAFC274A0723',
'AAFC275K0723',
'AAFC299H1123',
'AAFC300A1123',
'AAFC378L1124',
'AAFC379L1124',
'AAYC213A0524',
'AAYC216A0524',
'ABHC268K0723',
'ABHD267A0723',
'ABHC038A1123',
'ABHC038C111',
'ABHC038D1123',
'ABHC038J1123',
'ABHC038P1223',
'ABHC278D1223',
'ABHC368K0724',
'ABHC282L0823',
'ABHC339L0324',
'ABHC303L1223',
'ABHC304L1223',
'ABHC370A0724',
'ABHC382K0724',
'ABHG366L0924',
'ABHG367L0924',
'BAAC313M1123',
'BAAC314M1123',
'BAAC317M1123',
'BAAC318M1123',
'BAAC442A0823',
'BAAC442K0823',
'BAAC444C0823',
'BABC442A0823',
'BABC442C0823',
'BABC442K0823',
'BABC443C0823',
'BAVC181H0923',
'BAVC182H0923',
'BBAC297M1123',
'BBAC298M1123',
'BBAC299M1123',
'BBAC300M1123',
'BBAC307M1123',
'BBAC308M1123',
'BBHC269K0723',
'BBHD270D0723',
'BBHD271A0723',
'BBHC284L0823',
'BBHC339L0324',
'GBHC339L0324',
'BBHC371A0724',
'BBHC383K0724',
'CAUC202A0324',
'CAUC202F0324',
'CAUC202H0324',
'EAUC204A0324',
'EAUC204F0324',
'EAUC204G0324',
'EAUC204H0324',
'AAUC211A0324',
'AAUC211H0324',
'BAUC211A0324',
'BAUC211G0324',
'AAUD164A0723',
'AAUD164A0923',
'AAUD164G0723',
'AAUD164G0923',
'AAUD164H0723',
'AAUD164H0923',
'AAUC165A0723',
'AAUC165A0923',
'AAUC165H0723',
'AAUC165H0923',
'BAUD166A0923',
'BAUD166A1223',
'BAUD166G0923',
'BAUD166G1223',
'GAUD166A1223',
'GAUD166G1223',
'BAUC167A0923',
'BAUC167A1223',
'BAUC167H0923',
'BAUC167H1223',
'GAUC167A1223',
'GAUC167H1223',
'BAUC188A1023',
'BAUC188J1023',
'AAUC187A1023',
'AAUC187J1023',
'BRUD007A1024',
'BRUD007C1024',
'BRUD007E1024',
'BRUD007F1024',
'BRUD007G1024',
'BRUD007H1024',
'AAUC217A1124',
'AAUC217F1124',
'AAUC217J1124',
'AAUC225E1124',
'AAUC225H1124',
'AAUC225J1124',
'BAUC224A1124',
'BAUC224E1124',
'BAUC224J1124',
'GAUC223A0125',
'GAUC223G0125',
'GAUC223J0125',
'AAUC170A1223',
'AAUC170G1223',
'AAUC170H1223',
'12713219001',
'12713239001',
'12713259001',
'BAUC171A0124',
'BAUC171F0124',
'BAUC171H0124',
'GAUC171A0124',
'GAUC171F0124',
'GAUC171H0124',
'AAUD177A0124',
'AAUD177H0124',
'AAUD177J0124',
'AAUD177K0124',
'11713129052',
'11713189052',
'11713219052',
'81713129053',
'81713189053',
'81713219053',
'11713129056',
'11713219056',
'11713229056',
'11713239056',
'11713259056',
'12713179071',
'12713189071',
'12713229071',
'12713239071',
'12713259071',
'BAUD178A0224',
'BAUD178H0224',
'BAUD178J0224',
'BAUD178K0224',
'GAUD178A0224',
'GAUD178H0224',
'GAUD178J0224',
'GAUD178K0224',
'CAUC200A0224',
'CAUC200E0224',
'CAUC200F0224',
'CAUC200H0224',
'AAUC206A0224',
'AAUC206E0224',
'AAUC206G0224',
'AAUC206H0224',
'11713189060',
'11713219060',
'11713259060',
'11713269060',
'81713179061',
'81713239061',
'81713259061',
'31713129062',
'31713179062',
'31713189062',
'31713229062',
'32623519049',
'82623519049',
'BRVC007A1024',
'BRVC007B0524',
'BRVC007C022',
'BRVC007E1024',
'BRVC007F022',
'BRVC007F1024',
'BRVC007G022',
'BRVC007H022',
'BRVC007H1024',
'BRVC007K022',
'BRVC009A1024',
'BAAC398M0124',
'GAAC398M0124',
'BAAC401M0124',
'GAAC401M0124',
'BAAC427M0224',
'GAAC427M0224',
'BAAG483E0324',
'BAAG483H0324',
'BAAG483J0324',
'GAAG483E0324',
'GAAG483H0324',
'GAAG483J0324',
'AAAG484A0224',
'AAAG484F0224',
'BAAG484G0224',
'BAAG484J0224',
'AABC193A0523',
'AABC193J0523',
'BABC402M0124',
'GABC402M0124',
'BABC420M0224',
'GABC420M0224',
'BABC428M0224',
'GABC428M0224',
'BABC433M0124',
'GABC433M0124',
'BABC466K1223',
'GABC466K1223',
'BABC508M1024',
'GABC508M1024',
'BABC548M1224',
'GABC548M1224',
'BABC549M1224',
'GABC549M1224',
'BABC599M0225',
'GABC599M0225',
'BADC467K1223',
'GADC467K1223',
'BAFC273A0723',
'BAFC273K0723',
'BAFC380L1124',
'BAFC381L1124',
'GAFC380L1124',
'GAFC381L1124',
'AAHC357G0924',
'AAHC357H0924',
'AAHC357J0924',
'BAHC392H1124',
'GAHC392H1124',
'BAJC169E1223',
'GAJC169E1223',
'BAWD185A1023',
'GAWD185A1023',
'BBAB430M0224',
'GBAB430M0224',
'BBAB435M0224',
'GBAB435M0224',
'BBAB486C0324',
'GBAB486C0324',
'BBAC190A0223',
'BBAC190F0223',
'BBAC367L0624',
'GBAC367L0624',
'BBAC373M1223',
'GBAC373M1223',
'BBAC376M1123',
'GBAC376M1123',
'BBAC380M1123',
'GBAC380M1123',
'BBAC382M1223',
'GBAC382M1223',
'BBAC384C1223',
'GBAC384C1223',
'BBAC404M0124',
'GBAC404M0124',
'BBAC418M0224',
'GBAC418M0224',
'BBAC434M0124',
'GBAC434M0124',
'BBAC505M1124',
'GBAC505M1124',
'BBAC547M1224',
'GBAC547M1224',
'BBAC613A0125',
'GBAC613A0125',
'BBAC615A1124',
'GBAC615A1124',
'BBCC362C1123',
'GBCC362C1123',
'BBCC482L1223',
'GBCC482L1223',
'BBHC272L0423',
'BBHC272L1023',
'GBHC272L1023',
'BBHC279A1123',
'BBHC279D1123',
'BBHC279J1123',
'BBHC280D1223',
'BBHC280P1223',
'GBHC279J1124',
'GBHC280P1124',
'BBHC298A1123',
'GBHC298A1123',
'BBHC315K1023',
'GBHC315K1023',
'BBHC319K0124',
'GBHC319K0124',
'BBHC320A0124',
'GBHC320A0124',
'BBHC321K0124',
'GBHC321K0124',
'BBHC326L0224',
'GBHC326L0224',
'ABHC327A0224',
'ABHC327H0224',
'ABHC327J0224',
'BBHC329A0224',
'BBHC329C0224',
'BBHC329H0224',
'GBHC329A0224',
'GBHC329C0224',
'GBHC329H0224',
'BBHC330F0224',
'GBHC330F0224',
'BAHC357G0424',
'BAHC357H0424',
'BAHC357J0424',
'GAHC357G0424',
'GAHC357H0424',
'GAHC357J0424',
'BBHC359E0424',
'GBHC359E0424',
'BBHC369D0724',
'GBHC369D0724',
'22443439014',
'22443439015',
'32443439014',
'32443439015',
'82443439014',
'82443439015',
'BBHC388N1024',
'GBHC388N1024',
'BBHG358C0424',
'BBHG358J0424',
'BBHG360J0424',
'GBHG358C0424',
'GBHG358J0424',
'GBHG360J0424',
'BALD220A0824',
'BALD220F0824',
'BALC227P1124',
'GALC227P1124',
'BBJC126V1023',
'GBJC126V1023',
'BBJC127X1123',
'GBJC127X1123',
'ABRD094X0123',
'ABRD094X0723',
'ABRD094X1023',
'BBRD123V0723',
'GBRD123V0723',
'ABRE041T1022',
'ABRE041T1023',
'ABRE041T1223',
'ABRE054V0124',
'ABRE054V1122',
'ABRE054V1123',
'ABRE095X0123',
'ABRE095X0723',
'ABRE095X1023',
'BBSC314L1123',
'GBSC314L1123',
'ABTC050T1022',
'ABTC050T1023',
'ABTC050T1223',
'22654769013',
'82654769013',
'AAYR221A0924',
'AAYR221H0924',
'AAYR221J0924',
'AAZC160A0823',
'AAZC160A1023',
'AAZC160F0823',
'AAZC160F1023',
'AAZC160H0823',
'AAZC160H1023',
'BAZC161A0923',
'BAZC161A1123',
'BAZC161H0923',
'BAZC161H1123',
'BAZC161J0923',
'BAZC161J1123',
'AAZC207A0224',
'AAZC207F0224',
'AAZC207H0224',
'BAZC208A0224',
'BAZC208F0224',
'BAZC208H0224',
'AAWD212A0324',
'AAWD212E0324',
'AAWD212H0324',
'CAWD201A0324',
'CAWD201F0324',
'CAWD201J0324',
'HAWD209F0324',
'HAWD209G0324',
'HAWD209H0324',
'HAWD209J0324',
'12734189020',
'12734219020',
'21734239011',
'21734259011',
'31734239011',
'31734259011',
'AAWD126A0523',
'AAWD126G0523',
'AAWD126H0523',
'BAWD127A0523',
'BAWD127E0523',
'BAWD127F0523',
'AAWD157A0623',
'AAWD157F0623',
'AAWD157G0623',
'AAWD157J0623',
'AAWD158A0723',
'AAWD158A0923',
'AAWD158F0723',
'AAWD158F0923',
'AAWD158H0723',
'AAWD158H0923',
'BAWD159A0823',
'BAWD159A1023',
'BAWD159F0823',
'BAWD159F1023',
'BAWD159H1023',
'GAWD159H1023',
'EAWD195A1223',
'EAWD195E1223',
'EAWD195G1223',
'EAWD195H1223',
'HAWD195A1223',
'HAWD195E1223',
'HAWD195G1223',
'HAWD195H1223',
'AAWD228A1024',
'AAWD228E1024',
'AAWD228H1024',
'AAWD228J1024',
'AAWD231E0225',
'AAWD231F0225',
'AAWD231H0225',
'AAWD231J0225',
'CAWD193A1123',
'CAWD193F1123',
'CAWD193H1123',
'CAWD193J1123',
'AAWD218E1124',
'AAWD218H1124',
'AAWD218J1124',
'GAWD222A1124',
'GAWD222F1124',
'GAWD222H1124',
'BAWD230F1124',
'BAWD230H1124',
'AAWD172A1223',
'AAWD172F1223',
'AAWD172H1223',
'AAWD229F1224',
'AAWD229H1224',
'AAWD229J1224',
'11734189050',
'11734229050',
'11734239050',
'11734259050',
'BAWD173A0124',
'BAWD173F0124',
'BAWD173G0124',
'GAWD173A0124',
'GAWD173F0124',
'GAWD173G0124',
'31734129063',
'31734189063',
'31734219063',
'31734229063',
'81734129063',
'81734189063',
'81734219063',
'81734229063',
'11734179054',
'11734219054',
'11734259054',
'81734189055',
'81734219055',
'81734229055',
'EAWD205A0224',
'EAWD205E0224',
'EAWD205F0224',
'EAWD205H0224',
'CAWD197A0224',
'CAWD197E0224',
'CAWD197F0224',
'CAWD197H0324',
'EAWD196A0224',
'EAWD196E0224',
'EAWD196G0224',
'EAWD196H0224',
'EAWD198A0224',
'EAWD198E0224',
'EAWD198H0224',
'11734189059',
'11734229059',
'11734239059',
'CAXC199G0224',
'CAXC199J0224',
'31743149027',
'31743189027',
'31743239027',
'81743149027',
'81743189027',
'81743239027',
'86743259030',
'ARXC007A1024',
'ARXC007G1024',
'ARXC007H1024',
'ARXC007N1024',
'ARXC008A1024',
'ARXC008C1024',
'11743129028',
'11743179028',
'11743239028',
'11743259028',
'11743189045',
'11743219045',
'11743259045',
'31743129029',
'31743179029',
'31743239029',
'31743259029',
'81743129029',
'81743179029',
'81743239029',
'81743259029',
'31743189046',
'31743219046',
'31743259046',
'81743189046',
'81743219046',
'81743259046',
'CAXC193A1223',
'CAXC193H1223',
'CAXC193J1223',
'AAXC235E1124',
'AAXC235F1124',
'AAXC235H1124',
'AAXC235J1124',
'BAXC233A1224',
'BAXC233E1224',
'BAXC233G1224',
'BAXC233J1224',
'16743129007',
'16743129008',
'16743179007',
'16743179008',
'16743189007',
'16743189008',
'16743219008',
'16743229008',
'16743239007',
'16743259007',
'16743259008',
'16743439008',
'26743129007',
'26743129008',
'26743439009',
'EBRD133T0124',
'HBRD133T0124',
'EBRD132V0124',
'EBRE132V0324',
'HBRD132V0124',
'HBRE132V0324',
'BBRD089Q0223',
'GBRD089Q0324',
'ABRD052V0723',
'ABRD052V0923',
'ABRE045V0723',
'ABRE045V0923',
'ABRF046V0723',
'ABRF046V0923',
'ABRC118Q0723',
'ABRC118Q1023',
'BBRE097V0823',
'BBRE097V1023',
'BBRE100V0823',
'GBRE100V1023',
'BBRD107X0823',
'BBRD107X1223',
'BBRD108X0823',
'GBRD108X1223',
'32613719047',
'82613719047',
'32613779048',
'82613779048',
'32613739046',
'82613739046',
'ABRD039V1022',
'ABRD039V1023',
'ABRD099V1023',
'ABRD099V1223',
'BBRD096V1023',
'BBRD096V1223',
'GBRD096V1223',
'32614719063',
'82614719063',
'32614749064',
'82614749064',
'32614749065',
'82614749065',
'BBRD138X1123',
'GBRD138X1123',
'BBRD139W1123',
'GBRD139W1123',
'BBRD098V0323',
'BBRD098V1023',
'BBRD101V0323',
'GBRD101V1023',
'BBRC195W1224',
'GBRC195W1224',
'BBRD147V0124',
'GBRD147V0124',
'22613719008',
'22613749009',
'32613719008',
'32613749009',
'32613769076',
'82613719008',
'82613749009',
'82613769076',
'22614569004',
'82614569004',
'BBTC201V1224',
'GBTC201V1224',
'BBTC202Y1224',
'GBTC202Y1224',
'BBTC203T1224',
'GBTC203T1224',
'11758129047',
'11758179047',
'11758219047',
'11758239047',
'81758129048',
'81758179048',
'81758219048',
'81758230948',
'11423419052',
'11423419053',
'32223169016',
'32223269015',
'82223169016',
'82223269015',
'81223129044',
'81223149044',
'81223259044',
'12443419033',
'12443419034',
'32443419035',
'32443419036',
'82443419035',
'82443419036',
'82443129031',
'82443269032',
'11443189031',
'11443219031',
'11443259031',
'12443149062',
'12443189062',
'12443219062',
'12443239062',
'12443259062',
'32443149063',
'32443189063',
'32443219063',
'32443229063',
'32443239063',
'82443149063',
'82443189063',
'82443219063',
'82443229063',
'82443239063',
'12443189059',
'12443259059',
'AAJC210A0324',
'AAJC210G0324',
'AAJC210J0324',
'AAJC162E0823',
'AAJC162E1023',
'AAJC162H0823',
'AAJC162H1023',
'BAJC163A0923',
'BAJC163A1123',
'BAJC163F0923',
'BAJC163F1123',
'GAJC163A1123',
'GAJC163F1123',
'BABC375M1123',
'GABC375M1123',
'BABC381M1123',
'GABC381M1123',
'12613719005',
'12613749007',
'12613769006',
'1261476E9082P',
'ABRD128V1123',
'ABRE128V0124',
]

## Cek

In [ ]:
df_wh

In [ ]:
df_art=pd.read_excel('list artikel aging (stok toko & gudang).xlsx')

In [ ]:
df_finale=df_art.merge(df, on='fv_barcode', how='left').fillna(0)
df_finale=df_finale.merge(df_wh, on='fv_barcode', how='left').fillna(0)
df_finale[['stocks','fn_qty']]=df_finale[['stocks','fn_qty']].astype(int)
df_finale=df_finale.rename(columns={'fv_barcode':'Artikel','stocks':'Stock Store','fn_qty':'Stock Warehouse',})

In [ ]:
df_finale.to_excel('stok artikel aging toko & gudang.xlsx',index=False)

# AS 13 excel tracking

# AS 14 Cek Stock WH

In [ ]:
query = f"""
        SELECT 
            fv_whtypename, fv_artsizecode, fn_qty 
        FROM gis_db.stockarticle_tb a 
        LEFT JOIN gis_db.warehousetype_tb b 
            ON a.fn_whtypeid = b.fn_whtypeid
        WHERE 1=1 
          AND a.fn_whtypeid not in (0,1) 
          AND fn_qty > 0
    """
with get_engine('write') as engine:  # Menggunakan context manager dengan with
    df = pd.read_sql(query, engine)


In [ ]:
df.sort_values('fn_qty',ascending=False)

# AS 15 db group alokator

## upsert

In [ ]:
base_path = fr'/opt/share/PB DB/'
base_path = r'\\192.168.1.254\Share\_From Valdi\PB DB\group alokator.xlsx'
df = pd.read_excel(base_path)

In [ ]:
with get_engine('imp') as engine: 
    df.to_sql('group_alokator', engine, if_exists='replace', index=False)

In [ ]:
df

In [ ]:

query = f'''
SELECT * FROM user_tb
'''
with get_engine('read') as engine:  # Menggunakan context manager dengan with
    df_user = pd.read_sql(query, engine)

# AS 16 sales qty value

In [ ]:
# list_to_sql_in(df, col_name, astype='str')
df_cus=df__customers_pb_('fn_whconsiid,fc_code,fv_toko',distinct=False,where='')
df_art=df_artikel_pb_('fv_artsizecode, fv_barcode, fv_sizename, fn_articleid',distinct=False,where='')
query = f'''
        SELECT fn_artsizeid,fn_articleid
        FROM gis_db.articlesize_tb
        '''
    # df = pd.read_sql(query, engine_imp)
with get_engine('write') as engine:  # Menggunakan context manager dengan with
    df_artsiz = pd.read_sql(query, engine)
df_art=df_art.merge(df_artsiz,how='left',on='fn_articleid')

In [ ]:
year=2025
def df__pb_penjualan_(year):
    query = f'''SELECT
        fn_whconsiid, fv_artsizecode, fc_monthname, 
        sum(fn_jualpersize) qty, sum(fn_totalpenjualan) 'value' FROM gis_local.pb_penjualan_{year}
        group by fn_whconsiid, fv_artsizecode, fc_monthname
        order by fn_whconsiid, fc_monthname, fv_artsizecode'''
    with get_engine('imp') as engine:  # Menggunakan context manager dengan with
        df = pd.read_sql(query, engine)
    return df
df=df__pb_penjualan_(year)
# Pivot qty dan value sekaligus
df=df.merge(df_cus,how='left',on='fn_whconsiid')
df=df.merge(df_art,how='left',on='fv_artsizecode')
df['pk_id']=df['fn_whconsiid'].astype(str)+"/"+df['fn_artsizeid'].astype(str)
df_pivot = df.pivot_table(
    index=['pk_id','fv_toko','fc_code','fv_barcode', 'fv_sizename'],
    columns='fc_monthname',
    values=['qty','value'],
    aggfunc='sum',
    fill_value=0
)

# Flatten multiindex columns jadi satu level
df_pivot.columns = [f"{month}_{metric}" for metric, month in df_pivot.columns]

In [ ]:
# Reset index agar jadi DataFrame normal
df_final = df_pivot.reset_index()

# --- Urutkan kolom sesuai urutan bulan kalender ---
month_order = [
    "Janu", "February", "March", "April", "May", "June",
    "July", "August", "September", "October", "November", "December"
]

# Buat urutan kolom berdasarkan bulan yang tersedia di data
ordered_cols = ['pk_id','fv_toko','fc_code','fv_barcode', 'fv_sizename']
for m in month_order:
    qty_col = f"{m}_qty"
    val_col = f"{m}_value"
    if qty_col in df_final.columns:
        ordered_cols.append(qty_col)
    if val_col in df_final.columns:
        ordered_cols.append(val_col)

# Susun ulang kolom
df_final = df_final[ordered_cols]


In [ ]:
nama_file = fr'\\192.168.1.254\Share\_From Valdi\PB DB\Sales Qty & Value {year}.xlsx'
total_rows = len(df_final)
chunks = 4
chunk_size = math.ceil(total_rows / chunks)  # otomatis bagi 3

with pd.ExcelWriter(nama_file, engine='xlsxwriter') as writer:
    for i in range(chunks):
        start_row = i * chunk_size
        end_row = min(start_row + chunk_size, total_rows)
        sheet_df = df_final.iloc[start_row:end_row]
        sheet_df.to_excel(writer, index=False, sheet_name=f'data_{i+1}')

# AS 17 Ost OB

In [ ]:
# list_to_sql_in(df, col_name, astype='str')
df_cus=df__customers_pb_('fn_cusid,fc_code "Code Toko",fv_toko Toko',distinct=False,where='')
df_art=df_artikel_pb_('fv_artsizecode, fv_barcode "Artikel", fv_sizename "Size"' ,distinct=False,where='')

query = f'''
        SELECT fn_userid,fv_username "User"
        FROM gis_db.user_tb
        '''
    # df = pd.read_sql(query, engine_imp)
with get_engine('write') as engine:  # Menggunakan context manager dengan with
    df_user = pd.read_sql(query, engine)
# df_art=df_art.merge(df_artsiz,how='left',on='fn_articleid')

In [ ]:
query='''SELECT distinct  
        fn_cusid, fv_artsizecode, fn_userid,
        b.fv_noorder "No OB",
        fd_tglorder "Tgl Pengajuan", 
        fd_tgldelivery "Tgl On Store",
        sum(fn_orderpersize) OstOB,
          fv_keteranganumum "Keterangan Umum"  FROM gis_db.t_orderdtl_goods_tmp a 
        inner join gis_db.t_ordermst b on a.fv_noorder=b.fv_noorder
        left join gis_db.domst_tb c on b.fv_noorder=c.fc_noreff
        where fc_nodoc is null
        group by fn_cusid, fv_artsizecode'''
with get_engine('write') as engine:  # Menggunakan context manager dengan with
    df_ostob_ = pd.read_sql(query, engine)

In [ ]:
df_ostob_=df_art.merge(df_ostob_,how='right',on='fv_artsizecode').drop(columns='fv_artsizecode')
df_ostob_=df_cus.merge(df_ostob_,how='right',on='fn_cusid').drop(columns='fn_cusid')
df_ostob_=df_ostob_.merge(df_user,how='left',on='fn_userid').drop(columns='fn_userid')

In [ ]:
nama_file = fr'\\192.168.1.254\Share\_From Valdi\For Alokator\Out Standing OB.xlsx'
df_ostob_.to_excel(f'{nama_file}',index=False)

In [ ]:
df_ostob_

# AS 18 Stock ornal mothly

In [ ]:
query = f'''
        SELECT distinct fv_barcode,fn_markid,fd_effectivedate,fd_dateupdate FROM gis_db1.hs_article_tb;
        '''
    # df = pd.read_sql(query, engine_imp)
with get_engine('read') as engine:  # Menggunakan context manager dengan with
    df_ex = pd.read_sql(query, engine)
    


In [ ]:
df=df_ex.copy()

In [ ]:
df = df.sort_values(by=['fv_barcode', 'fn_markid', 'fd_effectivedate', 'fd_dateupdate'])
df = df.drop_duplicates(
    subset=['fv_barcode',  'fd_effectivedate'],
    keep='last'  # ambil fd_dateupdate paling baru
)
df = df.dropna(subset=['fd_effectivedate'])


In [ ]:
# pastikan tanggal jadi datetime
df['fd_effectivedate'] = pd.to_datetime(df['fd_effectivedate'], errors='coerce')
df['fd_dateupdate'] = pd.to_datetime(df['fd_dateupdate'], errors='coerce')

# sort:
# fd_effectivedate ASC → ambil yg paling lama
# fd_dateupdate DESC → jika sama pilih update terbaru
df_sorted = df.sort_values(
    by=['fv_barcode', 'fn_markid', 'fd_effectivedate', 'fd_dateupdate'],
    ascending=[True, True, True, False]
)

# ambil yang pertama (paling lama)
df_unique = df_sorted.drop_duplicates(
    subset=['fv_barcode', 'fn_markid'],
    keep='first'
)


In [ ]:
df_unique = df_unique.dropna(subset=['fd_effectivedate'])

In [ ]:
df_unique

# AS 19 sales thru bz

In [ ]:
query = f"""
        SELECT fc_nodoc, fn_bulan, fn_tahun FROM gis_db1.stockarticlebzpostingmst_tb 
        where fn_bulan<>0 and fn_tahun>2023
        order by fn_tahun, fn_bulan
    """
with get_engine('read') as engine:  # Menggunakan context manager dengan with
    df = pd.read_sql(query, engine)

In [ ]:
grouped = grouped.groupby(['fn_tahun', 'fn_bulan'])

In [ ]:
import calendar
import pandas as pd

grouped = df.groupby(['fn_tahun', 'fn_bulan'])

for (tahun, bulan), grp in grouped:
    
    # -----------------------------
    # 1. Ambil daftar nodoc
    # -----------------------------
    list_nodoc = grp['fc_nodoc'].drop_duplicates().tolist()
    list_nodoc_sql = "'" + "', '".join(list_nodoc) + "'"

    # -----------------------------
    # 2. Query SQL
    # -----------------------------
    query = f"""
        SELECT fn_whconsiid, fv_artsizecode, fn_qtyskrg 
        FROM gis_db1.stockarticlebzpostingdtl_tb 
        WHERE fc_nodoc IN ({list_nodoc_sql})
          AND fn_qtyskrg > 0
    """

    with get_engine('read') as engine:
        df_out = pd.read_sql(query, engine)

    # -----------------------------
    # 3. Generate tanggal akhir bulan → tgltrx
    # -----------------------------
    last_day = calendar.monthrange(tahun, bulan)[1]  
    # contoh hasil: 31

    df_out['tgltrx'] = f"{tahun}-{bulan:02d}-{last_day}"

    # -----------------------------
    # 4. Rename kolom fn_qtyskrg → stock {bulan3huruf}_{tahun2digit}
    # -----------------------------
    nama_bulan_3 = calendar.month_abbr[bulan]   # Jan, Feb, Mar, ...
    tahun_2digit = str(tahun)[2:]               # 2024 → 24

    new_col_name = f"stock {nama_bulan_3}_{tahun_2digit}"

    df_out = df_out.rename(columns={'fn_qtyskrg': new_col_name})

    # -----------------------------
    # 5. Save ke Excel
    # -----------------------------
    filename = f"stock_bz_{tahun}_{bulan:02d}.xlsx"
    df_out.to_excel(filename, index=False)

    print("saved:", filename)


# AS 20 data mentah memo

In [ ]:
memos = ["076/costacc/acara/09/2025", 
         "077/costacc/acara/09/2025", 
         "078/costacc/acara/10/2025", 
         "079/costacc/acara/10/2025",
         "081/costacc/acara/10/2025", 
         "082/costacc/acara/10/2025", 
         "083/costacc/acara/11/2025", 
         "083/costacc/acara/11/2025", 
         "084/costacc/acara/11/2025"]
where_like = " OR ".join([f"fv_descevent LIKE '%%{m}%%'" for m in memos])


In [ ]:
query = f"""
    SELECT 
    fv_descevent as Nama_Memo,fn_cusid fn_whconsiid, a.fd_tgljual "Tgl Jual", fv_artsizecode, d.fv_namemark 'event',
    fn_hargasatuan "Nett Kasa",fn_jualpersize Qty FROM t_penjualandtl_goods a 
    left join t_penjualanmst b on a.fv_nopenjualan=b.fv_nopenjualan
    LEFT JOIN 
        (SELECT a.fn_discid, b.fv_namemark FROM discount_tb a
            INNER JOIN markdown_tb b ON a.fn_markid = b.fn_markid) d ON a.fn_discid = d.fn_discid
    where {where_like}
"""
with get_engine('read') as engine:
    df = pd.read_sql(query, engine)


In [ ]:
df_cus=df__customers_pb_('fn_whconsiid, fv_toko Toko',distinct=False,where='')
df_art=df_artikel_pb_('fv_artsizecode, fv_barcode Artikel, fv_sizename size, fm_price Gross, fm_amount "Nett Master", fv_namemark "status acara"',distinct=False,where='')

In [ ]:
df['Nama_Memo'] = df['Nama_Memo'].str.lower()
df['fv_artsizecode']=df['fv_artsizecode'].str.upper()
for m in memos:
    df.loc[df['Nama_Memo'].str.contains(m, na=False), 'Memo'] = m

In [ ]:
df=df.merge(df_cus, on='fn_whconsiid',how='left')
df=df.merge(df_art, on='fv_artsizecode',how='left')

In [ ]:
df.loc[df['size'].isnull()]

In [ ]:
df=df[['Memo','Toko','Tgl Jual','Artikel','size','status acara','event','Gross','Nett Master',
       'Nett Kasa','Qty']]

In [ ]:
df.to_excel('data penjualan memo.xlsx',index=False)

In [ ]:
df

In [ ]:
df.info()

# AS 21 stock aging all

In [ ]:
input_value = datetime.now().strftime("%Y-%m-%d")   
year = datetime.now().year
sales_thru=f'sales_thru_pb_{str(year)[-2:]}'
trxstx = f"trxstx_pb{str(year)[-2:]}"
stock = f"fn_stock_{input_value[5:7]}"

main_query = f"""
    SELECT stock_backdate.fn_whconsiid,stock_backdate.fv_artsizecode,fv_barcode, fv_sizename, a.fv_configname,stocks
    FROM (
        SELECT fn_whconsiid, {stock} AS stock, b.fv_artsizecode 
        FROM gis_local.{sales_thru} b
        LEFT JOIN artikel_pb a ON b.fv_artsizecode = a.fv_artsizecode
        where 1=1
        UNION ALL
        SELECT fn_whconsiid, fn_stock AS stock, b.fv_artsizecode 
        FROM gis_local.{trxstx} b
        LEFT JOIN artikel_pb a ON b.fv_artsizecode = a.fv_artsizecode
        WHERE fd_tgltrx BETWEEN '{str(year)[-2:]}-{input_value[5:7]}-01' AND '{str(year)[-2:]}-{input_value[5:10]}'
        UNION ALL
        SELECT fn_whconsiid, fn_stock AS stock, b.fv_artsizecode 
        FROM gis_local.trxrt_pb b
        LEFT JOIN artikel_pb a ON b.fv_artsizecode = a.fv_artsizecode
        WHERE fd_tgltrx < '{str(year)[-2:]}-{input_value[5:10]}' 
        UNION ALL
        SELECT fn_whconsiid, fn_stock AS stock, b.fv_artsizecode 
        FROM gis_local.Stock_varian_{str(year)[-2:]} b
        LEFT JOIN artikel_pb a ON b.fv_artsizecode = a.fv_artsizecode
        WHERE fd_tgl BETWEEN '{str(year)[-2:]}-{input_value[5:7]}-01' AND '{str(year)[-2:]}-{input_value[5:10]}'
    ) stock_backdate
    LEFT JOIN artikel_pb a ON a.fv_artsizecode = stock_backdate.fv_artsizecode
    LEFT JOIN customers_pb b ON stock_backdate.fn_whconsiid = b.fn_whconsiid
    WHERE 1=1 
    """
with get_engine('imp') as engine:
    df = pd.read_sql(main_query, engine)

In [ ]:
def df__aging(input_value):
    query = f"""
    SELECT a.fn_whconsiid, fv_barcode, fn_stock, fd_tgltrx FROM 
    (SELECT * FROM gis_local.trxstx_pb24 
    where fv_stat like 'receive%%' and fd_tgltrx <= '{input_value}'
    union all
    SELECT fn_whconsiid, fv_artsizecode, fn_stock, fd_tgltrx, fv_stat FROM gis_local.trxstx_pb25 
    where fv_stat like 'receive%%' and fd_tgltrx <= '{input_value}') a
    left join artikel_pb b on a.fv_artsizecode = b.fv_artsizecode
    WHERE 1=1
    group by fn_whconsiid,fv_barcode, fd_tgltrx 
    """
    # df = pd.read_sql(query, engine_imp)
    with get_engine('imp') as engine:
        df = pd.read_sql(query, engine)
    return df
def proc_add_aging(df,df_aging):
### Proses untuk menambahkan kolom '1st DO', 'last_receive', dan 'stock_on_1st_do' ke DataFrame df.
    if not df_aging.empty:
        # Group df_aging by fn_whconsiid and fv_barcode, and get the minimum fd_tgltrx
        aging_min_date = df_aging.groupby(['fn_whconsiid', 'fv_barcode'])['fd_tgltrx'].min().reset_index()
        aging_min_date.rename(columns={'fd_tgltrx': '1st DO'}, inplace=True)

        # Group df_aging by fn_whconsiid and fv_barcode, and get the maximum fd_tgltrx (last_receive)
        aging_max_date = df_aging.groupby(['fn_whconsiid', 'fv_barcode'])['fd_tgltrx'].max().reset_index()
        aging_max_date.rename(columns={'fd_tgltrx': 'last_receive'}, inplace=True)

        # Merge df with aging_min_date on fn_whconsiid and fv_barcode
        df = pd.merge(df, aging_min_date, on=['fn_whconsiid', 'fv_barcode'], how='left')
        df = pd.merge(df, aging_max_date, on=['fn_whconsiid', 'fv_barcode'], how='left')

        # Get stock on 1st DO
        # Merge aging_df with aging_min_date to get stock on 1st DO
        aging_stock_on_1st_do = pd.merge(
            df_aging,
            aging_min_date,
            on=['fn_whconsiid', 'fv_barcode'],
            how='left'
        )

        # Filter rows where fd_tgltrx matches the 1st DO
        aging_stock_on_1st_do = aging_stock_on_1st_do[
        aging_stock_on_1st_do['fd_tgltrx'] == aging_stock_on_1st_do['1st DO']]

        # Select only the required columns
        aging_stock_on_1st_do = aging_stock_on_1st_do[['fn_whconsiid', 'fv_barcode', 'fn_stock']]
        # aging_stock_on_1st_do.rename(columns={'fn_stock': 'stock_on_1st_do'}, inplace=True)

        # Merge df with aging_stock_on_1st_do to add stock_on_1st_do column
        df = pd.merge(df, aging_stock_on_1st_do, on=['fn_whconsiid', 'fv_barcode'], how='left')
        df_sum = df.groupby(['fn_whconsiid', 'fv_barcode'], as_index=False)['fn_stock'].sum().drop_duplicates(subset=['fv_barcode'])
        df = pd.merge(df, df_sum, on=['fn_whconsiid', 'fv_barcode'], how='left')
        df = df.rename(columns={'fn_stock_y': 'stock_on_1st_do'}).drop(columns=['fn_stock_x'])
        df['stock_on_1st_do'] = df['stock_on_1st_do'].fillna(0)
        # Convert '1st DO' to datetime
        df['1st DO'] = pd.to_datetime(df['1st DO'])
        df['last_receive'] = pd.to_datetime(df['last_receive'])

        # Hitung aging (selisih hari dari last_receive ke hari ini)
        df['aging'] = (pd.Timestamp.today().normalize() - df['last_receive']).dt.days.fillna(0).astype(int)

        # Format '1st DO' to 'YYYY-MM-DD'
        df['1st DO'] = df['1st DO'].dt.strftime('%Y-%m-%d')
        df['last_receive'] = df['last_receive'].dt.strftime('%Y-%m-%d')
        return df
    else:
        # Jika aging_df kosong, buat kolom dengan nilai default
        df['1st DO'] = '2024-01-01'
        df['last_receive'] = '2024-01-01'
        df['aging'] = '2024-01'
        df['stock_on_1st_do'] = 0
        return df
def proc_pivot_table(df):
    all_sizes_bottom = ['25', '26', '27', '28', '29', '30', '31', '32', '33', '34', '35', '36', '37', '38',
                        '39', '40', '42', '44', 'ALLSIZE']
    all_sizes_top = ['FS', 'XS', 'SS', 'S', 'M', 'L', 'XL', 'XXL', 'XXXL']
    all_sizes = all_sizes_bottom + all_sizes_top

    # Buat pivot table
    pivot_table = pd.pivot_table(
        df,
        values='stocks',
        index=['fv_barcode', 'fn_whconsiid'],
        columns=['fv_sizename'],
        aggfunc='sum',
        fill_value=0
    )

    # Reset index agar bisa merge
    pivot_table = pivot_table.reset_index()

    # Kolom detail yang ingin ditambahkan kembali
    kolom_detail = ['fv_barcode', '1st DO', 'last_receive', 'aging','fv_configname',
                    'stock_on_1st_do', 'fn_whconsiid']
    # Merge detail
    pivot_table = pd.merge(pivot_table, df[kolom_detail], on=['fv_barcode', 'fn_whconsiid'], how='left')
    pivot_table = pivot_table.drop_duplicates()

    # Urutkan kolom agar kolom ukuran di akhir
    urutan_kolom = kolom_detail + [col for col in pivot_table.columns if col not in kolom_detail]
    pivot_table = pivot_table[urutan_kolom]

    # Pastikan semua ukuran muncul
    for size in all_sizes:
        if size not in pivot_table.columns:
            pivot_table[size] = 0
        pivot_table[size] = pd.to_numeric(pivot_table[size], errors='coerce').fillna(0).astype(int)

    # Atur ulang urutan: non-size columns dulu, lalu size columns
    size_columns = [size for size in all_sizes if size in pivot_table.columns]
    other_columns = [col for col in pivot_table.columns if col not in size_columns]
    pivot_table = pivot_table[other_columns + size_columns]

    # Hitung status (BROKEN / SEHAT)
    top_columns = ["L", "M", "S", "XL", "XS", "XXL", "XXXL"]
    bottom_columns = ['25', '26', '27', '28', '29', '30', '31', '32', '33', '34',
                        '35', '36', '37', '38', '39', '40', '42', '44']

    pivot_table['status'] = np.where(
        (pivot_table["fv_configname"] == "TOP") & (pivot_table[top_columns].gt(0).sum(axis=1) < 3), "BROKEN",
        np.where(
            (pivot_table["fv_configname"] == "BOTTOM") & (pivot_table[bottom_columns].gt(0).sum(axis=1) < 5), "BROKEN",
            "SEHAT"
        )
    )

    # Hitung total jumlah qty size (sumQty)
    pivot_table['sumQty'] = pivot_table[size_columns].sum(axis=1)
    pivot_table=pivot_table.drop(columns=['fv_configname'])

    return pivot_table

In [ ]:
df_aging = df__aging(input_value)
df = proc_add_aging(df, df_aging)
pivot_table = proc_pivot_table(df)

In [ ]:
pivot_table['tahun_float'] = pivot_table['aging'] / 365
pivot_table['Aging Tahun'] = (pivot_table['tahun_float'] * 2).astype(int) / 2


In [ ]:
df_cus=df__customers_pb_('fn_whconsiid, concat(fc_code," - ", fv_toko," - ",fv_kota)  Toko',distinct=False,where='')
pivot_table=pivot_table.merge(df_cus,how='left',on='fn_whconsiid')
nama_file = fr'\\192.168.1.254\Share\_From Valdi\For Alokator\Data broken aging.xlsx'
pivot_table[['Toko','fv_barcode','status','sumQty','aging','tahun_round',
             '27','28','29','30','31','32','33','34','36','38','40','S','M','L','XL','XXL']].rename(columns={
                 'fv_barcode':'Artikel',
                 'sumQty':'Jumlah stock'
             }).to_excel(f'{nama_file}',index=False)

In [ ]:
pivot_table=pivot_table.merge(df_cus,how='left',on='fn_whconsiid')

In [ ]:
nama_file = fr'\\192.168.1.254\Share\_From Valdi\For Alokator\Data broken aging.xlsx'

In [ ]:
pivot_table[['Toko','fv_barcode','status','sumQty','aging','tahun_round',
             '27','28','29','30','31','32','33','34','36','38','40','S','M','L','XL','XXL']].rename(columns={
                 'fv_barcode':'Artikel',
                 'sumQty':'Jumlah stock'
             }).to_excel(f'{nama_file}',index=False)

# AS 22 Master gudang

In [ ]:
query = f"""
        SELECT 
            fv_whtypename, fv_artsizecode, fn_qty 
        FROM gis_db.stockarticle_tb a 
        LEFT JOIN gis_db.warehousetype_tb b 
            ON a.fn_whtypeid = b.fn_whtypeid
        WHERE 1=1 
          AND fn_qty > 0
    """
with get_engine('write') as engine:  # Menggunakan context manager dengan with
    df = pd.read_sql(query, engine)

In [ ]:
df_artsize= df_artikel_pb_('fv_barcode,fn_articleid,fv_artsizecode,fv_sizename,fv_configname')

In [ ]:
df=df.merge(df_artsize,how='left',on='fv_artsizecode')

In [ ]:
df['fv_whtypename'].value_counts()

In [ ]:
df_piv=pd.pivot_table(df,
    values='fn_qty',
    index=['fv_barcode','fv_whtypename'],
    columns='fv_sizename',
    aggfunc='sum',
    fill_value=0).reset_index()

In [ ]:
df_art= df_artikel_pb_('distinct fv_barcode,fv_catname,fv_brandname,fv_divname,fv_colorname,fv_rating,' \
'fv_configname,fv_namemark,fm_price,fm_amount,fv_fitting,fv_status, fv_ornal, motif')

In [ ]:
query = f"""
        SELECT 
            fn_articleid, fn_hari 
        FROM gis_local.aging_gudang_pb 
    """
with get_engine('imp') as engine:  # Menggunakan context manager dengan with
    df_aging = pd.read_sql(query, engine)

In [ ]:
today = datetime.now()

In [ ]:
df_aging=df_aging.merge(df_artsize[['fv_barcode','fn_articleid']].drop_duplicates(),on='fn_articleid',how='left')

In [ ]:
df_piv2=pd.pivot_table(df.loc[df['fv_whtypename'].isin(['PUSAT','RETUR','KONSINETTO'])],
    values='fn_qty',
    index=['fv_barcode'],
    columns='fv_sizename',
    aggfunc='sum',
    fill_value=0).reset_index()

In [ ]:
df_broken=proc_pivot_table_n(df.loc[df['fv_whtypename'].isin(['PUSAT','RETUR','KONSINETTO'])],
                values='fn_qty',
                col_index=['fv_barcode'],
                columns=['fv_sizename'],
                col_ket=['fv_configname'])

In [ ]:
df_piv3=df_piv.merge(df_art,how='left',on='fv_barcode')

In [ ]:
df_piv3=df_piv3.merge(df_aging[['fv_barcode','fn_hari']],how='left',on='fv_barcode')
df_piv3=df_piv3.merge(df_broken[['fv_barcode','status','sumQty']],how='left',on='fv_barcode')

In [ ]:
all_sizes_bottom = ['25', '26', '27', '28', '29', '30', '31', '32', '33', '34', '35', '36', '37', '38',
                        '39', '40', '42', '44', 'ALLSIZE']
all_sizes_top = ['FS', 'XS',  'S', 'M', 'L', 'XL', 'XXL', 'XXXL']
all_sizes = all_sizes_bottom + all_sizes_top
df_piv3=df_piv3[['fv_barcode','fv_whtypename','fv_catname','fv_brandname','fv_divname','fv_colorname','fv_namemark',
                 'fv_fitting','fv_status','fv_ornal','motif','fv_configname','fv_rating','fn_hari','status','fm_price','fm_amount',*all_sizes,'sumQty']
                 ].rename(columns={
                     'fv_barcode':'Artikel',
                     'fv_whtypename':'Gudang',
                     'fv_brandname':'Brand',
                     'fv_divname':'Divisi',
                     'fv_colorname':'Color',
                     'fv_namemark':'Acara',
                     'fv_fitting':'Fitting',
                     'fv_status':'Status Produksi',
                     'fv_ornal':'Status',
                     'motif':'Motif',
                     'fn_hari':'Aging gundang',
                     'status':'Status barang',
                     'fm_price':'Harga Gross',
                     'fm_amount':'Harga Nett',
                     'sumQty':'Jumlah',
                     'fv_configname':'Top/Bottom',
                     'fv_rating':'Rating',
                 })

In [ ]:
df_piv3.info()

In [ ]:
nama_file = fr'\\192.168.1.254\Share\_From Valdi\For Alokator\Data Master Gudang.xlsx'
df_piv3.to_excel(f'{nama_file}',index=False)

# AS 23 penju table

In [31]:
queri=f'''
    SELECT fv_nopenjualan,fd_tgltrx,fc_code,fn_tabel 
    FROM gis_local.pb_penjualan_2025 a
    left join gis_local.customers_pb b on a. fn_whconsiid=b.fn_whconsiid where fn_month in (10)'''
with get_engine('imp') as engine:  # Menggunakan context manager dengan with
    df = pd.read_sql(queri, engine)

In [32]:
df=df.drop_duplicates('fn_tabel')

In [33]:
df.sort_values('fn_tabel')

,fv_nopenjualan,fd_tgltrx,fc_code,fn_tabel
0,PJ/DPS/2510/00001/120,2025-10-01,MD501,1
1,PJ/DPS/2510/00001/193,2025-10-01,R0039,2
44,PJ/DPS/2510/00001/188,2025-10-01,R0034,3
52,PJ/DPS/2510/00002/188,2025-10-01,R0034,4
53,PJ/DPS/2510/00002/188,2025-10-01,R0034,5
87,PJ/DPS/2510/00001/42,2025-10-01,MD293,6
265,PJ/DPS/2510/00004/151,2025-10-01,MD653,7
691,PJ/DPS/2510/00003/136,2025-10-02,MD567,8


In [22]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 15 entries, 0 to 3143
Data columns (total 2 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   fv_nopenjualan  15 non-null     object
 1   fn_tabel        15 non-null     int64 
dtypes: int64(1), object(1)
memory usage: 360.0+ bytes


# AS 24 artikel mutasi hold

In [3]:
query = f"""
SELECT fv_barcode FROM gis_db.article_tb where fv_barcode in ('1151317A162D',
'1151323A162D',
'1175221A0629D',
'1176318A160D',
'1176323A160D',
'1256312A1056D',
'1261476A052D',
'1261476A099D',
'1261578A095D',
'1271121A1622D',
'1273323A1881D',
'8151312A163D',
'8172318A037D',
'8174317A036D',
'8174318A038D',
'8176323A161D',
'8176325A161D',
'8231313A395D') and fc_hold="F"
    """
with get_engine('write') as engine:  # Menggunakan context manager dengan with
    df = pd.read_sql(query, engine)

In [4]:
df

,fv_barcode
0,1151317A162D
1,1151323A162D
2,1175221A0629D
3,1176318A160D
4,1176323A160D
5,1256312A1056D
6,1261476A052D
7,1261476A099D
8,1261578A095D
9,1271121A1622D


# troubel

In [ ]:
d

In [1]:
from modules.queries import (df__penju, df__tokoawal, df__transaction, df__cus, get_monthly_sales_data, 
                             df__sales_thru_24, df__sales_thru_25, df__artsizecode, df__gdmd, fetch_data_margin,
                             df__sales_, df__customers_pb, df__artikel_pb, df__artikel_pb_read, 
                             df__customers_pb_read, df__cus_user, df__sales_invoice,  df__sales_bz_disc2nd,
                             df__sales_bz, df__sales_consi_disc2nd, df__sales_online_disc2nd, df__otsdm, df__otsdo,
                             df__artikel_status ) 
from modules.proses import (save_to_sql, add_index_and_foreign_keys, clean_and_transform_data, 
                            insert_disc2_grouping, insert_margin, rename_col_py_to_sql, insert_pk,
                            get_tgl, get_stock, get_stock_data, merge_and_calculate, merge_detail, 
                            generate_rename_dict,trx_rt, process_data_margin,rename_col_sql_to_xlsx_gis,
                            get_artikel_md, get_new_or_changed_rows,upsert_data, process_data_cus,
                            sls_disctok)
from modules.db_connection import get_engine, get_db
from main import md_artikel_size2
from datetime import date, timedelta

✅ .env loaded from: c:\Users\IT-ANALIS\Documents\airflow-DPS\.env
36.93.148.91


## stock monthly

In [ ]:
def get_stock_monthly(tglEND,expot_excel):
    '''Main Stock Monthly'''
    tglEND2= datetime.strptime(tglEND, "%Y-%m-%d").date()
    tglEND = datetime.strptime(tglEND, '%Y-%m-%d').strftime('%Y-%m-%d 23:59:59')
    # Dapatkan tanggal kemarin
    today = datetime.now()
    d2ago = today - timedelta(days=2)
    # Ubah ke string (format default: YYYY-MM-DD)
    # kemarin_str = kemarin.strftime("%Y-%m-%d")
    # d2ago_str = d2ago.strftime("%Y-%m-%d")
    year = int(tglEND[:4])
    month = int(tglEND[5:7])
    day = int(tglEND[8:10])
    last_day = calendar.monthrange(year, month)[1]
    if day == last_day:
        if month == 12:  
            month = 1
            year += 1
        else:
            month += 1
    bulan = calendar.month_name[month].lower()[:3]
    expot_excel = expot_excel.lower()
    if expot_excel not in ['yes', 'no']:
        raise ValueError("Parameter 'expot_excel' harus berupa 'yes' atau 'no'.")
    tglnow = datetime.today().date() # Tanggal Perhari Ini 
    df_tokoawal = df__tokoawal()
    idwhtoko_list_awal = df_tokoawal['fn_whconsiid'].tolist() # Ubah kedalam List

    # Mencari tanggal SO terakhir utnuk masing-masing toko yang masih buka per tanggal yang dipilih
    df_tglLSO = pd.concat([get_tgl(idwhtoko, tglnow, tglEND) for idwhtoko in idwhtoko_list_awal])
    df_tglLSO['fd_tglflagIndo'] = pd.to_datetime(df_tglLSO['fd_tglflagIndo'])

    if tglEND2 > tglnow:  # Jika tanggal yang dipilih lebih besar dari hari ini
        df_tglLSO.loc[df_tglLSO['fc_status']=="T",'fd_tglflagIndo'] -= pd.Timedelta(days=1)

    df_tglLSO['fd_tglflagIndo'] = df_tglLSO['fd_tglflagIndo'].dt.strftime('%Y-%m-%d')
    idwhtoko_list = df_tglLSO['fn_whconsiid'].tolist() # Ubah kedalam List untuk kode toko
    tgl_list = df_tglLSO['fd_tglflagIndo'].tolist() # Ubah kedalam List untuk tanggal SO terakhir

    # Mencari Stock pertanggal yang dipilih dengan menjumlah stock awal SO terakhir dengan transaksi dari SO terakhir sampai tanggal yang dipilih
    df_tok = pd.DataFrame() # Dataframe Kosong
    for idwhtoko, tglSTR in zip(idwhtoko_list, tgl_list): # Looping untuk masing-masing toko dan tanggal SO terakhir
        df_stock = get_stock(idwhtoko, tglSTR, tglEND) # Mencari Stock awal
        df_stock['fv_artsizecode'] = df_stock['fv_artsizecode'].str.upper()
        df_stock['fv_artsizecode'] = df_stock['fv_artsizecode'].replace({'BBRD103V042333-': 'BBRD103V042333'})
        df_transactions = df__transaction(idwhtoko, tglSTR, tglEND) # Mencari semua transaksi
        df_transactions['fv_artsizecode'] = df_transactions['fv_artsizecode'].str.upper()
        df_transactions['fv_artsizecode'] = df_transactions['fv_artsizecode'].replace({'BBRD103V042333-': 'BBRD103V042333'})
        df_total = pd.concat([df_stock, df_transactions]).groupby('fv_artsizecode').sum().reset_index() # menjumlah stock awal dan transaksi
        df_total['fn_whconsiid'] = idwhtoko 
        df_total['tgltrx'] = tglEND
        
        df_tok = pd.concat([df_tok, df_total], ignore_index=True)
    fdate= str(year)[2:4]
    df_tok=df_tok.loc[~((df_tok['fn_whconsiid'].isin([1131,1130,1129]))&(df_tok['qty']==0))]
    df_tok= df_tok[['fn_whconsiid','fv_artsizecode','qty','tgltrx']].rename(columns={'qty':f'stock {bulan}_{fdate}'})
    
    if expot_excel == 'yes':
        max_rows = 1048570
        output_file = f'/opt/share/PB DB/pb_stock_{bulan}_{fdate}.xlsx'
        print(len(df_tok))
        print(max_rows)
        if len(df_tok) <= max_rows:
            # Simpan 1 sheet saja
            df_tok.to_excel(output_file, index=False)
        else:
            # Potong menjadi 2 bagian
            df1 = df_tok.iloc[:max_rows]
            df2 = df_tok.iloc[max_rows:]
            print(df1.info())
            print(df2.info())
            with pd.ExcelWriter(output_file) as writer:
                df1.to_excel(writer, sheet_name='Sheet1', index=False)
                df2.to_excel(writer, sheet_name='Sheet2', index=False)
        print(f'pb_stock_{bulan}_{fdate} Selesai')
        return df_tok
    print(f'pb_stock_{bulan}_{fdate} Selesai')
    return df_tok

def get_last_three_months_including_current():
    timestart = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    print("[INFO] Script get_last_three_months_including_current dimulai.")
    print(f"[INFO] Timestart: {timestart}\n")

    today = datetime.now()
    last_dates = []

    for i in range(2, -1, -1):  # Loop mundur dari 2 bulan lalu sampai bulan ini
        target_month = (today.month - i - 1) % 12 + 1
        target_year = today.year - ((today.month - i - 1) < 0)

        # Dapatkan hari terakhir bulan tersebut
        last_day = calendar.monthrange(target_year, target_month)[1]
        last_dates.append(f"{target_year}-{target_month:02d}-{last_day:02d}")
    for tgl in last_dates:
        get_stock_monthly(tgl, 'yes')
    # Print tanggal yang dipilih sebelum return
    print("[INFO] Tanggal yang dipilih:")
    print("\n".join(last_dates))

    return last_dates

In [ ]:
get_last_three_months_including_current()

In [ ]:
tglEND='2025-08-31'
expot_excel='yes'


In [ ]:
'''Main Stock Monthly'''
tglEND2= datetime.strptime(tglEND, "%Y-%m-%d").date()
tglEND = datetime.strptime(tglEND, '%Y-%m-%d').strftime('%Y-%m-%d 23:59:59')
# Dapatkan tanggal kemarin
today = datetime.now()
d2ago = today - timedelta(days=2)
# Ubah ke string (format default: YYYY-MM-DD)
# kemarin_str = kemarin.strftime("%Y-%m-%d")
# d2ago_str = d2ago.strftime("%Y-%m-%d")
year = int(tglEND[:4])
month = int(tglEND[5:7])
day = int(tglEND[8:10])
last_day = calendar.monthrange(year, month)[1]
if day == last_day:
    if month == 12:  
        month = 1
        year += 1
    else:
        month += 1
bulan = calendar.month_name[month].lower()[:3]
expot_excel = expot_excel.lower()
if expot_excel not in ['yes', 'no']:
    raise ValueError("Parameter 'expot_excel' harus berupa 'yes' atau 'no'.")
tglnow = datetime.today().date() # Tanggal Perhari Ini 

In [ ]:

df_tokoawal = df__tokoawal()
idwhtoko_list_awal = df_tokoawal['fn_whconsiid'].tolist() # Ubah kedalam List

# Mencari tanggal SO terakhir utnuk masing-masing toko yang masih buka per tanggal yang dipilih
df_tglLSO = pd.concat([get_tgl(idwhtoko, tglnow, tglEND) for idwhtoko in idwhtoko_list_awal])
df_tglLSO['fd_tglflagIndo'] = pd.to_datetime(df_tglLSO['fd_tglflagIndo'])

if tglEND2 > tglnow:  # Jika tanggal yang dipilih lebih besar dari hari ini
    df_tglLSO.loc[df_tglLSO['fc_status']=="T",'fd_tglflagIndo'] -= pd.Timedelta(days=1)

In [ ]:
df_tglLSO['fd_tglflagIndo'] = df_tglLSO['fd_tglflagIndo'].dt.strftime('%Y-%m-%d')
idwhtoko_list = df_tglLSO['fn_whconsiid'].tolist() # Ubah kedalam List untuk kode toko
tgl_list = df_tglLSO['fd_tglflagIndo'].tolist() # Ubah kedalam List untuk tanggal SO terakhir

In [ ]:
def get_stock(idwhtoko, tglSTR, tglEND):
    df_stock = df__stock_so(idwhtoko, tglSTR, tglEND)
    if df_stock['qty'].sum() == 0:
        df_stock = df__stock_sa(idwhtoko, tglEND)
    return df_stock

In [ ]:
def df__stock_so(idwhtoko, tglSTR, tglEND):
    query = '''
    select c.fv_artsizecode, IF(a.fn_qtyopname IS NULL, c.fn_qtyskrg, a.fn_qtyopname) AS qty
    from gis_db1.t_historysimpanstokopnamecounter a
    right join gis_db1.fl_stockarticleconsipostingmst b on a.fc_nodoc=b.fc_nodoc
    right join gis_db1.fl_stockarticleconsipostingdtl c on c.fc_nodoc=b.fc_nodoc
    where b.fn_whconsiid= '%s' AND SUBSTR(b.fd_tglflag, 1, 10) between %s AND %s
    GROUP BY c.fv_artsizecode
    '''
    with get_engine('read') as engine:  # Menggunakan context manager dengan with
                # Mencari tanggal Opname terakhir per tglnow
        df = pd.read_sql(query, engine, params=(idwhtoko, tglSTR, tglEND))
    return df

def df__stock_sa(idwhtoko, tglEND):
    query = '''
    select a.fv_artsizecode, a.fn_qty qty
    from saldoawalarticleconsi_tb a
    where a.fn_whconsiid= '%s' AND SUBSTR(fd_tglinject, 1, 10) <= %s
    '''
    with get_engine('read') as engine:  # Menggunakan context manager dengan with
                # Mencari tanggal Opname terakhir per tglnow
        df = pd.read_sql(query, engine, params=(idwhtoko, tglEND))
    return df

In [ ]:
# Mencari Stock pertanggal yang dipilih dengan menjumlah stock awal SO terakhir dengan transaksi dari SO terakhir sampai tanggal yang dipilih
df_tok = pd.DataFrame() # Dataframe Kosong
for idwhtoko, tglSTR in zip(idwhtoko_list, tgl_list): # Looping untuk masing-masing toko dan tanggal SO terakhir
    df_stock = get_stock(idwhtoko, tglSTR, tglEND) # Mencari Stock awal
    df_stock['fv_artsizecode'] = df_stock['fv_artsizecode'].str.upper()
    df_stock['fv_artsizecode'] = df_stock['fv_artsizecode'].replace({'BBRD103V042333-': 'BBRD103V042333'})
    df_transactions = df__transaction(idwhtoko, tglSTR, tglEND) # Mencari semua transaksi
    df_transactions['fv_artsizecode'] = df_transactions['fv_artsizecode'].str.upper()
    df_transactions['fv_artsizecode'] = df_transactions['fv_artsizecode'].replace({'BBRD103V042333-': 'BBRD103V042333'})
    df_total = pd.concat([df_stock, df_transactions]).groupby('fv_artsizecode').sum().reset_index() # menjumlah stock awal dan transaksi
    df_total['fn_whconsiid'] = idwhtoko 
    df_total['tgltrx'] = tglEND
    
    df_tok = pd.concat([df_tok, df_total], ignore_index=True)
fdate= str(year)[2:4]
df_tok=df_tok.loc[~((df_tok['fn_whconsiid'].isin([1131,1130,1129]))&(df_tok['qty']==0))]
df_tok= df_tok[['fn_whconsiid','fv_artsizecode','qty','tgltrx']].rename(columns={'qty':f'stock {bulan}_{fdate}'})

In [ ]:

if expot_excel == 'yes':
    max_rows = 1048570
    output_file = fr'\\192.168.1.254\Share\_From Valdi\PB DB\pb_stock_{bulan}_{fdate}.xlsx'
    print(len(df_tok))
    print(max_rows)
    if len(df_tok) <= max_rows:
        # Simpan 1 sheet saja
        df_tok.to_excel(output_file, index=False)
    else:
        # Potong menjadi 2 bagian
        df1 = df_tok.iloc[:max_rows]
        df2 = df_tok.iloc[max_rows:]
        print(df1.info())
        print(df2.info())
        with pd.ExcelWriter(output_file) as writer:
            df1.to_excel(writer, sheet_name='Sheet1', index=False)
            df2.to_excel(writer, sheet_name='Sheet2', index=False)
    print(f'pb_stock_{bulan}_{fdate} Selesai')
print(f'pb_stock_{bulan}_{fdate} Selesai')

In [ ]:
def df__customers_pb_n(placeholders):
        query = f'''
            SELECT fn_cusid,fn_whconsiid FROM gis_local.customers_pb
            where fc_code in ({placeholders})
            '''
        results = execute_raw_sql(query)
        df = pd.DataFrame(results)
        return df
df_cus=df__customers_pb_n(placeholders)
cus_list= df_cus['fn_cusid'].tolist()
cus_list =  ", ".join([f"'{s}'" for s in cus_list])
whconsi_list=df_cus['fn_whconsiid'].tolist()
whconsi_list =  ", ".join([f"'{s}'" for s in whconsi_list])
def change_cusid_to_whconsiid(df):
    df_cus=df__customers_pb_n(placeholders)
    df=pd.merge(df,df_cus,how='left',on='fn_cusid')
    df['fn_cusid']=df['fn_whconsiid']
    df=df.drop(columns='fn_whconsiid')
    df=df.rename(columns={'fn_cusid':'fn_whconsiid'})
    return df
def get_ostdo_(cus_list):
    query = f'''
        SELECT fv_artsizecode,fv_barcode from artikel_pb
        '''
    results = execute_raw_sql(query)
    df_art = pd.DataFrame(results)
    query_1= f'''SELECT 
    c.fn_cusid ,fv_artsizecode, sum(fn_qty) fn_qty
    FROM gis_db.dodtl_tb a 
    inner join gis_db.domst_tb b on a.fc_nodoc=b.fc_nodoc
    inner join gis_db.t_ordermst c on b.fc_noreff=fv_noorder
    left join gis_db.t_receivingmst d on b.fc_nodoc=d.fc_nosj
    where  fv_noreceiving is null and c.fn_cusid in ({cus_list}) and b.fc_status="F" 
    group by fn_cusid, fv_artsizecode, fd_date 
    '''
    # DatabaseConnector.execute_raw_sql(OutstandingDO_query, using=StockDashboardAPIView.dbsave)
    results_1 = DatabaseConnector.execute_raw_sql(query_1, using=StockDashboardAPIView.dbsave)
    df = pd.DataFrame(results_1)
    df=change_cusid_to_whconsiid(df)
    df=pd.merge(df, df_art, on='fv_artsizecode', how='left')
    df=df.groupby(['fn_whconsiid','fv_barcode',], as_index=False)['fn_qty'].sum()
    return df
OutstandingDO_df=get_ostdo_(cus_list)

In [1]:
# Main
import pandas as pd
import numpy as np

from datetime import date, timedelta
import calendar
from contextlib import contextmanager
from sqlalchemy import create_engine
from sqlalchemy import Integer, String, CHAR, Date, Text, text, Float
import os
from datetime import datetime
from modules.queries import (df__penju, df__tokoawal, df__transaction, df__cus, get_monthly_sales_data, 
                             df__sales_thru_24, df__sales_thru_25, df__artsizecode, df__gdmd, fetch_data_margin,
                             df__sales_, df__customers_pb, df__artikel_pb, df__artikel_pb_read, 
                             df__customers_pb_read, df__cus_user) 
from modules.proses import (save_to_sql, add_index_and_foreign_keys, clean_and_transform_data, 
                            insert_disc2_grouping, insert_margin, rename_col_py_to_sql, insert_pk,
                            get_tgl, get_stock, get_stock_data, merge_and_calculate, merge_detail, 
                            generate_rename_dict,trx_rt, process_data_margin,rename_col_sql_to_xlsx_gis,
                            get_artikel_md, get_new_or_changed_rows,upsert_data, process_data_cus)
# Proses
from modules.queries import (df__tglso, df__tglsa, df__disc2, df__margin, df__artsizecode, df__stock_so,
                             df__stock_sa, df__retur, df__size, generate_months_for_year, df__rtt)
from modules.db_connection import get_engine
from sqlalchemy import Integer, String, CHAR, Date, Text, text, Float
import numpy as np
import pandas as pd
from datetime import datetime
from sqlalchemy import MetaData
from sqlalchemy.dialects.mysql import insert as mysql_insert
from sqlalchemy.orm import sessionmaker
import traceback
import os

✅ .env loaded from: c:\Users\IT-ANALIS\Documents\airflow-DPS\.env
36.93.148.91


## (M) sales_to_sql

In [ ]:
year = 2023
start_time2= datetime.now().strftime('%Y-%m-%d %H:%M:%S')    
print("\n[INFO] Script sales_to_sql mulai dijalankan.")
print(f"[INFO] Timestamp: {start_time2}\n")

df1= df__penju(year)


In [ ]:
df= df1.copy()
df= clean_and_transform_data(df)
# df=insert_disc2_grouping(df,'sum')
# margin_tb(year)
# df=insert_margin(df,year)
# sum_col=['fn_jualpersize','total_penjualan2','total_disc2']
# agg_col=['fn_whconsiid', 'fv_artsizecode', 'tgltrx']
# fisrt_col=[col for col in df.columns if col not in sum_col + agg_col]
# col_col=fisrt_col+[col for col in df.columns if col not in fisrt_col]
# df=df[col_col]
# df['event']=df['event'].fillna(df['status_artikel'])
# df['fn_margin']=df['fn_margin'].fillna(0)
# df=rename_col_py_to_sql(df).sort_values(by='fn_idurut')
# df=insert_pk(df)
# df['fn_nett_amount'] = (df['fn_totalpenjualan']-df['fn_total_disc2']) * (1-df['fn_margin']/100)
# df['fn_sim_amount'] = np.where(df['fc_status'] == "X", df['fn_totalpenjualan'], df['fn_nett_amount'])
# df['fn_dpp'] = df['fn_nett_amount']/1.11
# df['fn_dpp2'] = df['fn_sim_amount']/1.11
# table_name = f"pb_penjualan_{year}"

### (P) insert_disc2_grouping


In [ ]:
df_disc2= df__disc2()
df = pd.merge(df, df_disc2, on='fv_nopenjualan', how='left')
df = df.loc[~df.duplicated()]
df['artsize_disc2nd'] = df['artsize_disc2nd'].fillna(0)
df['total_disc2']=df['artsize_disc2nd']*df['fn_jualpersize']
sum_col=['fn_jualpersize','total_penjualan2','total_disc2']
agg_col=['fv_nopenjualan','fn_whconsiid', 'fv_artsizecode', 'tgltrx']
fisrt_col=[col for col in df.columns if col not in sum_col + agg_col]
print(f" {sum_col}")
print(f" {fisrt_col}")

# df = df.groupby(['fv_nopenjualan','fn_whconsiid', 'fv_artsizecode', 'tgltrx']).agg({
#         **{col: 'sum' for col in sum_col} , **{col: 'first' for col in fisrt_col}
#         }).reset_index()

df = df.groupby(['fv_nopenjualan','fn_whconsiid', 'fv_artsizecode', 'tgltrx']).agg({
        **{col: 'sum' for col in sum_col} , **{col: 'first' for col in fisrt_col}
        }).reset_index()
# if type == 'sum':
#     df = df.groupby(['fv_nopenjualan','fn_whconsiid', 'fv_artsizecode', 'tgltrx']).agg({
#         **{col: 'sum' for col in sum_col} , **{col: 'first' for col in fisrt_col}
#         }).reset_index()
# else: df


In [ ]:
df.loc[df['fv_nopenjualan']=='PJ/DPS/2305/00014/77',
         ['fv_artsizecode','fn_hargasatuan','fn_jualpersize','total_penjualan2']]

In [ ]:
df1

## (M) upsert_customers_pb

### old

In [ ]:
base_path = fr'\\192.168.1.254\Share\_From Valdi\PB DB\\'
excel_path = f'{base_path}customers_pb_sup.xlsx'
csv_path = f'{base_path}list user & employee.csv'
pic_path = f'{base_path}LIST PIC MARKETING.xlsx'
mic_path = f'{base_path}PIC MIC.xlsx'
table_name = 'customers_pb'
df = df__customers_pb_read()
df_excel = pd.read_excel(excel_path)
emp_tb=pd.read_csv(csv_path)
emp_tb['fn_userid']=emp_tb['fn_userid'].astype(int)
df_user=df__cus_user()
df_user=pd.merge(df_user,emp_tb[['fn_userid','fv_nameemployees']],on='fn_userid',how='left')
df_user['fn_cusid']=df_user['fn_cusid'].astype(int)
df=pd.merge(df,df_user[['fn_cusid','fv_nameemployees']],on='fn_cusid',how='left')
df=df.loc[~df['fn_whconsiid'].duplicated()]
df=pd.merge(df, df_excel, on='fn_cusid', how='left')
print(df.loc[df['fn_whconsiid']=="-",['fn_whconsiid']].value_counts())
df_picmkt=pd.read_excel(pic_path, header=3).dropna(how='all')
df_picmkt['SUPERVISOR/ OJT**'] = df_picmkt['SUPERVISOR/ OJT**'].str.replace(r'[^A-Za-z\s]', '', regex=True).str.strip().str.title()
df=pd.merge(df,df_picmkt[['MANAGER','SUPERVISOR/ OJT**','KODE TOKO GIS']],left_on='code',right_on='KODE TOKO GIS',how='left')
print(df.loc[df['fn_whconsiid']=="-",['fn_whconsiid']].value_counts())
df=df.rename(columns={'MANAGER_y':'MANAGER'})
df['SUPERVISOR']=df['SUPERVISOR/ OJT**']



### New

In [2]:
base_path = fr'\\192.168.1.254\Share\_From Valdi\PB DB\\'
# base_path = fr'/opt/share/PB DB/'
excel_path = f'{base_path}customers_pb_sup.xlsx'
csv_path = f'{base_path}list user & employee.csv'
pic_path = f'{base_path}LIST PIC MARKETING.xlsx'
mic_path = f'{base_path}PIC MIC.xlsx'
table_name = 'customers_pb'
df = df__customers_pb_read()
df_excel = pd.read_excel(excel_path)
emp_tb=pd.read_csv(csv_path)
emp_tb['fn_userid']=emp_tb['fn_userid'].astype(int)
df_user=df__cus_user()
df_user=pd.merge(df_user,emp_tb[['fn_userid','fv_nameemployees']],on='fn_userid',how='left')
df_user['fn_cusid']=df_user['fn_cusid'].astype(int)
df=pd.merge(df,df_user[['fn_cusid','fv_nameemployees']],on='fn_cusid',how='left')
df=df.loc[~df['fn_whconsiid'].duplicated()]
df=pd.merge(df, df_excel, on='fn_cusid', how='left')

df_picmkt=pd.read_excel(pic_path, header=3).dropna(how='all')
df_picmkt['SUPERVISOR/ OJT**'] = df_picmkt['SUPERVISOR/ OJT**'].str.replace(r'[^A-Za-z\s]', '', regex=True).str.strip().str.title()
df=pd.merge(df,df_picmkt[['MANAGER','SUPERVISOR/ OJT**','KODE TOKO GIS']],left_on='code',right_on='KODE TOKO GIS',how='left')
df=df.rename(columns={'MANAGER_y':'MANAGER'})
df['SUPERVISOR']=df['SUPERVISOR/ OJT**']

df_picmic=pd.read_excel(mic_path,).dropna(how='all')
df=pd.merge(df,df_picmic[['PIC MIC','STORE GIS','SPG Konsinetto']],left_on='code',right_on='STORE GIS',how='left').fillna('-')
df['Alokator']=df['PIC MIC']
df.loc[df['fn_cusid']==915,'fn_whconsiid']= 9000
df=df.drop(columns=['KODE TOKO GIS', 'MANAGER_x', 'SUPERVISOR/ OJT**','STORE GIS','PIC MIC'])
df.loc[df['fn_grpcusid']==9,'Neo Toko']= "Bazar"
df.loc[df['Toko'].str.startswith('RJ-FO'),'fc_codecus']=df['code']
df.loc[df['fc_codecus'].str.startswith('-'),'fc_codecus']=df['code']
df=df.rename(columns={'SPG Konsinetto':'fv_spg_kosninetto'})
if df is None or df.empty:
        print("❌ DataFrame kosong! Upsert dibatalkan.")
else:
    print("✅ Data berhasil di-load dengan kolom:", df.columns.tolist())
df = process_data_cus(df)
df = df.where(pd.notnull(df), None)
upsert_data('imp',table_name,df, primary_key='customers_pb')


        SELECT a.fn_whconsiid, f.fc_hold, f.fv_namecus AS Toko, f.fn_grpcusid,
            c.fv_provincename AS Provinsi, d.fv_cityname AS Kota, f.fn_cusid, f.fc_codecus AS code,
            e.fc_consinetto, e.fv_namegrpcus, h.fv_configname
        FROM warehouseconsi_tb a
        RIGHT JOIN customer_tb f ON a.fn_cusid = f.fn_cusid
        LEFT JOIN province_tb c ON f.fn_provinceid = c.fn_provinceid
        LEFT JOIN city_tb d ON f.fn_cityid = d.fn_cityid
        LEFT JOIN groupcustomer_tb e ON f.fn_grpcusid = e.fn_grpcusid
        LEFT JOIN config_tb h ON f.fn_typestoreid = h.fn_configid
        
✅ Data berhasil di-load dengan kolom: ['fn_whconsiid', 'fc_hold', 'Toko', 'fn_grpcusid', 'Provinsi', 'Kota', 'fn_cusid', 'code', 'fc_consinetto', 'fv_namegrpcus', 'fv_configname', 'fv_nameemployees', 'fv_whconsiname', 'fc_codecus', 'Neo Toko', 'CHIEF SUPERVISOR', 'SUPERVISOR', 'Alokator', 'WADIR', 'KOREM', 'MMD', 'CSE/IC', 'MANAGER', 'fv_spg_kosninetto']
✅ Upsert berhasil: 798 baris.


In [85]:
upsert_data('imp',table_name,df, primary_key='customers_pb')

✅ Upsert berhasil: 784 baris.


In [2]:
base_path = fr'\\192.168.1.254\Share\_From Valdi\PB DB\\'
# base_path = fr'/opt/share/PB DB/'
excel_path = f'{base_path}customers_pb_sup.xlsx'
csv_path = f'{base_path}list user & employee.csv'
pic_path = f'{base_path}LIST PIC MARKETING.xlsx'
mic_path = f'{base_path}PIC MIC.xlsx'
table_name = 'customers_pb'
df = df__customers_pb_read()

In [10]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 905 entries, 0 to 904
Data columns (total 11 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   fn_whconsiid   764 non-null    float64
 1   fc_hold        905 non-null    object 
 2   Toko           905 non-null    object 
 3   fn_grpcusid    905 non-null    int64  
 4   Provinsi       903 non-null    object 
 5   Kota           891 non-null    object 
 6   fn_cusid       905 non-null    int64  
 7   code           905 non-null    object 
 8   fc_consinetto  905 non-null    object 
 9   fv_namegrpcus  905 non-null    object 
 10  fv_configname  262 non-null    object 
dtypes: float64(1), int64(2), object(8)
memory usage: 77.9+ KB


## (M) stock and sales

In [ ]:
from modules.queries import (df__penju, df__tokoawal, df__transaction, df__cus, get_monthly_sales_data, 
                             df__sales_thru_24, df__sales_thru_25, df__artsizecode, df__gdmd, fetch_data_margin,
                             df__sales_, df__customers_pb, df__artikel_pb, df__artikel_pb_read, 
                             df__customers_pb_read, df__cus_user, df__sales_invoice,  df__sales_bz_disc2nd,
                             df__sales_bz, df__sales_consi_disc2nd, df__sales_online_disc2nd, df__otsdm, df__otsdo,
                             df__artikel_status ) 
from modules.proses import (save_to_sql, add_index_and_foreign_keys, clean_and_transform_data, 
                            insert_disc2_grouping, insert_margin, rename_col_py_to_sql, insert_pk,
                            get_tgl, get_stock, get_stock_data, merge_and_calculate, merge_detail, 
                            generate_rename_dict,trx_rt, process_data_margin,rename_col_sql_to_xlsx_gis,
                            get_artikel_md, get_new_or_changed_rows,upsert_data, process_data_cus,
                            sls_disctok)
from main import md_artikel_size2
from datetime import date, timedelta

In [ ]:
def md_artikel_size2():
    df_artsq= df__artsizecode('imp')
    df_sqq24= df__sales_thru_24()
    df_sqq25= df__sales_thru_25()
    df_sg= df__gdmd()

    df_sq24=df_sqq24.copy()
    df_sq25=df_sqq25.copy()
    df_sq24=df_sq24.drop(columns=[df_sq24.columns[1]])
    df_sq25=df_sq25.drop(columns=[df_sq25.columns[1]])
    df_sq24['fn_artsizeid']=df_sq24['fn_artsizeid'].astype(str)
    df_sq25['fn_artsizeid']=df_sq25['fn_artsizeid'].astype(str)
    df_sq24 = df_sq24.groupby(['fv_artsizecode','fn_artsizeid'])[df_sq24.columns[2:]].sum().reset_index()
    df_sq25 = df_sq25.groupby(['fv_artsizecode','fn_artsizeid'])[df_sq25.columns[2:]].sum().reset_index()

    # Buat dictionary rename berdasarkan data
    df_sq24 = generate_rename_dict(2024, 2024, df_sq24)
    df_sq25 = generate_rename_dict(2025, 2025, df_sq25)
    df=pd.merge(df_sq24,df_sq25,how='outer',on=['fn_artsizeid','fv_artsizecode'])
    # df['qty retur']=df['qty retur_x'].fillna(0)+df['qty retur_y'].fillna(0)
    print(df.columns)
    df=df.rename(columns={
        'qty retur_x': 'retur 2024',
        'qty retur_y': 'retur 2025',})
    df['stock akhir']=df['stock akhir_y']
    df['total sales qty']=df['total sales qty_x'].fillna(0)+df['total sales qty_y'].fillna(0)
    df['total stock']=df['total sales qty']+df['stock akhir'].fillna(0)
    df=df.drop(columns=['total stock_x', 'stock akhir_x', 'stock akhir_y',
                        'total stock_y','total sales qty_x','total sales qty_y'])
    df_sg['fv_artsizecode']=df_sg['fv_artsizecode'].str.upper()
    df = df.groupby(['fv_artsizecode'], as_index=False).sum()
    df = pd.merge(df, df_sg, on='fv_artsizecode', how='outer')
    df = pd.merge(df, df_artsq, on='fv_artsizecode', how='left')
    df =generate_rename_dict(2025, 2025, df)
    df = df.loc[~(df['artikel'].isnull())]
    df2=df.fillna(0)
    df2=df2[df2['artikel']!=0]
    kolom_baru24 = ['artikel', 'size'] + [col for col in df2.columns if col not in ['artikel', 'size','fv_artsizecode','fn_artsizeid']]
    df2 = df2[kolom_baru24]
    df2r=df2.select_dtypes(include='number').sum()
    df2r.to_excel(fr'\\192.168.1.254\Share\_From Valdi\For MD\artsize_gd_result.xlsx')
    df2.to_excel(fr'\\192.168.1.254\Share\_From Valdi\For MD\stock artikel all 2025.xlsx', index=False)

In [ ]:
df_artsq= df__artsizecode('imp')
df_sqq24= df__sales_thru_24()
df_sqq25= df__sales_thru_25()
df_sg= df__gdmd()

df_sq24=df_sqq24.copy()
df_sq25=df_sqq25.copy()
df_sq24=df_sq24.drop(columns=[df_sq24.columns[1]])
df_sq25=df_sq25.drop(columns=[df_sq25.columns[1]])
df_sq24['fn_artsizeid']=df_sq24['fn_artsizeid'].astype(str)
df_sq25['fn_artsizeid']=df_sq25['fn_artsizeid'].astype(str)
df_sq24 = df_sq24.groupby(['fv_artsizecode','fn_artsizeid'])[df_sq24.columns[2:]].sum().reset_index()
df_sq25 = df_sq25.groupby(['fv_artsizecode','fn_artsizeid'])[df_sq25.columns[2:]].sum().reset_index()

# Buat dictionary rename berdasarkan data
df_sq24 = generate_rename_dict(2024, 2024, df_sq24)
df_sq25 = generate_rename_dict(2025, 2025, df_sq25)
df=pd.merge(df_sq24,df_sq25,how='outer',on=['fn_artsizeid','fv_artsizecode'])
# df['qty retur']=df['qty retur_x'].fillna(0)+df['qty retur_y'].fillna(0)
print(df.columns)
df=df.rename(columns={
    'qty retur_x': 'retur 2024',
    'qty retur_y': 'retur 2025',})
df['stock akhir']=df['stock akhir_y']
df['total sales qty']=df['total sales qty_x'].fillna(0)+df['total sales qty_y'].fillna(0)
df['total stock']=df['total sales qty']+df['stock akhir'].fillna(0)
df=df.drop(columns=['total stock_x', 'stock akhir_x', 'stock akhir_y',
                    'total stock_y','total sales qty_x','total sales qty_y'])
df_sg['fv_artsizecode']=df_sg['fv_artsizecode'].str.upper()
df = df.groupby(['fv_artsizecode'], as_index=False).sum()
df = pd.merge(df, df_sg, on='fv_artsizecode', how='outer')
df = pd.merge(df, df_artsq, on='fv_artsizecode', how='left')
df =generate_rename_dict(2025, 2025, df)
df = df.loc[~(df['artikel'].isnull())]
df2=df.fillna(0)
df2=df2[df2['artikel']!=0]
kolom_baru24 = ['artikel', 'size'] + [col for col in df2.columns if col not in ['artikel', 'size','fv_artsizecode','fn_artsizeid']]
df2 = df2[kolom_baru24]
df2r=df2.select_dtypes(include='number').sum()
df2r.to_excel(fr'\\192.168.1.254\Share\_From Valdi\For MD\artsize_gd_result.xlsx')
df2.to_excel(fr'\\192.168.1.254\Share\_From Valdi\For MD\stock artikel all 2025.xlsx', index=False)

In [ ]:
md_artikel_size2()

In [ ]:
def get_stock_monthly(tglEND,expot_excel):
    '''Main Stock Monthly'''
    tglEND2= datetime.strptime(tglEND, "%Y-%m-%d").date()
    tglEND = datetime.strptime(tglEND, '%Y-%m-%d').strftime('%Y-%m-%d 23:59:59')
    # Dapatkan tanggal kemarin
    today = datetime.now()
    d2ago = today - timedelta(days=2)
    # Ubah ke string (format default: YYYY-MM-DD)
    # kemarin_str = kemarin.strftime("%Y-%m-%d")
    # d2ago_str = d2ago.strftime("%Y-%m-%d")
    year = int(tglEND[:4])
    month = int(tglEND[5:7])
    day = int(tglEND[8:10])
    last_day = calendar.monthrange(year, month)[1]
    if day == last_day:
        if month == 12:  
            month = 1
            year += 1
        else:
            month += 1
    bulan = calendar.month_name[month].lower()[:3]
    expot_excel = expot_excel.lower()
    if expot_excel not in ['yes', 'no']:
        raise ValueError("Parameter 'expot_excel' harus berupa 'yes' atau 'no'.")
    tglnow = datetime.today().date() # Tanggal Perhari Ini 
    df_tokoawal = df__tokoawal()
    idwhtoko_list_awal = df_tokoawal['fn_whconsiid'].tolist() # Ubah kedalam List

    # Mencari tanggal SO terakhir utnuk masing-masing toko yang masih buka per tanggal yang dipilih
    df_tglLSO = pd.concat([get_tgl(idwhtoko, tglnow, tglEND) for idwhtoko in idwhtoko_list_awal])
    df_tglLSO['fd_tglflagIndo'] = pd.to_datetime(df_tglLSO['fd_tglflagIndo'])

    if tglEND2 > tglnow:  # Jika tanggal yang dipilih lebih besar dari hari ini
        df_tglLSO.loc[df_tglLSO['fc_status']=="T",'fd_tglflagIndo'] -= pd.Timedelta(days=1)

    df_tglLSO['fd_tglflagIndo'] = df_tglLSO['fd_tglflagIndo'].dt.strftime('%Y-%m-%d')
    idwhtoko_list = df_tglLSO['fn_whconsiid'].tolist() # Ubah kedalam List untuk kode toko
    tgl_list = df_tglLSO['fd_tglflagIndo'].tolist() # Ubah kedalam List untuk tanggal SO terakhir

    # Mencari Stock pertanggal yang dipilih dengan menjumlah stock awal SO terakhir dengan transaksi dari SO terakhir sampai tanggal yang dipilih
    df_tok = pd.DataFrame() # Dataframe Kosong
    for idwhtoko, tglSTR in zip(idwhtoko_list, tgl_list): # Looping untuk masing-masing toko dan tanggal SO terakhir
        df_stock = get_stock(idwhtoko, tglSTR, tglEND) # Mencari Stock awal
        df_stock['fv_artsizecode'] = df_stock['fv_artsizecode'].str.upper()
        df_stock['fv_artsizecode'] = df_stock['fv_artsizecode'].replace({'BBRD103V042333-': 'BBRD103V042333'})
        df_transactions = df__transaction(idwhtoko, tglSTR, tglEND) # Mencari semua transaksi
        df_transactions['fv_artsizecode'] = df_transactions['fv_artsizecode'].str.upper()
        df_transactions['fv_artsizecode'] = df_transactions['fv_artsizecode'].replace({'BBRD103V042333-': 'BBRD103V042333'})
        df_total = pd.concat([df_stock, df_transactions]).groupby('fv_artsizecode').sum().reset_index() # menjumlah stock awal dan transaksi
        df_total['fn_whconsiid'] = idwhtoko 
        df_total['tgltrx'] = tglEND
        
        df_tok = pd.concat([df_tok, df_total], ignore_index=True)
    fdate= str(year)[2:4]
    df_tok=df_tok.loc[~((df_tok['fn_whconsiid'].isin([1131,1130,1129]))&(df_tok['qty']==0))]
    df_tok= df_tok[['fn_whconsiid','fv_artsizecode','qty','tgltrx']].rename(columns={'qty':f'stock {bulan}_{fdate}'})
    
    if expot_excel == 'yes':
        max_rows = 1048570
        output_file = f'/opt/share/PB DB/pb_stock_{bulan}_{fdate}.xlsx'
        print(len(df_tok))
        print(max_rows)
        if len(df_tok) <= max_rows:
            # Simpan 1 sheet saja
            df_tok.to_excel(output_file, index=False)
        else:
            # Potong menjadi 2 bagian
            df1 = df_tok.iloc[:max_rows]
            df2 = df_tok.iloc[max_rows:]
            print(df1.info())
            print(df2.info())
            with pd.ExcelWriter(output_file) as writer:
                df1.to_excel(writer, sheet_name='Sheet1', index=False)
                df2.to_excel(writer, sheet_name='Sheet2', index=False)
        print(f'pb_stock_{bulan}_{fdate} Selesai')
        return df_tok
    print(f'pb_stock_{bulan}_{fdate} Selesai')
    return df_tok

In [ ]:
timestart = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
print("[INFO] Script get_last_three_months_including_current dimulai.")
print(f"[INFO] Timestart: {timestart}\n")

today = datetime.now()
last_dates = []

for i in range(2, -1, -1):  # Loop mundur dari 2 bulan lalu sampai bulan ini
    target_month = (today.month - i - 1) % 12 + 1
    target_year = today.year - ((today.month - i - 1) < 0)

    # Dapatkan hari terakhir bulan tersebut
    last_day = calendar.monthrange(target_year, target_month)[1]
    last_dates.append(f"{target_year}-{target_month:02d}-{last_day:02d}")
for tgl in last_dates:
    get_stock_monthly(tgl, 'yes')

In [ ]:
last_dates = get_last_three_months_including_current()

In [ ]:
last_dates

In [ ]:
def merge_detail(final_data, stock_data, start_year, end_year):
    df_articlesize = df__artsizecode('read')
    df_size = df__size()

    final_data = pd.merge(final_data, df_articlesize, on='fv_artsizecode', how='left')
    final_data = pd.merge(final_data, df_size, on='fn_sizeid', how='left')
    final_data = final_data[~final_data['fv_sizename'].isnull()]

    # Membuat kolom secara dinamis untuk semua bulan di rentang tahun
    columns = ['fn_artsizeid', 'fn_whconsiid', 'fv_artsizecode']
    rename_columns = {'qty': 'fn_qty_retur'}
    month_names = ['jan', 'feb', 'mar', 'apr', 'may', 'jun', 'jul', 'aug', 'sep', 'oct', 'nov', 'dec']

    current_year = datetime.now().year
    current_month = datetime.now().month +1
    # last_stock_col = None  # Variabel untuk menyimpan kolom stock terakhir

    for year in range(start_year, end_year + 1):
        last_month = current_month if year == current_year else 12  # Sampai bulan saat ini di tahun berjalan

        for month in range(1, last_month + 1):
            month_str = f"{month_names[month-1]}_{str(year)[-2:]}"  # Format jan_25, feb_25, dst.
            col_prefix = f'{month:02d}'
            
            stock_col = f'stock {month_str}'

            # Hanya menyimpan stock bulan terakhir
            if month == last_month:
                last_stock_col = stock_col
                columns.append(stock_col)
                rename_columns[stock_col] = 'fn_stock_at'
            else:
                total_stock_col = f'total_stock_{month_str}'
                qty_col = f'qty_{month_str}'
                
                columns.extend([stock_col, total_stock_col, qty_col])
                rename_columns.update({
                    qty_col: f'fn_qty_{col_prefix}',
                    stock_col: f'fn_stock_{col_prefix}',
                    total_stock_col: f'fn_ts_{col_prefix}'
                })
            
            if month == 12:
                columns[:-1]
                available_stock_cols = [col for col in stock_data.columns if col.startswith('stock ')]

                # Pilih kolom stok terakhir (sesuai urutan bulan)
                if available_stock_cols:
                    last_stock_col = (available_stock_cols)[-1]  # Ambil stok terakhir berdasarkan nama kolom
                else:
                    last_stock_col = None  # Jika tidak ada stok, gunakan None
                
                total_stock_dec = f'total_stock_dec_{str(year)[-2:]}'
                qty_dec = f'qty_dec_{str(year)[-2:]}'
                stock_dec=f'stock dec_{str(year)[-2:]}'
                columns.extend([total_stock_dec, qty_dec,last_stock_col])
                rename_columns.update({
                    stock_dec:'fn_stock_12',
                    total_stock_dec: 'fn_ts_12',
                    qty_dec: 'fn_qty_12',
                    last_stock_col:'fn_stock_at'
                })
    # additional condition
    if tgl_today=tgl1diawalbulan:
        columns.extend([])
        columns.extend([f'fn_stock_12','qty', 'total_stock', 'total_qty'])
        rename_columns.update({
            'total_qty': 'fn_total_sales_qty',
            'total_stock': 'fn_ts_all'})
    else:
        columns.extend([])
        columns.extend(['qty', 'total_stock', 'total_qty'])
        rename_columns.update({
            'total_qty': 'fn_total_sales_qty',
            'total_stock': 'fn_ts_all'
        })
    final_data = final_data[columns]
    final_data = final_data.rename(columns=rename_columns).sort_values(by='fn_whconsiid')
    return final_data

In [ ]:
start_year=2025
end_year=2025
timestamp = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
print("\n[INFO] Script stock_and_sales mulai dijalankan.")
print(f"[INFO] Timestamp: {timestamp}\n")
base_path = fr'\\192.168.1.254\Share\_From Valdi\PB DB\\'
print("Base path:", base_path)
print(os.path.exists(base_path)) 

stock_data_25 = get_stock_data(base_path, start_year, end_year)
print(stock_data_25.info())
print("stock_data prosess complite")

sales_data_25 = get_monthly_sales_data(start_year, end_year)
print(sales_data_25.columns)
print("Sales_data prosess complite")




In [ ]:
sales_data_25['qty_nov_25']=0

In [ ]:
sales_data_25

In [ ]:
stock_data_25['stock nov_25']=stock_data_25['stock oct_25']

In [ ]:
stock_data_25=stock_data_25.drop(columns='qty_nov_25')

In [ ]:
# Menggabungkan dan menghitung
final_data_25 = merge_and_calculate(sales_data_25, stock_data_25, start_year, end_year)
print("final_data prosess complite")

In [ ]:


finale_25 = merge_detail(final_data_25, stock_data_25, start_year, end_year )
tanggal_selesai = datetime.now().strftime("%d-%m-%Y %H:%M:%S")
print(f"finale prosess complite {tanggal_selesai}")

nama_tabel=f'sales_thru_pb_{str(start_year)[-2:]}'
# finale_25.to_sql(nama_tabel, engine2, if_exists='replace', index=False, dtype=sql_types)
save_to_sql(finale_25, nama_tabel, 'imp')
add_index_and_foreign_keys(
'imp',
nama_tabel,
index_cols=["fn_whconsiid", "fv_artsizecode"],
foreign_keys=[
    {
        "column": "fn_whconsiid",
        "ref_table": "customers_pb",
        "ref_column": "fn_whconsiid",
        "constraint_name": f"fk_whconsiid_pb_sales_thru_{start_year}"
    },
    {
        "column": "fv_artsizecode",
        "ref_table": "artikel_pb",
        "ref_column": "fv_artsizecode",
        "constraint_name": f"fk_artsize_pb_sales_thru_{start_year}"
    }
]
)
timestamp = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
print("\n[INFO] Script telah di input di gis.local.")
print(f"[INFO] Timestamp: {timestamp}")
    # Mendapatkan tanggal saat ini

save_to_sql(finale_25, nama_tabel, 'read')
# add_index_and_foreign_keys(nama_tabel, ['fn_whconsiid', 'fv_artsizecode'])

tanggal_selesai = datetime.now().strftime("%d-%m-%Y %H:%M:%S")
    # Menambahkan pesan status berhasil
timestamp = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
print("\n[INFO] Script telah berhasil dijalankan.")
print(f"[INFO] Timestamp: {timestamp}")

In [ ]:
# finale_25.to_sql(nama_tabel, engine2, if_exists='replace', index=False, dtype=sql_types)
save_to_sql(finale_25, nama_tabel, 'imp')
add_index_and_foreign_keys(
'imp',
nama_tabel,
index_cols=["fn_whconsiid", "fv_artsizecode"],
foreign_keys=[
    {
        "column": "fn_whconsiid",
        "ref_table": "customers_pb",
        "ref_column": "fn_whconsiid",
        "constraint_name": f"fk_whconsiid_pb_{start_year}"
    },
    {
        "column": "fv_artsizecode",
        "ref_table": "artikel_pb",
        "ref_column": "fv_artsizecode",
        "constraint_name": f"fk_artsize_pb_{start_year}"
    }
]
)
timestamp = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
print("\n[INFO] Script telah di input di gis.local.")
print(f"[INFO] Timestamp: {timestamp}")
    # Mendapatkan tanggal saat ini

save_to_sql(finale_25, nama_tabel, 'read')
# add_index_and_foreign_keys(nama_tabel, ['fn_whconsiid', 'fv_artsizecode'])

tanggal_selesai = datetime.now().strftime("%d-%m-%Y %H:%M:%S")
    # Menambahkan pesan status berhasil
timestamp = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
print("\n[INFO] Script telah berhasil dijalankan.")
print(f"[INFO] Timestamp: {timestamp}")


# System API

In [ ]:
input_param={
            # 'alokator': alokator,
            'store_list': ['MD255'],
            # 'start': start.strftime("%Y-%m-%d"),
            # 'end': ends.strftime("%Y-%m-%d"),
            # 'type': tipe,
            # 'format': output_format,
            # 'artikel':artikel,
            # 'input_date': input_date.strftime("%Y-%m-%d"),
            # 'subid':subid,
            # 's_all':s_all,
            'divisi':'JEANS / YOUTH',
            'ornal':'Normal',
            # 'bulan':bulan,
            # 'category':category
        }

In [ ]:
def df__customers_pb_n(placeholders):
    query = f'''
        SELECT fn_cusid,fn_whconsiid FROM gis_local.customers_pb
        where fc_code in ({placeholders})
        '''
    with get_engine('imp') as engine:
        df= pd.read_sql(query, engine)
        return df

# 01 add whare artikel

In [ ]:
def add_where(input_param,cus='',art=''):
    if input_param['store_list'] is not None and input_param['divisi']!='':
        if store_list:
            placeholders = ", ".join([f"'{s}'" for s in store_list])
            df_cus = df__customers_pb_n(placeholders)
            whconsi_list = df_cus['fn_whconsiid'].tolist()
            
            if whconsi_list:
                whconsi_placeholders = ", ".join([str(s) for s in whconsi_list])
                whconsi = f"and {cus}fn_whconsiid IN ({whconsi_placeholders})"
            else: whconsi = ""  # atau kondisi default kalau kosong
    else: whconsi = ""  # jika input_param['store_list'] None

    if input_param['divisi'] is not None and input_param['divisi']!='':
        divisi = input_param['divisi']
        divisi = f'and {art}fv_divname="{divisi}"'
    else: divisi = ''

    if input_param['ornal'] is not None and input_param['divisi']!='':
        ornal = input_param['ornal']
        ornal = f'and {art}fv_ornal="{ornal}"'
    else: ornal = ''
    return {
        'whconsi':whconsi,
        'divisi':divisi,
        'ornal':ornal}

In [ ]:
where=add_where(input_param,cus='cus.')

# Debug API

## Input dan def global

In [ ]:
from modules.db_connection import get_engine
input_value = datetime.today().strftime("%Y-%m-%d")
input_date = datetime.strptime(input_value, "%Y-%m-%d")
# Konversi kembali ke string
start=datetime.today()
ends=start - relativedelta(months=3)
# start_str = start.strftime("%Y-%m-%d")
# ends_str = ends.strftime("%Y-%m-%d")
alokator=''
store_list = ['MD255']
input_param={'alokator': alokator,
'store_list': store_list,
'start': start.strftime("%Y-%m-%d"),
'end': ends.strftime("%Y-%m-%d"),
'type': 'default',
# 'format': output_format,
'input_date': input_date.strftime("%Y-%m-%d"),
'artikel':'QY02066W'}

In [ ]:
whconsi_list=168

In [ ]:
def df__customers_pb_n(placeholders):
    query = f'''
        SELECT fn_cusid,fn_whconsiid FROM gis_local.customers_pb
        where fc_code in ({placeholders})
        '''
    with get_engine('imp') as engine:
        df= pd.read_sql(query, engine)
    # results = DatabaseConnector.execute_raw_sql(query, using=dbLoad)
    # df = pd.DataFrame(results)
    return df

In [ ]:
def df__all_customers(col):
    query = f'''
            SELECT {col} FROM gis_local.customers_pb
            '''
    with get_engine('imp') as engine:
        df= pd.read_sql(query, engine)
    return df

In [ ]:
def df_artikel_pb_(colom):
    query = f'''
        SELECT {colom}
        FROM gis_local.artikel_pb
        '''
    with get_engine('imp') as engine:
        df= pd.read_sql(query, engine)
    return df

## (Q) df__req_stock

In [ ]:
qstore_list=f'and fn_whconsiid in ({whconsi_list})'
query = f'''
        SELECT fn_whconsiid, (CAST(fn_qtykirim AS SIGNED) * -1) as stocks, fv_artsizecode from t_returdtl_pre t
        LEFT JOIN t_returmst_pre m on t.fv_noretur = m.fv_noretur
        WHERE t.fc_status in ('T') {qstore_list}
        '''

with get_engine('write') as engine:
        df_freestock= pd.read_sql(query, engine)

In [ ]:
df_freestock

## (Q) API retur filter artikel

In [ ]:
input_value = input_param['input_date']
alokator = input_param['alokator']
artikel=input_param['artikel']
query = f"""
select a.fn_whconsiid, fv_barcode, sum(qty)*-1 as retur FROM gis_local.trx_retur a
left join artikel_pb b on a.fv_artsizecode=b.fv_artsizecode 
left join customers_pb c on a.fn_whconsiid=c.fn_whconsiid
WHERE tgltrx <= '{input_value}' and fv_barcode in ('{artikel}')
group by fv_toko, fv_barcode
"""
with get_engine('imp') as engine:
        df_retur= pd.read_sql(query, engine)

In [ ]:
df_retur

## (Q) df__main

In [ ]:
input_value = input_param['input_date']
store_list = input_param['store_list']
if store_list:
    placeholders = ", ".join([f"'{s}'" for s in store_list])
df_cus = df__customers_pb_n(placeholders)
cus_list= df_cus['fn_cusid'].tolist()
cus_list =  ", ".join([f"'{s}'" for s in cus_list])
whconsi_list=df_cus['fn_whconsiid'].tolist()
whconsi_list =  ", ".join([f"'{s}'" for s in whconsi_list])
sales_thru = f"sales_thru_pb_{input_value[2:4]}"
trxstx = f"trxstx_pb{input_value[2:4]}"
stock = f"fn_stock_{input_value[5:7]}"

input_value = '2025-07-10'
whconsi_list=168

main_query_ = f"""
SELECT 
stock_backdate.fn_whconsiid,
SUM(stock) AS stocks,
stock_backdate.fv_artsizecode, 
fv_barcode, fv_sizename, fv_configname

FROM (
SELECT fn_whconsiid, {stock} AS stock, fv_artsizecode 
FROM gis_local.{sales_thru}
UNION ALL
SELECT fn_whconsiid, fn_stock AS stock, fv_artsizecode 
FROM gis_local.{trxstx} 
WHERE fd_tgltrx BETWEEN '{input_value[0:4]}-{input_value[5:7]}-01' AND '{input_value}'
UNION ALL
SELECT fn_whconsiid, fn_stock AS stock, fv_artsizecode 
FROM gis_local.trxrt_pb
WHERE fd_tgltrx BETWEEN '{input_value[0:4]}-{input_value[5:7]}-01' AND '{input_value}'
UNION ALL
SELECT fn_whconsiid, fn_stock AS stock, fv_artsizecode 
FROM gis_local.Stock_varian_25
WHERE fd_tgl BETWEEN '{input_value[0:4]}-{input_value[5:7]}-01' AND '{input_value}'
) stock_backdate
left join artikel_pb b on stock_backdate.fv_artsizecode = b.fv_artsizecode
WHERE 1=1 and fn_whconsiid in ({whconsi_list})
GROUP BY fn_whconsiid, fv_artsizecode
"""
with get_engine('imp') as engine:
    df_main= pd.read_sql(main_query_, engine)

In [ ]:
df_main

## (Q) df__trx_freestock

In [ ]:
input_value = '2025-07-11'
input_now = datetime.today().strftime("%Y-%m-%d")
# input_value = input_param['input_date']
trxstx = f"trxstx_pb{input_value[2:4]}"
# artikel=input_param['artikel']
artikel=None
if artikel==None:
    store_list = input_param['store_list']
    if store_list:
        placeholders = ", ".join([f"'{s}'" for s in store_list])
    df_cus = df__customers_pb_n(placeholders)
    cus_list= df_cus['fn_cusid'].tolist()
    cus_list =  ", ".join([f"'{s}'" for s in cus_list])
    # whconsi_list=df_cus['fn_whconsiid'].tolist()
    # whconsi_list =  ", ".join([f"'{s}'" for s in whconsi_list])
    qstore_list= f'and b.fn_whconsiid in ({whconsi_list})'
else:
    qstore_list= f'and fv_barcode in ("{artikel}")'
main_query = f"""
SELECT 
    stock_backdate.fn_whconsiid,
    SUM(stock) AS stocks,
    stock_backdate.fv_artsizecode
FROM (
    SELECT fn_whconsiid, fn_stock AS stock, fv_artsizecode , fv_stat stat
    FROM gis_local.{trxstx} 
    WHERE fd_tgltrx BETWEEN '{input_value}' AND '{input_now}'
    UNION ALL
    SELECT fn_whconsiid, fn_stock AS stock, fv_artsizecode , 'txrt' stat
    FROM gis_local.trxrt_pb 
    WHERE fd_tgltrx BETWEEN '{input_value}' AND '{input_now}'
    UNION ALL
    SELECT fn_whconsiid, fv_freestock AS stock, fv_artsizecode , 'var' stat
    FROM gis_local.Freestock_varian_25
    WHERE fd_tgl BETWEEN '{input_value}' AND '{input_now}'
) stock_backdate
LEFT JOIN artikel_pb a ON a.fv_artsizecode = stock_backdate.fv_artsizecode
LEFT JOIN customers_pb b ON stock_backdate.fn_whconsiid = b.fn_whconsiid
where 1=1 {qstore_list}
GROUP BY fn_whconsiid, fv_artsizecode
"""
with get_engine('imp') as engine:
    df_frees= pd.read_sql(main_query, engine)

In [ ]:
print(main_query_)

In [ ]:
print(main_query)

In [ ]:
df_frees.loc[(df_frees['fn_whconsiid']==168)]

In [ ]:
df_main.loc[df_main['fv_artsizecode']=='GB02378FS']

In [ ]:
df_combi=pd.concat([df_main, df_frees,df_freestock], ignore_index=True)

In [ ]:
df_result = df_combi.groupby(['fn_whconsiid', 'fv_artsizecode'], as_index=False)['stocks'].sum()


WB03395GV

In [ ]:
df_result

## (Q) df__main_filt_arts

In [ ]:
select Nama_Memo, col_a, col_b, col_c, col_d from(

SELECT '076/09/2025' as Nama_Memo, col_a, col_b, col_c, col_d FROM table_dtl a 
left join table_mst b on a.col_f=b.col_f
where memo like '%076/09/2025%'
union all

SELECT '077/09/2025' as Nama_Memo, col_a, col_b, col_c, col_d FROM table_dtl a 
left join table_mst b on a.col_f=b.col_f
where memo like '%077/09/2025%'
union all

SELECT '078/10/2025' as Nama_Memo, col_a, col_b, col_c, col_d FROM table_dtl a 
left join table_mst b on a.col_f=b.col_f
where memo like '%078/10/2025%'
union all

SELECT '079/10/2025' as Nama_Memo, col_a, col_b, col_c, col_d FROM table_dtl a 
left join table_mst b on a.col_f=b.col_f
where memo like '%079/10/2025%') a

In [ ]:
# input_value = input_param['input_date']
    # Konversi kembali ke string
input_value = '2025-07-10'
sales_thru = f"sales_thru_pb_{input_value[2:4]}"
trxstx = f"trxstx_pb{input_value[2:4]}"
stock = f"fn_stock_{input_value[5:7]}"
artikel=input_param['artikel']
whconsi_list=168
main_query = f"""
SELECT 
        stock_backdate.fn_whconsiid,
        SUM(stock) AS stocks,
        stock_backdate.fv_artsizecode,
        a.fv_barcode,
        a.fv_sizename,
        a.fv_configname,
        a.fv_namemark,
        b.fc_code,
        concat(fc_code, ' - ', fv_toko) fv_toko, stat
    FROM (
        SELECT fn_whconsiid, {stock} AS stock, fv_artsizecode, 'st' stat
        FROM gis_local.{sales_thru}
        UNION ALL
        SELECT fn_whconsiid, fn_stock AS stock, fv_artsizecode, fv_stat stat
        FROM gis_local.{trxstx} 
        WHERE fd_tgltrx BETWEEN '{input_value[0:4]}-{input_value[5:7]}-01' AND '{input_value}'
        UNION ALL
        SELECT fn_whconsiid, fn_stock AS stock, fv_artsizecode, 'trs' stat
        FROM gis_local.trxrt_pb 
        WHERE fd_tgltrx BETWEEN '{input_value[0:4]}-{input_value[5:7]}-01' AND '{input_value}'
        UNION ALL
        SELECT fn_whconsiid, fn_stock AS stock, fv_artsizecode, 'var' stat
        FROM gis_local.Stock_varian_25
        WHERE fd_tgl BETWEEN '{input_value[0:4]}-{input_value[5:7]}-01' AND '{input_value}'
    ) stock_backdate
    LEFT JOIN artikel_pb a ON a.fv_artsizecode = stock_backdate.fv_artsizecode
    LEFT JOIN customers_pb b ON stock_backdate.fn_whconsiid = b.fn_whconsiid
    where fv_barcode in ('{artikel}')
    GROUP BY fn_whconsiid, fv_artsizecode
"""
with get_engine('imp') as engine:
    df= pd.read_sql(main_query, engine)

In [ ]:
input_value

In [ ]:
df

In [ ]:
df.loc[(df['fn_whconsiid']==168)]

In [ ]:
# if input_param==None:
qstore_list=''
# else:
#     store_list = input_param['store_list']
#     if store_list:
#         placeholders = ", ".join([f"'{s}'" for s in store_list])
#     df_cus = df__customers_pb_n(placeholders)
#     cus_list= df_cus['fn_cusid'].tolist()
#     cus_list =  ", ".join([f"'{s}'" for s in cus_list])
#     qstore_list= f'and fn_cusid in ({cus_list})'
query = f'''
        SELECT fv_artsizecode,fv_barcode from artikel_pb
        '''
with get_engine('imp') as engine:
    df= pd.read_sql(query, engine)
# results = DatabaseConnector.execute_raw_sql(query)
# df_art = pd.DataFrame(results)
query_1= f'''SELECT 
    c.fn_cusid ,fv_artsizecode, sum(fn_qty) OstDO
    FROM gis_db.dodtl_tb a 
    inner join gis_db.domst_tb b on a.fc_nodoc=b.fc_nodoc
    inner join gis_db.t_ordermst c on b.fc_noreff=fv_noorder
    left join gis_db.t_receivingmst d on b.fc_nodoc=d.fc_nosj
    where  fv_noreceiving is null {qstore_list} and b.fc_status="F" 
    group by fn_cusid, fv_artsizecode, fd_date 
'''
with get_engine('write') as engine:
    df= pd.read_sql(query_1, engine)

In [ ]:
print(main_query)

In [ ]:
input_now = datetime.today().strftime("%Y-%m-%d")
input_value = '2025-05-05'

In [ ]:
main_query = f"""
SELECT 
        stock_backdate.fn_whconsiid,
        SUM(stock) AS stocks,
        stock_backdate.fv_artsizecode,
        a.fv_barcode,
        a.fv_sizename,
        a.fv_configname,
        a.fv_namemark,
        b.fc_code,
        concat(fc_code, ' - ', fv_toko) fv_toko, stat
    FROM (
        SELECT fn_whconsiid, fn_stock AS stock, fv_artsizecode, fv_stat stat
        FROM gis_local.{trxstx} 
        WHERE fd_tgltrx BETWEEN '{input_value}' AND '{input_now}'
        UNION ALL
        SELECT fn_whconsiid, fn_stock AS stock, fv_artsizecode, 'trs' stat
        FROM gis_local.trxrt_pb 
        WHERE fd_tgltrx BETWEEN '{input_value}' AND '{input_now}'
        UNION ALL
        SELECT fn_whconsiid, fn_stock AS stock, fv_artsizecode, 'var' stat
        FROM gis_local.Stock_varian_25
        WHERE fd_tgl BETWEEN '{input_value}' AND '{input_now}'
    ) stock_backdate
    LEFT JOIN artikel_pb a ON a.fv_artsizecode = stock_backdate.fv_artsizecode
    LEFT JOIN customers_pb b ON stock_backdate.fn_whconsiid = b.fn_whconsiid
    where fv_barcode in ('{artikel}')
    GROUP BY fn_whconsiid, fv_artsizecode, stat
"""
with get_engine('imp') as engine:
    df_trx= pd.read_sql(main_query, engine)

In [ ]:
query = '''SELECT * FROM gis_local.Stock_varian_25;
'''
with get_engine('imp') as engine:  # Menggunakan context manager dengan with
                # Mencari tanggal Opname terakhir per tglnow
        dvar = pd.read_sql(query, engine)

## (Q) API df_sales

In [ ]:
start_str = input_param['start']
ends_str = input_param['end']
artikel=input_param['artikel']
query = f"""
    select fv_barcode, sum(fn_jualpersize) as sales, fn_jualpersize, fv_sizename, fn_whconsiid, trx.fv_artsizecode from (
    SELECT fv_barcode, fn_jualpersize, fd_tgltrx, fv_sizename, fn_whconsiid, a.fv_artsizecode FROM gis_local.pb_penjualan_2025 a
    left join artikel_pb b on a.fv_artsizecode=b.fv_artsizecode where fv_barcode in ('{artikel}')
    union all 
    SELECT fv_barcode, fn_jualpersize, fd_tgltrx, fv_sizename, fn_whconsiid, a.fv_artsizecode FROM gis_local.pb_penjualan_2024 a
    left join artikel_pb b on a.fv_artsizecode=b.fv_artsizecode where fv_barcode in ('{artikel}')) trx
    where trx.fd_tgltrx between '{ends_str}' and '{start_str}' 
    group by fv_barcode, fn_whconsiid"""
with get_engine('imp') as engine:
    df_sales= pd.read_sql(query, engine)

In [ ]:
df_sales

## (P) API proc_sales_stock

In [ ]:
input_value = input_param['input_date']
artikel=input_param['artikel']
query = f"""
SELECT a.fn_whconsiid, fv_barcode, fn_stock, fd_tgltrx FROM 
(SELECT * FROM gis_local.trxstx_pb24 
where fv_stat like 'receive%%' and fd_tgltrx <= '{input_value}'
union all
SELECT * FROM gis_local.trxstx_pb25 
where fv_stat like 'receive%%' and fd_tgltrx <= '{input_value}') a
left join artikel_pb b on a.fv_artsizecode = b.fv_artsizecode
WHERE 1=1 and fv_barcode in ('{artikel}')
group by fn_whconsiid,fv_barcode, fd_tgltrx 
"""
with get_engine('imp') as engine:
    df_aging= pd.read_sql(query, engine)

In [ ]:
if not df_aging.empty:
    # Group df_aging by fn_whconsiid and fv_barcode, and get the minimum fd_tgltrx
    aging_min_date = df_aging.groupby(['fn_whconsiid', 'fv_barcode'])['fd_tgltrx'].min().reset_index()
    aging_min_date.rename(columns={'fd_tgltrx': '1st DO'}, inplace=True)

    # Group df_aging by fn_whconsiid and fv_barcode, and get the maximum fd_tgltrx (last_receive)
    aging_max_date = df_aging.groupby(['fn_whconsiid', 'fv_barcode'])['fd_tgltrx'].max().reset_index()
    aging_max_date.rename(columns={'fd_tgltrx': 'last_receive'}, inplace=True)

    # Merge df with aging_min_date on fn_whconsiid and fv_barcode
    df = pd.merge(df, aging_min_date, on=['fn_whconsiid', 'fv_barcode'], how='left')
    df = pd.merge(df, aging_max_date, on=['fn_whconsiid', 'fv_barcode'], how='left')

    # Get stock on 1st DO
    # Merge aging_df with aging_min_date to get stock on 1st DO
    aging_stock_on_1st_do = pd.merge(
        df_aging,
        aging_min_date,
        on=['fn_whconsiid', 'fv_barcode'],
        how='left'
    )

    # Filter rows where fd_tgltrx matches the 1st DO
    aging_stock_on_1st_do = aging_stock_on_1st_do[
    aging_stock_on_1st_do['fd_tgltrx'] == aging_stock_on_1st_do['1st DO']]

    # Select only the required columns
    aging_stock_on_1st_do = aging_stock_on_1st_do[['fn_whconsiid', 'fv_barcode', 'fn_stock']]
    # aging_stock_on_1st_do.rename(columns={'fn_stock': 'stock_on_1st_do'}, inplace=True)

    # Merge df with aging_stock_on_1st_do to add stock_on_1st_do column
    df = pd.merge(df, aging_stock_on_1st_do, on=['fn_whconsiid', 'fv_barcode'], how='left')
    df_sum = df.groupby(['fn_whconsiid', 'fv_barcode'], as_index=False)['fn_stock'].sum().drop_duplicates(subset=['fv_barcode'])
    df = pd.merge(df, df_sum, on=['fn_whconsiid', 'fv_barcode'], how='left')
    df = df.rename(columns={'fn_stock_y': 'stock_on_1st_do'}).drop(columns=['fn_stock_x'])
    df['stock_on_1st_do'] = df['stock_on_1st_do'].fillna(0)
    # Convert '1st DO' to datetime
    df['1st DO'] = pd.to_datetime(df['1st DO'])
    df['last_receive'] = pd.to_datetime(df['last_receive'])

    # Hitung aging (selisih hari dari last_receive ke hari ini)
    df['aging'] = (pd.Timestamp.today().normalize() - df['last_receive']).dt.days.fillna(0).astype(int)

    # Format '1st DO' to 'YYYY-MM-DD'
    df['1st DO'] = df['1st DO'].dt.strftime('%Y-%m-%d')
    df['last_receive'] = df['last_receive'].dt.strftime('%Y-%m-%d')
else:
    # Jika aging_df kosong, buat kolom dengan nilai default
    df['1st DO'] = '2024-01-01'
    df['last_receive'] = '2024-01-01'
    df['aging'] = '2024-01'
    df['stock_on_1st_do'] = 0

In [ ]:
all_sizes_bottom = ['25', '26', '27', '28', '29', '30', '31', '32', '33', '34', '35', '36', '37', '38',
                        '39', '40', '42', '44', 'ALLSIZE']
all_sizes_top = ['FS', 'XS', 'SS', 'S', 'M', 'L', 'XL', 'XXL', 'XXXL']
all_sizes = all_sizes_bottom + all_sizes_top

# Buat pivot table
pivot_table = pd.pivot_table(
    df,
    values='stocks',
    index=['fv_barcode', 'fn_whconsiid'],
    columns=['fv_sizename'],
    aggfunc='sum',
    fill_value=0
)

# Reset index agar bisa merge
pivot_table = pivot_table.reset_index()

# Kolom detail yang ingin ditambahkan kembali
kolom_detail = ['fv_barcode', '1st DO', 'last_receive', 'aging','fv_configname',
                'stock_on_1st_do', 'fn_whconsiid']
# Merge detail
pivot_table = pd.merge(pivot_table, df[kolom_detail], on=['fv_barcode', 'fn_whconsiid'], how='left')
pivot_table = pivot_table.drop_duplicates()

# Urutkan kolom agar kolom ukuran di akhir
urutan_kolom = kolom_detail + [col for col in pivot_table.columns if col not in kolom_detail]
pivot_table = pivot_table[urutan_kolom]

# Pastikan semua ukuran muncul
for size in all_sizes:
    if size not in pivot_table.columns:
        pivot_table[size] = 0
    pivot_table[size] = pd.to_numeric(pivot_table[size], errors='coerce').fillna(0).astype(int)

# Atur ulang urutan: non-size columns dulu, lalu size columns
size_columns = [size for size in all_sizes if size in pivot_table.columns]
other_columns = [col for col in pivot_table.columns if col not in size_columns]
pivot_table = pivot_table[other_columns + size_columns]

# Hitung status (BROKEN / SEHAT)
top_columns = ["L", "M", "S", "XL", "XS", "XXL", "XXXL"]
bottom_columns = ['25', '26', '27', '28', '29', '30', '31', '32', '33', '34',
                    '35', '36', '37', '38', '39', '40', '42', '44']

pivot_table['status'] = np.where(
    (pivot_table["fv_configname"] == "TOP") & (pivot_table[top_columns].gt(0).sum(axis=1) < 3), "BROKEN",
    np.where(
        (pivot_table["fv_configname"] == "BOTTOM") & (pivot_table[bottom_columns].gt(0).sum(axis=1) < 5), "BROKEN",
        "SEHAT"
    )
)

# Hitung total jumlah qty size (sumQty)
pivot_table['sumQty'] = pivot_table[size_columns].sum(axis=1)
pivot_table=pivot_table.drop(columns=['fv_configname'])

In [ ]:
pivot_table

In [ ]:
df=pivot_table

## (P) API proc_sales_stock

In [ ]:
if not df_sales.empty:
    # Merge df with df_sales on 'fv_barcode' and 'fn_whconsiid'
    df = pd.merge(df, df_sales[['fv_barcode', 'sales', 'fn_whconsiid']], on=['fv_barcode', 'fn_whconsiid'], how='left')
    df['sales'] = df['sales'].fillna(0)
    # Hitung total stock per barcode
    df['sumQty'] = pd.to_numeric(df['sumQty'], errors='coerce')
    df['sales'] = pd.to_numeric(df['sales'], errors='coerce')
    df['sumQty'] = df['sumQty'].fillna(0)
    df['sales'] = df['sales'].fillna(0)
    df['stockAwal'] = df['sumQty'] + df['sales']
    df = df.reset_index(drop=True)
else: 
    df['sales'] = 0
    df['sumQty'] = 0
    df['stockAwal'] = df['sumQty'] + df['sales']

In [ ]:
df

In [ ]:
if not df_retur.empty:
    # Merge df with df_retur on 'fv_barcode' and 'fn_whconsiid'
    df = pd.merge(df, df_retur[['fv_barcode', 'retur', 'fn_whconsiid']], on=['fv_barcode', 'fn_whconsiid'], how='left')
    df['retur'] = df['retur'].fillna(0)
else:
    df['retur'] = 0

In [ ]:
df

In [ ]:
df.loc[(df['fv_artsizecode']=='AAAC096H0822S') & (df['fn_whconsiid']==1)]

## (Q) df_main_draft

In [ ]:
whconsi_list=73

In [ ]:
input_value = input_param['input_date']
store_list = input_param['store_list']
if store_list:
    placeholders = ", ".join([f"'{s}'" for s in store_list])
df_cus = df__customers_pb_n(placeholders)
cus_list= df_cus['fn_cusid'].tolist()
cus_list =  ", ".join([f"'{s}'" for s in cus_list])
whconsi_list=df_cus['fn_whconsiid'].tolist()
whconsi_list =  ", ".join([f"'{s}'" for s in whconsi_list])
sales_thru = f"sales_thru_pb_{input_value[2:4]}"
trxstx = f"trxstx_pb{input_value[2:4]}"
stock = f"fn_stock_{input_value[5:7]}"

main_query = f"""
SELECT 
fn_whcosiid_asal fn_whconsiid,
SUM(fn_qtykirim) AS stocks,
a.fv_artsizecode, fv_barcode, fv_sizename
from gis_local.t_draftdtl a
left join artikel_pb b on a.fv_artsizecode = b.fv_artsizecode
WHERE 1=1 and fn_whcosiid_asal in ({whconsi_list})
GROUP BY fn_whcosiid_asal , a.fv_artsizecode
"""
with get_engine('imp') as engine:
    df= pd.read_sql(main_query, engine)

In [ ]:
kolom_formalitas = ['1st DO', 'last_receive', 'aging', 'fv_configname', 'stock_on_1st_do']

# Tambahkan kolom kosong jika belum ada
for kolom in kolom_formalitas:
    if kolom not in df.columns:
        df[kolom] = None  # atau np.nan jika ingin float kosong

In [ ]:
all_sizes_bottom = ['25', '26', '27', '28', '29', '30', '31', '32', '33', '34', '35', '36', '37', '38',
                    '39', '40', '42', '44', 'ALLSIZE']
all_sizes_top = ['FS', 'XS', 'SS', 'S', 'M', 'L', 'XL', 'XXL', 'XXXL']
all_sizes = all_sizes_bottom + all_sizes_top

# Buat pivot table
pivot_table = pd.pivot_table(
    df,
    values='stocks',
    index=['fv_barcode', 'fn_whconsiid'],
    columns=['fv_sizename'],
    aggfunc='sum',
    fill_value=0
)

# Reset index agar bisa merge
pivot_table = pivot_table.reset_index()

# Kolom detail yang ingin ditambahkan kembali
kolom_detail = ['fv_barcode', '1st DO', 'last_receive', 'aging','fv_configname',
                'stock_on_1st_do', 'fn_whconsiid']
# Merge detail
pivot_table = pd.merge(pivot_table, df[kolom_detail], on=['fv_barcode', 'fn_whconsiid'], how='left')
pivot_table = pivot_table.drop_duplicates()

# Urutkan kolom agar kolom ukuran di akhir
urutan_kolom = kolom_detail + [col for col in pivot_table.columns if col not in kolom_detail]
pivot_table = pivot_table[urutan_kolom]

# Pastikan semua ukuran muncul
for size in all_sizes:
    if size not in pivot_table.columns:
        pivot_table[size] = 0
    pivot_table[size] = pd.to_numeric(pivot_table[size], errors='coerce').fillna(0).astype(int)

# Atur ulang urutan: non-size columns dulu, lalu size columns
size_columns = [size for size in all_sizes if size in pivot_table.columns]
other_columns = [col for col in pivot_table.columns if col not in size_columns]
pivot_table = pivot_table[other_columns + size_columns]

# Hitung status (BROKEN / SEHAT)
top_columns = ["L", "M", "S", "XL", "XS", "XXL", "XXXL"]
bottom_columns = ['25', '26', '27', '28', '29', '30', '31', '32', '33', '34',
                    '35', '36', '37', '38', '39', '40', '42', '44']

pivot_table['status'] = np.where(
    (pivot_table["fv_configname"] == "TOP") & (pivot_table[top_columns].gt(0).sum(axis=1) < 3), "BROKEN",
    np.where(
        (pivot_table["fv_configname"] == "BOTTOM") & (pivot_table[bottom_columns].gt(0).sum(axis=1) < 5), "BROKEN",
        "SEHAT"
    )
)

# Hitung total jumlah qty size (sumQty)
pivot_table['sumQty'] = pivot_table[size_columns].sum(axis=1)
pivot_table=pivot_table.drop(columns=['fv_configname'])

In [ ]:
pivot_table

In [ ]:
df=pivot_table
type_table='detail'
col='fv_barcode'
df_cus = df__all_customers("concat(fc_code, ' - ', fv_toko) fv_toko, fn_whconsiid, fc_code")
if col == 'fv_artsizecode':
    df_art= df_artikel_pb_('fv_barcode,fv_catname,fv_brandname,fv_divname,fv_colorname,fv_rating,fv_configname,fv_namemark,fm_price,fv_sizename,fv_artsizecode,fv_fitname')
    df = pd.merge(df, df_art, on='fv_artsizecode', how='left')
else:
    df_art= df_artikel_pb_('fv_barcode,fv_catname,fv_brandname,fv_divname,fv_colorname,fv_rating,fv_configname,fv_namemark,fm_price,fv_status').drop_duplicates('fv_barcode')
    df = pd.merge(df, df_art, on='fv_barcode', how='left')
df = pd.merge(df, df_cus, on='fn_whconsiid', how='left')
if type_table == 'mobile_view':
    df['fv_sizename_x']=df['fv_sizename_x'].fillna(df['fv_sizename_y'])
    df=df.rename(columns={'fv_sizename_x': 'fv_sizename'})
df['artikel']= df['fv_barcode']
df['fv_artsizecode']= df.empty

In [ ]:
df=df[['fv_toko',
        'fv_barcode',
        '25',
        '26',
        '27',
        '28',
        '29',
        '30',
        '31',
        '32',
        '33',
        '34',
        '35',
        '36',
        '37',
        '38',
        '39',
        '40',
        '42',
        '44',
        'ALLSIZE',
        'FS',
        'XS',
        'SS',
        'S',
        'M',
        'L',
        'XL',
        'XXL',
        'XXXL',
        'fn_whconsiid',
        ]]

In [ ]:
df

In [ ]:
print(main_query)

In [ ]:
pic_mic_dict = {
    3079: 'IREN',
    26: 'TYAS / YUNI',
    3348: 'TYAS / YUNI',
    200: 'VINDRA',
    1585: 'DIAN/ARUM',
    3107: 'NOPAL',
    147: 'SHOFI',
    3350: 'IMAS / ANGGI',
    1804: 'MILI/NENI',
    3341: 'MILI/NENI',
    1759: 'NOVA',
    40: 'SYAFII',
    1385: 'RILYA'
}
# Buat dict baru dengan key = user id, value = nama PIC
# user_id_to_name = {uid: name for name, uid in pic_mic_dict.items() if uid is not None}


In [ ]:
pic_mic_dict

In [ ]:
user_id = pic_mic_dict[3079]
print(user_id)


In [ ]:
df_main

# AR

In [ ]:
query = f'''SELECT distinct main.fv_nomutasi, fd_tgltransaksi, a.fc_nodoc, c.fv_noreceiving, fd_tglreceive, fn_whconsiid, fn_whconsiid2, d.fv_artsizecode, sub_main.fn_qtykirim, fn_qtyterima FROM gis_db.t_mutasimst main

left join gis_db.domst_tb a on main.fv_nomutasi=  a.fc_noreff 
left join gis_db.t_mutasidtl sub_main on main.fv_nomutasi=  sub_main.fv_nomutasi
left join gis_db.t_receivingmst c on a.fc_nodoc=  c.fc_nosj
left join gis_db.t_receivingdtl d on c.fv_noreceiving=  d.fv_noreceiving and sub_main.fv_artsizecode=d.fv_artsizecode
where a.fd_date between '2025-04-01 00:00:00' and '2025-09-31 23:59:59' and 
a.fc_status <> "F"'''
with get_engine('write') as engine:  # Menggunakan context manager dengan with
    df = pd.read_sql(query, engine)

In [ ]:
df_cuspb=df__customers_pb('fn_whconsiid,fv_toko "Toko Pengirim"')

In [ ]:
df_cuspb

In [ ]:
df

In [ ]:
df.info()

In [ ]:
df=pd.merge(df,df_cuspb, on='fn_whconsiid', how='left')

In [ ]:
df_cuspb=df__customers_pb('fn_whconsiid,fv_toko "Toko Terima"')

In [ ]:
df=pd.merge(df,df_cuspb, left_on='fn_whconsiid2', right_on='fn_whconsiid', how='left')

In [ ]:
name_col = {
        'fv_nomutasi': 'No Mutasi',
        'fc_nodoc': 'No DM',
        'fv_noreceiving': 'No Receiving',
        'fv_artsizecode': 'Artikel Size',
        'fn_qtykirim':'Qty Kirim',
        'fn_qtyterima': 'Qty Terima',
        'fd_tgltransaksi':'Tgl Mutasi',
        'fd_tglreceive':'Tgl Receive'}
    # Lakukan rename dan sort
df = df.rename(columns=name_col)


In [ ]:
df=df[['No Mutasi','Tgl Mutasi','No DM','No Receiving','Tgl Receive','Toko Pengirim','Toko Terima','Artikel Size','Qty Kirim','Qty Terima'
    
]]

In [ ]:
df.to_excel('riwayat mutasi.xlsx',index=False)

# ML

In [ ]:
query = f'''select a.fv_artsizecode, a.fn_qtyopname qty, b.fc_status, fn_qtyskrg qty_fl
    from gis_db1.t_historysimpanstokopnamecounter a
    right join gis_db1.fl_stockarticleconsipostingmst b on a.fc_nodoc=b.fc_nodoc
    right join gis_db1.fl_stockarticleconsipostingdtl c on c.fc_nodoc=b.fc_nodoc
    where b.fn_whconsiid= 84 AND SUBSTR(b.fd_tglflag, 1, 10) between '2025-07-01 00:00:01' AND '2025-09-31 00:00:01'
    GROUP BY a.fv_artsizecode'''
with get_engine('read') as engine:  # Menggunakan context manager dengan with
    df = pd.read_sql(query, engine)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error

# ==== Contoh data sederhana ====
data = {
    "Year": [2023]*12 + [2024]*12,
    "Month": list(range(1,13))*2,
    "Category": ["Shirt"]*24,
    "Size": ["M"]*24,
    "Price": [200,200,200,180,200,200,200,190,190,200,200,200] * 2,
    "Season": ["Rainy","Rainy","Dry","Dry","Dry","Dry","Dry","Dry","Dry","Rainy","Rainy","Rainy"]*2,
    "Promo": [0,0,1,0,0,0,1,0,0,0,1,0]*2,
    "Holiday": [0,0,0,1,0,0,0,0,0,0,0,0]*2,
    "Qty_Sold": [
        # 2023 (training)
        120,130,180,220,160,150,200,210,205,190,195,210,
        # 2024 (actual for evaluation)
        125,135,190,230,170,160,210,220,215,200,210,225
    ]
}
df = pd.DataFrame(data)

# ==== Pilih kategori & size tertentu ====
cat = "Shirt"
size = "M"
df_cs = df[(df["Category"]==cat) & (df["Size"]==size)].copy()

# Encode categorical (Season)
df_cs = pd.get_dummies(df_cs, columns=["Season"], drop_first=True)

# Feature set
features = ["Month","Price","Promo","Holiday","Season_Rainy"]
X_train = df_cs[df_cs["Year"]==2023][features]
y_train = df_cs[df_cs["Year"]==2023]["Qty_Sold"]

X_test = df_cs[df_cs["Year"]==2024][features]
y_test = df_cs[df_cs["Year"]==2024]["Qty_Sold"]

# ==== Model ====
model = XGBRegressor(objective="reg:squarederror", n_estimators=200, random_state=42)
model.fit(X_train, y_train)

# Prediksi
y_pred = model.predict(X_test)

# ==== Evaluasi ====
mae = mean_absolute_error(y_test, y_pred)
print(f"MAE (Mean Absolute Error): {mae:.2f}")

# ==== Plot ====
plt.figure(figsize=(8,5))
plt.plot(X_train["Month"], y_train, label="Actual 2023", marker="o")
# plt.plot(X_test["Month"], y_test, label="Actual 2024", marker="o")
plt.plot(X_test["Month"], y_pred, label="Predicted 2024", marker="x")
plt.xlabel("Month")
plt.ylabel("Qty Sold")
plt.title(f"Forecast vs Actual - {cat} Size {size}")
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from xgboost import XGBRegressor
import matplotlib.pyplot as plt

# ==== contoh data ====
data = {
    "Month": [1,2,3,4,5],
    "Category": ["Shirt","Shirt","Shirt","Pants","Pants"],
    "Size": ["M","M","L","M","L"],
    "Color": ["Black","White","Blue","Black","White"],
    "Fitting": ["Regular","Slim","Oversize","Regular","Slim"],
    "Price": [199000,229000,199000,259000,279000],
    "Season": ["Rainy","Rainy","Holiday","Dry","Dry"],
    "Promo": [0,1,1,0,1],
    "Holiday": [0,0,1,0,0],
    "Qty_Sold": [120,170,210,90,130]
}
df = pd.DataFrame(data)

# ==== definisi fitur & target ====
X = df.drop("Qty_Sold", axis=1)
y = df["Qty_Sold"]

# Kolom kategorikal
categorical = ["Category","Size","Color","Fitting","Season"]
numeric = ["Month","Price","Promo","Holiday"]

# One-hot encoding
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(drop="first"), categorical),
        ("num", "passthrough", numeric)
    ]
)

# ==== Model 1: Linear Regression untuk interpretasi ====
linreg_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("regressor", LinearRegression())
])

linreg_model.fit(X, y)

# Ambil koefisien Linear Regression
feature_names = (linreg_model.named_steps["preprocessor"]
                 .get_feature_names_out())
coef = linreg_model.named_steps["regressor"].coef_

coef_df = pd.DataFrame({"Feature": feature_names, "Coefficient": coef})
print("=== Pengaruh Fitur (Linear Regression) ===")
print(coef_df.sort_values(by="Coefficient", ascending=False))

# ==== Model 2: XGBoost untuk prediksi ====
xgb_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("regressor", XGBRegressor(n_estimators=50, random_state=42))
])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

xgb_model.fit(X_train, y_train)
y_pred = xgb_model.predict(X_test)

print("\n=== Hasil Prediksi vs Data Asli (XGBoost) ===")
comparison = pd.DataFrame({"Actual": y_test, "Predicted": y_pred})
print(comparison)

# Plot
comparison.plot(kind="bar", figsize=(6,4))
plt.title("Prediksi vs Aktual")
plt.ylabel("Qty Sold")
plt.show()


# Last Prog

In [ ]:

@contextmanager
def get_engine(stat):
    """Context manager untuk membuka dan menutup SQLAlchemy engine."""
    if stat == 'read':
        engine = create_engine(f'{DB_TYPE}://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_SYNC}')
    elif stat == 'write':
        engine = create_engine(f'{DB_TYPE}://{DB_USER}:{DB_PASSWORD}@{DB_HOST3}:{DB_PORT}/{DB_RT}')
    else:
        engine = create_engine(f'{DB_TYPE}://{DB_USER}:{DB_PASSWORD}@{DB_HOST2}:{DB_PORT}/{DB_LOCAL}')

    try:
        # Mengembalikan engine untuk digunakan dalam blok with
        yield engine
    finally:
        # Menutup koneksi pool engine untuk membersihkan sumber daya
        engine.dispose()

def define_sql_types(df):
    """Menentukan tipe data untuk SQL sesuai dengan tipe kolom."""
    sql_types = {}
    for col in df.columns:
        col_dtype = df[col].dtype

        # Jika kolom tipe float
        if col_dtype == 'float64':
            sql_types[col] = Numeric(15, 2)
        # Jika kolom diawali 'fn' dan bukan float
        elif col.startswith('fn'):
            sql_types[col] = Integer
        # Jika kolom diawali 'fv'
        elif col.startswith('fv'):
            max_length = df[col].astype(str).map(len).max()
            if max_length > 255:
                sql_types[col] = Text
            else:
                sql_types[col] = String(255)
        # Jika kolom diawali 'fc'
        elif col.startswith('fc'):
            sql_types[col] = CHAR(20)
        # Jika kolom diawali 'fd'
        elif col.startswith('fd'):
            sql_types[col] = DateTime
        # Kalau tidak masuk kriteria di atas
        else:
            sql_types[col] = String(50)
    
    return sql_types

def save_to_sql(df, table_name, engine):
    """Menyimpan data ke dalam database menggunakan SQLAlchemy."""
    with get_engine(engine) as engine:
        sql_types = define_sql_types(df)
        df.to_sql(table_name, con=engine, if_exists='replace', index=False, dtype=sql_types)
from sqlalchemy import text
def df__rtt():
    # def trxstx_pb(year):
    today_str = date.today().strftime('%Y-%m-%d')
    with get_engine('write') as engine:
        query=f'''
        SELECT fn_whconsiid, fv_artsizecode, IFNULL(SUM(totalplus), 0) AS stock, tgltrx, stat, no_doc
    FROM (
    select
        SUBSTRING_INDEX(a.fv_noreceiving, '/', -1) AS fn_whconsiid,
        b.fv_artsizecode, 
        b.fn_qtyterima AS totalplus,
        DATE_FORMAT(a.fd_tglreceive, '%%m-%%d-%%Y') tgltrx, 'recieve' as stat,
        a.fv_noreceiving no_doc
        FROM     t_receivingmst a
        INNER JOIN t_receivingdtl b ON a.fv_noreceiving = b.fv_noreceiving
        WHERE a.fc_status <> 'F'
        AND (a.fd_tglsistem BETWEEN '{today_str} 00:00:00' and '{today_str} 23:59:00')
        union all
    SELECT 
        a.fn_whconsiid,
        b.fv_artsizecode, 
        b.fn_qty as totalplus,
        DATE_FORMAT(a.fd_tgladj, '%%m-%%d-%%Y') tgltrx, 'adj_in' as stat,
        a.fc_nodoc no_doc
        FROM 
        t_adjustconsi_mst a 
        INNER JOIN 
        t_adjustconsi_dtl b ON a.fc_nodoc = b.fc_nodoc
        WHERE 
        a.fc_status = 'X' 
        AND b.fc_tipeadj = 'I' 
        AND b.fn_useridbatal IS NULL
        and (a.fd_dateinput between '{today_str} 00:00:00' and '{today_str} 23:59:00')
        union all
    SELECT 
        a.fn_cusid fn_whconsiid,
        b.fv_artsizecode, 
        b.fn_jualpersize as totalplus,
        DATE_FORMAT(a.fd_tgljual, '%%m-%%d-%%Y') tgltrx, 'b_penjualan' as stat,
        a.fv_nopenjualan no_doc
        FROM 
        t_penjualanmst a 
        INNER JOIN 
        t_penjualandtl_goods b ON a.fv_nopenjualan = b.fv_nopenjualan
        WHERE 
        b.fc_status = 'F'
        and (b.fd_tglbatal between '{today_str} 00:00:00' and '{today_str} 23:59:00')
        union all
    SELECT 
        a.fn_whconsiid,
        b.fv_artsizecode, 
        b.fn_qtykirim as totalplus,
        DATE_FORMAT(a.fd_tgltransaksi, '%%m-%%d-%%Y') tgltrx, 'b_retur' as stat,
        a.fv_noretur no_doc
        FROM 
        t_returmst a 
        INNER JOIN 
        t_returdtl b ON a.fv_noretur=b.fv_noretur
        WHERE 
        b.fc_status='F' and 
        (b.fd_tglbatal BETWEEN '{today_str} 00:00:00' and '{today_str} 23:59:00')
        union all
    SELECT 
        a.fn_whconsiid,
        b.fv_artsizecode, 
        (b.fn_stok - b.fn_qtykirim) as totalplus,
        DATE_FORMAT(a.fd_tgltransaksi, '%%m-%%d-%%Y') tgltrx, 'b_retur2' as stat,
        a.fv_noretur no_doc
        FROM 
        t_returmst a 
        INNER JOIN 
        t_returdtl b ON a.fv_noretur=b.fv_noretur
        WHERE 
        b.fn_useridbatal is not null and 
        (b.fd_tglbatal BETWEEN '{today_str} 00:00:00' and '{today_str} 23:59:00')
        union all
    SELECT 
        a.fn_whconsiid,
        b.fv_artsizecode, 
        b.fn_qtykirim as totalplus,
        DATE_FORMAT(a.fd_tgltransaksi, '%%m-%%d-%%Y') tgltrx, 'b_mutasi' as stat,
        a.fv_nomutasi no_doc
        FROM 
        t_mutasimst a 
        INNER JOIN 
        t_mutasidtl b ON a.fv_nomutasi=b.fv_nomutasi
        WHERE 
        b.fc_status='F'
        AND (a.fd_tglbatal BETWEEN '{today_str} 00:00:00' and '{today_str} 23:59:00')
        union all
    SELECT 
        a.fn_whconsiid,
        b.fv_artsizecode, 
        (b.fn_stok - b.fn_qtykirim ) as totalplus,
        DATE_FORMAT(a.fd_tgltransaksi, '%%m-%%d-%%Y') tgltrx, 'b_mutasi2' as stat,
        a.fv_nomutasi no_doc
        FROM 
        t_mutasimst a 
        INNER JOIN 
        t_mutasidtl b ON a.fv_nomutasi=b.fv_nomutasi
        WHERE 
        b.fn_useridbatal is not null
        AND (b.fd_tglbatal BETWEEN '{today_str} 00:00:00' and '{today_str} 23:59:00')
        union all
    SELECT 
        fn_cusid fn_whconsiid,
        b.fv_artsizecode, 
        -1 * b.fn_jualpersize as totalplus,
        DATE_FORMAT(a.fd_tgljual, '%%m-%%d-%%Y') tgltrx, 'penjualan' as stat,
        a.fv_nopenjualan no_doc
        FROM 
        t_penjualanmst a 
        INNER JOIN 
        t_penjualandtl_goods b ON a.fv_nopenjualan = b.fv_nopenjualan
        WHERE 
        a.fv_nopenjualan LIKE 'PJ%%' and b.fd_tglbatal is null
        AND (a.fd_tglinput BETWEEN  '{today_str} 00:00:00' and '{today_str} 23:59:00')
        union all
    SELECT 
        fn_whconsiid,
        b.fv_artsizecode, 
        -1 * b.fn_qty as totalplus,
        DATE_FORMAT(a.fd_tgladj, '%%m-%%d-%%Y') tgltrx, 'adj_out' as stat,
        a.fc_nodoc no_doc
        FROM 
        t_adjustconsi_mst a 
        INNER JOIN 
        t_adjustconsi_dtl b ON a.fc_nodoc = b.fc_nodoc
        WHERE 
        a.fc_status = 'X' 
        AND b.fc_tipeadj = 'O' 
        AND b.fn_useridbatal IS NULL
        AND (a.fd_dateinput BETWEEN '{today_str} 00:00:00' and '{today_str} 23:59:00')
        union all
    SELECT 
        a.fn_whconsiid,
        b.fv_artsizecode, 
        -1 * b.fn_qtykirim as totalplus,
        DATE_FORMAT(a.fd_tgltransaksi, '%%m-%%d-%%Y') tgltrx, 'retur' as stat,
        a.fv_noretur no_doc
        FROM 
        t_returmst a 
        INNER JOIN 
        t_returdtl b ON a.fv_noretur=b.fv_noretur
        WHERE 
        a.fv_noretur LIKE 'RT%%' 
        AND (a.fd_tglinput BETWEEN '{today_str} 00:00:00' and '{today_str} 23:59:00')
        union all
    SELECT 
        a.fn_whconsiid,
        b.fv_artsizecode, 
        -1 * b.fn_qtykirim as totalplus,
        DATE_FORMAT(a.fd_tgltransaksi, '%%m-%%d-%%Y') tgltrx, 'mutasi' as stat,
        a.fv_nomutasi no_doc
        FROM 
        t_mutasimst a 
        inner JOIN 
        t_mutasidtl b ON a.fv_nomutasi=b.fv_nomutasi
        WHERE 
        a.fv_nomutasi LIKE 'MT%%'
        AND (a.fd_tglinput BETWEEN '{today_str} 00:00:00' and '{today_str} 23:59:00')
        ) AS subquery
    GROUP BY fn_whconsiid, fv_artsizecode, tgltrx, no_doc;
        '''
        df= pd.read_sql(query, engine)
        print(today_str)
        print(query)
    return df
def trx_rt():
    df_trxstx= df__rtt()
    df_trxstx['tgltrx'] = pd.to_datetime(df_trxstx['tgltrx'], format='%m-%d-%Y')
    df_trxstx['fn_whconsiid']=df_trxstx['fn_whconsiid'].astype(int)
    df_trxstx['stock']=df_trxstx['stock'].astype(int)
    df_trxstx=df_trxstx.rename(columns={
        'stock':'fn_stock', 
        'tgltrx':'fd_tgltrx', 
        'stat':'fv_stat'
    })
    save_to_sql(df_trxstx, 'trxrt_pb', 'imp')